# V1
评估+重新生成

In [3]:
! modelscope download --model Qwen/Qwen3.5-2B --local_dir "D:\Jupyter profile\汽车信息安全风险评估\models"
! modelscope download --model BAAI/bge-large-zh-v1.5 --local_dir "D:\Jupyter profile\汽车信息安全风险评估\models\rag"


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model Qwen/Qwen3.5-2B.



Processing 13 items:   0%|          | 0.00/13.0 [00:00<?, ?it/s]







































Processing 13 items:   8%|▊         | 1.00/13.0 [00:00<00:07, 1.57it/s]

















































Processing 13 items:  46%|████▌     | 6.00/13.0 [00:00<00:00, 9.08it/s]


















Processing 13 items:  62%|██████▏   | 8.00/13.0 [00:01<00:00, 6.56it/s]


























































































Processing 13 items:  85%|████████▍ | 11.0/13.0 [00:02<00:00, 3.13it/s]




























Processing 13 items:  92%|█████████▏| 12.0/13.0 [00:03<00:00, 2.74it/s]

































































































































































































































































































































































 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model BAAI/bge-large-zh


Processing 12 items:   0%|          | 0.00/12.0 [00:00<?, ?it/s]























































Processing 12 items:   8%|▊         | 1.00/12.0 [00:00<00:07, 1.55it/s]











































Processing 12 items:  67%|██████▋   | 8.00/12.0 [00:01<00:00, 7.61it/s]











Processing 12 items:  92%|█████████▏| 11.0/12.0 [00:01<00:00, 10.3it/s]



































































































































































































































































































































































































































































































































































































































In [52]:
import json
import os
import re
import glob
import logging
import time
from typing import TypedDict, Literal, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import requests
from requests.exceptions import ConnectionError, Timeout, HTTPError
import hashlib
from functools import lru_cache

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 1.1 LLM 配置 ──
llm = ChatOpenAI(
    model="Qwen/Qwen3.5-35B-A3B",
    openai_api_key="sk-xuuvffmjkclemojvbvheodqncswwqqdbknqineabswodcorb",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={
        "enable_thinking": False,
        "stream": True  
    })

# ── 1.2 Embedding 配置 ──
ENABLE_RAG = False
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

# ── 1.3 RAG 知识库路径 ──
RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

# ── 1.4 工作流参数 ──
MAX_EVAL_RETRIES = 1
ENABLE_EVALUATION = False  # 设 False 可跳过评估环节加速

# ── 1.5 并发与分批配置 ──
MAX_WORKERS = 3          # 并发
CALL_INTERVAL = 0.8      # 间隔
BATCH_SIZE_RATING = 5    # 批次调用次数

# ── 1.6 输出路径 ──
OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1.7 资产类型 → 安全属性映射 ──
SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

# ── 1.8 安全属性 → STRIDE 映射 ──
ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,          # 单次最多请求 8 条文本，防 413 限流
                request_timeout=45,    # 防网络卡死
                max_retries=2          # 自动重试
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,         # 重叠减少，防 Token 膨胀
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:  # 硬拦截 > 字符的块
                valid_chunks.append(c)
            else:
                # 兜底：超长块强制截断（保命策略）
                if len(content) > 128:
                    c.page_content = content[:127] + "..."
                    valid_chunks.append(c)
        
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True)
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs)
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                # 字典按顶层 key 拆分
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: dict
    external_interface_json: dict
    asset_results: list
    damage_scenarios: list
    ds_eval_passed: bool
    ds_retry_count: int
    ds_feedback: str
    threat_scenarios: list
    ts_eval_passed: bool
    ts_retry_count: int
    ts_feedback: str
    attack_paths: list
    ap_eval_passed: bool
    ap_retry_count: int
    ap_feedback: str
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    cleaned = text.strip()
    cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"

# 添加简单缓存
from threading import Lock

# 全局缓存映射 + 线程锁
hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()

def hash_prompt(text: str) -> str:
    """生成 prompt 的 MD5 哈希（用于缓存键）"""
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def register_prompt_for_cache(prompt: str) -> str:
    """
    注册 prompt 到全局映射，返回其 hash
    用法: h = register_prompt_for_cache(my_prompt)
    """
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h

def clear_prompt_cache():
    """清空缓存映射（测试/重置时用）"""
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()  # 同时清空 lru_cache

@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    """
    基于 prompt 哈希的缓存调用
    ⚠️ 使用前必须先用 register_prompt_for_cache 注册 prompt
    """
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册 (sys={system_hash[:8]}..., user={user_hash[:8]}...)")
        return "[]"
    
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)

# 带重试的 LLM 调用
def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([...])
            return resp.content
        except ConnectionError as e:
            logger.error(f"  ❌ 连接失败: {e} - 检查网络/代理/防火墙")
        except Timeout as e:
            logger.error(f"  ❌ 请求超时: {e} - 尝试增加 request_timeout")
        except HTTPError as e:
            logger.error(f"  ❌ HTTP 错误: {e.response.status_code} - {e.response.text}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    
    return "[]"

# 并发 LLM 调用引擎
def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    results = []
    total = len(tasks)
    if total == 0:
        return results
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)

        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")

    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s, 平均 {elapsed/len(tasks):.2f}s/任务")
    return results


def save_checkpoint(data: Any, step_name: str):
    """按步骤名保存独立 checkpoint"""
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径。
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


# ──────  步骤2: 单条损害场景 Prompt (逐条调用) ──────
DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是“涉及车辆或车辆功能且影响道路使用者的不良后果”。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}
- 对应威胁类型：{threat_type}

## 参考知识
{rag_context}
{feedback_block}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体危害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- “存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。”

## 输出（严格 JSON 对象，不要数组）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "完整的损害场景描述",
  "threat_type": "{threat_type}"
}}"""

# ──────  步骤3: 单条威胁场景 Prompt (逐条调用) ──────
TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是“为实现损害场景，资产的信息安全属性遭到破坏的潜在原因”。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}
{feedback_block}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体{asset_name}，如“GW部件”、“前照灯请求信号”、“车辆数据”
   - 「{asset_name}」被破坏的「{security_attribute}」（必须与输入的安全属性完全一致）
   - 「{security_attribute}」被破坏的原因/攻击意图
2. - 描述中必须体现“破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果”逻辑链条。
   - 进一步的信息能被威胁场景包含或与威胁场景关联。例如:危害场景与资产、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系

## 参考示例：
- “攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。”

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "{damage_scenario}",
  "threat_type": "{threat_type}",
  "threat_scenario": "详细的威胁场景描述"
}}"""



# ──────  步骤4: 单条攻击路径 Prompt (逐条调用, 附带拓扑) ──────
AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」生成最短的的攻击路径（经过拓扑节点最少）,如有多条最短路径请生成多条，最多不超过4条，攻击路径是“为实现威胁场景的一组蓄意活动”。必须采用自上而下的方法（如攻击树分析），从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}（{asset_type}）
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}
{feedback_block}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产：
   - 以编号步骤形式呈现：1.…；2.…；3.…；…
   - 第一步：明确的外部攻击入口
   - 中间步骤：沿车辆拓扑经过的中间节点和通信协议
   - 最终步骤：到达「{asset_name}」并实施具体攻击技术，从而破坏「{security_attribute}」
2. 路径中的节点和连接关系必须与车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」一致，不得虚构不存在的节点或跳过拓扑中的网关/ECU。但输出协议时只输出名称，不要输出颜色编码（如："CANFD(#F53F3F)"输出为"CANFD"）
3. 同一个威胁场景的多条攻击路径不要重复

## 参考示例：
- “1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号(非预期的快速减速)。”

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "threat_scenario": "{threat_scenario}",
  "attack_paths": [
    "1.完整步骤链",
    "2.完整步骤链"
  ]
}}"""


# ────── 步骤5: 影响评级 Prompt (分批调用) ──────
IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数）
### Safety: 1=无伤害, 2=轻度/中度伤害, 3=严重伤害和有生命危险, 4=威胁生命的伤害（不确定是否幸存）或致命的伤害
### Finance: 1=可忽略, 2=个人可承受, 3=大量损失但受影响的道路使用者将能够克服这些后果, 4=灾难性后果受影响的道路使用者可能无法克服
#### 利益相关者包括：车主、驾驶员、乘员、行人、供应商、主机厂。
### Operation: 1=不影响, 2=导致车辆功能的部分退化, 3=导致车辆重要功能丧失或受损, 4=导致车辆核心功能丧失或受损
### Privacy: 1=隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体,
2=隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体； b)泄露的信息不敏感但很容易识别到PII主体,
3=隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体； b)泄露的信息敏感而且很容易识别到PII主体,
4=隐私侵犯会对道路使用者造成重大甚至不可逆转的影响 泄露的信息高度敏感，并且很容易识别到PII主体 
#### 需要考虑车主、驾驶员、乘员、行人、供应商、主机厂。

## 强约束条件
- 每个维度必须严格对照评级标准打分，不得使用其他标准
- 分值必须是整数 1~4
- impact_level = 四维最高分对应的等级名称（Negligible / Moderate / Major / Severe）

## 输出（严格 JSON 数组）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "damage_scenario": "损害场景",
    "influence_parameters": {{"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}},
    "impact_level": "Negligible/Moderate/Major/Severe"
  }}
]"""


# ────── 步骤6: 可行性评级 Prompt (分批调用) ──────
AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434（尤其是 Annex G 基于攻击潜力的攻击可行性评估方法） 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准
### Exposure_time: 
- 0分：≤1天（实现攻击行为的时间小于等于1天）  
- 1分：≤1周（实现攻击行为的时间小于等于1周）  
- 4分：≤1个月（实现攻击行为的时间小于等于1个月）  
- 17分：≤6个月（实现攻击行为的时间小于等于6个月）  
- 19分：>6个月（实现攻击行为的时间大于6个月）
### Professional_experience:
- 0分：非专业（Layman）——外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3分：精通/擅长（Proficient）——熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6分：专业（Expert）——熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 8分：复合型（Multiple experts）——一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0分：公开的（Public）——公共信息（例如互联网上获得的信息）  
- 3分：限制的（Restricted）——受限制的信息（制造商和供应商共享的内部文档）  
- 7分：敏感的（Sensitive）——机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11分：至关重要的（Critical）——严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0分：十分高（Critical）——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1分：高（High）——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4分：中（Medium）——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10分：低（Low）——困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0分：标准的（Standard）——标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4分：专用的（Specialized）——专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7分：定制的（Bespoke）——定制设备（厂家限制的工具、电子显微镜）  
- 9分：多重定制的（Multiple bespoke）——攻击不同步骤需要不同类型的定制设备

## 可行性等级: 五维求和 → ≤13=High, 14~19=Medium, 20~24=Low, ≥25=Very Low

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数（0、1、3、4、6、7、8、9、10、11、17、19）
- feasibility_score = 五维分值之和（整数）
- feasibility_level 必须使用精确英文名称：High / Medium / Low / Very Low

## 输出（严格 JSON 数组）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "attack_path": "攻击路径",
    "attack_parameters": {{
      "Exposure_time": 0, "Professional_experience": 0,
      "Required_information": 0, "Opportunity_window": 0, "Required_equipment": 0
    }},
    "feasibility_score": 0,
    "feasibility_level": "High/Medium/Low/Very Low"
  }}
]"""


# ────── 评估 Prompt (步骤2/3/4共用, 不变) ──────
EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估质量。
仅输出 JSON，不要任何其他文字。"""

DS_EVAL_USER_TEMPLATE = """## 评审任务
请评估以下损害场景是否符合 ISO 21434 要求。
## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{attribute_name}
## 待评审的损害场景
{damage_scenario}

## 评估标准
1. 完整性：每条{attribute_name}都有对应{damage_scenario}
2. 因果合理性：{damage_scenario}是否说明了{function_name}异常与后果的因果关系


## 输出格式
{{"passed": true或false, "feedback": "如不通过，列出具体问题"}}"""

TS_EVAL_USER_TEMPLATE = """## 评审任务
请评估以下威胁场景。
## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{attribute_name}
## 威胁类型
{threat_type}
## 损害场景
{damage_scenario}
## 待评审
{threat_scenario}

## 评估标准
1. 完整性：每条{damage_scenario}都有对应{threat_scenario}
2. 因果一致性：{damage_scenario}是“最终坏结果”，{threat_scenario}是“导致坏结果的网络安全威胁原因”

## 输出格式
{{"passed": true或false, "feedback": "如不通过，列出具体问题"}}"""

AP_EVAL_USER_TEMPLATE = """## 评审任务
请评估攻击路径质量。
## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{attribute_name}
## 损害场景
{damage_scenario}
## 威胁场景
{threat_scenario}
## 车辆拓扑图
{topo_info}

## 外部接口
{ext_info}

## 待评审
{attack_paths}

## 评估标准
1. 完整性：是否包含完整攻击链
2. 拓扑一致性：是否符合车辆拓扑图节点与协议传播路径
3. 攻击入口有效性：是否存在对应外部接口
4. 步骤连贯性：资产-威胁场景-攻击路径是否匹配
5. 因果一致性：{damage_scenario}是“最终坏结果”，{threat_scenario}是“导致坏结果的网络安全威胁原因”，{attack_paths}是“具体怎么实现这个威胁”的攻击步骤。{threat_scenario}是“资产的{security_attributes}被破坏”以实现{damage_scenario}的潜在原因；而{attack_paths}则是实现该{threat_scenario}的一组蓄意活动。

## 输出格式
{{"passed": true或false, "feedback": "如不通过，列出具体问题"}}"""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ────────── 步骤1: 资产识别 (规则引擎, 不变) ──────────

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": attrs,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "ds_retry_count": 0, "ts_retry_count": 0, "ap_retry_count": 0,
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


# ──────────  步骤2: 损害场景 (逐条并发生成) ──────────

def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]
    feedback = state.get("ds_feedback", "")
    feedback_block = f"\n## ⚠️ 上轮评审反馈\n{feedback}\n" if feedback else ""

    # RAG 检索一次，所有子任务共享
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    #  构建逐条任务
    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for attr in asset["security_attributes"]:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                prompt = DS_SINGLE_USER_TEMPLATE.format(
                    function_name=func["function"],
                    asset_name=asset["asset_name"],
                    asset_type=asset["asset_type"],
                    security_attribute=attr,
                    threat_type=threat_type,
                    rag_context=rag_context,
                    feedback_block=feedback_block,
                )
                tasks.append({"id": task_id, "system": BASE_SYSTEM, "user": prompt})
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    # ★ 并发调用
    raw_results = concurrent_llm_calls(tasks)

    # ★ 聚合结果
    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                scenarios.extend(parsed)
            else:
                scenarios.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {"damage_scenarios": scenarios}


def node_ds_evaluate(state: TaraState) -> dict:
    logger.info("[步骤2-评估] 损害场景评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ds_eval_passed": True, "ds_feedback": ""}

    scenarios = state.get("damage_scenarios", [])
    assets = state.get("asset_results", [])

    if not scenarios:
        return {"ds_eval_passed": False, "ds_feedback": "损害场景为空，请重新生成。"}

    expected = sum(len(a.get("security_attributes", [])) for f in assets for a in f.get("assets", []))

    user_prompt = DS_EVAL_USER_TEMPLATE.format(
        expected_count=expected, actual_count=len(scenarios),
        asset_info=json.dumps(assets, ensure_ascii=False, indent=2),
        scenarios=json.dumps(scenarios, ensure_ascii=False, indent=2),
    )

    raw = llm_call(EVAL_SYSTEM, user_prompt)
    try:
        result = safe_parse_json(raw)
        passed = bool(result.get("passed", False))
        feedback = result.get("feedback", "")
    except Exception:
        passed, feedback = True, ""

    logger.info(f"  → {'✅ 通过' if passed else '❌ 不通过'}")
    return {"ds_eval_passed": passed, "ds_feedback": feedback}


def node_ds_correct(state: TaraState) -> dict:
    new_count = state.get("ds_retry_count", 0) + 1
    logger.info(f"  [损害场景修正] 第 {new_count}/{MAX_EVAL_RETRIES} 次重试")
    return {"ds_retry_count": new_count}


# ──────────  步骤3: 威胁场景 (逐条并发生成) ──────────

def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    feedback = state.get("ts_feedback", "")
    feedback_block = f"\n## ⚠️ 上轮评审反馈\n{feedback}\n" if feedback else ""
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    # ★ 逐条构建任务
    tasks = []
    for i, ds in enumerate(damage_scenarios):
        prompt = TS_SINGLE_USER_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            asset_type=ds.get("asset_type", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
            rag_context=rag_context,
            feedback_block=feedback_block,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ts_list.extend(parsed)
            else:
                ts_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {"threat_scenarios": ts_list}


def node_ts_evaluate(state: TaraState) -> dict:
    logger.info("[步骤3-评估] 威胁场景评估")

    if not ENABLE_EVALUATION:
        return {"ts_eval_passed": True, "ts_feedback": ""}

    ts = state.get("threat_scenarios", [])
    ds = state.get("damage_scenarios", [])
    if not ts:
        return {"ts_eval_passed": False, "ts_feedback": "威胁场景为空。"}

    user_prompt = TS_EVAL_USER_TEMPLATE.format(
        ds_count=len(ds), ts_count=len(ts),
        scenarios=json.dumps(ts, ensure_ascii=False, indent=2),
    )
    raw = llm_call(EVAL_SYSTEM, user_prompt)
    try:
        result = safe_parse_json(raw)
        passed = bool(result.get("passed", False))
        feedback = result.get("feedback", "")
    except Exception:
        passed, feedback = True, ""

    logger.info(f"  → {'✅ 通过' if passed else '❌ 不通过'}")
    return {"ts_eval_passed": passed, "ts_feedback": feedback}


def node_ts_correct(state: TaraState) -> dict:
    new_count = state.get("ts_retry_count", 0) + 1
    logger.info(f"  [威胁场景修正] 第 {new_count}/{MAX_EVAL_RETRIES} 次重试")
    return {"ts_retry_count": new_count}


# ──────────  步骤4: 攻击路径 (逐条并发生成 — 核心改动) ──────────

def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发 — 解决重复问题)")

    threat_scenarios = state.get("threat_scenarios", [])
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    feedback = state.get("ap_feedback", "")
    feedback_block = f"\n## ⚠️ 上轮评审反馈\n{feedback}\n" if feedback else ""
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    #  每条威胁场景独立生成攻击路径
    tasks = []
    for i, ts in enumerate(threat_scenarios):
        prompt = AP_SINGLE_USER_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            asset_type=ts.get("asset_type", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_scenario=ts.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            rag_context=rag_context,
            feedback_block=feedback_block,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ap_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ap_list.extend(parsed)
            else:
                ap_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成攻击路径 {len(ap_list)}/{len(tasks)} 组")
    save_checkpoint(ap_list, "step4_attack_paths")
    return {"attack_paths": ap_list}


def node_ap_evaluate(state: TaraState) -> dict:
    logger.info("[步骤4-评估] 攻击路径评估")

    if not ENABLE_EVALUATION:
        return {"ap_eval_passed": True, "ap_feedback": ""}

    ap = state.get("attack_paths", [])
    ts = state.get("threat_scenarios", [])
    if not ap:
        return {"ap_eval_passed": False, "ap_feedback": "攻击路径为空。"}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)

    user_prompt = AP_EVAL_USER_TEMPLATE.format(
        ts_count=len(ts), ap_count=len(ap),
        topo_info=topo_info, ext_info=ext_info,
        scenarios=json.dumps(ap, ensure_ascii=False, indent=2),
    )
    raw = llm_call(EVAL_SYSTEM, user_prompt)
    try:
        result = safe_parse_json(raw)
        passed = bool(result.get("passed", False))
        feedback = result.get("feedback", "")
    except Exception:
        passed, feedback = True, ""

    logger.info(f"  → {'✅ 通过' if passed else '❌ 不通过'}")
    return {"ap_eval_passed": passed, "ap_feedback": feedback}


def node_ap_correct(state: TaraState) -> dict:
    new_count = state.get("ap_retry_count", 0) + 1
    logger.info(f"  [攻击路径修正] 第 {new_count}/{MAX_EVAL_RETRIES} 次重试")
    return {"ap_retry_count": new_count}


# ──────────  步骤5: 影响评级 (分批并发) ──────────

def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响参数评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    # ★ 分批
    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = IR_BATCH_USER_TEMPLATE.format(
            ds_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # 校验 impact_level
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    for r in ratings:
        params = r.get("influence_parameters", {})
        max_val = max(int(params.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
        r["impact_level"] = IMPACT_LABELS.get(min(max_val, 4), "Negligible")

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


# ──────────  步骤6: 攻击可行性评级 (分批并发) ──────────

def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (分批并发)")

    # 展平攻击路径
    flat_paths = []
    for ap_group in state.get("attack_paths", []):
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "function": ap_group.get("function", ""),
                "asset_name": ap_group.get("asset_name", ""),
                "security_attribute": ap_group.get("security_attribute", ""),
                "attack_path": path,
            })

    if not flat_paths:
        return {"feasibility_ratings": []}

    # ★ 分批
    batches = [flat_paths[i:i+BATCH_SIZE_RATING] for i in range(0, len(flat_paths), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(flat_paths)} 条攻击路径, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = AF_BATCH_USER_TEMPLATE.format(
            ap_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": AF_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # 校验 feasibility_level
    for r in ratings:
        params = r.get("attack_parameters", {})
        total = sum(int(params.get(k, 0)) for k in [
            "Exposure_time", "Professional_experience",
            "Required_information", "Opportunity_window", "Required_equipment",
        ])
        r["feasibility_score"] = total
        if total <= 13:
            r["feasibility_level"] = "High"
        elif total <= 19:
            r["feasibility_level"] = "Medium"
        elif total <= 24:
            r["feasibility_level"] = "Low"
        else:
            r["feasibility_level"] = "Very Low"

    logger.info(f"  → 完成可行性评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": ratings}


# ────────── 步骤7: 风险判定 (规则引擎, 不变) ──────────

def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险判定 (交叉矩阵)")

    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    RISK_MATRIX = {
        ("Severe", "High"): 5, ("Severe", "Medium"): 4, ("Severe", "Low"): 3,
        ("Severe", "Very Low"): 2, 
        ("Major", "High"): 4, ("Major", "Medium"): 3, ("Major", "Low"): 2,
        ("Major", "Very Low"): 1, 
        ("Moderate", "High"): 3, ("Moderate", "Medium"): 2, ("Moderate", "Low"): 2,
        ("Moderate", "Very Low"): 1, 
        ("Negligible", "High"): 1, ("Negligible", "Medium"): 1, ("Negligible", "Low"): 1,
        ("Negligible", "Very Low"): 1,
    }
    RISK_LABEL = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
    TREATMENT = {
        "1": "风险接受",
        "2": "风险接受",
        "3": "风险接受",
        "4": "风险缓解",
        "5": "风险缓解",
    }

    FEAS_ORDER = {"High": 4, "Medium": 3, "Low": 2, "Very Low": 1}
    feas_map: dict[tuple, str] = {}
    for fr in feasibility_ratings:
        key = (fr.get("function", ""), fr.get("asset_name", ""), fr.get("security_attribute", ""))
        level = fr.get("feasibility_level", "Very Low")
        if FEAS_ORDER.get(level, 1) > FEAS_ORDER.get(feas_map.get(key, "Very Low"), 1):
            feas_map[key] = level

    risk_results = []
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        impact_level = ir.get("impact_level", "Negligible")
        feasibility_level = feas_map.get(key, "Very Low")
        risk_val = RISK_MATRIX.get((impact_level, feasibility_level), 1)
        risk_label = RISK_LABEL.get(risk_val, "1")
        treatment = TREATMENT.get(risk_label, "风险接受")

        risk_results.append({
            "function": ir.get("function", ""),
            "asset_name": ir.get("asset_name", ""),
            "security_attribute": ir.get("security_attribute", ""),
            "impact_level": impact_level,
            "feasibility_level": feasibility_level,
            "risk_value": risk_label,
            "risk_treatment": treatment,
        })

    logger.info(f"  → 风险判定完成 {len(risk_results)} 条")
    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


# ────────── 步骤8: 报告汇总──────────

def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    rr_map = {_key(r): r for r in risk_results}

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for attr in asset.get("security_attributes", []):
                key = (func_name, asset["asset_name"], attr)
                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": "", "Professional_experience": "",
                            "Required_information": "", "Opportunity_window": "",
                            "Required_equipment": "",
                        }),
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "threat_type": ds.get("threat_type", ""),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": "", "Finance": "", "Operation": "", "Privacy": "",
                    }),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "attack": attack_list,
                    "risk_value": rr.get("risk_value", ""),
                    "risk_treatment": rr.get("risk_treatment", ""),
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")
    return {"tara_report": report}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：路由函数 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def route_ds_eval(state: TaraState) -> Literal["node_ds_correct", "node_ts_generate"]:
    if state.get("ds_eval_passed") or state.get("ds_retry_count", 0) >= MAX_EVAL_RETRIES:
        if not state.get("ds_eval_passed"):
            logger.warning("  ⚠ 损害场景重试已达上限，强制继续")
        return "node_ts_generate"
    return "node_ds_correct"


def route_ts_eval(state: TaraState) -> Literal["node_ts_correct", "node_ap_generate"]:
    if state.get("ts_eval_passed") or state.get("ts_retry_count", 0) >= MAX_EVAL_RETRIES:
        if not state.get("ts_eval_passed"):
            logger.warning("  ⚠ 威胁场景重试已达上限，强制继续")
        return "node_ap_generate"
    return "node_ts_correct"


def route_ap_eval(state: TaraState) -> Literal["node_ap_correct", "node_impact_rating"]:
    if state.get("ap_eval_passed") or state.get("ap_retry_count", 0) >= MAX_EVAL_RETRIES:
        if not state.get("ap_eval_passed"):
            logger.warning("  ⚠ 攻击路径重试已达上限，强制继续")
        return "node_impact_rating"
    return "node_ap_correct"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：工作流构建 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_evaluate",         node_ds_evaluate)
    builder.add_node("node_ds_correct",          node_ds_correct)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_evaluate",         node_ts_evaluate)
    builder.add_node("node_ts_correct",          node_ts_correct)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_evaluate",         node_ap_evaluate)
    builder.add_node("node_ap_correct",          node_ap_correct)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")

    builder.add_edge("node_ds_generate", "node_ds_evaluate")
    builder.add_conditional_edges("node_ds_evaluate", route_ds_eval)
    builder.add_edge("node_ds_correct", "node_ds_generate")

    builder.add_edge("node_ts_generate", "node_ts_evaluate")
    builder.add_conditional_edges("node_ts_evaluate", route_ts_eval)
    builder.add_edge("node_ts_correct", "node_ts_generate")

    builder.add_edge("node_ap_generate", "node_ap_evaluate")
    builder.add_conditional_edges("node_ap_evaluate", route_ap_eval)
    builder.add_edge("node_ap_correct", "node_ap_generate")

    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_json: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: dict | None = None,
    asset_results: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": data_flow_json or [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or {},
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_eval_passed": False, "ds_retry_count": 0, "ds_feedback": "",
        "threat_scenarios": [], "ts_eval_passed": False, "ts_retry_count": 0, "ts_feedback": "",
        "attack_paths": [],    "ap_eval_passed": False, "ap_retry_count": 0, "ap_feedback": "",
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    # ★ 保存最终报告
    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    # ★ 同时保存完整中间状态（便于调试）
    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":

    sample_assets = [
        {
            "function": "交通标识及信号灯识别",
            "assets": [
                {"asset_type": "信号", "asset_name": "弹窗/音效"},
            ],
        },
        {
            "function": "智能泊车辅助",
            "assets": [
                {"asset_type": "数据", "asset_name": "用户账户信息"},
            ],
        },
    ]

    sample_topology = {
        "CANFD(#F53F3F)": ["AMP", "CDC", "IC"],
        "A2B(#016AFF)": ["AMP", "CDC"],
        "FPDLINK(#00B42A)": ["CDC", "IC"],
        "FPDLINK(#FF7D00)": ["CDC", "IVI"],
        "GMSL(#000000)": ["CDC", "DMS"],
        "GMSL(#00B0F0)": ["CDC", "OMS"],
        "GMSL(#7F7F7F)": ["CDC", "MDC"],
        "ETH(#8F8F8F)": ["CDC", "CEM"],
        "CANFD(#5470C6)": ["BDCR", "CDC", "CEM", "REA"],
        "CAN(#91CC75)": ["BDCR", "BLE", "CEM", "CMSL", "CMSR", "DSM", "ECALL", "PDSM", "TRM", "WLCM"],
        "CANFD(#FAC858)": ["MDC", "USS ECU"],
        "CANFD(#EE6666)": ["FR", "MDC"],
        "GMSL(#73C0DE)": ["LRC-FC", "MDC"],
        "GMSL(#FCDBC9)": ["MDC", "SRC-FC"],
        "GMSL(#9A60B4)": ["MDC", "MRC-SC"],
        "GMSL(#BC85FE)": ["MDC", "MRC-RC"],
        "GMSL(#3088DA)": ["FEC", "MDC"],
        "ETH(#5E7C61)": ["CEM", "MDC"],
        "CANFD(#831A5C)": ["CEM", "EPS", "IBCU", "MDC", "SRS", "VCU", "VMC"],
        "ETH(#8D62E8)": ["CEM", "VCU"],
        "CANFD(#0D21E3)": ["BCU", "EVCC", "FMIPU", "HPMU", "ITMS", "PDU", "RDPEU", "RMIPU", "VCU"],
        "CAN(#01F2E3)": ["BCU", "EVCC"],
        "CAN(#29FC11)": ["ACP", "ITMS", "PTC"],
        "CANFD(#973D5D)": ["BDCR", "CDC", "CEM", "LBMS"],
        "ETH(#CDED23)": ["Lidar", "MDC"],
        "蜂窝(#AF21CE)": ["BCALL中心", "CDC", "CEM", "ECALL中心", "华为车云", "诊断仪", "阿维塔车云"],
    }

    sample_ext_interfaces = [
        {"component": "CDC", "interfaces": ["Cellular Netword  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
        {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
        {"component": "UWB", "interfaces": ["RF  射频"]},
        {"component": "CHLIL", "interfaces": ["JTAG"]},
        {"component": "CHLIR", "interfaces": ["JTAG"]},
        {"component": "ITMS", "interfaces": ["JTAG"]},
    ]

    report = run_tara(
        asset_results=sample_assets,
        topology_json=sample_topology,
        external_interface_json=sample_ext_interfaces,
        output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
        build_rag_index=True,
    )

    logger.info(f"📋 最终报告包含 {len(report)} 个功能模块，已保存为 JSON 文件")

2026-04-10 08:34:11,331 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-10 08:34:11,332 [INFO] ============================================================
2026-04-10 08:34:11,332 [INFO]   并发配置: MAX_WORKERS=3, BATCH_SIZE_RATING=5
2026-04-10 08:34:11,333 [INFO]   评估开关: ENABLE_EVALUATION=False
2026-04-10 08:34:11,333 [INFO] [初始化] RAG 已禁用，跳过
2026-04-10 08:34:11,372 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-10 08:34:11,380 [INFO] ============================================================
2026-04-10 08:34:11,381 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-10 08:34:11,381 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-10 08:34:11,383 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V2\checkpoint_step1_assets.json
2026-04-10 08:34:11,384 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-10 08:34:11,384 [INFO]   → 共 7 个子任务, 并发数 3
2026-04-10 08:34:11,387 [ERROR]   ❌ 未知错误: NotImplementedError: Unsupported message type: <class 'ellipsis'>
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_

# V2
并发+缓存避免重复

In [60]:
import json
import os
import re
import glob
import logging
import time
from typing import TypedDict, Literal, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache
import hashlib

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import requests
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 1.1 LLM 配置 ──
llm = ChatOpenAI(
    model="Qwen/Qwen3.5-35B-A3B",
    openai_api_key="sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={
        "enable_thinking": False
    },
)

# ── 1.2 Embedding 配置 ──
ENABLE_RAG = False
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

# ── 1.3 RAG 知识库路径 ──
RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

# ── 1.4 工作流参数 ──
MAX_EVAL_RETRIES = 1
ENABLE_EVALUATION = False  # 设 False 可跳过评估环节加速

# ── 1.5 并发与分批配置 ──
MAX_WORKERS = 3          # 最大并发 LLM 调用数
CALL_INTERVAL = 0.8      # 提交任务间隔(秒), 用于控制 API 速率
BATCH_SIZE_RATING = 5    # 评级步骤的批大小

# ── 1.6 输出路径 ──
OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1.7 资产类型 → 安全属性映射 ──
SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

# ── 1.8 安全属性 → STRIDE 映射 ──
ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,          # 单次最多请求 8 条文本，防 413 限流
                request_timeout=45,    # 防网络卡死
                max_retries=2          # 自动重试
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,         # 重叠减少，防 Token 膨胀
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:  # 硬拦截 >128 字符的块
                valid_chunks.append(c)
            else:
                # 兜底：超长块强制截断（保命策略）
                if len(content) > 128:
                    c.page_content = content[:127] + "..."
                    valid_chunks.append(c)
        
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True)
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs)
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                # 字典按顶层 key 拆分
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: dict
    external_interface_json: dict
    asset_results: list
    damage_scenarios: list
    ds_eval_passed: bool
    ds_retry_count: int
    ds_feedback: str
    threat_scenarios: list
    ts_eval_passed: bool
    ts_retry_count: int
    ts_feedback: str
    attack_paths: list
    ap_eval_passed: bool
    ap_retry_count: int
    ap_feedback: str
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
        
    cleaned = text.strip()
    cleaned = text.strip()
    cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


# ── 缓存机制 ──
hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()

def hash_prompt(text: str) -> str:
    """生成 prompt 的 MD5 哈希（用于缓存键）"""
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def register_prompt_for_cache(prompt: str) -> str:
    """注册 prompt 到全局映射，返回其 hash"""
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h

def clear_prompt_cache():
    """清空缓存映射（测试/重置时用）"""
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()

@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    """基于 prompt 哈希的缓存调用，使用前必须先用 register_prompt_for_cache 注册 prompt"""
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册 (sys={system_hash[:8]}..., user={user_hash[:8]}...)")
        return "[]"
    
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


# 带重试的 LLM 调用
def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
                
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: langchain 返回类型不匹配 - {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    
    return "[]"


# 并发 LLM 调用引擎（增强日志统计）
def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    """
    并发执行多个 LLM 调用
    tasks: [{"id": str/int, "system": str, "user": str}, ...]
    返回: [{"id": ..., "raw": str}, ...]  按 id 排序
    """
    results = []
    total = len(tasks)
    if total == 0:
        return results
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)

        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")

    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s, 平均 {elapsed/len(tasks):.2f}s/任务")
    return results


def save_checkpoint(data: Any, step_name: str):
    """按步骤名保存独立 checkpoint"""
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径。
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


# ──────  步骤2: 单条损害场景 Prompt (逐条调用) ──────
DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是“涉及车辆或车辆功能且影响道路使用者的不良后果”。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}
- 对应威胁类型：{threat_type}

## 参考知识
{rag_context}
{feedback_block}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体危害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- “存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。”

## 输出（严格 JSON 对象，不要数组）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "完整的损害场景描述",
  "threat_type": "{threat_type}"
}}"""

# ──────  步骤3: 单条威胁场景 Prompt (逐条调用) ──────
TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是“为实现损害场景，资产的信息安全属性遭到破坏的潜在原因”。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}
{feedback_block}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体{asset_name}，如“GW部件”、“前照灯请求信号”、“车辆数据”
   - 「{asset_name}」被破坏的「{security_attribute}」（必须与输入的安全属性完全一致）
   - 「{security_attribute}」被破坏的原因/攻击意图
2. - 描述中必须体现“破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果”逻辑链条。
   - 进一步的信息能被威胁场景包含或与威胁场景关联。例如:危害场景与资产、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系

## 参考示例：
- “攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。”

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "{damage_scenario}",
  "threat_type": "{threat_type}",
  "threat_scenario": "详细的威胁场景描述"
}}"""



# ──────  步骤4: 单条攻击路径 Prompt (逐条调用, 附带拓扑) ──────
AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」生成最短的的攻击路径（经过拓扑节点最少）,如有多条最短路径请生成多条，最多不超过4条，攻击路径是“为实现威胁场景的一组蓄意活动”。必须采用自上而下的方法（如攻击树分析），从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}（{asset_type}）
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}
{feedback_block}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产：
   - 以编号步骤形式呈现：1.…；2.…；3.…；…
   - 第一步：明确的外部攻击入口
   - 中间步骤：沿车辆拓扑经过的中间节点和通信协议
   - 最终步骤：到达「{asset_name}」并实施具体攻击技术，从而破坏「{security_attribute}」
2. 路径中的节点和连接关系必须与车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」一致，不得虚构不存在的节点或跳过拓扑中的网关/ECU。但输出协议时只输出名称，不要输出颜色编码（如："CANFD(#F53F3F)"输出为"CANFD"）
3. 同一个威胁场景的多条攻击路径不要重复

## 参考示例：
- “1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号(非预期的快速减速)。”

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "threat_scenario": "{threat_scenario}",
  "attack_paths": [
    "1.完整步骤链",
    "2.完整步骤链"
  ]
}}"""


# ────── 步骤5: 影响评级 Prompt (分批调用) ──────
IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数）
### Safety: 1=无伤害, 2=轻度/中度伤害, 3=严重伤害和有生命危险, 4=威胁生命的伤害（不确定是否幸存）或致命的伤害
### Finance: 1=可忽略, 2=个人可承受, 3=大量损失但受影响的道路使用者将能够克服这些后果, 4=灾难性后果受影响的道路使用者可能无法克服
#### 利益相关者包括：车主、驾驶员、乘员、行人、供应商、主机厂。
### Operation: 1=不影响, 2=导致车辆功能的部分退化, 3=导致车辆重要功能丧失或受损, 4=导致车辆核心功能丧失或受损
### Privacy: 1=隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体,
2=隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体； b)泄露的信息不敏感但很容易识别到PII主体,
3=隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体； b)泄露的信息敏感而且很容易识别到PII主体,
4=隐私侵犯会对道路使用者造成重大甚至不可逆转的影响 泄露的信息高度敏感，并且很容易识别到PII主体 
#### 需要考虑车主、驾驶员、乘员、行人、供应商、主机厂。

## 强约束条件
- 每个维度必须严格对照评级标准打分，不得使用其他标准
- 分值必须是整数 1~4
- impact_level = 四维最高分对应的等级名称（Negligible / Moderate / Major / Severe）

## 输出（严格 JSON 数组）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "damage_scenario": "损害场景",
    "influence_parameters": {{"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}},
    "impact_level": "Negligible/Moderate/Major/Severe"
  }}
]"""


# ────── 步骤6: 可行性评级 Prompt (分批调用) ──────
AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434（尤其是 Annex G 基于攻击潜力的攻击可行性评估方法） 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准
### Exposure_time: 
- 0分：≤1天（实现攻击行为的时间小于等于1天）  
- 1分：≤1周（实现攻击行为的时间小于等于1周）  
- 4分：≤1个月（实现攻击行为的时间小于等于1个月）  
- 17分：≤6个月（实现攻击行为的时间小于等于6个月）  
- 19分：>6个月（实现攻击行为的时间大于6个月）
### Professional_experience:
- 0分：非专业（Layman）——外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3分：精通/擅长（Proficient）——熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6分：专业（Expert）——熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 8分：复合型（Multiple experts）——一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0分：公开的（Public）——公共信息（例如互联网上获得的信息）  
- 3分：限制的（Restricted）——受限制的信息（制造商和供应商共享的内部文档）  
- 7分：敏感的（Sensitive）——机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11分：至关重要的（Critical）——严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0分：十分高（Critical）——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1分：高（High）——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4分：中（Medium）——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10分：低（Low）——困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0分：标准的（Standard）——标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4分：专用的（Specialized）——专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7分：定制的（Bespoke）——定制设备（厂家限制的工具、电子显微镜）  
- 9分：多重定制的（Multiple bespoke）——攻击不同步骤需要不同类型的定制设备

## 可行性等级: 五维求和 → ≤13=High, 14~19=Medium, 20~24=Low, ≥25=Very Low

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数（0、1、3、4、6、7、8、9、10、11、17、19）
- feasibility_score = 五维分值之和（整数）
- feasibility_level 必须使用精确英文名称：High / Medium / Low / Very Low

## 输出（严格 JSON 数组）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "attack_path": "攻击路径",
    "attack_parameters": {{
      "Exposure_time": 0, "Professional_experience": 0,
      "Required_information": 0, "Opportunity_window": 0, "Required_equipment": 0
    }},
    "feasibility_score": 0,
    "feasibility_level": "High/Medium/Low/Very Low"
  }}
]"""


# ────── 评估 Prompt (步骤2/3/4共用, 不变) ──────
EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估质量。
仅输出 JSON，不要任何其他文字。"""

DS_EVAL_USER_TEMPLATE = """## 评审任务
请评估以下损害场景是否符合 ISO 21434 要求。
## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{attribute_name}
## 待评审的损害场景
{damage_scenario}

## 评估标准
1. 完整性：每条{attribute_name}都有对应{damage_scenario}
2. 因果合理性：{damage_scenario}是否说明了{function_name}异常与后果的因果关系


## 输出格式
{{"passed": true或false, "feedback": "如不通过，列出具体问题"}}"""

TS_EVAL_USER_TEMPLATE = """## 评审任务
请评估以下威胁场景。
## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{attribute_name}
## 威胁类型
{threat_type}
## 损害场景
{damage_scenario}
## 待评审
{threat_scenario}

## 评估标准
1. 完整性：每条{damage_scenario}都有对应{threat_scenario}
2. 因果一致性：{damage_scenario}是“最终坏结果”，{threat_scenario}是“导致坏结果的网络安全威胁原因”

## 输出格式
{{"passed": true或false, "feedback": "如不通过，列出具体问题"}}"""

AP_EVAL_USER_TEMPLATE = """## 评审任务
请评估攻击路径质量。
## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{attribute_name}
## 损害场景
{damage_scenario}
## 威胁场景
{threat_scenario}
## 车辆拓扑图
{topo_info}

## 外部接口
{ext_info}

## 待评审
{attack_paths}

## 评估标准
1. 完整性：是否包含完整攻击链
2. 拓扑一致性：是否符合车辆拓扑图节点与协议传播路径
3. 攻击入口有效性：是否存在对应外部接口
4. 步骤连贯性：资产-威胁场景-攻击路径是否匹配
5. 因果一致性：{damage_scenario}是“最终坏结果”，{threat_scenario}是“导致坏结果的网络安全威胁原因”，{attack_paths}是“具体怎么实现这个威胁”的攻击步骤。{threat_scenario}是“资产的{security_attributes}被破坏”以实现{damage_scenario}的潜在原因；而{attack_paths}则是实现该{threat_scenario}的一组蓄意活动。

## 输出格式
{{"passed": true或false, "feedback": "如不通过，列出具体问题"}}"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ────────── 步骤1: 资产识别 (规则引擎, 不变) ──────────

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": attrs,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "ds_retry_count": 0, "ts_retry_count": 0, "ap_retry_count": 0,
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


# ──────────  步骤2: 损害场景 (逐条并发生成) ──────────

def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]
    feedback = state.get("ds_feedback", "")
    feedback_block = f"\n## ⚠️ 上轮评审反馈\n{feedback}\n" if feedback else ""

    # RAG 检索一次，所有子任务共享
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    #  构建逐条任务
    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for attr in asset["security_attributes"]:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                prompt = DS_SINGLE_USER_TEMPLATE.format(
                    function_name=func["function"],
                    asset_name=asset["asset_name"],
                    asset_type=asset["asset_type"],
                    security_attribute=attr,
                    threat_type=threat_type,
                    rag_context=rag_context,
                    feedback_block=feedback_block,
                )
                tasks.append({"id": task_id, "system": BASE_SYSTEM, "user": prompt})
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    # ★ 并发调用
    raw_results = concurrent_llm_calls(tasks)

    # ★ 聚合结果
    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                scenarios.extend(parsed)
            else:
                scenarios.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {"damage_scenarios": scenarios}


def node_ds_evaluate(state: TaraState) -> dict:
    logger.info("[步骤2-评估] 损害场景评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ds_eval_passed": True, "ds_feedback": ""}

    scenarios = state.get("damage_scenarios", [])
    assets = state.get("asset_results", [])

    if not scenarios:
        return {"ds_eval_passed": False, "ds_feedback": "损害场景为空，请重新生成。"}

    expected = sum(len(a.get("security_attributes", [])) for f in assets for a in f.get("assets", []))

    user_prompt = DS_EVAL_USER_TEMPLATE.format(
        expected_count=expected, actual_count=len(scenarios),
        asset_info=json.dumps(assets, ensure_ascii=False, indent=2),
        scenarios=json.dumps(scenarios, ensure_ascii=False, indent=2),
    )

    raw = llm_call(EVAL_SYSTEM, user_prompt)
    try:
        result = safe_parse_json(raw)
        passed = bool(result.get("passed", False))
        feedback = result.get("feedback", "")
    except Exception:
        passed, feedback = True, ""

    logger.info(f"  → {'✅ 通过' if passed else '❌ 不通过'}")
    return {"ds_eval_passed": passed, "ds_feedback": feedback}


def node_ds_correct(state: TaraState) -> dict:
    new_count = state.get("ds_retry_count", 0) + 1
    logger.info(f"  [损害场景修正] 第 {new_count}/{MAX_EVAL_RETRIES} 次重试")
    return {"ds_retry_count": new_count}


# ──────────  步骤3: 威胁场景 (逐条并发生成) ──────────

def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    feedback = state.get("ts_feedback", "")
    feedback_block = f"\n## ⚠️ 上轮评审反馈\n{feedback}\n" if feedback else ""
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    # ★ 逐条构建任务
    tasks = []
    for i, ds in enumerate(damage_scenarios):
        prompt = TS_SINGLE_USER_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            asset_type=ds.get("asset_type", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
            rag_context=rag_context,
            feedback_block=feedback_block,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ts_list.extend(parsed)
            else:
                ts_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {"threat_scenarios": ts_list}


def node_ts_evaluate(state: TaraState) -> dict:
    logger.info("[步骤3-评估] 威胁场景评估")

    if not ENABLE_EVALUATION:
        return {"ts_eval_passed": True, "ts_feedback": ""}

    ts = state.get("threat_scenarios", [])
    ds = state.get("damage_scenarios", [])
    if not ts:
        return {"ts_eval_passed": False, "ts_feedback": "威胁场景为空。"}

    user_prompt = TS_EVAL_USER_TEMPLATE.format(
        ds_count=len(ds), ts_count=len(ts),
        scenarios=json.dumps(ts, ensure_ascii=False, indent=2),
    )
    raw = llm_call(EVAL_SYSTEM, user_prompt)
    try:
        result = safe_parse_json(raw)
        passed = bool(result.get("passed", False))
        feedback = result.get("feedback", "")
    except Exception:
        passed, feedback = True, ""

    logger.info(f"  → {'✅ 通过' if passed else '❌ 不通过'}")
    return {"ts_eval_passed": passed, "ts_feedback": feedback}


def node_ts_correct(state: TaraState) -> dict:
    new_count = state.get("ts_retry_count", 0) + 1
    logger.info(f"  [威胁场景修正] 第 {new_count}/{MAX_EVAL_RETRIES} 次重试")
    return {"ts_retry_count": new_count}


# ──────────  步骤4: 攻击路径 (逐条并发生成 — 核心改动) ──────────

def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发 — 解决重复问题)")

    threat_scenarios = state.get("threat_scenarios", [])
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    feedback = state.get("ap_feedback", "")
    feedback_block = f"\n## ⚠️ 上轮评审反馈\n{feedback}\n" if feedback else ""
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    #  每条威胁场景独立生成攻击路径
    tasks = []
    for i, ts in enumerate(threat_scenarios):
        prompt = AP_SINGLE_USER_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            asset_type=ts.get("asset_type", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_scenario=ts.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            rag_context=rag_context,
            feedback_block=feedback_block,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ap_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ap_list.extend(parsed)
            else:
                ap_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成攻击路径 {len(ap_list)}/{len(tasks)} 组")
    save_checkpoint(ap_list, "step4_attack_paths")
    return {"attack_paths": ap_list}


def node_ap_evaluate(state: TaraState) -> dict:
    logger.info("[步骤4-评估] 攻击路径评估")

    if not ENABLE_EVALUATION:
        return {"ap_eval_passed": True, "ap_feedback": ""}

    ap = state.get("attack_paths", [])
    ts = state.get("threat_scenarios", [])
    if not ap:
        return {"ap_eval_passed": False, "ap_feedback": "攻击路径为空。"}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)

    user_prompt = AP_EVAL_USER_TEMPLATE.format(
        ts_count=len(ts), ap_count=len(ap),
        topo_info=topo_info, ext_info=ext_info,
        scenarios=json.dumps(ap, ensure_ascii=False, indent=2),
    )
    raw = llm_call(EVAL_SYSTEM, user_prompt)
    try:
        result = safe_parse_json(raw)
        passed = bool(result.get("passed", False))
        feedback = result.get("feedback", "")
    except Exception:
        passed, feedback = True, ""

    logger.info(f"  → {'✅ 通过' if passed else '❌ 不通过'}")
    return {"ap_eval_passed": passed, "ap_feedback": feedback}


def node_ap_correct(state: TaraState) -> dict:
    new_count = state.get("ap_retry_count", 0) + 1
    logger.info(f"  [攻击路径修正] 第 {new_count}/{MAX_EVAL_RETRIES} 次重试")
    return {"ap_retry_count": new_count}


# ──────────  步骤5: 影响评级 (分批并发) ──────────

def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响参数评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    # ★ 分批
    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = IR_BATCH_USER_TEMPLATE.format(
            ds_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # 校验 impact_level
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    for r in ratings:
        params = r.get("influence_parameters", {})
        max_val = max(int(params.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
        r["impact_level"] = IMPACT_LABELS.get(min(max_val, 4), "Negligible")

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


# ──────────  步骤6: 攻击可行性评级 (分批并发) ──────────

def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (分批并发)")

    # 展平攻击路径
    flat_paths = []
    for ap_group in state.get("attack_paths", []):
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "function": ap_group.get("function", ""),
                "asset_name": ap_group.get("asset_name", ""),
                "security_attribute": ap_group.get("security_attribute", ""),
                "attack_path": path,
            })

    if not flat_paths:
        return {"feasibility_ratings": []}

    # ★ 分批
    batches = [flat_paths[i:i+BATCH_SIZE_RATING] for i in range(0, len(flat_paths), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(flat_paths)} 条攻击路径, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = AF_BATCH_USER_TEMPLATE.format(
            ap_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": AF_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # 校验 feasibility_level
    for r in ratings:
        params = r.get("attack_parameters", {})
        total = sum(int(params.get(k, 0)) for k in [
            "Exposure_time", "Professional_experience",
            "Required_information", "Opportunity_window", "Required_equipment",
        ])
        r["feasibility_score"] = total
        if total <= 13:
            r["feasibility_level"] = "High"
        elif total <= 19:
            r["feasibility_level"] = "Medium"
        elif total <= 24:
            r["feasibility_level"] = "Low"
        else:
            r["feasibility_level"] = "Very Low"

    logger.info(f"  → 完成可行性评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": ratings}


# ────────── 步骤7: 风险判定 (规则引擎, 不变) ──────────

def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险判定 (交叉矩阵)")

    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    RISK_MATRIX = {
        ("Severe", "High"): 5, ("Severe", "Medium"): 4, ("Severe", "Low"): 3,
        ("Severe", "Very Low"): 2, 
        ("Major", "High"): 4, ("Major", "Medium"): 3, ("Major", "Low"): 2,
        ("Major", "Very Low"): 1, 
        ("Moderate", "High"): 3, ("Moderate", "Medium"): 2, ("Moderate", "Low"): 2,
        ("Moderate", "Very Low"): 1, 
        ("Negligible", "High"): 1, ("Negligible", "Medium"): 1, ("Negligible", "Low"): 1,
        ("Negligible", "Very Low"): 1,
    }
    RISK_LABEL = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
    TREATMENT = {
        "1": "风险接受",
        "2": "风险接受",
        "3": "风险接受",
        "4": "风险缓解",
        "5": "风险缓解",
    }

    FEAS_ORDER = {"High": 4, "Medium": 3, "Low": 2, "Very Low": 1}
    feas_map: dict[tuple, str] = {}
    for fr in feasibility_ratings:
        key = (fr.get("function", ""), fr.get("asset_name", ""), fr.get("security_attribute", ""))
        level = fr.get("feasibility_level", "Very Low")
        if FEAS_ORDER.get(level, 1) > FEAS_ORDER.get(feas_map.get(key, "Very Low"), 1):
            feas_map[key] = level

    risk_results = []
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        impact_level = ir.get("impact_level", "Negligible")
        feasibility_level = feas_map.get(key, "Very Low")
        risk_val = RISK_MATRIX.get((impact_level, feasibility_level), 1)
        risk_label = RISK_LABEL.get(risk_val, "1")
        treatment = TREATMENT.get(risk_label, "风险接受")

        risk_results.append({
            "function": ir.get("function", ""),
            "asset_name": ir.get("asset_name", ""),
            "security_attribute": ir.get("security_attribute", ""),
            "impact_level": impact_level,
            "feasibility_level": feasibility_level,
            "risk_value": risk_label,
            "risk_treatment": treatment,
        })

    logger.info(f"  → 风险判定完成 {len(risk_results)} 条")
    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


# ────────── 步骤8: 报告汇总──────────

def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    rr_map = {_key(r): r for r in risk_results}

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for attr in asset.get("security_attributes", []):
                key = (func_name, asset["asset_name"], attr)
                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": "", "Professional_experience": "",
                            "Required_information": "", "Opportunity_window": "",
                            "Required_equipment": "",
                        }),
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "threat_type": ds.get("threat_type", ""),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": "", "Finance": "", "Operation": "", "Privacy": "",
                    }),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "attack": attack_list,
                    "risk_value": rr.get("risk_value", ""),
                    "risk_treatment": rr.get("risk_treatment", ""),
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")
    return {"tara_report": report}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：路由函数 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def route_ds_eval(state: TaraState) -> Literal["node_ds_correct", "node_ts_generate"]:
    if state.get("ds_eval_passed") or state.get("ds_retry_count", 0) >= MAX_EVAL_RETRIES:
        if not state.get("ds_eval_passed"):
            logger.warning("  ⚠ 损害场景重试已达上限，强制继续")
        return "node_ts_generate"
    return "node_ds_correct"


def route_ts_eval(state: TaraState) -> Literal["node_ts_correct", "node_ap_generate"]:
    if state.get("ts_eval_passed") or state.get("ts_retry_count", 0) >= MAX_EVAL_RETRIES:
        if not state.get("ts_eval_passed"):
            logger.warning("  ⚠ 威胁场景重试已达上限，强制继续")
        return "node_ap_generate"
    return "node_ts_correct"


def route_ap_eval(state: TaraState) -> Literal["node_ap_correct", "node_impact_rating"]:
    if state.get("ap_eval_passed") or state.get("ap_retry_count", 0) >= MAX_EVAL_RETRIES:
        if not state.get("ap_eval_passed"):
            logger.warning("  ⚠ 攻击路径重试已达上限，强制继续")
        return "node_impact_rating"
    return "node_ap_correct"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：工作流构建 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_evaluate",         node_ds_evaluate)
    builder.add_node("node_ds_correct",          node_ds_correct)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_evaluate",         node_ts_evaluate)
    builder.add_node("node_ts_correct",          node_ts_correct)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_evaluate",         node_ap_evaluate)
    builder.add_node("node_ap_correct",          node_ap_correct)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")

    builder.add_edge("node_ds_generate", "node_ds_evaluate")
    builder.add_conditional_edges("node_ds_evaluate", route_ds_eval)
    builder.add_edge("node_ds_correct", "node_ds_generate")

    builder.add_edge("node_ts_generate", "node_ts_evaluate")
    builder.add_conditional_edges("node_ts_evaluate", route_ts_eval)
    builder.add_edge("node_ts_correct", "node_ts_generate")

    builder.add_edge("node_ap_generate", "node_ap_evaluate")
    builder.add_conditional_edges("node_ap_evaluate", route_ap_eval)
    builder.add_edge("node_ap_correct", "node_ap_generate")

    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_json: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: dict | None = None,
    asset_results: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": data_flow_json or [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or {},
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_eval_passed": False, "ds_retry_count": 0, "ds_feedback": "",
        "threat_scenarios": [], "ts_eval_passed": False, "ts_retry_count": 0, "ts_feedback": "",
        "attack_paths": [],    "ap_eval_passed": False, "ap_retry_count": 0, "ap_feedback": "",
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    # ★ 保存最终报告
    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    # ★ 同时保存完整中间状态（便于调试）
    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":

    sample_assets = [
        {
            "function": "交通标识及信号灯识别",
            "assets": [
                {"asset_type": "信号", "asset_name": "弹窗/音效"},
            ],
        },
        {
            "function": "智能泊车辅助",
            "assets": [
                {"asset_type": "数据", "asset_name": "用户账户信息"},
            ],
        },
    ]

    sample_topology = [
  {
    "protocol": "CANFD",
    "color": "#F53F3F",
    "controller": [
      "AMP",
      "CDC",
      "IC"
    ]
  },
  {
    "protocol": "A2B",
    "color": "#016AFF",
    "controller": [
      "AMP",
      "CDC"
    ]
  },
  {
    "protocol": "FPDLINK",
    "color": "#00B42A",
    "controller": [
      "CDC",
      "IC"
    ]
  },
  {
    "protocol": "FPDLINK",
    "color": "#FF7D00",
    "controller": [
      "CDC",
      "IVI"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#000000",
    "controller": [
      "CDC",
      "DMS"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#00B0F0",
    "controller": [
      "CDC",
      "OMS"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#7F7F7F",
    "controller": [
      "CDC",
      "MDC"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#8F8F8F",
    "controller": [
      "CDC",
      "CEM"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#5470C6",
    "controller": [
      "BDCR",
      "CDC",
      "CEM",
      "REA"
    ]
  },
  {
    "protocol": "CAN",
    "color": "#91CC75",
    "controller": [
      "BDCR",
      "BLE",
      "CEM",
      "CMSL",
      "CMSR",
      "DSM",
      "ECALL",
      "PDSM",
      "TRM",
      "WLCM"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#FAC858",
    "controller": [
      "MDC",
      "USS ECU"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#EE6666",
    "controller": [
      "FR",
      "MDC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#73C0DE",
    "controller": [
      "LRC-FC",
      "MDC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#FCDBC9",
    "controller": [
      "MDC",
      "SRC-FC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#9A60B4",
    "controller": [
      "MDC",
      "MRC-SC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#BC85FE",
    "controller": [
      "MDC",
      "MRC-RC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#3088DA",
    "controller": [
      "FEC",
      "MDC"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#5E7C61",
    "controller": [
      "CEM",
      "MDC"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#831A5C",
    "controller": [
      "CEM",
      "EPS",
      "IBCU",
      "MDC",
      "SRS",
      "VCU",
      "VMC"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#8D62E8",
    "controller": [
      "CEM",
      "VCU"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#0D21E3",
    "controller": [
      "BCU",
      "EVCC",
      "FMIPU",
      "HPMU",
      "ITMS",
      "PDU",
      "RDPEU",
      "RMIPU",
      "VCU"
    ]
  },
  {
    "protocol": "CAN",
    "color": "#01F2E3",
    "controller": [
      "BCU",
      "EVCC"
    ]
  },
  {
    "protocol": "CAN",
    "color": "#29FC11",
    "controller": [
      "ACP",
      "ITMS",
      "PTC"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#973D5D",
    "controller": [
      "BDCR",
      "CDC",
      "CEM",
      "LBMS"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#CDED23",
    "controller": [
      "Lidar",
      "MDC"
    ]
  },
  {
    "protocol": "蜂窝",
    "color": "#AF21CE",
    "controller": [
      "BCALL中心",
      "CDC",
      "CEM",
      "ECALL中心",
      "华为车云",
      "诊断仪",
      "阿维塔车云"
    ]
  }
]

    sample_ext_interfaces = [
        {"component": "CDC", "interfaces": ["Cellular Netword  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
        {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
        {"component": "UWB", "interfaces": ["RF  射频"]},
        {"component": "CHLIL", "interfaces": ["JTAG"]},
        {"component": "CHLIR", "interfaces": ["JTAG"]},
        {"component": "ITMS", "interfaces": ["JTAG"]},
    ]

    report = run_tara(
        asset_results=sample_assets,
        topology_json=sample_topology,
        external_interface_json=sample_ext_interfaces,
        output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
        build_rag_index=True,
    )

    logger.info(f"📋 最终报告包含 {len(report)} 个功能模块，已保存为 JSON 文件")

2026-04-10 09:29:03,773 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-10 09:29:03,774 [INFO] ============================================================
2026-04-10 09:29:03,775 [INFO]   并发配置: MAX_WORKERS=3, BATCH_SIZE_RATING=5
2026-04-10 09:29:03,775 [INFO]   评估开关: ENABLE_EVALUATION=False
2026-04-10 09:29:03,776 [INFO] [初始化] RAG 已禁用，跳过
2026-04-10 09:29:03,800 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-10 09:29:03,802 [INFO] ============================================================
2026-04-10 09:29:03,803 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-10 09:29:03,803 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-10 09:29:03,805 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V2\checkpoint_step1_assets.json
2026-04-10 09:29:03,805 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-10 09:29:03,806 [INFO]   → 共 7 个子任务, 并发数 3
2026-04-10 09:29:07,287 [INFO] HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 09:29:08,617 [INFO]     ✅ [1/7] 任务 0 完成
2026-04-10 09:30:04,732 [INFO] Re

KeyError: 'damage_scenario'

# V3
置信度评估+人工审核

In [62]:
import json
import os
import re
import glob
import logging
import time
from typing import TypedDict, Literal, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache
import hashlib

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import requests
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 1.1 LLM 配置 ──
llm = ChatOpenAI(
    model="Qwen/Qwen3.5-35B-A3B",
    openai_api_key="sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={
        "enable_thinking": False
    },
)

# ── 1.2 Embedding 配置 ──
ENABLE_RAG = False
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

# ── 1.3 RAG 知识库路径 ──
RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

# ── 1.4 工作流参数 ──
# 移除 MAX_EVAL_RETRIES，不再需要重试机制
ENABLE_EVALUATION = True  # 启用置信度评估

# ── 1.5 并发与分批配置 ──
MAX_WORKERS = 10         # 最大并发 LLM 调用数
CALL_INTERVAL = 0.9      # 提交任务间隔(秒), 用于控制 API 速率
BATCH_SIZE_RATING = 10    # 评级步骤的批大小

# ── 1.6 输出路径 ──
OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1.7 资产类型 → 安全属性映射 ──
SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

# ── 1.8 安全属性 → STRIDE 映射 ──
ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}

# ── 1.9 置信度评估配置 ──
CONFIDENCE_THRESHOLD = 85  # 初始阈值设为85

# 置信度评估权重
CONFIDENCE_WEIGHTS = {
    "要素完整性": 0.30,  # 30%
    "一致性": 0.30,      # 30%
    "真实性": 0.30,      # 30%
    "清晰度": 0.10,      # 10%
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,          # 单次最多请求 8 条文本，防 413 限流
                request_timeout=45,    # 防网络卡死
                max_retries=2          # 自动重试
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,         # 重叠减少，防 Token 膨胀
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:  # 硬拦截 >128 字符的块
                valid_chunks.append(c)
            else:
                # 兜底：超长块强制截断（保命策略）
                if len(content) > 128:
                    c.page_content = content[:127] + "..."
                    valid_chunks.append(c)
        
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True)
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs)
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                # 字典按顶层 key 拆分
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: dict
    external_interface_json: dict
    asset_results: list
    damage_scenarios: list
    # 移除 ds_eval_passed, ds_retry_count, ds_feedback
    ds_confidence: dict  # 存储损害场景的置信度评估结果
    threat_scenarios: list
    # 移除 ts_eval_passed, ts_retry_count, ts_feedback
    ts_confidence: dict  # 存储威胁场景的置信度评估结果
    attack_paths: list
    # 移除 ap_eval_passed, ap_retry_count, ap_feedback
    ap_confidence: dict  # 存储攻击路径的置信度评估结果
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list
    # 新增：需要人工审核的场景列表
    manual_review_items: list


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
        
    cleaned = text.strip()
    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


# ── 缓存机制 ──
hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()

def hash_prompt(text: str) -> str:
    """生成 prompt 的 MD5 哈希（用于缓存键）"""
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def register_prompt_for_cache(prompt: str) -> str:
    """注册 prompt 到全局映射，返回其 hash"""
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h

def clear_prompt_cache():
    """清空缓存映射（测试/重置时用）"""
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()

@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    """基于 prompt 哈希的缓存调用，使用前必须先用 register_prompt_for_cache 注册 prompt"""
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册 (sys={system_hash[:8]}..., user={user_hash[:8]}...)")
        return "[]"
    
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


# 带重试的 LLM 调用
def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
                
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: langchain 返回类型不匹配 - {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    
    return "[]"


# 并发 LLM 调用引擎（增强日志统计）
def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    """
    并发执行多个 LLM 调用
    tasks: [{"id": str/int, "system": str, "user": str}, ...]
    返回: [{"id": ..., "raw": str}, ...]  按 id 排序
    """
    results = []
    total = len(tasks)
    if total == 0:
        return results
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)

        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")

    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s, 平均 {elapsed/len(tasks):.2f}s/任务")
    return results


def save_checkpoint(data: Any, step_name: str):
    """按步骤名保存独立 checkpoint"""
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Python计算函数（新增）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def calculate_impact_value(influence_parameters: dict) -> int:
    """
    计算影响值（四维最高分）
    """
    if not influence_parameters:
        return 1
    max_val = max(int(influence_parameters.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
    return min(max(max_val, 1), 4)  # 确保在1-4范围内


def calculate_impact_level(impact_value: int) -> str:
    """
    根据影响值映射影响等级
    """
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    return IMPACT_LABELS.get(impact_value, "Negligible")


def calculate_feasibility_score(attack_parameters: dict) -> int:
    """
    计算可行性分数（五维求和）
    """
    if not attack_parameters:
        return 0
    return sum(int(attack_parameters.get(k, 0)) for k in [
        "Exposure_time", "Professional_experience",
        "Required_information", "Opportunity_window", "Required_equipment",
    ])


def calculate_feasibility_level(feasibility_score: int) -> str:
    """
    根据可行性分数映射可行性等级
    """
    if feasibility_score <= 13:
        return "High"
    elif feasibility_score <= 19:
        return "Medium"
    elif feasibility_score <= 24:
        return "Low"
    else:
        return "Very Low"


def calculate_confidence_score(scores: dict) -> float:
    """
    计算加权置信度分数（0-100）
    scores: {"要素完整性": 0-100, "一致性": 0-100, "真实性": 0-100, "清晰度": 0-100}
    """
    total = 0.0
    for dimension, weight in CONFIDENCE_WEIGHTS.items():
        score = scores.get(dimension, 0)
        total += score * weight
    return round(total, 2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径。
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


# ──────  步骤2: 单条损害场景 Prompt (逐条调用) ──────
DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是"涉及车辆或车辆功能且影响道路使用者的不良后果"。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}
- 对应威胁类型：{threat_type}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体危害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"

## 输出（严格 JSON 对象，不要数组）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "完整的损害场景描述",
  "threat_type": "{threat_type}"
}}"""


# ──────  步骤3: 单条威胁场景 Prompt (逐条调用) ──────
TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是"为实现损害场景，资产的信息安全属性遭到破坏的潜在原因"。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体{asset_name}，如"GW部件"、"前照灯请求信号"、"车辆数据"
   - 「{asset_name}」被破坏的「{security_attribute}」（必须与输入的安全属性完全一致）
   - 「{security_attribute}」被破坏的原因/攻击意图
2. - 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条。
   - 进一步的信息能被威胁场景包含或与威胁场景关联。例如:危害场景与资产、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "{damage_scenario}",
  "threat_type": "{threat_type}",
  "threat_scenario": "详细的威胁场景描述"
}}"""



# ──────  步骤4: 单条攻击路径 Prompt (逐条调用, 附带拓扑) ──────
AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」生成最短的的攻击路径（经过拓扑节点最少）,攻击路径是"为实现威胁场景的一组蓄意活动"。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}（{asset_type}）
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产：
   - 以编号步骤形式呈现：1.…；2.…；3.…；…
   - 第一步：明确的外部攻击入口
   - 中间步骤：沿车辆拓扑经过的中间节点和通信协议
   - 最终步骤：到达「{asset_name}」并实施具体攻击技术，从而破坏「{security_attribute}」
2. 路径中的节点和连接关系必须与车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」一致，不得虚构不存在的节点或跳过拓扑中的网关/ECU。
3. 必须采用自上而下的方法（如攻击树分析），从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的{threat_scenario}
4. {topo_info}中相同名称{protocol}的不同{color}是不同的信道协议
5. 只输出最短的的攻击路径（经过拓扑节点最少）,如有多条最短路径请生成多条，最多不超过4条,同一个威胁场景的多条攻击路径不要重复

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号(非预期的快速减速)。"

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "threat_scenario": "{threat_scenario}",
  "attack_paths": [
    "1.完整步骤链",
    "2.完整步骤链"
  ]
}}"""


# ────── 步骤5: 影响评级 Prompt (分批调用) - 修改为只输出参数 ──────
IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数）
### Safety: 1=无伤害, 2=轻度/中度伤害, 3=严重伤害和有生命危险, 4=威胁生命的伤害（不确定是否幸存）或致命的伤害
### Finance: 1=可忽略, 2=个人可承受, 3=大量损失但受影响的道路使用者将能够克服这些后果, 4=灾难性后果受影响的道路使用者可能无法克服
#### 利益相关者包括：车主、驾驶员、乘员、行人、供应商、主机厂。
### Operation: 1=不影响, 2=导致车辆功能的部分退化, 3=导致车辆重要功能丧失或受损, 4=导致车辆核心功能丧失或受损
### Privacy: 1=隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体,
2=隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体； b)泄露的信息不敏感但很容易识别到PII主体,
3=隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体； b)泄露的信息敏感而且很容易识别到PII主体,
4=隐私侵犯会对道路使用者造成重大甚至不可逆转的影响 泄露的信息高度敏感，并且很容易识别到PII主体 
#### 需要考虑车主、驾驶员、乘员、行人、供应商、主机厂。

## 强约束条件
- 每个维度必须严格对照评级标准打分，不得使用其他标准
- 分值必须是整数 1~4

## 输出（严格 JSON 数组，只输出参数）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "influence_parameters": {{"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}}
  }}
]"""


# ────── 步骤6: 可行性评级 Prompt (分批调用) - 修改为只输出参数 ──────
AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434（尤其是 Annex G 基于攻击潜力的攻击可行性评估方法） 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准
### Exposure_time: 
- 0分：≤1天（实现攻击行为的时间小于等于1天）  
- 1分：≤1周（实现攻击行为的时间小于等于1周）  
- 4分：≤1个月（实现攻击行为的时间小于等于1个月）  
- 17分：≤6个月（实现攻击行为的时间小于等于6个月）  
- 19分：>6个月（实现攻击行为的时间大于6个月）
### Professional_experience:
- 0分：非专业（Layman）——外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3分：精通/擅长（Proficient）——熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6分：专业（Expert）——熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 8分：复合型（Multiple experts）——一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0分：公开的（Public）——公共信息（例如互联网上获得的信息）  
- 3分：限制的（Restricted）——受限制的信息（制造商和供应商共享的内部文档）  
- 7分：敏感的（Sensitive）——机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11分：至关重要的（Critical）——严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0分：十分高（Critical）——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1分：高（High）——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4分：中（Medium）——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10分：低（Low）——困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0分：标准的（Standard）——标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4分：专用的（Specialized）——专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7分：定制的（Bespoke）——定制设备（厂家限制的工具、电子显微镜）  
- 9分：多重定制的（Multiple bespoke）——攻击不同步骤需要不同类型的定制设备

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数（0、1、3、4、6、7、8、9、10、11、17、19）

## 输出（严格 JSON 数组，只输出参数）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "attack_path": "攻击路径",
    "attack_parameters": {{
      "Exposure_time": 0, "Professional_experience": 0,
      "Required_information": 0, "Opportunity_window": 0, "Required_equipment": 0
    }}
  }}
]"""


# ────── 置信度评估 Prompt (新增) ──────
CONFIDENCE_EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估场景质量并给出各维度的置信度评分。
仅输出 JSON，不要任何其他文字。"""

# 损害场景置信度评估模板
DS_CONFIDENCE_EVAL_TEMPLATE = """## 评审任务
请评估以下损害场景的质量，并给出各维度的置信度评分（0-100分）。

## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{security_attribute}
## 威胁类型
{threat_type}
## 待评审的损害场景
{damage_scenario}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
损害场景必须包含以下要素：
- 功能与不良后果的因果关系
- 对道路使用者的具体危害
- 涉及的目标资产
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 与功能/资产/STRIDE是否匹配
- 损害场景描述是否与输入信息一致
评分标准：完全匹配100分，部分不匹配扣20-50分，严重不一致扣60-80分

### 3. 真实性（满分100分）
- 危害对道路使用者影响是否合理
- 是否有明显幻觉或不合理描述
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 描述是否清晰易懂
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式
{{
  "要素完整性": 0-100,
  "一致性": 0-100,
  "真实性": 0-100,
  "清晰度": 0-100,
  "confidence_score": 加权总分,
  "passed": true或false（总分>=85为true）,
  "feedback": "如不通过，列出具体问题"
}}"""

# 威胁场景置信度评估模板
TS_CONFIDENCE_EVAL_TEMPLATE = """## 评审任务
请评估以下威胁场景的质量，并给出各维度的置信度评分（0-100分）。

## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{security_attribute}
## 威胁类型
{threat_type}
## 损害场景
{damage_scenario}
## 待评审的威胁场景
{threat_scenario}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
威胁场景必须包含以下要素：
- 目标资产
- 被破坏的信息安全属性
- 信息安全属性被破坏的原因
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 是否直接导致损害场景
- 与损害场景的因果关系是否清晰
评分标准：完全一致100分，部分不一致扣20-50分，因果关系断裂扣60-80分

### 3. 真实性（满分100分）
- 攻击意图是否现实
- 是否有明显幻觉或不合理描述
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 描述是否清晰易懂
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式
{{
  "要素完整性": 0-100,
  "一致性": 0-100,
  "真实性": 0-100,
  "清晰度": 0-100,
  "confidence_score": 加权总分,
  "passed": true或false（总分>=85为true）,
  "feedback": "如不通过，列出具体问题"
}}"""

# 攻击路径置信度评估模板
AP_CONFIDENCE_EVAL_TEMPLATE = """## 评审任务
请评估以下攻击路径的质量，并给出各维度的置信度评分（0-100分）。

## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{security_attribute}
## 损害场景
{damage_scenario}
## 威胁场景
{threat_scenario}
## 车辆拓扑图
{topo_info}
## 外部接口
{ext_info}
## 待评审的攻击路径
{attack_paths}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
攻击路径必须包含以下要素：
- 起点（具体攻击面）
- 逻辑步骤
- 终点（目标资产）
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 是否能真正实现威胁场景
- 是否符合拓扑图
- 攻击入口是否存在于外部接口中
评分标准：完全一致100分，部分不一致扣20-50分，严重偏离扣60-80分

### 3. 真实性（满分100分）
- 攻击面是否存在
- 步骤是否可行
- 是否有明显幻觉或不合理描述
评分标准：完全可行100分，部分不可行扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 步骤描述是否清晰连贯
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式
{{
  "要素完整性": 0-100,
  "一致性": 0-100,
  "真实性": 0-100,
  "清晰度": 0-100,
  "confidence_score": 加权总分,
  "passed": true或false（总分>=85为true）,
  "feedback": "如不通过，列出具体问题"
}}"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ────────── 步骤1: 资产识别 (规则引擎, 不变) ──────────

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": attrs,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "manual_review_items": [],  # 初始化人工审核列表
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


# ──────────  步骤2: 损害场景 (逐条并发生成) ──────────

def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]

    # RAG 检索一次，所有子任务共享
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    #  构建逐条任务
    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for attr in asset["security_attributes"]:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                prompt = DS_SINGLE_USER_TEMPLATE.format(
                    function_name=func["function"],
                    asset_name=asset["asset_name"],
                    asset_type=asset["asset_type"],
                    security_attribute=attr,
                    threat_type=threat_type,
                    rag_context=rag_context,
                )
                tasks.append({"id": task_id, "system": BASE_SYSTEM, "user": prompt})
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    # ★ 并发调用
    raw_results = concurrent_llm_calls(tasks)

    # ★ 聚合结果
    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                scenarios.extend(parsed)
            else:
                scenarios.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {"damage_scenarios": scenarios}


def node_ds_confidence_eval(state: TaraState) -> dict:
    """损害场景置信度评估节点"""
    logger.info("[步骤2-评估] 损害场景置信度评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ds_confidence": {"all_passed": True, "items": []}}

    scenarios = state.get("damage_scenarios", [])
    if not scenarios:
        return {"ds_confidence": {"all_passed": True, "items": []}}

    # 构建评估任务
    tasks = []
    for i, ds in enumerate(scenarios):
        prompt = DS_CONFIDENCE_EVAL_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
        )
        tasks.append({"id": i, "system": CONFIDENCE_EVAL_SYSTEM, "user": prompt})

    logger.info(f"  → 评估 {len(tasks)} 个损害场景")
    raw_results = concurrent_llm_calls(tasks)

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            # 计算加权置信度分数
            scores = {
                "要素完整性": parsed.get("要素完整性", 0),
                "一致性": parsed.get("一致性", 0),
                "真实性": parsed.get("真实性", 0),
                "清晰度": parsed.get("清晰度", 0),
            }
            confidence_score = calculate_confidence_score(scores)
            passed = confidence_score >= CONFIDENCE_THRESHOLD
            
            eval_item = {
                "id": r["id"],
                "scenario": scenarios[r["id"]],
                "scores": scores,
                "confidence_score": confidence_score,
                "passed": passed,
                "feedback": parsed.get("feedback", ""),
            }
            eval_results.append(eval_item)
            
            # 记录需要人工审核的项目
            if not passed:
                manual_review_items.append({
                    "type": "damage_scenario",
                    "id": r["id"],
                    "confidence_score": confidence_score,
                    "feedback": parsed.get("feedback", ""),
                    "scenario": scenarios[r["id"]],
                })
                
        except Exception as e:
            logger.warning(f"  ⚠ 评估任务 {r['id']} 解析失败: {e}")

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个损害场景需要人工审核")
        for item in manual_review_items:
            logger.warning(f"    - ID {item['id']}: 置信度 {item['confidence_score']}, 反馈: {item['feedback'][:50]}...")

    save_checkpoint(eval_results, "step2_ds_confidence_eval")
    
    return {
        "ds_confidence": {
            "all_passed": all_passed,
            "items": eval_results,
        },
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤3: 威胁场景 (逐条并发生成) ──────────

def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    # ★ 逐条构建任务
    tasks = []
    for i, ds in enumerate(damage_scenarios):
        prompt = TS_SINGLE_USER_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            asset_type=ds.get("asset_type", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
            rag_context=rag_context,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ts_list.extend(parsed)
            else:
                ts_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {"threat_scenarios": ts_list}


def node_ts_confidence_eval(state: TaraState) -> dict:
    """威胁场景置信度评估节点"""
    logger.info("[步骤3-评估] 威胁场景置信度评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ts_confidence": {"all_passed": True, "items": []}}

    ts_list = state.get("threat_scenarios", [])
    if not ts_list:
        return {"ts_confidence": {"all_passed": True, "items": []}}

    # 构建评估任务
    tasks = []
    for i, ts in enumerate(ts_list):
        prompt = TS_CONFIDENCE_EVAL_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_type=ts.get("threat_type", ""),
            damage_scenario=ts.get("damage_scenario", ""),
            threat_scenario=ts.get("threat_scenario", ""),
        )
        tasks.append({"id": i, "system": CONFIDENCE_EVAL_SYSTEM, "user": prompt})

    logger.info(f"  → 评估 {len(tasks)} 个威胁场景")
    raw_results = concurrent_llm_calls(tasks)

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            scores = {
                "要素完整性": parsed.get("要素完整性", 0),
                "一致性": parsed.get("一致性", 0),
                "真实性": parsed.get("真实性", 0),
                "清晰度": parsed.get("清晰度", 0),
            }
            confidence_score = calculate_confidence_score(scores)
            passed = confidence_score >= CONFIDENCE_THRESHOLD
            
            eval_item = {
                "id": r["id"],
                "scenario": ts_list[r["id"]],
                "scores": scores,
                "confidence_score": confidence_score,
                "passed": passed,
                "feedback": parsed.get("feedback", ""),
            }
            eval_results.append(eval_item)
            
            if not passed:
                manual_review_items.append({
                    "type": "threat_scenario",
                    "id": r["id"],
                    "confidence_score": confidence_score,
                    "feedback": parsed.get("feedback", ""),
                    "scenario": ts_list[r["id"]],
                })
                
        except Exception as e:
            logger.warning(f"  ⚠ 评估任务 {r['id']} 解析失败: {e}")

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个威胁场景需要人工审核")

    save_checkpoint(eval_results, "step3_ts_confidence_eval")
    
    return {
        "ts_confidence": {
            "all_passed": all_passed,
            "items": eval_results,
        },
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤4: 攻击路径 (逐条并发生成) ──────────

def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发)")

    threat_scenarios = state.get("threat_scenarios", [])
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    #  每条威胁场景独立生成攻击路径
    tasks = []
    for i, ts in enumerate(threat_scenarios):
        prompt = AP_SINGLE_USER_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            asset_type=ts.get("asset_type", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_scenario=ts.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            rag_context=rag_context,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ap_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ap_list.extend(parsed)
            else:
                ap_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成攻击路径 {len(ap_list)}/{len(tasks)} 组")
    save_checkpoint(ap_list, "step4_attack_paths")
    return {"attack_paths": ap_list}


def node_ap_confidence_eval(state: TaraState) -> dict:
    """攻击路径置信度评估节点"""
    logger.info("[步骤4-评估] 攻击路径置信度评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ap_confidence": {"all_passed": True, "items": []}}

    ap_list = state.get("attack_paths", [])
    if not ap_list:
        return {"ap_confidence": {"all_passed": True, "items": []}}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)

    # 构建评估任务
    tasks = []
    for i, ap in enumerate(ap_list):
        prompt = AP_CONFIDENCE_EVAL_TEMPLATE.format(
            function_name=ap.get("function", ""),
            asset_name=ap.get("asset_name", ""),
            security_attribute=ap.get("security_attribute", ""),
            damage_scenario=ap.get("damage_scenario", "") or state.get("threat_scenarios", [{}])[i].get("damage_scenario", ""),
            threat_scenario=ap.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            attack_paths=json.dumps(ap.get("attack_paths", []), ensure_ascii=False),
        )
        tasks.append({"id": i, "system": CONFIDENCE_EVAL_SYSTEM, "user": prompt})

    logger.info(f"  → 评估 {len(tasks)} 个攻击路径")
    raw_results = concurrent_llm_calls(tasks)

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            scores = {
                "要素完整性": parsed.get("要素完整性", 0),
                "一致性": parsed.get("一致性", 0),
                "真实性": parsed.get("真实性", 0),
                "清晰度": parsed.get("清晰度", 0),
            }
            confidence_score = calculate_confidence_score(scores)
            passed = confidence_score >= CONFIDENCE_THRESHOLD
            
            eval_item = {
                "id": r["id"],
                "scenario": ap_list[r["id"]],
                "scores": scores,
                "confidence_score": confidence_score,
                "passed": passed,
                "feedback": parsed.get("feedback", ""),
            }
            eval_results.append(eval_item)
            
            if not passed:
                manual_review_items.append({
                    "type": "attack_path",
                    "id": r["id"],
                    "confidence_score": confidence_score,
                    "feedback": parsed.get("feedback", ""),
                    "scenario": ap_list[r["id"]],
                })
                
        except Exception as e:
            logger.warning(f"  ⚠ 评估任务 {r['id']} 解析失败: {e}")

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个攻击路径需要人工审核")

    save_checkpoint(eval_results, "step4_ap_confidence_eval")
    
    return {
        "ap_confidence": {
            "all_passed": all_passed,
            "items": eval_results,
        },
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤5: 影响评级 (分批并发) - 修改为Python计算 ──────────

def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响参数评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    # ★ 分批
    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = IR_BATCH_USER_TEMPLATE.format(
            ds_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # ★ 使用Python计算 impact_value 和 impact_level
    for r in ratings:
        params = r.get("influence_parameters", {})
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        r["impact_value"] = str(impact_value)
        r["impact_level"] = impact_level

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


# ──────────  步骤6: 攻击可行性评级 (分批并发) - 修改为Python计算 ──────────

def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (分批并发)")

    # 展平攻击路径
    flat_paths = []
    for ap_group in state.get("attack_paths", []):
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "function": ap_group.get("function", ""),
                "asset_name": ap_group.get("asset_name", ""),
                "security_attribute": ap_group.get("security_attribute", ""),
                "attack_path": path,
            })

    if not flat_paths:
        return {"feasibility_ratings": []}

    # ★ 分批
    batches = [flat_paths[i:i+BATCH_SIZE_RATING] for i in range(0, len(flat_paths), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(flat_paths)} 条攻击路径, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = AF_BATCH_USER_TEMPLATE.format(
            ap_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": AF_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # ★ 使用Python计算 feasibility_score 和 feasibility_level
    for r in ratings:
        params = r.get("attack_parameters", {})
        feasibility_score = calculate_feasibility_score(params)
        feasibility_level = calculate_feasibility_level(feasibility_score)
        r["feasibility_score"] = feasibility_score
        r["feasibility_level"] = feasibility_level

    logger.info(f"  → 完成可行性评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": ratings}


# ────────── 步骤7: 风险判定 (规则引擎, 不变) ──────────

def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险判定 (交叉矩阵)")

    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    RISK_MATRIX = {
        ("Severe", "High"): 5, ("Severe", "Medium"): 4, ("Severe", "Low"): 3,
        ("Severe", "Very Low"): 2, 
        ("Major", "High"): 4, ("Major", "Medium"): 3, ("Major", "Low"): 2,
        ("Major", "Very Low"): 1, 
        ("Moderate", "High"): 3, ("Moderate", "Medium"): 2, ("Moderate", "Low"): 2,
        ("Moderate", "Very Low"): 1, 
        ("Negligible", "High"): 1, ("Negligible", "Medium"): 1, ("Negligible", "Low"): 1,
        ("Negligible", "Very Low"): 1,
    }
    RISK_LABEL = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
    TREATMENT = {
        "1": "风险接受",
        "2": "风险接受",
        "3": "风险接受",
        "4": "风险缓解",
        "5": "风险缓解",
    }

    FEAS_ORDER = {"High": 4, "Medium": 3, "Low": 2, "Very Low": 1}
    feas_map: dict[tuple, str] = {}
    for fr in feasibility_ratings:
        key = (fr.get("function", ""), fr.get("asset_name", ""), fr.get("security_attribute", ""))
        level = fr.get("feasibility_level", "Very Low")
        if FEAS_ORDER.get(level, 1) > FEAS_ORDER.get(feas_map.get(key, "Very Low"), 1):
            feas_map[key] = level

    risk_results = []
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        impact_level = ir.get("impact_level", "Negligible")
        feasibility_level = feas_map.get(key, "Very Low")
        risk_val = RISK_MATRIX.get((impact_level, feasibility_level), 1)
        risk_label = RISK_LABEL.get(risk_val, "1")
        treatment = TREATMENT.get(risk_label, "风险接受")

        risk_results.append({
            "function": ir.get("function", ""),
            "asset_name": ir.get("asset_name", ""),
            "security_attribute": ir.get("security_attribute", ""),
            "impact_level": impact_level,
            "feasibility_level": feasibility_level,
            "risk_value": risk_label,
            "risk_treatment": treatment,
        })

    logger.info(f"  → 风险判定完成 {len(risk_results)} 条")
    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


# ────────── 步骤8: 报告汇总 ──────────

def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    rr_map = {_key(r): r for r in risk_results}

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for attr in asset.get("security_attributes", []):
                key = (func_name, asset["asset_name"], attr)
                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "threat_type": ds.get("threat_type", ""),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "attack": attack_list,
                    "risk_value": rr.get("risk_value", ""),
                    "risk_treatment": rr.get("risk_treatment", ""),
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")
    return {"tara_report": report}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：工作流构建 (修改：移除correct节点)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_confidence_eval",  node_ds_confidence_eval)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_confidence_eval",  node_ts_confidence_eval)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_confidence_eval",  node_ap_confidence_eval)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")

    # 移除评估后的correct节点，直接连接到下一步
    builder.add_edge("node_ds_generate", "node_ds_confidence_eval")
    builder.add_edge("node_ds_confidence_eval", "node_ts_generate")

    builder.add_edge("node_ts_generate", "node_ts_confidence_eval")
    builder.add_edge("node_ts_confidence_eval", "node_ap_generate")

    builder.add_edge("node_ap_generate", "node_ap_confidence_eval")
    builder.add_edge("node_ap_confidence_eval", "node_impact_rating")

    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_json: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: dict | None = None,
    asset_results: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")
    logger.info(f"  置信度阈值: CONFIDENCE_THRESHOLD={CONFIDENCE_THRESHOLD}")

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": data_flow_json or [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or {},
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_confidence": {},
        "threat_scenarios": [], "ts_confidence": {},
        "attack_paths": [],    "ap_confidence": {},
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
        "manual_review_items": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    # ★ 保存最终报告
    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    # ★ 保存完整中间状态（便于调试）
    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
            "manual_review_items": final_state.get("manual_review_items", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    # ★ 输出需要人工审核的项目
    manual_items = final_state.get("manual_review_items", [])
    if manual_items:
        logger.warning("=" * 60)
        logger.warning(f"⚠️  共 {len(manual_items)} 个场景需要人工审核:")
        for item in manual_items:
            logger.warning(f"  - [{item['type']}] ID: {item['id']}, 置信度: {item['confidence_score']}")
            logger.warning(f"    反馈: {item['feedback'][:100]}...")
        logger.warning("=" * 60)

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":

    sample_assets = [
        {
            "function": "交通标识及信号灯识别",
            "assets": [
                {"asset_type": "信号", "asset_name": "弹窗/音效"},
            ],
        },
        {
            "function": "智能泊车辅助",
            "assets": [
                {"asset_type": "数据", "asset_name": "用户账户信息"},
            ],
        },
    ]

    sample_topology = [
  {
    "protocol": "CANFD",
    "color": "#F53F3F",
    "controller": [
      "AMP",
      "CDC",
      "IC"
    ]
  },
  {
    "protocol": "A2B",
    "color": "#016AFF",
    "controller": [
      "AMP",
      "CDC"
    ]
  },
  {
    "protocol": "FPDLINK",
    "color": "#00B42A",
    "controller": [
      "CDC",
      "IC"
    ]
  },
  {
    "protocol": "FPDLINK",
    "color": "#FF7D00",
    "controller": [
      "CDC",
      "IVI"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#000000",
    "controller": [
      "CDC",
      "DMS"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#00B0F0",
    "controller": [
      "CDC",
      "OMS"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#7F7F7F",
    "controller": [
      "CDC",
      "MDC"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#8F8F8F",
    "controller": [
      "CDC",
      "CEM"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#5470C6",
    "controller": [
      "BDCR",
      "CDC",
      "CEM",
      "REA"
    ]
  },
  {
    "protocol": "CAN",
    "color": "#91CC75",
    "controller": [
      "BDCR",
      "BLE",
      "CEM",
      "CMSL",
      "CMSR",
      "DSM",
      "ECALL",
      "PDSM",
      "TRM",
      "WLCM"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#FAC858",
    "controller": [
      "MDC",
      "USS ECU"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#EE6666",
    "controller": [
      "FR",
      "MDC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#73C0DE",
    "controller": [
      "LRC-FC",
      "MDC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#FCDBC9",
    "controller": [
      "MDC",
      "SRC-FC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#9A60B4",
    "controller": [
      "MDC",
      "MRC-SC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#BC85FE",
    "controller": [
      "MDC",
      "MRC-RC"
    ]
  },
  {
    "protocol": "GMSL",
    "color": "#3088DA",
    "controller": [
      "FEC",
      "MDC"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#5E7C61",
    "controller": [
      "CEM",
      "MDC"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#831A5C",
    "controller": [
      "CEM",
      "EPS",
      "IBCU",
      "MDC",
      "SRS",
      "VCU",
      "VMC"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#8D62E8",
    "controller": [
      "CEM",
      "VCU"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#0D21E3",
    "controller": [
      "BCU",
      "EVCC",
      "FMIPU",
      "HPMU",
      "ITMS",
      "PDU",
      "RDPEU",
      "RMIPU",
      "VCU"
    ]
  },
  {
    "protocol": "CAN",
    "color": "#01F2E3",
    "controller": [
      "BCU",
      "EVCC"
    ]
  },
  {
    "protocol": "CAN",
    "color": "#29FC11",
    "controller": [
      "ACP",
      "ITMS",
      "PTC"
    ]
  },
  {
    "protocol": "CANFD",
    "color": "#973D5D",
    "controller": [
      "BDCR",
      "CDC",
      "CEM",
      "LBMS"
    ]
  },
  {
    "protocol": "ETH",
    "color": "#CDED23",
    "controller": [
      "Lidar",
      "MDC"
    ]
  },
  {
    "protocol": "蜂窝",
    "color": "#AF21CE",
    "controller": [
      "BCALL中心",
      "CDC",
      "CEM",
      "ECALL中心",
      "华为车云",
      "诊断仪",
      "阿维塔车云"
    ]
  }
]

    sample_ext_interfaces = [
        {"component": "CDC", "interfaces": ["Cellular Netword  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
        {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
        {"component": "UWB", "interfaces": ["RF  射频"]},
        {"component": "CHLIL", "interfaces": ["JTAG"]},
        {"component": "CHLIR", "interfaces": ["JTAG"]},
        {"component": "ITMS", "interfaces": ["JTAG"]},
    ]

    report = run_tara(
        asset_results=sample_assets,
        topology_json=sample_topology,
        external_interface_json=sample_ext_interfaces,
        output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
        build_rag_index=True,
    )


2026-04-10 13:12:08,642 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-10 13:12:08,645 [INFO] ============================================================
2026-04-10 13:12:08,646 [INFO]   并发配置: MAX_WORKERS=3, BATCH_SIZE_RATING=5
2026-04-10 13:12:08,647 [INFO]   评估开关: ENABLE_EVALUATION=True
2026-04-10 13:12:08,647 [INFO]   置信度阈值: CONFIDENCE_THRESHOLD=85
2026-04-10 13:12:08,647 [INFO] [初始化] RAG 已禁用，跳过
2026-04-10 13:12:08,685 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-10 13:12:08,693 [INFO] ============================================================
2026-04-10 13:12:08,694 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-10 13:12:08,695 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-10 13:12:08,697 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V3\checkpoint_step1_assets.json
2026-04-10 13:12:08,698 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-10 13:12:08,699 [INFO]   → 共 7 个子任务, 并发数 3
2026-04-10 13:12:10,693 [INFO] HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 13:12:12,1

# V4
Prompt 缓存增强（减少重复计算）+批量评估

In [68]:
import json
import os
import re
import glob
import logging
import time
from typing import TypedDict, Literal, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache
import hashlib

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import requests
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 1.1 LLM 配置 ──
llm = ChatOpenAI(
    model="Qwen/Qwen3.5-35B-A3B",
    openai_api_key="sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={
        "enable_thinking": False
    },
)

# ── 1.2 Embedding 配置 ──
ENABLE_RAG = False
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

# ── 1.3 RAG 知识库路径 ──
RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

# ── 1.4 工作流参数 ──
ENABLE_EVALUATION = True  # 置信度评估

# ── 1.5 并发与分批配置 ──
MAX_WORKERS = 8          # 最大并发 LLM 调用数
CALL_INTERVAL = 0.5      # 提交任务间隔(秒), 用于控制 API 速率
BATCH_SIZE_RATING = 5    # 评级步骤的批大小

# ── 1.6 输出路径 ──
OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1.7 资产类型 → 安全属性映射 ──
SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

# ── 1.8 安全属性 → STRIDE 映射 ──
ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}

# ── 1.9 置信度评估配置 ──
CONFIDENCE_THRESHOLD = 85  # 初始阈值设为85

# 置信度评估权重
CONFIDENCE_WEIGHTS = {
    "要素完整性": 0.30,  # 30%
    "一致性": 0.30,      # 30%
    "真实性": 0.30,      # 30%
    "清晰度": 0.10,      # 10%
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,          # 单次最多请求 8 条文本，防 413 限流
                request_timeout=45,    # 防网络卡死
                max_retries=2          # 自动重试
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,         # 重叠减少，防 Token 膨胀
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:  # 硬拦截 >128 字符的块
                valid_chunks.append(c)
            else:
                # 兜底：超长块强制截断（保命策略）
                if len(content) > 128:
                    c.page_content = content[:127] + "..."
                    valid_chunks.append(c)
        
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True)
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs)
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                # 字典按顶层 key 拆分
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: dict
    external_interface_json: dict
    asset_results: list
    damage_scenarios: list
    # 移除 ds_eval_passed, ds_retry_count, ds_feedback
    ds_confidence: dict  # 存储损害场景的置信度评估结果
    threat_scenarios: list
    # 移除 ts_eval_passed, ts_retry_count, ts_feedback
    ts_confidence: dict  # 存储威胁场景的置信度评估结果
    attack_paths: list
    # 移除 ap_eval_passed, ap_retry_count, ap_feedback
    ap_confidence: dict  # 存储攻击路径的置信度评估结果
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list
    # 新增：需要人工审核的场景列表
    manual_review_items: list


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
        
    cleaned = text.strip()
    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


# ── 缓存机制 ──
hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()

def hash_prompt(text: str) -> str:
    """生成 prompt 的 MD5 哈希（用于缓存键）"""
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def register_prompt_for_cache(prompt: str) -> str:
    """注册 prompt 到全局映射，返回其 hash"""
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h

def clear_prompt_cache():
    """清空缓存映射（测试/重置时用）"""
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()

@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    """基于 prompt 哈希的缓存调用，使用前必须先用 register_prompt_for_cache 注册 prompt"""
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册 (sys={system_hash[:8]}..., user={user_hash[:8]}...)")
        return "[]"
    
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


# 带重试的 LLM 调用
def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
                
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: langchain 返回类型不匹配 - {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    
    return "[]"


# 并发 LLM 调用引擎（增强日志统计）
def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    """
    并发执行多个 LLM 调用
    tasks: [{"id": str/int, "system": str, "user": str}, ...]
    返回: [{"id": ..., "raw": str}, ...]  按 id 排序
    """
    results = []
    total = len(tasks)
    if total == 0:
        return results
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)

        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")

    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s, 平均 {elapsed/len(tasks):.2f}s/任务")
    return results


def save_checkpoint(data: Any, step_name: str):
    """按步骤名保存独立 checkpoint"""
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Python计算函数（新增）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def calculate_impact_value(influence_parameters: dict) -> int:
    """
    计算影响值（四维最高分）
    """
    if not influence_parameters:
        return 1
    max_val = max(int(influence_parameters.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
    return min(max(max_val, 1), 4)  # 确保在1-4范围内


def calculate_impact_level(impact_value: int) -> str:
    """
    根据影响值映射影响等级
    """
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    return IMPACT_LABELS.get(impact_value, "Negligible")


def calculate_feasibility_score(attack_parameters: dict) -> int:
    """
    计算可行性分数（五维求和）
    """
    if not attack_parameters:
        return 0
    return sum(int(attack_parameters.get(k, 0)) for k in [
        "Exposure_time", "Professional_experience",
        "Required_information", "Opportunity_window", "Required_equipment",
    ])


def calculate_feasibility_level(feasibility_score: int) -> str:
    """
    根据可行性分数映射可行性等级
    """
    if feasibility_score <= 13:
        return "High"
    elif feasibility_score <= 19:
        return "Medium"
    elif feasibility_score <= 24:
        return "Low"
    else:
        return "Very Low"


def calculate_confidence_score(scores: dict) -> float:
    """
    计算加权置信度分数（0-100）
    scores: {"要素完整性": 0-100, "一致性": 0-100, "真实性": 0-100, "清晰度": 0-100}
    """
    total = 0.0
    for dimension, weight in CONFIDENCE_WEIGHTS.items():
        score = scores.get(dimension, 0)
        total += score * weight
    return round(total, 2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径。
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


# ──────  步骤2: 单条损害场景 Prompt (逐条调用) ──────
DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是"涉及车辆或车辆功能且影响道路使用者的不良后果"。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}
- 对应威胁类型：{threat_type}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体危害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"

## 输出（严格 JSON 对象，不要数组）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "完整的损害场景描述",
  "threat_type": "{threat_type}"
}}"""


# ──────  步骤3: 单条威胁场景 Prompt (逐条调用) ──────
TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是"为实现损害场景，资产的信息安全属性遭到破坏的潜在原因"。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体{asset_name}，如"GW部件"、"前照灯请求信号"、"车辆数据"
   - 「{asset_name}」被破坏的「{security_attribute}」（必须与输入的安全属性完全一致）
   - 「{security_attribute}」被破坏的原因/攻击意图
2. - 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条。
   - 进一步的信息能被威胁场景包含或与威胁场景关联。例如:危害场景与资产、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "{damage_scenario}",
  "threat_type": "{threat_type}",
  "threat_scenario": "详细的威胁场景描述"
}}"""



# ──────  步骤4: 单条攻击路径 Prompt (逐条调用, 附带拓扑) ──────
AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」生成最短的（经过拓扑节点最少）但符合现实场景和逻辑的攻击路径,如有多条最短路径请生成多条，最多不超过4条，攻击路径是"为实现威胁场景的一组蓄意活动"。必须采用自上而下的方法（如攻击树分析），从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}（{asset_type}）
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产：
   - 以编号步骤形式呈现：1.…；2.…；3.…；…
   - 第一步：明确的外部攻击入口
   - 中间步骤：沿车辆拓扑经过的中间节点和通信协议
   - 最终步骤：到达「{asset_name}」并实施具体攻击技术，从而破坏「{security_attribute}」
2. 路径中的节点和连接关系必须与车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」一致，不得虚构不存在的节点或跳过拓扑中的网关/ECU
3. 同一个威胁场景的多条攻击路径不要重复
4. 同一名称（如"CAN"）不同id的通信协议，代表不同的总线，输出时只输出名称就行，不要输出id
5. 攻击路径必须符合现实中的实际场景

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号(非预期的快速减速)。"

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "threat_scenario": "{threat_scenario}",
  "attack_paths": [
    "1.完整步骤链",
    "2.完整步骤链"
  ]
}}"""


# ────── 步骤5: 影响评级 Prompt (分批调用) - 修改为只输出参数 ──────
IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数）
### Safety: 1=无伤害, 2=轻度/中度伤害, 3=严重伤害和有生命危险, 4=威胁生命的伤害（不确定是否幸存）或致命的伤害
### Finance: 1=可忽略, 2=个人可承受, 3=大量损失但受影响的道路使用者将能够克服这些后果, 4=灾难性后果受影响的道路使用者可能无法克服
#### 利益相关者包括：车主、驾驶员、乘员、行人、供应商、主机厂。
### Operation: 1=不影响, 2=导致车辆功能的部分退化, 3=导致车辆重要功能丧失或受损, 4=导致车辆核心功能丧失或受损
### Privacy: 1=隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体,
2=隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体； b)泄露的信息不敏感但很容易识别到PII主体,
3=隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体； b)泄露的信息敏感而且很容易识别到PII主体,
4=隐私侵犯会对道路使用者造成重大甚至不可逆转的影响 泄露的信息高度敏感，并且很容易识别到PII主体 
#### 需要考虑车主、驾驶员、乘员、行人、供应商、主机厂。

## 强约束条件
- 每个维度必须严格对照评级标准打分，不得使用其他标准
- 分值必须是整数 1~4

## 输出（严格 JSON 数组，只输出参数）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "influence_parameters": {{"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}}
  }}
]"""


# ────── 步骤6: 可行性评级 Prompt (分批调用) - 修改为只输出参数 ──────
AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434（尤其是 Annex G 基于攻击潜力的攻击可行性评估方法） 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准
### Exposure_time: 
- 0分：≤1天（实现攻击行为的时间小于等于1天）  
- 1分：≤1周（实现攻击行为的时间小于等于1周）  
- 4分：≤1个月（实现攻击行为的时间小于等于1个月）  
- 17分：≤6个月（实现攻击行为的时间小于等于6个月）  
- 19分：>6个月（实现攻击行为的时间大于6个月）
### Professional_experience:
- 0分：非专业（Layman）——外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3分：精通/擅长（Proficient）——熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6分：专业（Expert）——熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 8分：复合型（Multiple experts）——一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0分：公开的（Public）——公共信息（例如互联网上获得的信息）  
- 3分：限制的（Restricted）——受限制的信息（制造商和供应商共享的内部文档）  
- 7分：敏感的（Sensitive）——机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11分：至关重要的（Critical）——严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0分：十分高（Critical）——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1分：高（High）——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4分：中（Medium）——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10分：低（Low）——困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0分：标准的（Standard）——标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4分：专用的（Specialized）——专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7分：定制的（Bespoke）——定制设备（厂家限制的工具、电子显微镜）  
- 9分：多重定制的（Multiple bespoke）——攻击不同步骤需要不同类型的定制设备

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数（0、1、3、4、6、7、8、9、10、11、17、19）

## 输出（严格 JSON 数组，只输出参数）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "attack_path": "攻击路径",
    "attack_parameters": {{
      "Exposure_time": 0, "Professional_experience": 0,
      "Required_information": 0, "Opportunity_window": 0, "Required_equipment": 0
    }}
  }}
]"""


# ────── 置信度评估 Prompt (新增) ──────
CONFIDENCE_EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估场景质量并给出各维度的置信度评分。
仅输出 JSON，不要任何其他文字。"""

# 损害场景置信度评估模板
DS_CONFIDENCE_EVAL_TEMPLATE = """## 评审任务
请评估以下损害场景的质量，并给出各维度的置信度评分（0-100分）。

## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{security_attribute}
## 威胁类型
{threat_type}
## 待评审的损害场景
{damage_scenario}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
损害场景必须包含以下要素：
- 功能与不良后果的因果关系
- 对道路使用者的具体危害
- 涉及的目标资产
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 与功能/资产/STRIDE是否匹配
- 损害场景描述是否与输入信息一致
评分标准：完全匹配100分，部分不匹配扣20-50分，严重不一致扣60-80分

### 3. 真实性（满分100分）
- 危害对道路使用者影响是否合理
- 是否有明显幻觉或不合理描述
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 描述是否清晰易懂
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式
{{
  "要素完整性": 0-100,
  "一致性": 0-100,
  "真实性": 0-100,
  "清晰度": 0-100,
  "confidence_score": 加权总分,
  "passed": true或false（总分>=85为true）,
  "feedback": "如不通过，列出具体问题"
}}"""

# 威胁场景置信度评估模板
TS_CONFIDENCE_EVAL_TEMPLATE = """## 评审任务
请评估以下威胁场景的质量，并给出各维度的置信度评分（0-100分）。

## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{security_attribute}
## 威胁类型
{threat_type}
## 损害场景
{damage_scenario}
## 待评审的威胁场景
{threat_scenario}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
威胁场景必须包含以下要素：
- 目标资产
- 被破坏的信息安全属性
- 信息安全属性被破坏的原因
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 是否直接导致损害场景
- 与损害场景的因果关系是否清晰
评分标准：完全一致100分，部分不一致扣20-50分，因果关系断裂扣60-80分

### 3. 真实性（满分100分）
- 攻击意图是否现实
- 是否有明显幻觉或不合理描述
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 描述是否清晰易懂
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式
{{
  "要素完整性": 0-100,
  "一致性": 0-100,
  "真实性": 0-100,
  "清晰度": 0-100,
  "confidence_score": 加权总分,
  "passed": true或false（总分>=85为true）,
  "feedback": "如不通过，列出具体问题"
}}"""

# 攻击路径置信度评估模板
AP_CONFIDENCE_EVAL_TEMPLATE = """## 评审任务
请评估以下攻击路径的质量，并给出各维度的置信度评分（0-100分）。

## 功能
{function_name}
## 资产名称
{asset_name}
## 安全属性
{security_attribute}
## 损害场景
{damage_scenario}
## 威胁场景
{threat_scenario}
## 车辆拓扑图
{topo_info}
## 外部接口
{ext_info}
## 待评审的攻击路径
{attack_paths}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
攻击路径必须包含以下要素：
- 起点（具体攻击面）
- 逻辑步骤
- 终点（目标资产）
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 是否能真正实现威胁场景
- 是否符合拓扑图
- 攻击入口是否存在于外部接口中
评分标准：完全一致100分，部分不一致扣20-50分，严重偏离扣60-80分

### 3. 真实性（满分100分）
- 攻击面是否存在
- 步骤是否可行
- 是否有明显幻觉或不合理描述
评分标准：完全可行100分，部分不可行扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 步骤描述是否清晰连贯
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式
{{
  "要素完整性": 0-100,
  "一致性": 0-100,
  "真实性": 0-100,
  "清晰度": 0-100,
  "confidence_score": 加权总分,
  "passed": true或false（总分>=85为true）,
  "feedback": "如不通过，列出具体问题"
}}"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ────────── 步骤1: 资产识别 (规则引擎, 不变) ──────────

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": attrs,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "manual_review_items": [],  # 初始化人工审核列表
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


# ──────────  步骤2: 损害场景 (逐条并发生成) ──────────

def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]

    # RAG 检索一次，所有子任务共享
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    #  构建逐条任务
    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for attr in asset["security_attributes"]:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                prompt = DS_SINGLE_USER_TEMPLATE.format(
                    function_name=func["function"],
                    asset_name=asset["asset_name"],
                    asset_type=asset["asset_type"],
                    security_attribute=attr,
                    threat_type=threat_type,
                    rag_context=rag_context,
                )
                tasks.append({"id": task_id, "system": BASE_SYSTEM, "user": prompt})
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    # ★ 并发调用
    raw_results = concurrent_llm_calls(tasks)

    # ★ 聚合结果
    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                scenarios.extend(parsed)
            else:
                scenarios.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {"damage_scenarios": scenarios}


def node_ds_confidence_eval(state: TaraState) -> dict:
    """损害场景置信度评估节点"""
    logger.info("[步骤2-评估] 损害场景置信度评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ds_confidence": {"all_passed": True, "items": []}}

    scenarios = state.get("damage_scenarios", [])
    if not scenarios:
        return {"ds_confidence": {"all_passed": True, "items": []}}

    # 构建评估任务
    tasks = []
    for i, ds in enumerate(scenarios):
        prompt = DS_CONFIDENCE_EVAL_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
        )
        tasks.append({"id": i, "system": CONFIDENCE_EVAL_SYSTEM, "user": prompt})

    logger.info(f"  → 评估 {len(tasks)} 个损害场景")
    raw_results = concurrent_llm_calls(tasks)

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            # 计算加权置信度分数
            scores = {
                "要素完整性": parsed.get("要素完整性", 0),
                "一致性": parsed.get("一致性", 0),
                "真实性": parsed.get("真实性", 0),
                "清晰度": parsed.get("清晰度", 0),
            }
            confidence_score = calculate_confidence_score(scores)
            passed = confidence_score >= CONFIDENCE_THRESHOLD
            
            eval_item = {
                "id": r["id"],
                "scenario": scenarios[r["id"]],
                "scores": scores,
                "confidence_score": confidence_score,
                "passed": passed,
                "feedback": parsed.get("feedback", ""),
            }
            eval_results.append(eval_item)
            
            # 记录需要人工审核的项目
            if not passed:
                manual_review_items.append({
                    "type": "damage_scenario",
                    "id": r["id"],
                    "confidence_score": confidence_score,
                    "feedback": parsed.get("feedback", ""),
                    "scenario": scenarios[r["id"]],
                })
                
        except Exception as e:
            logger.warning(f"  ⚠ 评估任务 {r['id']} 解析失败: {e}")

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个损害场景需要人工审核")
        for item in manual_review_items:
            logger.warning(f"    - ID {item['id']}: 置信度 {item['confidence_score']}, 反馈: {item['feedback'][:50]}...")

    save_checkpoint(eval_results, "step2_ds_confidence_eval")
    
    return {
        "ds_confidence": {
            "all_passed": all_passed,
            "items": eval_results,
        },
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤3: 威胁场景 (逐条并发生成) ──────────

def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    # ★ 逐条构建任务
    tasks = []
    for i, ds in enumerate(damage_scenarios):
        prompt = TS_SINGLE_USER_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            asset_type=ds.get("asset_type", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
            rag_context=rag_context,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ts_list.extend(parsed)
            else:
                ts_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {"threat_scenarios": ts_list}


def node_ts_confidence_eval(state: TaraState) -> dict:
    """威胁场景置信度评估节点"""
    logger.info("[步骤3-评估] 威胁场景置信度评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ts_confidence": {"all_passed": True, "items": []}}

    ts_list = state.get("threat_scenarios", [])
    if not ts_list:
        return {"ts_confidence": {"all_passed": True, "items": []}}

    # 构建评估任务
    tasks = []
    for i, ts in enumerate(ts_list):
        prompt = TS_CONFIDENCE_EVAL_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_type=ts.get("threat_type", ""),
            damage_scenario=ts.get("damage_scenario", ""),
            threat_scenario=ts.get("threat_scenario", ""),
        )
        tasks.append({"id": i, "system": CONFIDENCE_EVAL_SYSTEM, "user": prompt})

    logger.info(f"  → 评估 {len(tasks)} 个威胁场景")
    raw_results = concurrent_llm_calls(tasks)

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            scores = {
                "要素完整性": parsed.get("要素完整性", 0),
                "一致性": parsed.get("一致性", 0),
                "真实性": parsed.get("真实性", 0),
                "清晰度": parsed.get("清晰度", 0),
            }
            confidence_score = calculate_confidence_score(scores)
            passed = confidence_score >= CONFIDENCE_THRESHOLD
            
            eval_item = {
                "id": r["id"],
                "scenario": ts_list[r["id"]],
                "scores": scores,
                "confidence_score": confidence_score,
                "passed": passed,
                "feedback": parsed.get("feedback", ""),
            }
            eval_results.append(eval_item)
            
            if not passed:
                manual_review_items.append({
                    "type": "threat_scenario",
                    "id": r["id"],
                    "confidence_score": confidence_score,
                    "feedback": parsed.get("feedback", ""),
                    "scenario": ts_list[r["id"]],
                })
                
        except Exception as e:
            logger.warning(f"  ⚠ 评估任务 {r['id']} 解析失败: {e}")

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个威胁场景需要人工审核")

    save_checkpoint(eval_results, "step3_ts_confidence_eval")
    
    return {
        "ts_confidence": {
            "all_passed": all_passed,
            "items": eval_results,
        },
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤4: 攻击路径 (逐条并发生成) ──────────

def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发)")

    threat_scenarios = state.get("threat_scenarios", [])
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    #  每条威胁场景独立生成攻击路径
    tasks = []
    for i, ts in enumerate(threat_scenarios):
        prompt = AP_SINGLE_USER_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            asset_type=ts.get("asset_type", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_scenario=ts.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            rag_context=rag_context,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")

    raw_results = concurrent_llm_calls(tasks)

    ap_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ap_list.extend(parsed)
            else:
                ap_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成攻击路径 {len(ap_list)}/{len(tasks)} 组")
    save_checkpoint(ap_list, "step4_attack_paths")
    return {"attack_paths": ap_list}


def node_ap_confidence_eval(state: TaraState) -> dict:
    """攻击路径置信度评估节点"""
    logger.info("[步骤4-评估] 攻击路径置信度评估")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ap_confidence": {"all_passed": True, "items": []}}

    ap_list = state.get("attack_paths", [])
    if not ap_list:
        return {"ap_confidence": {"all_passed": True, "items": []}}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)

    # 构建评估任务
    tasks = []
    for i, ap in enumerate(ap_list):
        prompt = AP_CONFIDENCE_EVAL_TEMPLATE.format(
            function_name=ap.get("function", ""),
            asset_name=ap.get("asset_name", ""),
            security_attribute=ap.get("security_attribute", ""),
            damage_scenario=ap.get("damage_scenario", "") or state.get("threat_scenarios", [{}])[i].get("damage_scenario", ""),
            threat_scenario=ap.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            attack_paths=json.dumps(ap.get("attack_paths", []), ensure_ascii=False),
        )
        tasks.append({"id": i, "system": CONFIDENCE_EVAL_SYSTEM, "user": prompt})

    logger.info(f"  → 评估 {len(tasks)} 个攻击路径")
    raw_results = concurrent_llm_calls(tasks)

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            scores = {
                "要素完整性": parsed.get("要素完整性", 0),
                "一致性": parsed.get("一致性", 0),
                "真实性": parsed.get("真实性", 0),
                "清晰度": parsed.get("清晰度", 0),
            }
            confidence_score = calculate_confidence_score(scores)
            passed = confidence_score >= CONFIDENCE_THRESHOLD
            
            eval_item = {
                "id": r["id"],
                "scenario": ap_list[r["id"]],
                "scores": scores,
                "confidence_score": confidence_score,
                "passed": passed,
                "feedback": parsed.get("feedback", ""),
            }
            eval_results.append(eval_item)
            
            if not passed:
                manual_review_items.append({
                    "type": "attack_path",
                    "id": r["id"],
                    "confidence_score": confidence_score,
                    "feedback": parsed.get("feedback", ""),
                    "scenario": ap_list[r["id"]],
                })
                
        except Exception as e:
            logger.warning(f"  ⚠ 评估任务 {r['id']} 解析失败: {e}")

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个攻击路径需要人工审核")

    save_checkpoint(eval_results, "step4_ap_confidence_eval")
    
    return {
        "ap_confidence": {
            "all_passed": all_passed,
            "items": eval_results,
        },
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤5: 影响评级 (分批并发) - 修改为Python计算 ──────────

def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响参数评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    # ★ 分批
    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = IR_BATCH_USER_TEMPLATE.format(
            ds_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # ★ 使用Python计算 impact_value 和 impact_level
    for r in ratings:
        params = r.get("influence_parameters", {})
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        r["impact_value"] = str(impact_value)
        r["impact_level"] = impact_level

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


# ──────────  步骤6: 攻击可行性评级 (分批并发) - 修改为Python计算 ──────────

def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (分批并发)")

    # 展平攻击路径
    flat_paths = []
    for ap_group in state.get("attack_paths", []):
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "function": ap_group.get("function", ""),
                "asset_name": ap_group.get("asset_name", ""),
                "security_attribute": ap_group.get("security_attribute", ""),
                "attack_path": path,
            })

    if not flat_paths:
        return {"feasibility_ratings": []}

    # ★ 分批
    batches = [flat_paths[i:i+BATCH_SIZE_RATING] for i in range(0, len(flat_paths), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(flat_paths)} 条攻击路径, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = AF_BATCH_USER_TEMPLATE.format(
            ap_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": AF_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    # ★ 使用Python计算 feasibility_score 和 feasibility_level
    for r in ratings:
        params = r.get("attack_parameters", {})
        feasibility_score = calculate_feasibility_score(params)
        feasibility_level = calculate_feasibility_level(feasibility_score)
        r["feasibility_score"] = feasibility_score
        r["feasibility_level"] = feasibility_level

    logger.info(f"  → 完成可行性评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": ratings}


# ────────── 步骤7: 风险判定 (规则引擎, 不变) ──────────

def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险判定 (交叉矩阵)")

    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    RISK_MATRIX = {
        ("Severe", "High"): 5, ("Severe", "Medium"): 4, ("Severe", "Low"): 3,
        ("Severe", "Very Low"): 2, 
        ("Major", "High"): 4, ("Major", "Medium"): 3, ("Major", "Low"): 2,
        ("Major", "Very Low"): 1, 
        ("Moderate", "High"): 3, ("Moderate", "Medium"): 2, ("Moderate", "Low"): 2,
        ("Moderate", "Very Low"): 1, 
        ("Negligible", "High"): 1, ("Negligible", "Medium"): 1, ("Negligible", "Low"): 1,
        ("Negligible", "Very Low"): 1,
    }
    RISK_LABEL = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
    TREATMENT = {
        "1": "风险接受",
        "2": "风险接受",
        "3": "风险接受",
        "4": "风险缓解",
        "5": "风险缓解",
    }

    FEAS_ORDER = {"High": 4, "Medium": 3, "Low": 2, "Very Low": 1}
    feas_map: dict[tuple, str] = {}
    for fr in feasibility_ratings:
        key = (fr.get("function", ""), fr.get("asset_name", ""), fr.get("security_attribute", ""))
        level = fr.get("feasibility_level", "Very Low")
        if FEAS_ORDER.get(level, 1) > FEAS_ORDER.get(feas_map.get(key, "Very Low"), 1):
            feas_map[key] = level

    risk_results = []
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        impact_level = ir.get("impact_level", "Negligible")
        feasibility_level = feas_map.get(key, "Very Low")
        risk_val = RISK_MATRIX.get((impact_level, feasibility_level), 1)
        risk_label = RISK_LABEL.get(risk_val, "1")
        treatment = TREATMENT.get(risk_label, "风险接受")

        risk_results.append({
            "function": ir.get("function", ""),
            "asset_name": ir.get("asset_name", ""),
            "security_attribute": ir.get("security_attribute", ""),
            "impact_level": impact_level,
            "feasibility_level": feasibility_level,
            "risk_value": risk_label,
            "risk_treatment": treatment,
        })

    logger.info(f"  → 风险判定完成 {len(risk_results)} 条")
    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


# ────────── 步骤8: 报告汇总 ──────────

def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    rr_map = {_key(r): r for r in risk_results}

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for attr in asset.get("security_attributes", []):
                key = (func_name, asset["asset_name"], attr)
                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "threat_type": ds.get("threat_type", ""),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "attack": attack_list,
                    "risk_value": rr.get("risk_value", ""),
                    "risk_treatment": rr.get("risk_treatment", ""),
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")
    return {"tara_report": report}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：工作流构建 (修改：移除correct节点)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_confidence_eval",  node_ds_confidence_eval)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_confidence_eval",  node_ts_confidence_eval)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_confidence_eval",  node_ap_confidence_eval)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")

    # 移除评估后的correct节点，直接连接到下一步
    builder.add_edge("node_ds_generate", "node_ds_confidence_eval")
    builder.add_edge("node_ds_confidence_eval", "node_ts_generate")

    builder.add_edge("node_ts_generate", "node_ts_confidence_eval")
    builder.add_edge("node_ts_confidence_eval", "node_ap_generate")

    builder.add_edge("node_ap_generate", "node_ap_confidence_eval")
    builder.add_edge("node_ap_confidence_eval", "node_impact_rating")

    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_json: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: dict | None = None,
    asset_results: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")
    logger.info(f"  置信度阈值: CONFIDENCE_THRESHOLD={CONFIDENCE_THRESHOLD}")

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": data_flow_json or [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or {},
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_confidence": {},
        "threat_scenarios": [], "ts_confidence": {},
        "attack_paths": [],    "ap_confidence": {},
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
        "manual_review_items": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    # ★ 保存最终报告
    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    # ★ 保存完整中间状态（便于调试）
    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
            "manual_review_items": final_state.get("manual_review_items", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    # ★ 输出需要人工审核的项目
    manual_items = final_state.get("manual_review_items", [])
    if manual_items:
        logger.warning("=" * 60)
        logger.warning(f"⚠️  共 {len(manual_items)} 个场景需要人工审核:")
        for item in manual_items:
            logger.warning(f"  - [{item['type']}] ID: {item['id']}, 置信度: {item['confidence_score']}")
            logger.warning(f"    反馈: {item['feedback'][:100]}...")
        logger.warning("=" * 60)

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":

    sample_assets = [
        {
            "function": "交通标识及信号灯识别",
            "assets": [
                {"asset_type": "信号", "asset_name": "弹窗/音效"},
            ],
        },
        {
            "function": "智能泊车辅助",
            "assets": [
                {"asset_type": "数据", "asset_name": "用户账户信息"},
            ],
        },
    ]

    sample_topology = [{"id": 1, "CANFD": ["AMP", "CDC", "IC"]}, {"id": 2, "A2B": ["AMP", "CDC"]}, {"id": 3, "FPDLINK": ["CDC", "IC"]}, {"id": 4, "FPDLINK": ["CDC", "IVI"]}, {"id": 5, "GMSL": ["CDC", "DMS"]}, {"id": 6, "GMSL": ["CDC", "OMS"]}, {"id": 7, "GMSL": ["CDC", "MDC"]}, {"id": 8, "ETH": ["CDC", "CEM"]}, {"id": 9, "CANFD": ["BDCR", "CDC", "CEM", "REA"]}, {"id": 10, "CAN": ["BDCR", "BLE", "CEM", "CMSL", "CMSR", "DSM", "ECALL", "PDSM", "TRM", "WLCM"]}, {"id": 11, "CANFD": ["MDC", "USS ECU"]}, {"id": 12, "CANFD": ["FR", "MDC"]}, {"id": 13, "GMSL": ["LRC-FC", "MDC"]}, {"id": 14, "GMSL": ["MDC", "SRC-FC"]}, {"id": 15, "GMSL": ["MDC", "MRC-SC"]}, {"id": 16, "GMSL": ["MDC", "MRC-RC"]}, {"id": 17, "GMSL": ["FEC", "MDC"]}, {"id": 18, "ETH": ["CEM", "MDC"]}, {"id": 19, "CANFD": ["CEM", "EPS", "IBCU", "MDC", "SRS", "VCU", "VMC"]}, {"id": 20, "ETH": ["CEM", "VCU"]}, {"id": 21, "CANFD": ["BCU", "EVCC", "FMIPU", "HPMU", "ITMS", "PDU", "RDPEU", "RMIPU", "VCU"]}, {"id": 22, "CAN": ["BCU", "EVCC"]}, {"id": 23, "CAN": ["ACP", "ITMS", "PTC"]}, {"id": 24, "CANFD": ["BDCR", "CDC", "CEM", "LBMS"]}, {"id": 25, "ETH": ["Lidar", "MDC"]}, {"id": 26, "蜂窝": ["BCALL中心", "CDC", "ECALL中心", "华为车云", "阿维塔车云"]}, {"id": 27, "CAN": ["CEM", "诊断仪"]}]

    sample_ext_interfaces = [
        {"component": "CDC", "interfaces": ["Cellular Netword  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
        {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
        {"component": "UWB", "interfaces": ["RF  射频"]},
        {"component": "CHLIL", "interfaces": ["JTAG"]},
        {"component": "CHLIR", "interfaces": ["JTAG"]},
        {"component": "ITMS", "interfaces": ["JTAG"]},
    ]

    report = run_tara(
        asset_results=sample_assets,
        topology_json=sample_topology,
        external_interface_json=sample_ext_interfaces,
        output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
        build_rag_index=True,
    )


2026-04-13 10:41:34,874 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-13 10:41:34,883 [INFO] ============================================================
2026-04-13 10:41:34,884 [INFO]   并发配置: MAX_WORKERS=8, BATCH_SIZE_RATING=5
2026-04-13 10:41:34,884 [INFO]   评估开关: ENABLE_EVALUATION=True
2026-04-13 10:41:34,884 [INFO]   置信度阈值: CONFIDENCE_THRESHOLD=85
2026-04-13 10:41:34,885 [INFO] [初始化] RAG 已禁用，跳过
2026-04-13 10:41:34,968 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-13 10:41:35,014 [INFO] ============================================================
2026-04-13 10:41:35,015 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-13 10:41:35,015 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-13 10:41:35,020 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V4\checkpoint_step1_assets.json
2026-04-13 10:41:35,022 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-13 10:41:35,023 [INFO]   → 共 7 个子任务, 并发数 8
2026-04-13 10:41:37,658 [INFO] HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 10:41:38,0

# V5

In [90]:
import json
import os
import re
import glob
import logging
import time
from typing import TypedDict, Literal, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache
import hashlib

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import requests
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── 1.1 LLM 配置 ──
llm = ChatOpenAI(
    model="Qwen/Qwen3.5-35B-A3B",
    openai_api_key="sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={
        "enable_thinking": False
    },
)

# ── 1.2 Embedding 配置 ──
ENABLE_RAG = False
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

# ── 1.3 RAG 知识库路径 ──
RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

# ── 1.4 工作流参数 ──
ENABLE_EVALUATION = True  # 置信度评估

# ── 1.5 并发与分批配置 ──
MAX_WORKERS = 5          # 最大并发 LLM 调用数
CALL_INTERVAL = 0.8      # 提交任务间隔(秒), 用于控制 API 速率
BATCH_SIZE_RATING = 5    # 评级步骤的批大小
EVAL_BATCH_SIZE = 3      # 【方案B】置信度评估的批大小

# ── 1.6 输出路径 ──
OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1.7 资产类型 → 安全属性映射 ──
SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

# ── 1.8 安全属性 → STRIDE 映射 ──
ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}

# ── 1.9 置信度评估配置 ──
CONFIDENCE_THRESHOLD = 85  # 初始阈值设为85

# 置信度评估权重
CONFIDENCE_WEIGHTS = {
    "要素完整性": 0.30,  # 30%
    "一致性": 0.30,      # 30%
    "真实性": 0.30,      # 30%
    "清晰度": 0.10,      # 10%
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库 
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,          # 单次最多请求 8 条文本，防 413 限流
                request_timeout=45,    # 防网络卡死
                max_retries=2          # 自动重试
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,         # 重叠减少，防 Token 膨胀
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:  # 硬拦截 >128 字符的块
                valid_chunks.append(c)
            else:
                # 兜底：超长块强制截断（保命策略）
                if len(content) > 128:
                    c.page_content = content[:127] + "..."
                    valid_chunks.append(c)
        
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True)
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs)
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                # 字典按顶层 key 拆分
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: dict
    external_interface_json: dict
    asset_results: list
    damage_scenarios: list
    ds_confidence: dict  # 存储损害场景的置信度评估结果
    threat_scenarios: list
    ts_confidence: dict  # 存储威胁场景的置信度评估结果
    attack_paths: list
    ap_confidence: dict  # 存储攻击路径的置信度评估结果
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list
    manual_review_items: list


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
        
    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


# ── 【方案C】增强缓存机制 ──
hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()

def hash_prompt(text: str) -> str:
    """生成 prompt 的 MD5 哈希（用于缓存键）"""
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()

def register_prompt_for_cache(prompt: str) -> str:
    """注册 prompt 到全局映射，返回其 hash"""
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h

def clear_prompt_cache():
    """清空缓存映射（测试/重置时用）"""
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()

@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    """基于 prompt 哈希的缓存调用，使用前必须先用 register_prompt_for_cache 注册 prompt"""
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册 (sys={system_hash[:8]}..., user={user_hash[:8]}...)")
        return "[]"
    
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


# 【方案C】智能缓存键生成器
def smart_cache_key(template: str, variables: dict, key_fields: list[str] = None) -> str:
    """
    生成智能缓存键：模板哈希 + 关键变量哈希
    避免长文本（如 rag_context、topo_info）污染缓存键
    """
    if key_fields is None:
        # 默认只缓存这些关键字段，其他长文本字段忽略
        key_fields = ["function", "asset_name", "security_attribute", "threat_type"]
    
    template_hash = hashlib.md5(template.encode('utf-8')).hexdigest()
    
    # 只提取关键变量生成哈希
    key_vars = {k: variables.get(k, "") for k in key_fields if k in variables}
    vars_str = json.dumps(key_vars, sort_keys=True, ensure_ascii=False)
    vars_hash = hashlib.md5(vars_str.encode('utf-8')).hexdigest()
    
    return f"{template_hash}:{vars_hash}"


# 带重试的 LLM 调用
def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
                
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: langchain 返回类型不匹配 - {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    
    return "[]"


# 并发 LLM 调用引擎（增强日志统计）
def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    """
    并发执行多个 LLM 调用
    tasks: [{"id": str/int, "system": str, "user": str}, ...]
    返回: [{"id": ..., "raw": str}, ...]  按 id 排序
    """
    results = []
    total = len(tasks)
    if total == 0:
        return results
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)

        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")

    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s, 平均 {elapsed/len(tasks):.2f}s/任务")
    return results


# 【方案B】批量置信度评估调用函数
# 【方案B】批量置信度评估调用函数（修复版）
def batch_confidence_eval(
    scenarios: list,
    eval_template: str,
    system_prompt: str,
    batch_size: int = EVAL_BATCH_SIZE,
    key_fields: list[str] = None,
    **template_vars  # ✅ 新增：支持额外模板变量
) -> list[dict]:
    """
    【方案B】将逐条置信度评估改为小批量调用，减少往返开销
    
    Args:
        **template_vars: 额外传递给模板的变量（如 topo_info, ext_info 等）
    """
    if not scenarios:
        return []
    
    # 分批
    batches = [scenarios[i:i+batch_size] for i in range(0, len(scenarios), batch_size)]
    logger.info(f"  → 批量评估: {len(scenarios)} 条场景, 分 {len(batches)} 批 (batch_size={batch_size})")
    
    tasks = []
    for batch_idx, batch in enumerate(batches):
        # 格式化批量输入
        batch_json = json.dumps(batch, ensure_ascii=False, indent=2)
        
        # ✅ 修复：合并所有模板变量
        format_vars = {"batch": batch_json, "batch_count": len(batch), **template_vars}
        user_prompt = eval_template.format(**format_vars)
        
        # 【方案C】使用智能缓存键（只哈希关键变量）
        cache_vars = {"batch_size": len(batch), **{k: v for k, v in template_vars.items() if k in (key_fields or [])}}
        cache_key = smart_cache_key(eval_template, cache_vars, key_fields)
        system_hash = register_prompt_for_cache(system_prompt)
        user_hash = register_prompt_for_cache(user_prompt)
        
        tasks.append({
            "id": batch_idx,
            "system": system_prompt,
            "user": user_prompt,
            "_cache_key": cache_key,
            "_batch_indices": list(range(batch_idx * batch_size, min((batch_idx + 1) * batch_size, len(scenarios))))
        })
    
    # 并发调用
    raw_results = concurrent_llm_calls(tasks)
    
    # 解析并展平结果（保持不变）
    eval_results = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    orig_idx = r.get("_batch_indices", [])[idx] if "_batch_indices" in r else len(eval_results) + idx
                    eval_results.append({
                        "id": orig_idx,
                        "raw_result": item,
                        "batch_id": r["id"],
                        "cache_key": r.get("_cache_key", "")
                    })
            else:
                eval_results.append({
                    "id": r["id"],
                    "raw_result": parsed,
                    "batch_id": r["id"],
                    "cache_key": r.get("_cache_key", "")
                })
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")
    
    eval_results.sort(key=lambda x: x["id"])
    return eval_results


def save_checkpoint(data: Any, step_name: str):
    """按步骤名保存独立 checkpoint"""
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Python计算函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def calculate_impact_value(influence_parameters: dict) -> int:
    """计算影响值（四维最高分）"""
    if not influence_parameters:
        return 1
    max_val = max(int(influence_parameters.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
    return min(max(max_val, 1), 4)


def calculate_impact_level(impact_value: int) -> str:
    """根据影响值映射影响等级"""
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    return IMPACT_LABELS.get(impact_value, "Negligible")


def calculate_feasibility_score(attack_parameters: dict) -> int:
    """计算可行性分数（五维求和）"""
    if not attack_parameters:
        return 0
    return sum(int(attack_parameters.get(k, 0)) for k in [
        "Exposure_time", "Professional_experience",
        "Required_information", "Opportunity_window", "Required_equipment",
    ])


def calculate_feasibility_level(feasibility_score: int) -> str:
    """根据可行性分数映射可行性等级"""
    if feasibility_score <= 13:
        return "High"
    elif feasibility_score <= 19:
        return "Medium"
    elif feasibility_score <= 24:
        return "Low"
    else:
        return "Very Low"


def calculate_confidence_score(scores: dict) -> float:
    """计算加权置信度分数（0-100）"""
    total = 0.0
    for dimension, weight in CONFIDENCE_WEIGHTS.items():
        score = scores.get(dimension, 0)
        total += score * weight
    return round(total, 2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，攻击路径分析基于:自上而下的方法:通过分析实现威胁场景的不同方式(例如:攻击树、攻击图)来推断攻击路径。
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


# ──────  步骤2: 单条损害场景 Prompt (逐条调用) ──────
DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是"涉及车辆或车辆功能且影响道路使用者的不良后果"。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}
- 对应威胁类型：{threat_type}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体危害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"

## 输出（严格 JSON 对象，不要数组）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "完整的损害场景描述",
  "threat_type": "{threat_type}"
}}"""


# ──────  步骤3: 单条威胁场景 Prompt (逐条调用) ──────
TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是"为实现损害场景，资产的信息安全属性遭到破坏的潜在原因"。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体{asset_name}，如"GW部件"、"前照灯请求信号"、"车辆数据"
   - 「{asset_name}」被破坏的「{security_attribute}」（必须与输入的安全属性完全一致）
   - 「{security_attribute}」被破坏的原因/攻击意图
2. - 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条。
   - 进一步的信息能被威胁场景包含或与威胁场景关联。例如:危害场景与资产、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "damage_scenario": "{damage_scenario}",
  "threat_type": "{threat_type}",
  "threat_scenario": "详细的威胁场景描述"
}}"""


# ──────  步骤4: 单条攻击路径 Prompt (逐条调用, 附带拓扑) ──────
AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」生成最短的（经过拓扑节点最少）必须符合现实场景和逻辑的攻击路径,如有多条最短路径请生成多条，最多不超过4条，攻击路径是"为实现威胁场景的一组蓄意活动"。必须采用自上而下的方法（如攻击树分析），从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}（{asset_type}）
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产：
   - 以编号步骤形式呈现：1.…；2.…；3.…；…
   - 第一步：明确的外部攻击入口
   - 中间步骤：沿车辆拓扑经过的中间节点和通信协议
   - 最终步骤：到达「{asset_name}」并实施具体攻击技术，从而破坏「{security_attribute}」
2. 路径中的节点和连接关系必须与车辆拓扑图「{topo_info}」和外部接口信息「{ext_info}」一致，不得虚构不存在的节点或跳过拓扑中的网关/ECU。
3. 同一个威胁场景的多条攻击路径不要重复
4. 输出协议时只输出名称，不要输出id编码
5. 输出的路径必须符合现实场景和逻辑

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号(非预期的快速减速)。"

## 输出（严格 JSON 对象）
{{
  "function": "{function_name}",
  "asset_name": "{asset_name}",
  "asset_type": "{asset_type}",
  "security_attribute": "{security_attribute}",
  "threat_scenario": "{threat_scenario}",
  "attack_paths": [
    "1.完整步骤链",
    "2.完整步骤链"
  ]
}}"""


# ────── 步骤5: 影响评级 Prompt (分批调用) ──────
IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数）
### Safety: 1=无伤害, 2=轻度/中度伤害, 3=严重伤害和有生命危险, 4=威胁生命的伤害（不确定是否幸存）或致命的伤害
### Finance: 1=可忽略, 2=个人可承受, 3=大量损失但受影响的道路使用者将能够克服这些后果, 4=灾难性后果受影响的道路使用者可能无法克服
#### 利益相关者包括：车主、驾驶员、乘员、行人、供应商、主机厂。
### Operation: 1=不影响, 2=导致车辆功能的部分退化, 3=导致车辆重要功能丧失或受损, 4=导致车辆核心功能丧失或受损
### Privacy: 1=隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体,
2=隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体； b)泄露的信息不敏感但很容易识别到PII主体,
3=隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体； b)泄露的信息敏感而且很容易识别到PII主体,
4=隐私侵犯会对道路使用者造成重大甚至不可逆转的影响 泄露的信息高度敏感，并且很容易识别到PII主体 
#### 需要考虑车主、驾驶员、乘员、行人、供应商、主机厂。

## 强约束条件
- 每个维度必须严格对照评级标准打分，不得使用其他标准
- 分值必须是整数 1~4

## 输出（严格 JSON 数组，只输出参数）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "influence_parameters": {{"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}}
  }}
]"""


# ────── 步骤6: 可行性评级 Prompt (分批调用) ──────
AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434（尤其是 Annex G 基于攻击潜力的攻击可行性评估方法） 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准
### Exposure_time: 
- 0分：≤1天（实现攻击行为的时间小于等于1天）  
- 1分：≤1周（实现攻击行为的时间小于等于1周）  
- 4分：≤1个月（实现攻击行为的时间小于等于1个月）  
- 17分：≤6个月（实现攻击行为的时间小于等于6个月）  
- 19分：>6个月（实现攻击行为的时间大于6个月）
### Professional_experience:
- 0分：非专业（Layman）——外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3分：精通/擅长（Proficient）——熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6分：专业（Expert）——熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 8分：复合型（Multiple experts）——一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0分：公开的（Public）——公共信息（例如互联网上获得的信息）  
- 3分：限制的（Restricted）——受限制的信息（制造商和供应商共享的内部文档）  
- 7分：敏感的（Sensitive）——机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11分：至关重要的（Critical）——严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0分：十分高（Critical）——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1分：高（High）——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4分：中（Medium）——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10分：低（Low）——困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0分：标准的（Standard）——标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4分：专用的（Specialized）——专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7分：定制的（Bespoke）——定制设备（厂家限制的工具、电子显微镜）  
- 9分：多重定制的（Multiple bespoke）——攻击不同步骤需要不同类型的定制设备

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数（0、1、3、4、6、7、8、9、10、11、17、19）

## 输出（严格 JSON 数组，只输出参数）
[
  {{
    "function": "功能名称",
    "asset_name": "资产名称",
    "security_attribute": "安全属性",
    "attack_path": "攻击路径",
    "attack_parameters": {{
      "Exposure_time": 0, "Professional_experience": 0,
      "Required_information": 0, "Opportunity_window": 0, "Required_equipment": 0
    }}
  }}
]"""


# ────── 【方案B】置信度评估 Prompt (批量版) ──────
CONFIDENCE_EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估场景质量并给出各维度的置信度评分。
仅输出 JSON 数组，不要任何其他文字。"""

# 损害场景置信度评估模板（批量版）
DS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下损害场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
损害场景必须包含以下要素：
- 功能与不良后果的因果关系
- 对道路使用者的具体危害
- 涉及的目标资产
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 与功能/资产/STRIDE是否匹配
- 损害场景描述是否与输入信息一致
评分标准：完全匹配100分，部分不匹配扣20-50分，严重不一致扣60-80分

### 3. 真实性（满分100分）
- 危害对道路使用者影响是否合理
- 是否有明显幻觉或不合理描述
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 描述是否清晰易懂
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过，列出具体问题"
  }},
  ...
]"""

# 威胁场景置信度评估模板（批量版）
TS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下威胁场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
威胁场景必须包含以下要素：
- 目标资产
- 被破坏的信息安全属性
- 信息安全属性被破坏的原因
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 是否直接导致损害场景
- 与损害场景的因果关系是否清晰
评分标准：完全一致100分，部分不一致扣20-50分，因果关系断裂扣60-80分

### 3. 真实性（满分100分）
- 攻击意图是否现实
- 是否有明显幻觉或不合理描述
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 描述是否清晰易懂
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过，列出具体问题"
  }},
  ...
]"""

# 攻击路径置信度评估模板（批量版）
AP_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下攻击路径列表的质量，对每条路径给出各维度的置信度评分（0-100分）。

## 车辆拓扑图（简略）
{topo_info}
## 外部接口（简略）
{ext_info}

## 待评审攻击路径列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
攻击路径必须包含以下要素：
- 起点（具体攻击面）
- 逻辑步骤
- 终点（目标资产）
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
- 是否能真正实现威胁场景
- 是否符合拓扑图
- 攻击入口是否存在于外部接口中
评分标准：完全一致100分，部分不一致扣20-50分，严重偏离扣60-80分

### 3. 真实性（满分100分）
- 攻击面是否存在
- 步骤是否可行
- 是否有明显幻觉或不合理描述
评分标准：完全可行100分，部分不可行扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
- 句子是否简洁、无冗余、无语法错误
- 步骤描述是否清晰连贯
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过，列出具体问题"
  }},
  ...
]"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ────────── 步骤1: 资产识别 (规则引擎, 不变) ──────────

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": attrs,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "manual_review_items": [],
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


# ──────────  步骤2: 损害场景 (逐条并发生成) ──────────

def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for attr in asset["security_attributes"]:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                prompt = DS_SINGLE_USER_TEMPLATE.format(
                    function_name=func["function"],
                    asset_name=asset["asset_name"],
                    asset_type=asset["asset_type"],
                    security_attribute=attr,
                    threat_type=threat_type,
                    rag_context=rag_context,
                )
                tasks.append({"id": task_id, "system": BASE_SYSTEM, "user": prompt})
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                scenarios.extend(parsed)
            else:
                scenarios.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {"damage_scenarios": scenarios}


def node_ds_confidence_eval(state: TaraState) -> dict:
    """【方案B】损害场景置信度评估节点（批量调用）"""
    logger.info("[步骤2-评估] 损害场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ds_confidence": {"all_passed": True, "items": []}}

    scenarios = state.get("damage_scenarios", [])
    if not scenarios:
        return {"ds_confidence": {"all_passed": True, "items": []}}

    # 【方案B】使用批量评估函数
    eval_results_raw = batch_confidence_eval(
        scenarios=scenarios,
        eval_template=DS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"]
    )

    # 解析评估结果
    eval_results = []
    manual_review_items = []
    
    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD
        
        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": scenarios[orig_id] if orig_id < len(scenarios) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
            "batch_id": item.get("batch_id"),
            "cache_hit": bool(item.get("cache_key")),  # 【方案C】标记是否命中缓存
        }
        eval_results.append(eval_item)
        
        if not passed:
            manual_review_items.append({
                "type": "damage_scenario",
                "id": orig_id,
                "confidence_score": confidence_score,
                "feedback": parsed.get("feedback", ""),
                "scenario": scenarios[orig_id] if orig_id < len(scenarios) else {},
            })

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个损害场景需要人工审核")

    save_checkpoint(eval_results, "step2_ds_confidence_eval")
    
    return {
        "ds_confidence": {"all_passed": all_passed, "items": eval_results},
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤3: 威胁场景 (逐条并发生成) ──────────

def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    tasks = []
    for i, ds in enumerate(damage_scenarios):
        prompt = TS_SINGLE_USER_TEMPLATE.format(
            function_name=ds.get("function", ""),
            asset_name=ds.get("asset_name", ""),
            asset_type=ds.get("asset_type", ""),
            security_attribute=ds.get("security_attribute", ""),
            threat_type=ds.get("threat_type", ""),
            damage_scenario=ds.get("damage_scenario", ""),
            rag_context=rag_context,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ts_list.extend(parsed)
            else:
                ts_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {"threat_scenarios": ts_list}


def node_ts_confidence_eval(state: TaraState) -> dict:
    """【方案B】威胁场景置信度评估节点（批量调用）"""
    logger.info("[步骤3-评估] 威胁场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ts_confidence": {"all_passed": True, "items": []}}

    ts_list = state.get("threat_scenarios", [])
    if not ts_list:
        return {"ts_confidence": {"all_passed": True, "items": []}}

    # 【方案B】使用批量评估函数
    eval_results_raw = batch_confidence_eval(
        scenarios=ts_list,
        eval_template=TS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"]
    )

    eval_results = []
    manual_review_items = []
    
    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD
        
        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": ts_list[orig_id] if orig_id < len(ts_list) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
            "batch_id": item.get("batch_id"),
            "cache_hit": bool(item.get("cache_key")),
        }
        eval_results.append(eval_item)
        
        if not passed:
            manual_review_items.append({
                "type": "threat_scenario",
                "id": orig_id,
                "confidence_score": confidence_score,
                "feedback": parsed.get("feedback", ""),
                "scenario": ts_list[orig_id] if orig_id < len(ts_list) else {},
            })

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个威胁场景需要人工审核")

    save_checkpoint(eval_results, "step3_ts_confidence_eval")
    
    return {
        "ts_confidence": {"all_passed": all_passed, "items": eval_results},
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤4: 攻击路径 (逐条并发生成) ──────────

def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发)")

    threat_scenarios = state.get("threat_scenarios", [])
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    tasks = []
    for i, ts in enumerate(threat_scenarios):
        prompt = AP_SINGLE_USER_TEMPLATE.format(
            function_name=ts.get("function", ""),
            asset_name=ts.get("asset_name", ""),
            asset_type=ts.get("asset_type", ""),
            security_attribute=ts.get("security_attribute", ""),
            threat_scenario=ts.get("threat_scenario", ""),
            topo_info=topo_info,
            ext_info=ext_info,
            rag_context=rag_context,
        )
        tasks.append({"id": i, "system": BASE_SYSTEM, "user": prompt})

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    ap_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ap_list.extend(parsed)
            else:
                ap_list.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成攻击路径 {len(ap_list)}/{len(tasks)} 组")
    save_checkpoint(ap_list, "step4_attack_paths")
    return {"attack_paths": ap_list}


def node_ap_confidence_eval(state: TaraState) -> dict:
    """【方案B】攻击路径置信度评估节点（批量调用）"""
    logger.info("[步骤4-评估] 攻击路径置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过 (ENABLE_EVALUATION=False)")
        return {"ap_confidence": {"all_passed": True, "items": []}}

    ap_list = state.get("attack_paths", [])
    if not ap_list:
        return {"ap_confidence": {"all_passed": True, "items": []}}

    # 准备拓扑和接口信息（简化版，避免过长）
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False)[:2000] + "..."
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False)[:1000] + "..."

    # 准备批量输入
    batch_input = []
    for ap in ap_list:
        batch_input.append({
            "function": ap.get("function", ""),
            "asset_name": ap.get("asset_name", ""),
            "security_attribute": ap.get("security_attribute", ""),
            "threat_scenario": ap.get("threat_scenario", ""),
            "attack_paths": ap.get("attack_paths", []),
        })

    # ✅ 修复：传入 topo_info 和 ext_info 作为额外模板变量
    eval_results_raw = batch_confidence_eval(
        scenarios=batch_input,
        eval_template=AP_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute"],
        topo_info=topo_info,      # ✅ 新增
        ext_info=ext_info,        # ✅ 新增
    )


    eval_results = []
    manual_review_items = []
    
    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD
        
        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": ap_list[orig_id] if orig_id < len(ap_list) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
            "batch_id": item.get("batch_id"),
            "cache_hit": bool(item.get("cache_key")),
        }
        eval_results.append(eval_item)
        
        if not passed:
            manual_review_items.append({
                "type": "attack_path",
                "id": orig_id,
                "confidence_score": confidence_score,
                "feedback": parsed.get("feedback", ""),
                "scenario": ap_list[orig_id] if orig_id < len(ap_list) else {},
            })

    all_passed = len(manual_review_items) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")
    
    if not all_passed:
        logger.warning(f"  ⚠ {len(manual_review_items)} 个攻击路径需要人工审核")

    save_checkpoint(eval_results, "step4_ap_confidence_eval")
    
    return {
        "ap_confidence": {"all_passed": all_passed, "items": eval_results},
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


# ──────────  步骤5: 影响评级 (分批并发) ──────────

def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响参数评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = IR_BATCH_USER_TEMPLATE.format(
            ds_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    for r in ratings:
        params = r.get("influence_parameters", {})
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        r["impact_value"] = str(impact_value)
        r["impact_level"] = impact_level

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


# ──────────  步骤6: 攻击可行性评级 (分批并发) ──────────

def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (分批并发)")

    flat_paths = []
    for ap_group in state.get("attack_paths", []):
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "function": ap_group.get("function", ""),
                "asset_name": ap_group.get("asset_name", ""),
                "security_attribute": ap_group.get("security_attribute", ""),
                "attack_path": path,
            })

    if not flat_paths:
        return {"feasibility_ratings": []}

    batches = [flat_paths[i:i+BATCH_SIZE_RATING] for i in range(0, len(flat_paths), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(flat_paths)} 条攻击路径, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = AF_BATCH_USER_TEMPLATE.format(
            ap_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": AF_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    for r in ratings:
        params = r.get("attack_parameters", {})
        feasibility_score = calculate_feasibility_score(params)
        feasibility_level = calculate_feasibility_level(feasibility_score)
        r["feasibility_score"] = feasibility_score
        r["feasibility_level"] = feasibility_level

    logger.info(f"  → 完成可行性评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": ratings}


# ────────── 步骤7: 风险判定 (规则引擎, 不变) ──────────

def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险判定 (交叉矩阵)")

    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    RISK_MATRIX = {
        ("Severe", "High"): 5, ("Severe", "Medium"): 4, ("Severe", "Low"): 3,
        ("Severe", "Very Low"): 2, 
        ("Major", "High"): 4, ("Major", "Medium"): 3, ("Major", "Low"): 2,
        ("Major", "Very Low"): 1, 
        ("Moderate", "High"): 3, ("Moderate", "Medium"): 2, ("Moderate", "Low"): 2,
        ("Moderate", "Very Low"): 1, 
        ("Negligible", "High"): 1, ("Negligible", "Medium"): 1, ("Negligible", "Low"): 1,
        ("Negligible", "Very Low"): 1,
    }
    RISK_LABEL = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
    TREATMENT = {
        "1": "风险接受",
        "2": "风险接受",
        "3": "风险接受",
        "4": "风险缓解",
        "5": "风险缓解",
    }

    FEAS_ORDER = {"High": 4, "Medium": 3, "Low": 2, "Very Low": 1}
    feas_map: dict[tuple, str] = {}
    for fr in feasibility_ratings:
        key = (fr.get("function", ""), fr.get("asset_name", ""), fr.get("security_attribute", ""))
        level = fr.get("feasibility_level", "Very Low")
        if FEAS_ORDER.get(level, 1) > FEAS_ORDER.get(feas_map.get(key, "Very Low"), 1):
            feas_map[key] = level

    risk_results = []
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        impact_level = ir.get("impact_level", "Negligible")
        feasibility_level = feas_map.get(key, "Very Low")
        risk_val = RISK_MATRIX.get((impact_level, feasibility_level), 1)
        risk_label = RISK_LABEL.get(risk_val, "1")
        treatment = TREATMENT.get(risk_label, "风险接受")

        risk_results.append({
            "function": ir.get("function", ""),
            "asset_name": ir.get("asset_name", ""),
            "security_attribute": ir.get("security_attribute", ""),
            "impact_level": impact_level,
            "feasibility_level": feasibility_level,
            "risk_value": risk_label,
            "risk_treatment": treatment,
        })

    logger.info(f"  → 风险判定完成 {len(risk_results)} 条")
    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


# ────────── 步骤8: 报告汇总 ──────────

def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    rr_map = {_key(r): r for r in risk_results}

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for attr in asset.get("security_attributes", []):
                key = (func_name, asset["asset_name"], attr)
                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "threat_type": ds.get("threat_type", ""),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "attack": attack_list,
                    "risk_value": rr.get("risk_value", ""),
                    "risk_treatment": rr.get("risk_treatment", ""),
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")
    return {"tara_report": report}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：工作流构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_confidence_eval",  node_ds_confidence_eval)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_confidence_eval",  node_ts_confidence_eval)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_confidence_eval",  node_ap_confidence_eval)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")
    builder.add_edge("node_ds_generate", "node_ds_confidence_eval")
    builder.add_edge("node_ds_confidence_eval", "node_ts_generate")
    builder.add_edge("node_ts_generate", "node_ts_confidence_eval")
    builder.add_edge("node_ts_confidence_eval", "node_ap_generate")
    builder.add_edge("node_ap_generate", "node_ap_confidence_eval")
    builder.add_edge("node_ap_confidence_eval", "node_impact_rating")
    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_json: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: dict | None = None,
    asset_results: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")
    logger.info(f"  置信度阈值: CONFIDENCE_THRESHOLD={CONFIDENCE_THRESHOLD}")
    logger.info(f"  【方案B】评估批大小: EVAL_BATCH_SIZE={EVAL_BATCH_SIZE}")
    logger.info(f"  【方案C】智能缓存: 启用 (maxsize=200)")

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": data_flow_json or [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or {},
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_confidence": {},
        "threat_scenarios": [], "ts_confidence": {},
        "attack_paths": [],    "ap_confidence": {},
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
        "manual_review_items": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    # 保存最终报告
    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    # 保存完整中间状态
    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
            "manual_review_items": final_state.get("manual_review_items", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    # 输出需要人工审核的项目
    manual_items = final_state.get("manual_review_items", [])
    if manual_items:
        logger.warning("=" * 60)
        logger.warning(f"⚠️  共 {len(manual_items)} 个场景需要人工审核:")
        for item in manual_items:
            logger.warning(f"  - [{item['type']}] ID: {item['id']}, 置信度: {item['confidence_score']}")
            logger.warning(f"    反馈: {item['feedback'][:100]}...")
        logger.warning("=" * 60)

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":

    sample_assets = [
        {
            "function": "交通标识及信号灯识别",
            "assets": [
                {"asset_type": "信号", "asset_name": "弹窗/音效"},
            ],
        },
        {
            "function": "智能泊车辅助",
            "assets": [
                {"asset_type": "数据", "asset_name": "用户账户信息"},
            ],
        },
    ]

    sample_topology = [{"id": 1, "CANFD": ["AMP", "CDC", "IC"]}, {"id": 2, "A2B": ["AMP", "CDC"]}, {"id": 3, "FPDLINK": ["CDC", "IC"]}, {"id": 4, "FPDLINK": ["CDC", "IVI"]}, {"id": 5, "GMSL": ["CDC", "DMS"]}, {"id": 6, "GMSL": ["CDC", "OMS"]}, {"id": 7, "GMSL": ["CDC", "MDC"]}, {"id": 8, "ETH": ["CDC", "CEM"]}, {"id": 9, "CANFD": ["BDCR", "CDC", "CEM", "REA"]}, {"id": 10, "CAN": ["BDCR", "BLE", "CEM", "CMSL", "CMSR", "DSM", "ECALL", "PDSM", "TRM", "WLCM"]}, {"id": 11, "CANFD": ["MDC", "USS ECU"]}, {"id": 12, "CANFD": ["FR", "MDC"]}, {"id": 13, "GMSL": ["LRC-FC", "MDC"]}, {"id": 14, "GMSL": ["MDC", "SRC-FC"]}, {"id": 15, "GMSL": ["MDC", "MRC-SC"]}, {"id": 16, "GMSL": ["MDC", "MRC-RC"]}, {"id": 17, "GMSL": ["FEC", "MDC"]}, {"id": 18, "ETH": ["CEM", "MDC"]}, {"id": 19, "CANFD": ["CEM", "EPS", "IBCU", "MDC", "SRS", "VCU", "VMC"]}, {"id": 20, "ETH": ["CEM", "VCU"]}, {"id": 21, "CANFD": ["BCU", "EVCC", "FMIPU", "HPMU", "ITMS", "PDU", "RDPEU", "RMIPU", "VCU"]}, {"id": 22, "CAN": ["BCU", "EVCC"]}, {"id": 23, "CAN": ["ACP", "ITMS", "PTC"]}, {"id": 24, "CANFD": ["BDCR", "CDC", "CEM", "LBMS"]}, {"id": 25, "ETH": ["Lidar", "MDC"]}, {"id": 26, "蜂窝": ["BCALL中心", "CDC", "ECALL中心", "华为车云", "阿维塔车云"]}, {"id": 27, "CAN": ["CEM", "诊断仪"]}]
    
    sample_ext_interfaces = [
        {"component": "CDC", "interfaces": ["Cellular Netword  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
        {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
        {"component": "UWB", "interfaces": ["RF  射频"]},
        {"component": "CHLIL", "interfaces": ["JTAG"]},
        {"component": "CHLIR", "interfaces": ["JTAG"]},
        {"component": "ITMS", "interfaces": ["JTAG"]},
    ]

    report = run_tara(
        asset_results=sample_assets,
        topology_json=sample_topology,
        external_interface_json=sample_ext_interfaces,
        output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
        build_rag_index=True,
    )

    logger.info(f"📋 最终报告包含 {len(report)} 个功能模块，已保存为 JSON 文件")

2026-04-13 16:10:55,851 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-13 16:10:55,852 [INFO] ============================================================
2026-04-13 16:10:55,853 [INFO]   并发配置: MAX_WORKERS=5, BATCH_SIZE_RATING=5
2026-04-13 16:10:55,853 [INFO]   评估开关: ENABLE_EVALUATION=True
2026-04-13 16:10:55,854 [INFO]   置信度阈值: CONFIDENCE_THRESHOLD=85
2026-04-13 16:10:55,854 [INFO]   【方案B】评估批大小: EVAL_BATCH_SIZE=3
2026-04-13 16:10:55,854 [INFO]   【方案C】智能缓存: 启用 (maxsize=200)
2026-04-13 16:10:55,855 [INFO] [初始化] RAG 已禁用，跳过
2026-04-13 16:10:55,879 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-13 16:10:55,880 [INFO] ============================================================
2026-04-13 16:10:55,880 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-13 16:10:55,881 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-13 16:10:55,883 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V4\checkpoint_step1_assets.json
2026-04-13 16:10:55,883 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-13 16:10:55,884 [INFO]   → 共 7 个子任务, 并发数 5
2026-04-13 1

# V6

In [5]:
import json
import os
import re
import glob
import logging
import time
import hashlib
from typing import TypedDict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import pandas as pd
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

llm = ChatOpenAI(
    model="Qwen/Qwen3.5-4B",
    openai_api_key="sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={"enable_thinking": False},
)

ENABLE_RAG = True
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

ENABLE_EVALUATION = True

MAX_WORKERS = 5
CALL_INTERVAL = 0.8
BATCH_SIZE_RATING = 5
EVAL_BATCH_SIZE = 3

OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V8"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}

CONFIDENCE_THRESHOLD = 85

CONFIDENCE_WEIGHTS = {
    "要素完整性": 0.30,
    "一致性": 0.30,
    "真实性": 0.30,
    "清晰度": 0.10,
}

MAX_CONFIDENCE_RETRIES = 1


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,
                request_timeout=45,
                max_retries=2,
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len,
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:
                valid_chunks.append(c)
            elif len(content) > 128:
                c.page_content = content[:127] + "..."
                valid_chunks.append(c)
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True,
            )
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs
            )
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: list
    external_interface_json: list
    asset_results: list
    damage_scenarios: list
    ds_confidence: dict
    ds_retry_count: int
    ds_feedback: dict
    threat_scenarios: list
    ts_confidence: dict
    ts_retry_count: int
    ts_feedback: dict
    attack_paths: list
    ap_confidence: dict
    ap_retry_count: int
    ap_feedback: dict
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list
    manual_review_items: list


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()


def hash_prompt(text: str) -> str:
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def register_prompt_for_cache(prompt: str) -> str:
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h


def clear_prompt_cache():
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()


@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册")
        return "[]"
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


def smart_cache_key(template: str, variables: dict, key_fields: list[str] = None) -> str:
    if key_fields is None:
        key_fields = ["function", "asset_name", "security_attribute", "threat_type"]
    template_hash = hashlib.md5(template.encode('utf-8')).hexdigest()
    key_vars = {k: variables.get(k, "") for k in key_fields if k in variables}
    vars_str = json.dumps(key_vars, sort_keys=True, ensure_ascii=False)
    vars_hash = hashlib.md5(vars_str.encode('utf-8')).hexdigest()
    return f"{template_hash}:{vars_hash}"


def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    return "[]"


def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    results = []
    total = len(tasks)
    if total == 0:
        return results
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)
        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")
    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s")
    return results


def batch_confidence_eval(
    scenarios: list,
    eval_template: str,
    system_prompt: str,
    batch_size: int = EVAL_BATCH_SIZE,
    key_fields: list[str] = None,
    **template_vars,
) -> list[dict]:
    if not scenarios:
        return []
    batches = [scenarios[i:i+batch_size] for i in range(0, len(scenarios), batch_size)]
    logger.info(f"  → 批量评估: {len(scenarios)} 条场景, 分 {len(batches)} 批")
    tasks = []
    for batch_idx, batch in enumerate(batches):
        batch_json = json.dumps(batch, ensure_ascii=False, indent=2)
        format_vars = {"batch": batch_json, "batch_count": len(batch), **template_vars}
        user_prompt = eval_template.format(**format_vars)
        cache_vars = {"batch_size": len(batch), **{k: v for k, v in template_vars.items() if k in (key_fields or [])}}
        cache_key = smart_cache_key(eval_template, cache_vars, key_fields)
        system_hash = register_prompt_for_cache(system_prompt)
        user_hash = register_prompt_for_cache(user_prompt)
        tasks.append({
            "id": batch_idx,
            "system": system_prompt,
            "user": user_prompt,
            "_cache_key": cache_key,
            "_batch_indices": list(range(batch_idx * batch_size, min((batch_idx + 1) * batch_size, len(scenarios)))),
        })
    raw_results = concurrent_llm_calls(tasks)
    eval_results = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    orig_idx = r.get("_batch_indices", [])[idx] if "_batch_indices" in r and idx < len(r["_batch_indices"]) else len(eval_results) + idx
                    eval_results.append({
                        "id": orig_idx,
                        "raw_result": item,
                        "batch_id": r["id"],
                        "cache_key": r.get("_cache_key", ""),
                    })
            else:
                eval_results.append({
                    "id": r["id"],
                    "raw_result": parsed,
                    "batch_id": r["id"],
                    "cache_key": r.get("_cache_key", ""),
                })
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")
    eval_results.sort(key=lambda x: x["id"])
    return eval_results


def save_checkpoint(data: Any, step_name: str):
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Python计算函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def calculate_impact_value(influence_parameters: dict) -> int:
    if not influence_parameters:
        return 1
    max_val = max(int(influence_parameters.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
    return min(max(max_val, 1), 4)


def calculate_impact_level(impact_value: int) -> str:
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    return IMPACT_LABELS.get(impact_value, "Negligible")


def calculate_feasibility_score(attack_parameters: dict) -> int:
    if not attack_parameters:
        return 0
    return sum(int(attack_parameters.get(k, 0)) for k in [
        "Exposure_time", "Professional_experience",
        "Required_information", "Opportunity_window", "Required_equipment",
    ])


def calculate_feasibility_level(feasibility_score: int) -> str:
    if feasibility_score <= 13:
        return "High"
    elif feasibility_score <= 19:
        return "Medium"
    elif feasibility_score <= 24:
        return "Low"
    else:
        return "Very Low"


def calculate_confidence_score(scores: dict) -> float:
    total = 0.0
    for dimension, weight in CONFIDENCE_WEIGHTS.items():
        score = scores.get(dimension, 0)
        total += score * weight
    return round(total, 2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：输入解析函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_dcd_json(input_dir: str) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号",
        "tm.Process": "部件",
        "tm.Store": "数据",
        "tm.Actor": "接口",
    }
    all_results = []
    json_files = glob.glob(os.path.join(input_dir, '*.json'))
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                logger.warning(f"⚠️ DCD解析失败: {os.path.basename(file_path)}")
                continue
        current_result = {'function': '', 'assets': []}
        if 'detail' in data and 'diagrams' in data['detail']:
            for diagram in data['detail']['diagrams']:
                if not current_result['function'] and 'title' in diagram:
                    current_result['function'] = diagram['title']
                if 'diagramJson' in diagram and 'cells' in diagram['diagramJson']:
                    for cell in diagram['diagramJson']['cells']:
                        if cell.get('outOfScope') is True:
                            continue
                        raw_cell_type = cell.get('type', '')
                        if raw_cell_type == 'tm.Boundary':
                            continue
                        finetermval_value = ""
                        for key, value in cell.items():
                            if key.startswith('propertyList') and isinstance(value, dict):
                                if 'finetermval' in value:
                                    finetermval_value = value['finetermval']
                                    break
                        if not finetermval_value:
                            continue
                        mapped_cell_type = TYPE_MAPPING.get(raw_cell_type, raw_cell_type)
                        current_result['assets'].append({
                            'asset_type': mapped_cell_type,
                            'asset_name': finetermval_value,
                        })
        if not current_result['function']:
            current_result['function'] = os.path.splitext(os.path.basename(file_path))[0]
        if current_result['assets']:
            all_results.append(current_result)
    logger.info(f"  DCD解析完成: {len(all_results)} 个功能, 来自 {len(json_files)} 个文件")
    return all_results


def parse_topology_json(input_dir: str) -> dict:
    all_topo = {}
    json_files = glob.glob(os.path.join(input_dir, '*.json'))
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                logger.warning(f"⚠️ 拓扑图解析失败: {os.path.basename(file_path)}")
                continue
        name = os.path.splitext(os.path.basename(file_path))[0]
        if isinstance(data, dict) and '拓扑图' in data:
            all_topo[name] = data['拓扑图']
        elif isinstance(data, list):
            all_topo[name] = data
        elif isinstance(data, dict):
            all_topo[name] = data
    logger.info(f"  拓扑图解析完成: {len(all_topo)} 个文件")
    return all_topo


def parse_external_interface_excel(excel_path: str) -> list:
    if not os.path.exists(excel_path):
        logger.warning(f"⚠️ 外部接口Excel不存在: {excel_path}")
        return []
    df = pd.read_excel(excel_path, header=1)
    df.columns = [str(col).replace('\n', ' ').strip() for col in df.columns]
    interface_columns = df.columns[2:]
    results = []
    for index, row in df.iterrows():
        component_name = str(row['零部件名称']).strip()
        if pd.isna(row['零部件名称']) or not component_name:
            continue
        active_interfaces = []
        for col in interface_columns:
            cell_value = str(row[col]).strip()
            if '√' in cell_value:
                active_interfaces.append(col)
        if active_interfaces:
            results.append({
                "component": component_name,
                "interfaces": active_interfaces,
            })
    logger.info(f"  外部接口解析完成: {len(results)} 个零部件")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，基于攻击树的方法推断
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是"涉及车辆或车辆功能且影响道路使用者的不良后果"。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体危害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"

## 输出（严格 JSON 对象，不要数组）
{{
  "damage_scenario": "完整的损害场景描述"
}}"""


DS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请为以下资产的指定安全属性，重新生成一条具体的损害场景。上一次生成未通过质量评估，请根据反馈改进。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）：
   - {function_name}如何因为{asset_name}的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的危害说明
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"

## 输出（严格 JSON 对象，不要包含输入字段）
{{
  "damage_scenario": "完整的损害场景描述"
}}"""


TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是"为实现损害场景，资产的信息安全属性遭到破坏的潜在原因"。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体「{asset_name}」
   - 「{asset_name}」被破坏的「{security_attribute}」
   - 「{security_attribute}」被破坏的原因/攻击意图
2. 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"

## 输出（严格 JSON 对象）
{{
  "threat_scenario": "详细的威胁场景描述"
}}"""


TS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请根据以下损害场景，重新生成一条详细的威胁场景描述。上一次生成未通过质量评估，请根据反馈改进。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体「{asset_name}」
   - 「{asset_name}」被破坏的「{security_attribute}」
   - 「{security_attribute}」被破坏的原因/攻击意图
2. 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"

## 输出（严格 JSON 对象，不要包含输入字段）
{{
  "threat_scenario": "详细的威胁场景描述"
}}"""


AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图和外部接口信息生成最短的（经过拓扑节点最少）但符合现实场景和逻辑的攻击路径，如有多条最短路径请生成多条，最多不超过4条。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产
2. 路径中的节点和连接关系必须与{topo_info}和{ext_info}一致
3. 同一个威胁场景的多条攻击路径不要重复（包括路径逻辑）
4. 不同id的通信协议，代表不同的总线，输出时只输出名称不要输出id
5. 攻击路径必须符合现实中的实际场景
6. 经过拓扑节点最少，但符合现实场景和逻辑

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。"

## 输出（严格 JSON 对象）
{{
  "attack_paths": [
    "1.完整步骤链",
    "1.完整步骤链"
  ]
}}"""


AP_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请为以下威胁场景重新生成攻击路径。上一次生成未通过质量评估，请根据反馈改进。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}
- 资产类型：{asset_type}
- 威胁场景：{threat_scenario}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 输出（严格 JSON 对象）
{{
  "attack_paths": [
    "1.完整步骤链",
    "1.完整步骤链"
  ]
}}"""


IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数）
### Safety: 1=无伤害, 2=轻度/中度伤害, 3=严重伤害和有生命危险, 4=威胁生命的伤害或致命的伤害
### Finance: 1=可忽略, 2=个人可承受, 3=大量损失但能克服, 4=灾难性后果
### Operation: 1=不影响, 2=部分退化, 3=重要功能丧失, 4=核心功能丧失
### Privacy: 1=不会带来不便, 2=带来很多不便, 3=很严重影响, 4=重大甚至不可逆转影响

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是整数 1~4

## 输出（严格 JSON 数组，只输出各维度参数）
[
  {{
    "influence_parameters": {{"Safety": 1, "Finance": 2, "Operation": 3, "Privacy": 4}}
  }}
]"""


AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准
### Exposure_time: 0=≤1天, 1=≤1周, 4=≤1个月, 17=≤6个月, 19=>6个月
### Professional_experience: 0=非专业, 3=精通, 6=专业, 8=复合型
### Required_information: 0=公开的, 3=限制的, 7=敏感的, 11=至关重要的
### Opportunity_window: 0=十分高(无限), 1=高(容易), 4=中(有限), 10=低(困难)
### Required_equipment: 0=标准的, 4=专用的, 7=定制的, 9=多重定制的

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数

## 输出（严格 JSON 数组，只输出各维度参数）
[
  {{
    "attack_parameters": {{
      "Exposure_time": 1, "Professional_experience": 3,
      "Required_information": 3, "Opportunity_window": 4, "Required_equipment": 9
    }}
  }}
]"""


CONFIDENCE_EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估场景质量并给出各维度的置信度评分。
仅输出 JSON 数组，不要任何其他文字。"""

DS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下损害场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
损害场景必须包含：功能与不良后果的因果关系、对道路使用者的具体危害、涉及的目标资产
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
与功能/资产/STRIDE是否匹配，描述是否与输入信息一致
评分标准：完全匹配100分，部分不匹配扣20-50分，严重不一致扣60-80分

### 3. 真实性（满分100分）
危害对道路使用者影响是否合理，是否有明显幻觉
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
句子是否简洁、无冗余、无语法错误
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过（"passed": false），列出具体问题"
  }}
]"""

TS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下威胁场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
威胁场景必须包含：目标资产、被破坏的信息安全属性、信息安全属性被破坏的原因
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
是否直接导致损害场景，因果关系是否清晰
评分标准：完全一致100分，部分不一致扣20-50分，因果关系断裂扣60-80分

### 3. 真实性（满分100分）
攻击意图是否现实，是否有明显幻觉
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
句子是否简洁、无冗余、无语法错误
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过（"passed": false），列出具体问题"
  }}
]"""

AP_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下攻击路径列表的质量，对**每条攻击路径**给出各维度的置信度评分（0-100 分）。

## 车辆拓扑图
{topo_info}
## 外部接口
{ext_info}

## 待评审攻击路径列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分 100 分）
攻击路径必须包含：起点（具体攻击面）、逻辑步骤、终点（目标资产）
评分标准：每缺失一个要素扣 30 分，要素不完整扣 10-20 分

### 2. 一致性（满分 100 分）
是否能真正实现威胁场景，是否符合拓扑图，攻击入口是否存在于外部接口中
评分标准：完全一致 100 分，部分不一致扣 20-50 分，严重偏离扣 60-80 分

### 3. 真实性（满分 100 分）
攻击面是否存在，步骤是否可行，是否有明显幻觉
评分标准：完全可行 100 分，部分不可行扣 20-50 分，存在幻觉扣 60-80 分

### 4. 清晰度（满分 100 分）
句子是否简洁、无冗余、无语法错误，步骤描述是否清晰连贯
评分标准：清晰流畅 100 分，有冗余或小错误扣 10-30 分，表达混乱扣 40-60 分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true 或 false（总分>=85 为 true）,
    "feedback": "如不通过（"passed": true 或 false），列出具体问题"
  }}
]"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            security_attributes = []
            for attr in attrs:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                security_attributes.append({
                    "attribute_name": attr,
                    "threat_type": threat_type,
                })
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": security_attributes,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "manual_review_items": [],
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]
    ds_feedback = state.get("ds_feedback", {})
    retry_count = state.get("ds_retry_count", 0)
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for sa in asset["security_attributes"]:
                attr_name = sa["attribute_name"]
                key = (func["function"], asset["asset_name"], attr_name)
                
                if ds_feedback and key in ds_feedback:
                    feedback_info = ds_feedback[key]
                    prompt = DS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                        function_name=func["function"],
                        asset_name=asset["asset_name"],
                        asset_type=asset["asset_type"],
                        security_attribute=attr_name,
                        previous_result=feedback_info.get("previous_result", ""),
                        feedback=feedback_info.get("feedback", ""),
                        rag_context=rag_context,
                    )
                else:
                    prompt = DS_SINGLE_USER_TEMPLATE.format(
                        function_name=func["function"],
                        asset_name=asset["asset_name"],
                        asset_type=asset["asset_type"],
                        security_attribute=attr_name,
                        rag_context=rag_context,
                    )
                tasks.append({
                    "id": task_id,
                    "system": BASE_SYSTEM,
                    "user": prompt,
                    "_meta": {
                        "function": func["function"],
                        "asset_name": asset["asset_name"],
                        "asset_type": asset["asset_type"],
                        "security_attribute": attr_name,
                        "threat_type": sa["threat_type"],
                    },
                })
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                scenarios.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    scenarios.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {"damage_scenarios": scenarios, "ds_feedback": {}}


def node_ds_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤2-评估] 损害场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ds_confidence": {"all_passed": True, "items": []}, "ds_retry_count": 0, "ds_feedback": {}}

    scenarios = state.get("damage_scenarios", [])
    if not scenarios:
        return {"ds_confidence": {"all_passed": True, "items": []}, "ds_retry_count": 0, "ds_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=scenarios,
        eval_template=DS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"],
    )

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ds_feedback = {}

    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD

        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": scenarios[orig_id] if orig_id < len(scenarios) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
        }
        eval_results.append(eval_item)

        if not passed:
            failed_indices.append(orig_id)
            sc = scenarios[orig_id] if orig_id < len(scenarios) else {}
            key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
            ds_feedback[key] = {
                "previous_result": sc.get("damage_scenario", ""),
                "feedback": parsed.get("feedback", ""),
            }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ds_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            sc = scenarios[idx] if idx < len(scenarios) else {}
            eval_item = next((e for e in eval_results if e["id"] == idx), {})
            manual_review_items.append({
                "type": "damage_scenario",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": sc,
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 个损害场景需要人工审核")

    save_checkpoint(eval_results, "step2_ds_confidence_eval")
    return {
        "ds_confidence": {"all_passed": all_passed, "items": eval_results},
        "ds_retry_count": retry_count + 1,
        "ds_feedback": ds_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    ts_feedback = state.get("ts_feedback", {})
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    tasks = []
    for i, ds in enumerate(damage_scenarios):
        key = (ds.get("function", ""), ds.get("asset_name", ""), ds.get("security_attribute", ""))
        
        if ts_feedback and key in ts_feedback:
            feedback_info = ts_feedback[key]
            prompt = TS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                function_name=ds.get("function", ""),
                asset_name=ds.get("asset_name", ""),
                security_attribute=ds.get("security_attribute", ""),
                threat_type=ds.get("threat_type", ""),
                damage_scenario=ds.get("damage_scenario", ""),
                previous_result=feedback_info.get("previous_result", ""),
                feedback=feedback_info.get("feedback", ""),
                rag_context=rag_context,
            )
        else:
            prompt = TS_SINGLE_USER_TEMPLATE.format(
                function_name=ds.get("function", ""),
                asset_name=ds.get("asset_name", ""),
                security_attribute=ds.get("security_attribute", ""),
                threat_type=ds.get("threat_type", ""),
                damage_scenario=ds.get("damage_scenario", ""),
                rag_context=rag_context,
            )
        tasks.append({
            "id": i,
            "system": BASE_SYSTEM,
            "user": prompt,
            "_meta": {
                "function": ds.get("function", ""),
                "asset_name": ds.get("asset_name", ""),
                "asset_type": ds.get("asset_type", ""),
                "security_attribute": ds.get("security_attribute", ""),
                "threat_type": ds.get("threat_type", ""),
                "damage_scenario": ds.get("damage_scenario", ""),
            },
        })

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                ts_list.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    ts_list.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {"threat_scenarios": ts_list, "ts_feedback": {}}


def node_ts_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤3-评估] 威胁场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ts_confidence": {"all_passed": True, "items": []}, "ts_retry_count": 0, "ts_feedback": {}}

    ts_list = state.get("threat_scenarios", [])
    if not ts_list:
        return {"ts_confidence": {"all_passed": True, "items": []}, "ts_retry_count": 0, "ts_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=ts_list,
        eval_template=TS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"],
    )

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ts_feedback = {}

    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD

        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": ts_list[orig_id] if orig_id < len(ts_list) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
        }
        eval_results.append(eval_item)

        if not passed:
            failed_indices.append(orig_id)
            ts = ts_list[orig_id] if orig_id < len(ts_list) else {}
            key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
            ts_feedback[key] = {
                "previous_result": ts.get("threat_scenario", ""),
                "feedback": parsed.get("feedback", ""),
            }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ts_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            ts = ts_list[idx] if idx < len(ts_list) else {}
            eval_item = next((e for e in eval_results if e["id"] == idx), {})
            manual_review_items.append({
                "type": "threat_scenario",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": ts,
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 个威胁场景需要人工审核")

    save_checkpoint(eval_results, "step3_ts_confidence_eval")
    return {
        "ts_confidence": {"all_passed": all_passed, "items": eval_results},
        "ts_retry_count": retry_count + 1,
        "ts_feedback": ts_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发)")

    threat_scenarios = state.get("threat_scenarios", [])
    ap_feedback = state.get("ap_feedback", {})
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    tasks = []
    for i, ts in enumerate(threat_scenarios):
        key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
        
        if ap_feedback and key in ap_feedback:
            feedback_info = ap_feedback[key]
            prompt = AP_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                function_name=ts.get("function", ""),
                asset_name=ts.get("asset_name", ""),
                asset_type=ts.get("asset_type", ""),
                security_attribute=ts.get("security_attribute", ""),
                threat_scenario=ts.get("threat_scenario", ""),
                previous_result=feedback_info.get("previous_result", ""),
                feedback=feedback_info.get("feedback", ""),
                topo_info=topo_info,
                ext_info=ext_info,
                rag_context=rag_context,
            )
        else:
            prompt = AP_SINGLE_USER_TEMPLATE.format(
                function_name=ts.get("function", ""),
                asset_name=ts.get("asset_name", ""),
                asset_type=ts.get("asset_type", ""),
                security_attribute=ts.get("security_attribute", ""),
                threat_scenario=ts.get("threat_scenario", ""),
                topo_info=topo_info,
                ext_info=ext_info,
                rag_context=rag_context,
            )
        tasks.append({
            "id": i,
            "system": BASE_SYSTEM,
            "user": prompt,
            "_meta": {
                "function": ts.get("function", ""),
                "asset_name": ts.get("asset_name", ""),
                "asset_type": ts.get("asset_type", ""),
                "security_attribute": ts.get("security_attribute", ""),
                "threat_type": ts.get("threat_type", ""),
                "threat_scenario": ts.get("threat_scenario", ""),
                "damage_scenario": ts.get("damage_scenario", ""),
            },
        })

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    ap_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                ap_list.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    ap_list.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成攻击路径 {len(ap_list)}/{len(tasks)} 组")
    save_checkpoint(ap_list, "step4_attack_paths")
    return {"attack_paths": ap_list, "ap_feedback": {}}


def node_ap_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤4-评估] 攻击路径置信度评估 (逐条模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    ap_list = state.get("attack_paths", [])
    if not ap_list:
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False)[:2000] + "..."
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False)[:1000] + "..."

    flat_paths = []
    path_id = 0
    for ap_group in ap_list:
        meta = {
            "function": ap_group.get("function", ""),
            "asset_name": ap_group.get("asset_name", ""),
            "security_attribute": ap_group.get("security_attribute", ""),
            "threat_type": ap_group.get("threat_type", ""),
            "threat_scenario": ap_group.get("threat_scenario", ""),
            "damage_scenario": ap_group.get("damage_scenario", ""),
            "asset_type": ap_group.get("asset_type", ""),
        }
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "_id": path_id,
                "_meta": meta,
                "attack_path": path,
            })
            path_id += 1

    if not flat_paths:
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=flat_paths,
        eval_template=AP_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute"],
        topo_info=topo_info,
        ext_info=ext_info,
    )

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ap_feedback = {}

    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD

        orig_id = item["id"]
        path_data = flat_paths[orig_id] if orig_id < len(flat_paths) else {}

        eval_item = {
            "id": orig_id,
            "attack_path": path_data.get("attack_path", ""),
            "meta": path_data.get("_meta", {}),
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
        }
        eval_results.append(eval_item)

        if not passed:
            failed_indices.append(orig_id)
            meta = path_data.get("_meta", {})
            key = (meta.get("function", ""), meta.get("asset_name", ""), meta.get("security_attribute", ""))
            if key not in ap_feedback:
                ap_feedback[key] = {
                    "previous_result": [],
                    "feedback": "",
                }
            ap_feedback[key]["previous_result"].append(path_data.get("attack_path", ""))
            ap_feedback[key]["feedback"] += parsed.get("feedback", "") + "; "

    for key in ap_feedback:
        ap_feedback[key]["previous_result"] = json.dumps(ap_feedback[key]["previous_result"], ensure_ascii=False)
        ap_feedback[key]["feedback"] = ap_feedback[key]["feedback"].strip().rstrip(";")

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成：{passed_count}/{len(eval_results)} 条路径通过，阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ap_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            path_data = flat_paths[idx] if idx < len(flat_paths) else {}
            eval_item = next((e for e in eval_results if e["id"] == idx), {})
            manual_review_items.append({
                "type": "attack_path",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "attack_path": path_data.get("attack_path", ""),
                "meta": path_data.get("_meta", {}),
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 条攻击路径需要人工审核")

    save_checkpoint(eval_results, "step4_ap_confidence_eval")
    return {
        "ap_confidence": {"all_passed": all_passed, "items": eval_results},
        "ap_retry_count": retry_count + 1,
        "ap_feedback": ap_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
    }


def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        ds_info = json.dumps([
            {"function": d.get("function", ""), "asset_name": d.get("asset_name", ""),
             "security_attribute": d.get("security_attribute", ""),
             "damage_scenario": d.get("damage_scenario", "")}
            for d in batch
        ], ensure_ascii=False, indent=2)
        prompt = IR_BATCH_USER_TEMPLATE.format(ds_info=ds_info)
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            batch_idx = r["id"]
            batch_start = batch_idx * BATCH_SIZE_RATING
            batch_scenarios = ds_list[batch_start:batch_start + BATCH_SIZE_RATING]

            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    if idx < len(batch_scenarios) and isinstance(item, dict):
                        meta = {
                            "function": batch_scenarios[idx].get("function", ""),
                            "asset_name": batch_scenarios[idx].get("asset_name", ""),
                            "security_attribute": batch_scenarios[idx].get("security_attribute", ""),
                        }
                        ratings.append({**meta, **item})
            elif isinstance(parsed, dict):
                meta = {
                    "function": batch_scenarios[0].get("function", "") if batch_scenarios else "",
                    "asset_name": batch_scenarios[0].get("asset_name", "") if batch_scenarios else "",
                    "security_attribute": batch_scenarios[0].get("security_attribute", "") if batch_scenarios else "",
                }
                ratings.append({**meta, **parsed})
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败：{e}")

    for r in ratings:
        params = r.get("influence_parameters", {})
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        r["impact_value"] = str(impact_value)
        r["impact_level"] = impact_level

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (分批并发)")

    flat_paths = []
    for ap_group in state.get("attack_paths", []):
        for path in ap_group.get("attack_paths", []):
            flat_paths.append({
                "function": ap_group.get("function", ""),
                "asset_name": ap_group.get("asset_name", ""),
                "security_attribute": ap_group.get("security_attribute", ""),
                "attack_path": path,
            })

    if not flat_paths:
        return {"feasibility_ratings": []}

    batches = [flat_paths[i:i+BATCH_SIZE_RATING] for i in range(0, len(flat_paths), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(flat_paths)} 条攻击路径, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        prompt = AF_BATCH_USER_TEMPLATE.format(
            ap_info=json.dumps(batch, ensure_ascii=False, indent=2))
        tasks.append({"id": i, "system": AF_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                ratings.extend(parsed)
            else:
                ratings.append(parsed)
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")

    for r in ratings:
        params = r.get("attack_parameters", {})
        feasibility_score = calculate_feasibility_score(params)
        feasibility_level = calculate_feasibility_level(feasibility_score)
        r["feasibility_score"] = feasibility_score
        r["feasibility_level"] = feasibility_level

    logger.info(f"  → 完成可行性评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": ratings}


def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险判定 (交叉矩阵)")

    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    RISK_MATRIX = {
        ("Severe", "High"): 5, ("Severe", "Medium"): 4, ("Severe", "Low"): 3,
        ("Severe", "Very Low"): 2,
        ("Major", "High"): 4, ("Major", "Medium"): 3, ("Major", "Low"): 2,
        ("Major", "Very Low"): 1,
        ("Moderate", "High"): 3, ("Moderate", "Medium"): 2, ("Moderate", "Low"): 2,
        ("Moderate", "Very Low"): 1,
        ("Negligible", "High"): 1, ("Negligible", "Medium"): 1, ("Negligible", "Low"): 1,
        ("Negligible", "Very Low"): 1,
    }
    RISK_LABEL = {1: "1", 2: "2", 3: "3", 4: "4", 5: "5"}
    TREATMENT = {"1": "风险接受", "2": "风险接受", "3": "风险接受", "4": "风险缓解", "5": "风险缓解"}

    FEAS_ORDER = {"High": 4, "Medium": 3, "Low": 2, "Very Low": 1}
    feas_map: dict[tuple, str] = {}
    for fr in feasibility_ratings:
        key = (fr.get("function", ""), fr.get("asset_name", ""), fr.get("security_attribute", ""))
        level = fr.get("feasibility_level", "Very Low")
        if FEAS_ORDER.get(level, 1) > FEAS_ORDER.get(feas_map.get(key, "Very Low"), 1):
            feas_map[key] = level

    risk_results = []
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        impact_level = ir.get("impact_level", "Negligible")
        feasibility_level = feas_map.get(key, "Very Low")
        risk_val = RISK_MATRIX.get((impact_level, feasibility_level), 1)
        risk_label = RISK_LABEL.get(risk_val, "1")
        treatment = TREATMENT.get(risk_label, "风险接受")

        risk_results.append({
            "function": ir.get("function", ""),
            "asset_name": ir.get("asset_name", ""),
            "security_attribute": ir.get("security_attribute", ""),
            "impact_level": impact_level,
            "feasibility_level": feasibility_level,
            "risk_value": risk_label,
            "risk_treatment": treatment,
        })

    logger.info(f"  → 风险判定完成 {len(risk_results)} 条")
    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    rr_map = {_key(r): r for r in risk_results}

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for sa in asset.get("security_attributes", []):
                attr = sa["attribute_name"] if isinstance(sa, dict) else sa
                threat_type = sa.get("threat_type", ATTRIBUTE_TO_THREAT.get(attr, "未知")) if isinstance(sa, dict) else ATTRIBUTE_TO_THREAT.get(attr, "未知")
                key = (func_name, asset["asset_name"], attr)

                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    matched_rr = rr if not attack_list else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "ap_score": "",
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                        "risk_value": matched_rr.get("risk_value", "") if not attack_list else "",
                        "risk_treatment": matched_rr.get("risk_treatment", "") if not attack_list else "",
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "threat_type": threat_type,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "ds_score": "",
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "ts_score": "",
                    "attack": attack_list,
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")

    manual_items = state.get("manual_review_items", [])
    if manual_items:
        manual_report = _build_manual_review_report(manual_items, asset_results, ds_map, ts_map, ap_map, ir_map, fr_map, rr_map)
        manual_path = os.path.join(OUTPUT_DIR, "tara_manual_review.json")
        with open(manual_path, "w", encoding="utf-8") as f:
            json.dump(manual_report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 人工审核报告已保存: {manual_path}")

    return {"tara_report": report}


def _build_manual_review_report(
    manual_items: list,
    asset_results: list,
    ds_map: dict,
    ts_map: dict,
    ap_map: dict,
    ir_map: dict,
    fr_map: dict,
    rr_map: dict,
) -> list:
    flagged_keys = set()
    for item in manual_items:
        sc = item.get("scenario", {})
        meta = item.get("meta", {})
        if sc:
            key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
            flagged_keys.add(key)
        if meta:
            key = (meta.get("function", ""), meta.get("asset_name", ""), meta.get("security_attribute", ""))
            flagged_keys.add(key)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for sa in asset.get("security_attributes", []):
                attr = sa["attribute_name"] if isinstance(sa, dict) else sa
                threat_type = sa.get("threat_type", ATTRIBUTE_TO_THREAT.get(attr, "未知")) if isinstance(sa, dict) else ATTRIBUTE_TO_THREAT.get(attr, "未知")
                key = (func_name, asset["asset_name"], attr)

                if key not in flagged_keys:
                    continue

                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                rr = rr_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    attack_list.append({
                        "attack_path": path_desc,
                        "ap_score": "",
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                        "risk_value": rr.get("risk_value", "") if not attack_list else "",
                        "risk_treatment": rr.get("risk_treatment", "") if not attack_list else "",
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "threat_type": threat_type,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "ds_score": "",
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "ts_score": "",
                    "attack": attack_list,
                    "_review_flag": True,
                    "_review_feedback": "; ".join([
                        item.get("feedback", "") for item in manual_items
                        if (item.get("scenario", {}).get("function", ""), item.get("scenario", {}).get("asset_name", ""), item.get("scenario", {}).get("security_attribute", "")) == key
                        or (item.get("meta", {}).get("function", ""), item.get("meta", {}).get("asset_name", ""), item.get("meta", {}).get("security_attribute", "")) == key
                    ]),
                }
                asset_entry["security_attributes"].append(attr_entry)
            if asset_entry["security_attributes"]:
                func_entry["assets"].append(asset_entry)
        if func_entry["assets"]:
            report.append(func_entry)
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：工作流构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _should_retry_ds(state: TaraState) -> str:
    ds_conf = state.get("ds_confidence", {})
    retry_count = state.get("ds_retry_count", 0)
    if not ds_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ds"
    return "next_ts"

def _should_retry_ts(state: TaraState) -> str:
    ts_conf = state.get("ts_confidence", {})
    retry_count = state.get("ts_retry_count", 0)
    if not ts_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ts"
    return "next_ap"

def _should_retry_ap(state: TaraState) -> str:
    ap_conf = state.get("ap_confidence", {})
    retry_count = state.get("ap_retry_count", 0)
    if not ap_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ap"
    return "next_ir"


def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_confidence_eval",  node_ds_confidence_eval)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_confidence_eval",  node_ts_confidence_eval)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_confidence_eval",  node_ap_confidence_eval)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")
    builder.add_edge("node_ds_generate", "node_ds_confidence_eval")
    builder.add_conditional_edges("node_ds_confidence_eval", _should_retry_ds, {
        "retry_ds": "node_ds_generate",
        "next_ts": "node_ts_generate",
    })
    builder.add_edge("node_ts_generate", "node_ts_confidence_eval")
    builder.add_conditional_edges("node_ts_confidence_eval", _should_retry_ts, {
        "retry_ts": "node_ts_generate",
        "next_ap": "node_ap_generate",
    })
    builder.add_edge("node_ap_generate", "node_ap_confidence_eval")
    builder.add_conditional_edges("node_ap_confidence_eval", _should_retry_ap, {
        "retry_ap": "node_ap_generate",
        "next_ir": "node_impact_rating",
    })
    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 10 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_dir: str | None = None,
    topology_dir: str | None = None,
    external_interface_excel: str | None = None,
    asset_results: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")
    logger.info(f"  置信度阈值: CONFIDENCE_THRESHOLD={CONFIDENCE_THRESHOLD}")
    logger.info(f"  最大重试次数: MAX_CONFIDENCE_RETRIES={MAX_CONFIDENCE_RETRIES}")

    clear_prompt_cache()

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    if asset_results is None and data_flow_dir:
        logger.info("[输入解析] DCD数据流图")
        asset_results = parse_dcd_json(data_flow_dir)

    if topology_json is None and topology_dir:
        logger.info("[输入解析] 拓扑图")
        topology_json = parse_topology_json(topology_dir)

    if external_interface_json is None and external_interface_excel:
        logger.info("[输入解析] 外部接口信息")
        external_interface_json = parse_external_interface_excel(external_interface_excel)

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or [],
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_confidence": {}, "ds_retry_count": 0, "ds_feedback": {},
        "threat_scenarios": [], "ts_confidence": {}, "ts_retry_count": 0, "ts_feedback": {},
        "attack_paths": [],    "ap_confidence": {}, "ap_retry_count": 0, "ap_feedback": {},
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
        "manual_review_items": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
            "manual_review_items": final_state.get("manual_review_items", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    manual_items = final_state.get("manual_review_items", [])
    if manual_items:
        logger.warning("=" * 60)
        logger.warning(f"⚠️  共 {len(manual_items)} 个场景需要人工审核:")
        for item in manual_items:
            logger.warning(f"  - [{item['type']}] ID: {item['id']}, 置信度: {item['confidence_score']}")
        logger.warning("=" * 60)

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":
# 🔧 测试配置开关
    USE_SAMPLE_DATA = True  # 改为 False 则使用完整数据
    
    if USE_SAMPLE_DATA:
        # ===== 样本数据 =====
        sample_assets = [
            {
                "function": "交通标识及信号灯识别",
                "assets": [
                    {"asset_type": "信号", "asset_name": "弹窗/音效"},
                ],
            },
            {
                "function": "智能泊车辅助",
                "assets": [
                    {"asset_type": "数据", "asset_name": "用户账户信息"},
                ],
            },
        ]
    
        sample_topology = [{"id": 1, "CANFD": ["AMP", "CDC", "IC"]}, {"id": 2, "A2B": ["AMP", "CDC"]}, {"id": 3, "FPDLINK": ["CDC", "IC"]}, {"id": 4, "FPDLINK": ["CDC", "IVI"]}, {"id": 5, "GMSL": ["CDC", "DMS"]}, {"id": 6, "GMSL": ["CDC", "OMS"]}, {"id": 7, "GMSL": ["CDC", "MDC"]}, {"id": 8, "ETH": ["CDC", "CEM"]}, {"id": 9, "CANFD": ["BDCR", "CDC", "CEM", "REA"]}, {"id": 10, "CAN": ["BDCR", "BLE", "CEM", "CMSL", "CMSR", "DSM", "ECALL", "PDSM", "TRM", "WLCM"]}, {"id": 11, "CANFD": ["MDC", "USS ECU"]}, {"id": 12, "CANFD": ["FR", "MDC"]}, {"id": 13, "GMSL": ["LRC-FC", "MDC"]}, {"id": 14, "GMSL": ["MDC", "SRC-FC"]}, {"id": 15, "GMSL": ["MDC", "MRC-SC"]}, {"id": 16, "GMSL": ["MDC", "MRC-RC"]}, {"id": 17, "GMSL": ["FEC", "MDC"]}, {"id": 18, "ETH": ["CEM", "MDC"]}, {"id": 19, "CANFD": ["CEM", "EPS", "IBCU", "MDC", "SRS", "VCU", "VMC"]}, {"id": 20, "ETH": ["CEM", "VCU"]}, {"id": 21, "CANFD": ["BCU", "EVCC", "FMIPU", "HPMU", "ITMS", "PDU", "RDPEU", "RMIPU", "VCU"]}, {"id": 22, "CAN": ["BCU", "EVCC"]}, {"id": 23, "CAN": ["ACP", "ITMS", "PTC"]}, {"id": 24, "CANFD": ["BDCR", "CDC", "CEM", "LBMS"]}, {"id": 25, "ETH": ["Lidar", "MDC"]}, {"id": 26, "蜂窝": ["BCALL中心", "CDC", "ECALL中心", "华为车云", "阿维塔车云"]}, {"id": 27, "CAN": ["CEM", "诊断仪"]}]
    
        sample_ext_interfaces = [
            {"component": "CDC", "interfaces": ["Cellular Network  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
            {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
            {"component": "UWB", "interfaces": ["RF  射频"]},
            {"component": "CHLIL", "interfaces": ["JTAG"]},
            {"component": "CHLIR", "interfaces": ["JTAG"]},
            {"component": "ITMS", "interfaces": ["JTAG"]},
        ]
    
        report = run_tara(
            asset_results=sample_assets,
            topology_json=sample_topology,
            external_interface_json=sample_ext_interfaces,
            output_path=os.path.join(OUTPUT_DIR, "tara_report_sample.json"),  # 区分输出文件
            build_rag_index=True,  # 样本测试关闭 RAG 索引加速
        )
    else:
        # ===== 完整数据 =====
        report = run_tara(
            data_flow_dir=DFD_DIR,
            topology_dir=TOPOLOGY_DIR,
            external_interface_excel=EXTERNAL_INTERFACE_EXCEL,
            output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
            build_rag_index=True,
        )
    
    logger.info(f"📋 最终报告包含 {len(report)} 个功能模块，已保存为 JSON 文件")


2026-04-16 13:38:31,582 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-16 13:38:31,583 [INFO] ============================================================
2026-04-16 13:38:31,583 [INFO]   并发配置: MAX_WORKERS=5, BATCH_SIZE_RATING=5
2026-04-16 13:38:31,584 [INFO]   评估开关: ENABLE_EVALUATION=True
2026-04-16 13:38:31,584 [INFO]   置信度阈值: CONFIDENCE_THRESHOLD=85
2026-04-16 13:38:31,585 [INFO]   最大重试次数: MAX_CONFIDENCE_RETRIES=1
2026-04-16 13:38:31,620 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-16 13:38:31,621 [INFO] ============================================================
2026-04-16 13:38:31,623 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-16 13:38:31,623 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-16 13:38:31,625 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V8\checkpoint_step1_assets.json
2026-04-16 13:38:31,626 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-16 13:38:31,627 [INFO]   → 共 7 个子任务, 并发数 5
2026-04-16 13:38:32,969 [INFO] HTTP Request: POST https://api.siliconflow.cn/v1/chat/completions "HTTP/1.1 200 OK"
202

# V8

In [1]:
import json
import os
import re
import glob
import logging
import time
import hashlib
from typing import TypedDict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import pandas as pd
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

llm = ChatOpenAI(
    model="Qwen/Qwen3.5-4B",
    openai_api_key="sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk",
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.1,
    max_tokens=8192,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={"enable_thinking": False},
)

ENABLE_RAG = False
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

ENABLE_EVALUATION = True

MAX_WORKERS = 4
CALL_INTERVAL = 0.9
BATCH_SIZE_RATING = 5
EVAL_BATCH_SIZE = 3

OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V9"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}

CONFIDENCE_THRESHOLD = 85

CONFIDENCE_WEIGHTS = {
    "要素完整性": 0.30,
    "一致性": 0.30,
    "真实性": 0.30,
    "清晰度": 0.10,
}

MAX_CONFIDENCE_RETRIES = 1


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,
                request_timeout=45,
                max_retries=2,
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len,
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:
                valid_chunks.append(c)
            elif len(content) > 128:
                c.page_content = content[:127] + "..."
                valid_chunks.append(c)
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True,
            )
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs
            )
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: list
    external_interface_json: list
    asset_results: list
    damage_scenarios: list
    ds_confidence: dict
    ds_retry_count: int
    ds_feedback: dict
    threat_scenarios: list
    ts_confidence: dict
    ts_retry_count: int
    ts_feedback: dict
    attack_paths: list
    ap_confidence: dict
    ap_retry_count: int
    ap_feedback: dict
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list
    manual_review_items: list
    ds_score_map: dict      # {idx: confidence_score}
    ts_score_map: dict      # {idx: confidence_score}
    ap_score_map: dict      # {(ts_idx, path_idx): confidence_score}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()


def hash_prompt(text: str) -> str:
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def register_prompt_for_cache(prompt: str) -> str:
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h


def clear_prompt_cache():
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()


@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册")
        return "[]"
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


def smart_cache_key(template: str, variables: dict, key_fields: list[str] = None) -> str:
    if key_fields is None:
        key_fields = ["function", "asset_name", "security_attribute", "threat_type"]
    template_hash = hashlib.md5(template.encode('utf-8')).hexdigest()
    key_vars = {k: variables.get(k, "") for k in key_fields if k in variables}
    vars_str = json.dumps(key_vars, sort_keys=True, ensure_ascii=False)
    vars_hash = hashlib.md5(vars_str.encode('utf-8')).hexdigest()
    return f"{template_hash}:{vars_hash}"


def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    return "[]"


def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    results = []
    total = len(tasks)
    if total == 0:
        return results
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)
        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")
    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s")
    return results


def batch_confidence_eval(
    scenarios: list,
    eval_template: str,
    system_prompt: str,
    batch_size: int = EVAL_BATCH_SIZE,
    key_fields: list[str] = None,
    **template_vars,
) -> list[dict]:
    """批量置信度评估 —— 修复版：用 batch_start + idx 精确计算原始索引"""
    if not scenarios:
        return []
    batches = [scenarios[i:i+batch_size] for i in range(0, len(scenarios), batch_size)]
    logger.info(f"  → 批量评估: {len(scenarios)} 条场景, 分 {len(batches)} 批")

    tasks = []
    for batch_idx, batch in enumerate(batches):
        batch_json = json.dumps(batch, ensure_ascii=False, indent=2)
        format_vars = {"batch": batch_json, "batch_count": len(batch), **template_vars}
        user_prompt = eval_template.format(**format_vars)
        tasks.append({
            "id": batch_idx,
            "system": system_prompt,
            "user": user_prompt,
        })

    raw_results = concurrent_llm_calls(tasks)

    eval_results = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        batch_idx = r["id"]
        # ===== 核心修复：从 batch_idx 直接算出起始偏移 =====
        batch_start = batch_idx * batch_size
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    orig_idx = batch_start + idx          # ← 关键修复
                    if orig_idx >= len(scenarios):
                        logger.warning(f"  ⚠ 批次 {batch_idx} 返回项数({len(parsed)})超出预期，跳过 idx={idx}")
                        break
                    eval_results.append({
                        "id": orig_idx,
                        "raw_result": item,
                        "batch_id": batch_idx,
                    })
            elif isinstance(parsed, dict):
                eval_results.append({
                    "id": batch_start,
                    "raw_result": parsed,
                    "batch_id": batch_idx,
                })
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {batch_idx} 解析失败: {e}")

    eval_results.sort(key=lambda x: x["id"])
    return eval_results


def save_checkpoint(data: Any, step_name: str):
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Python计算函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def build_confidence_lookup(confidence_data: dict) -> dict:
    """
    将置信度评估结果构建为 {id: confidence_score} 的查找表。
    
    confidence_data 结构示例:
    {
        "all_passed": True,
        "items": [
            {"id": 0, "confidence_score": 92.5, "passed": True, ...},
            {"id": 1, "confidence_score": 78.3, "passed": False, ...},
        ]
    }
    """
    if not confidence_data or "items" not in confidence_data:
        return {}
    return {item["id"]: item.get("confidence_score", 0) for item in confidence_data["items"]}
    
def calculate_impact_value(influence_parameters: dict) -> int:
    if not influence_parameters:
        return 1
    max_val = max(int(influence_parameters.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
    return min(max(max_val, 1), 4)


def calculate_impact_level(impact_value: int) -> str:
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    return IMPACT_LABELS.get(impact_value, "Negligible")


def calculate_feasibility_score(attack_parameters: dict) -> int:
    if not attack_parameters:
        return 0
    return sum(int(attack_parameters.get(k, 0)) for k in [
        "Exposure_time", "Professional_experience",
        "Required_information", "Opportunity_window", "Required_equipment",
    ])


def calculate_feasibility_level(feasibility_score: int) -> str:
    if feasibility_score <= 13:
        return "High"
    elif feasibility_score <= 19:
        return "Medium"
    elif feasibility_score <= 24:
        return "Low"
    else:
        return "Very Low"


def calculate_confidence_score(scores: dict) -> float:
    total = 0.0
    for dimension, weight in CONFIDENCE_WEIGHTS.items():
        score = scores.get(dimension, 0)
        total += score * weight
    return round(total, 2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：输入解析函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_dcd_json(input_dir: str) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号",
        "tm.Process": "部件",
        "tm.Store": "数据",
        "tm.Actor": "接口",
    }
    all_results = []
    json_files = glob.glob(os.path.join(input_dir, '*.json'))
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                logger.warning(f"⚠️ DCD解析失败: {os.path.basename(file_path)}")
                continue
        current_result = {'function': '', 'assets': []}
        if 'detail' in data and 'diagrams' in data['detail']:
            for diagram in data['detail']['diagrams']:
                if not current_result['function'] and 'title' in diagram:
                    current_result['function'] = diagram['title']
                if 'diagramJson' in diagram and 'cells' in diagram['diagramJson']:
                    for cell in diagram['diagramJson']['cells']:
                        if cell.get('outOfScope') is True:
                            continue
                        raw_cell_type = cell.get('type', '')
                        if raw_cell_type == 'tm.Boundary':
                            continue
                        finetermval_value = ""
                        for key, value in cell.items():
                            if key.startswith('propertyList') and isinstance(value, dict):
                                if 'finetermval' in value:
                                    finetermval_value = value['finetermval']
                                    break
                        if not finetermval_value:
                            continue
                        mapped_cell_type = TYPE_MAPPING.get(raw_cell_type, raw_cell_type)
                        current_result['assets'].append({
                            'asset_type': mapped_cell_type,
                            'asset_name': finetermval_value,
                        })
        if not current_result['function']:
            current_result['function'] = os.path.splitext(os.path.basename(file_path))[0]
        if current_result['assets']:
            all_results.append(current_result)
    logger.info(f"  DCD解析完成: {len(all_results)} 个功能, 来自 {len(json_files)} 个文件")
    return all_results


def parse_topology_json(input_dir: str) -> dict:
    all_topo = {}
    json_files = glob.glob(os.path.join(input_dir, '*.json'))
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                logger.warning(f"⚠️ 拓扑图解析失败: {os.path.basename(file_path)}")
                continue
        name = os.path.splitext(os.path.basename(file_path))[0]
        if isinstance(data, dict) and '拓扑图' in data:
            all_topo[name] = data['拓扑图']
        elif isinstance(data, list):
            all_topo[name] = data
        elif isinstance(data, dict):
            all_topo[name] = data
    logger.info(f"  拓扑图解析完成: {len(all_topo)} 个文件")
    return all_topo


def parse_external_interface_excel(excel_path: str) -> list:
    if not os.path.exists(excel_path):
        logger.warning(f"⚠️ 外部接口Excel不存在: {excel_path}")
        return []
    df = pd.read_excel(excel_path, header=1)
    df.columns = [str(col).replace('\n', ' ').strip() for col in df.columns]
    interface_columns = df.columns[2:]
    results = []
    for index, row in df.iterrows():
        component_name = str(row['零部件名称']).strip()
        if pd.isna(row['零部件名称']) or not component_name:
            continue
        active_interfaces = []
        for col in interface_columns:
            cell_value = str(row[col]).strip()
            if '√' in cell_value:
                active_interfaces.append(col)
        if active_interfaces:
            results.append({
                "component": component_name,
                "interfaces": active_interfaces,
            })
    logger.info(f"  外部接口解析完成: {len(results)} 个零部件")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 。

请你先仔细分析{topo_info}和{ext_info}理解整车架构，基于此完成TARA任务。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，基于攻击树的方法推断
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是"涉及车辆或车辆功能且影响道路使用者的不良后果"。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）且生成损害场景必须结合车辆实际域（如ADAS、T-Box、IVI、BCM、车身控制、电池管理等）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"
- “车辆夜间行驶时，前照灯控制功能因‘前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”

## 输出（严格 JSON 对象，不要数组）
{{
  "damage_scenario": "完整的损害场景描述"
}}"""


DS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请为以下资产的指定安全属性，重新生成一条具体的损害场景。上一次生成未通过质量评估，请根据反馈改进。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 参考知识
{rag_context}

## 生成规则
1. 损害场景必须同时包含以下三个要素（缺一不可）且生成损害场景必须结合车辆实际域（如ADAS、T-Box、IVI、BCM、车身控制、电池管理等）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   - 必须明确提到被破坏的{asset_name}
2. 以{previous_result}为基础，根据{feedback}改进

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"
- “车辆夜间行驶时，前照灯控制功能因‘前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”


## 输出（严格 JSON 对象，不要包含输入字段）
{{
  "damage_scenario": "完整的损害场景描述"
}}"""


TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是"为实现损害场景，资产的信息安全属性遭到破坏的潜在原因"。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体「{asset_name}」
   - 「{asset_name}」被破坏的「{security_attribute}」
   - 「{security_attribute}」被破坏的原因/攻击意图
2. 威胁场景必须能直接导致给定的{damage_scenario}，描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条
3. {damage_scenario}与{asset_name}、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系息能被威胁场景包含或与威胁场景关联

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"
- "伪装信号导致发送至电源开关控制器的“灯光请求”信号的数据通信完整性丢失,可能造成前照灯意外关闭"

## 输出（严格 JSON 对象）
{{
  "threat_scenario": "详细的威胁场景描述"
}}"""


TS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请根据以下损害场景，重新生成一条详细的威胁场景描述。上一次生成未通过质量评估，请根据反馈改进。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体「{asset_name}」
   - 「{asset_name}」被破坏的「{security_attribute}」
   - 「{security_attribute}」被破坏的原因/攻击意图
2. 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条
3. {damage_scenario}与{asset_name}、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系息能被威胁场景包含或与威胁场景关联
4. 以{previous_result}为基础，根据{feedback}改进

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"
- "伪装信号导致发送至电源开关控制器的“灯光请求”信号的数据通信完整性丢失,可能造成前照灯意外关闭"

## 输出（严格 JSON 对象，不要包含输入字段）
{{
  "threat_scenario": "详细的威胁场景描述"
}}"""


AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图和外部接口信息生成最短的（经过车辆拓扑图中的控制器最少）但符合现实逻辑的攻击路径，如有多条最短路径请生成多条，最多不超过4条。攻击路径是“为实现威胁场景的一组蓄意活动”。必须采用攻击树分析，从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的{threat_scenario}，每条路径的技术手段必须针对{asset_type}的特点设计
2. 路径中控制器的连接关系必须与{topo_info}一致，外部接口与控制器的连接关系必须与{ext_info}一致，不得虚构不存在的控制器或跳过拓扑中的网关/ECU。{topo_info}中不同id的通信协议，代表不同的总线，输出时只输出通信协议总线名称不要输出id。{topo_info}中通信协议总线是键，值代表这些控制器都在一条总线上可以相互传递
3. 同一个{threat_scenario}的多条攻击路径不要重复，避免冗余
4. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。"
- "1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- "1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

## 输出（严格 JSON 对象）
{{
  "attack_paths": [
    "1.完整步骤链",
    "1.完整步骤链"
  ]
}}"""


AP_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请为以下威胁场景重新生成攻击路径。上一次生成未通过质量评估，请根据反馈改进。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的{threat_scenario}，每条路径的技术手段必须针对{asset_type}的特点设计
2. 路径中控制器的连接关系必须与{topo_info}一致，外部接口与控制器的连接关系必须与{ext_info}一致，不得虚构不存在的控制器或跳过拓扑中的网关/ECU。{topo_info}中不同id的通信协议，代表不同的总线，输出时只输出通信协议总线名称不要输出id。{topo_info}中通信协议总线是键，值代表这些控制器都在一条总线上可以相互传递
3. 同一个{threat_scenario}的多条攻击路径不要重复，避免冗余
4. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）
5. 以{previous_result}为基础，根据{feedback}改进

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。"
- "1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- "1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

## 输出（严格 JSON 对象）
{{
  "attack_paths": [
    "1.完整步骤链",
    "1.完整步骤链"
  ]
}}"""



IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数，必须逐字对照、严格遵守，不得主观臆断）
### Safety:
- 1 =没有受伤  
- 2 =轻度、中度伤害  
- 3 =严重的和有生命危险的伤害（可能生存）  
- 4 =威胁生命的伤害（不确定是否幸存），致命的伤害  
### Finance:
- 1 =经济损失导致的影响不大，后果可忽略不计，或与道路使用者无关  
- 2 =经济损失导致不便的后果，受影响的道路使用者将能用有限的资源来克服  
- 3 =导致经济上的大量损失，受影响的道路使用者将能够克服这些后果  
- 4 =经济损失导致的灾难性后果，受影响的道路使用者可能无法克服
利益相关者：车主、驾驶员、乘员、行人、供应商、主机厂
### Operation:
- 1 =操作上的损坏导致车辆功能没有损害或无法感知的损害  
- 2 =操作上的损坏导致了车辆功能的部分退化（例：用户满意度受到负面影响）  
- 3 =操作上的损坏导致了车辆重要功能的丧失或受损（例：司机的重大烦扰）  
- 4 =操作上的损坏导致了车辆核心功能的丧失或受损（例：车辆不工作或出现核心功能的意外行为，如启用跛行回家模式或自主驾驶到一个非预期的位置）
### Privacy:
- 1 =隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体  
- 2 =隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体；b)泄露的信息不敏感但很容易识别到PII主体  
- 3 =隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体；b)泄露的信息敏感而且很容易识别到PII主体  
- 4 =隐私侵犯会对道路使用者造成重大甚至不可逆转的影响。泄露的信息高度敏感，并且很容易识别到PII主体
考虑车主、驾驶员、乘员、行人、供应商、主机厂

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是整数 1~4

## 输出（严格 JSON 数组，只输出各维度参数）
[
  {{
    "influence_parameters": {{"Safety": 1, "Finance": 2, "Operation": 3, "Privacy": 4}}
  }}
]"""


AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准（必须逐字对照、严格遵守，不得主观臆断）
### Exposure_time: 
- 0=实现攻击行为的时间小于等于1天 
- 1=实现攻击行为的时间小于等于1周 
- 4=实现攻击行为的时间小于等于1个月  
- 17=实现攻击行为的时间小于等于6个月  
- 19=实现攻击行为的时间大于6个月
### Professional_experience: 
- 0=外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3=熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6=熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 9=一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0=公共信息（例如互联网上获得的信息）  
- 3=受限制的信息（制造商和供应商共享的内部文档）  
- 7=机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11=—严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0=十分高——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1=高——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4=中——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10=低—困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0=标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4=专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7=定制设备（厂家限制的工具、电子显微镜）  
- 9=多重定制设备，攻击不同步骤需要不同类型的定制设备

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数

## 输出（严格 JSON 数组，只输出各维度参数）
[
  {{
    "attack_parameters": {{
      "Exposure_time": 1, "Professional_experience": 3,
      "Required_information": 3, "Opportunity_window": 4, "Required_equipment": 9
    }}
  }}
]"""


CONFIDENCE_EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估场景质量并给出各维度的置信度评分。
仅输出 JSON 数组，不要任何其他文字。"""

DS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下损害场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
损害场景必须包含：功能与不良后果的因果关系、对道路使用者的具体危害、涉及的目标资产
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
损害场景描述是否与功能、资产、安全属性信息相符
评分标准：完全匹配100分，部分不匹配扣20-50分，严重不一致扣60-80分

### 3. 真实性（满分100分）
危害对道路使用者影响是否合理，是否有明显幻觉
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
句子是否简洁、无冗余、无语法错误
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过（"passed": false），列出具体问题"
  }}
]"""

TS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下威胁场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
威胁场景必须包含：目标资产、被破坏的信息安全属性、信息安全属性被破坏的原因
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
是否直接导致损害场景，因果关系是否清晰
评分标准：完全一致100分，部分不一致扣20-50分，因果关系断裂扣60-80分

### 3. 真实性（满分100分）
攻击意图是否现实，是否有明显幻觉
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
句子是否简洁、无冗余、无语法错误
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过（"passed": false），列出具体问题"
  }}
]"""

AP_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下攻击路径列表的质量，对**每条攻击路径**给出各维度的置信度评分（0-100 分）。

## 车辆拓扑图
{topo_info}
## 外部接口
{ext_info}

## 待评审攻击路径列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分 100 分）
攻击路径必须包含：起点（具体攻击面）、逻辑步骤、终点（目标资产）
评分标准：每缺失一个要素扣 30 分，要素不完整扣 10-20 分

### 2. 一致性（满分 100 分）
攻击路径是否能真正实现威胁场景，路径中的控制器及通信协议连接关系是否符合拓扑图，攻击入口是否存在于外部接口中
评分标准：完全一致 100 分，部分不一致扣 20-50 分，严重偏离扣 60-80 分

### 3. 真实性（满分 100 分）
攻击面是否存在，步骤是否可行，是否有明显幻觉
评分标准：完全可行 100 分，部分不可行扣 20-50 分，存在幻觉扣 60-80 分

### 4. 清晰度（满分 100 分）
句子是否简洁、无冗余、无语法错误，步骤描述是否清晰连贯
评分标准：清晰流畅 100 分，有冗余或小错误扣 10-30 分，表达混乱扣 40-60 分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true 或 false（总分>=85 为 true）,
    "feedback": "如不通过（"passed": true 或 false），列出具体问题"
  }}
]"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            security_attributes = []
            for attr in attrs:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                security_attributes.append({
                    "attribute_name": attr,
                    "threat_type": threat_type,
                })
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": security_attributes,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "manual_review_items": [],
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]
    ds_feedback = state.get("ds_feedback", {})
    retry_count = state.get("ds_retry_count", 0)
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for sa in asset["security_attributes"]:
                attr_name = sa["attribute_name"]
                key = (func["function"], asset["asset_name"], attr_name)
                
                if ds_feedback and key in ds_feedback:
                    feedback_info = ds_feedback[key]
                    prompt = DS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                        function_name=func["function"],
                        asset_name=asset["asset_name"],
                        asset_type=asset["asset_type"],
                        security_attribute=attr_name,
                        previous_result=feedback_info.get("previous_result", ""),
                        feedback=feedback_info.get("feedback", ""),
                        rag_context=rag_context,
                    )
                else:
                    prompt = DS_SINGLE_USER_TEMPLATE.format(
                        function_name=func["function"],
                        asset_name=asset["asset_name"],
                        asset_type=asset["asset_type"],
                        security_attribute=attr_name,
                        rag_context=rag_context,
                    )
                tasks.append({
                    "id": task_id,
                    "system": BASE_SYSTEM,
                    "user": prompt,
                    "_meta": {
                        "function": func["function"],
                        "asset_name": asset["asset_name"],
                        "asset_type": asset["asset_type"],
                        "security_attribute": attr_name,
                        "threat_type": sa["threat_type"],
                    },
                })
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                scenarios.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    scenarios.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {
        "damage_scenarios": scenarios, 
        "ds_feedback": {},
        "ds_retry_count": state.get("ds_retry_count", 0) + 1,
    }


def node_ds_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤2-评估] 损害场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ds_confidence": {"all_passed": True, "items": []}, "ds_retry_count": 0, "ds_feedback": {}}

    scenarios = state.get("damage_scenarios", [])
    if not scenarios:
        return {"ds_confidence": {"all_passed": True, "items": []}, "ds_retry_count": 0, "ds_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=scenarios,
        eval_template=DS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"],
    )

    # ===== 构建 id → raw_result 查找表，方便兜底处理 =====
    raw_lookup = {item["id"]: item.get("raw_result", {}) for item in eval_results_raw}

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ds_feedback = {}
    ds_score_map = {}

    for orig_id in range(len(scenarios)):
        sc = scenarios[orig_id]
        parsed = raw_lookup.get(orig_id, {})

        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        # 如果该 ID 没有返回评估结果（LLM漏返），视为不通过
        has_eval = orig_id in raw_lookup
        passed = confidence_score >= CONFIDENCE_THRESHOLD if has_eval else False

        eval_item = {
            "id": orig_id,
            "scenario": sc,
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", "") if has_eval else "评估结果缺失，需要人工审核",
        }
        eval_results.append(eval_item)

        key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
        ds_score_map[key] = confidence_score

        if not passed:
            failed_indices.append(orig_id)
            ds_feedback[key] = {
                "previous_result": sc.get("damage_scenario", ""),
                "feedback": eval_item["feedback"],
            }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ds_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            sc = scenarios[idx]
            eval_item = eval_results[idx]
            manual_review_items.append({
                "type": "damage_scenario",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": sc,
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 个损害场景需要人工审核")

    save_checkpoint(eval_results, "step2_ds_confidence_eval")
    return {
        "ds_confidence": {"all_passed": all_passed, "items": eval_results},
        "ds_retry_count": retry_count,
        "ds_feedback": ds_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
        "ds_score_map": ds_score_map,
    }


def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    ts_feedback = state.get("ts_feedback", {})
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    tasks = []
    for i, ds in enumerate(damage_scenarios):
        key = (ds.get("function", ""), ds.get("asset_name", ""), ds.get("security_attribute", ""))
        
        if ts_feedback and key in ts_feedback:
            feedback_info = ts_feedback[key]
            prompt = TS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                function_name=ds.get("function", ""),
                asset_name=ds.get("asset_name", ""),
                asset_type=ds.get("asset_type", ""),        # ← 补上这行
                security_attribute=ds.get("security_attribute", ""),
                threat_type=ds.get("threat_type", ""),
                damage_scenario=ds.get("damage_scenario", ""),
                previous_result=feedback_info.get("previous_result", ""),
                feedback=feedback_info.get("feedback", ""),
                rag_context=rag_context,
            )
        else:
            prompt = TS_SINGLE_USER_TEMPLATE.format(
                function_name=ds.get("function", ""),
                asset_name=ds.get("asset_name", ""),
                asset_type=ds.get("asset_type", ""),        # ← 补上这行
                security_attribute=ds.get("security_attribute", ""),
                threat_type=ds.get("threat_type", ""),
                damage_scenario=ds.get("damage_scenario", ""),
                rag_context=rag_context,
            )
        tasks.append({
            "id": i,
            "system": BASE_SYSTEM,
            "user": prompt,
            "_meta": {
                "function": ds.get("function", ""),
                "asset_name": ds.get("asset_name", ""),
                "asset_type": ds.get("asset_type", ""),
                "security_attribute": ds.get("security_attribute", ""),
                "threat_type": ds.get("threat_type", ""),
                "damage_scenario": ds.get("damage_scenario", ""),
            },
        })

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                ts_list.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    ts_list.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {
        "threat_scenarios": ts_list, 
        "ts_feedback": {},
        "ts_retry_count": state.get("ts_retry_count", 0) + 1,
    }


def node_ts_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤3-评估] 威胁场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ts_confidence": {"all_passed": True, "items": []}, "ts_retry_count": 0, "ts_feedback": {}}

    ts_list = state.get("threat_scenarios", [])
    if not ts_list:
        return {"ts_confidence": {"all_passed": True, "items": []}, "ts_retry_count": 0, "ts_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=ts_list,
        eval_template=TS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"],
    )

    raw_lookup = {item["id"]: item.get("raw_result", {}) for item in eval_results_raw}

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ts_feedback = {}
    ts_score_map = {}

    for orig_id in range(len(ts_list)):
        ts = ts_list[orig_id]
        parsed = raw_lookup.get(orig_id, {})

        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        has_eval = orig_id in raw_lookup
        passed = confidence_score >= CONFIDENCE_THRESHOLD if has_eval else False

        eval_item = {
            "id": orig_id,
            "scenario": ts,
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", "") if has_eval else "评估结果缺失，需要人工审核",
        }
        eval_results.append(eval_item)

        key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
        ts_score_map[key] = confidence_score

        if not passed:
            failed_indices.append(orig_id)
            ts_feedback[key] = {
                "previous_result": ts.get("threat_scenario", ""),
                "feedback": eval_item["feedback"],
            }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ts_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            ts = ts_list[idx]
            eval_item = eval_results[idx]
            manual_review_items.append({
                "type": "threat_scenario",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": ts,
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 个威胁场景需要人工审核")

    save_checkpoint(eval_results, "step3_ts_confidence_eval")
    return {
        "ts_confidence": {"all_passed": all_passed, "items": eval_results},
        "ts_retry_count": retry_count,
        "ts_feedback": ts_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
        "ts_score_map": ts_score_map,
    }


def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发)")

    threat_scenarios = state.get("threat_scenarios", [])
    ap_feedback = state.get("ap_feedback", {})
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    tasks = []
    for i, ts in enumerate(threat_scenarios):
        key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
        if ap_feedback and key in ap_feedback:
            feedback_info = ap_feedback[key]
            prompt = AP_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                function_name=ts.get("function", ""), asset_name=ts.get("asset_name", ""),
                asset_type=ts.get("asset_type", ""), security_attribute=ts.get("security_attribute", ""),
                threat_scenario=ts.get("threat_scenario", ""),
                previous_result=feedback_info.get("previous_result", ""),
                feedback=feedback_info.get("feedback", ""),
                topo_info=topo_info, ext_info=ext_info, rag_context=rag_context,
            )
        else:
            prompt = AP_SINGLE_USER_TEMPLATE.format(
                function_name=ts.get("function", ""), asset_name=ts.get("asset_name", ""),
                asset_type=ts.get("asset_type", ""), security_attribute=ts.get("security_attribute", ""),
                threat_scenario=ts.get("threat_scenario", ""),
                topo_info=topo_info, ext_info=ext_info, rag_context=rag_context,
            )
        tasks.append({
            "id": i, "system": BASE_SYSTEM, "user": prompt,
            "_meta": {
                "function": ts.get("function", ""), "asset_name": ts.get("asset_name", ""),
                "asset_type": ts.get("asset_type", ""), "security_attribute": ts.get("security_attribute", ""),
                "threat_type": ts.get("threat_type", ""), "threat_scenario": ts.get("threat_scenario", ""),
            },
        })

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    # 保持嵌套结构返回给 State
    attack_paths = []
    for r in raw_results:
        if r["raw"] is None:
            attack_paths.append({"attack_paths": []})
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict) and "attack_paths" in parsed:
                paths = parsed["attack_paths"]
            elif isinstance(parsed, list):
                paths = parsed
            else:
                paths = []
            attack_paths.append({**meta, "attack_paths": paths if isinstance(paths, list) else [str(paths)]})
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")
            attack_paths.append({"attack_paths": []})

    total_paths = sum(len(a.get("attack_paths", [])) for a in attack_paths)
    logger.info(f"  → 成功生成: {len(attack_paths)} 个威胁场景, 共 {total_paths} 条攻击路径")
    save_checkpoint(attack_paths, "step4_attack_paths")
    return {
        "attack_paths": attack_paths, 
        "ap_feedback": {},
        "ap_retry_count": state.get("ap_retry_count", 0) + 1,  # ← 加上这行
    }


def node_ap_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤4-评估] 攻击路径置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    nested_attack_paths = state.get("attack_paths", [])
    if not nested_attack_paths:
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)

    # ===== 扁平化，同时记录每条路径的精确坐标 =====
    flat_eval_data = []
    for ts_idx, ap in enumerate(nested_attack_paths):
        meta = {
            "function": ap.get("function", ""),
            "asset_name": ap.get("asset_name", ""),
            "asset_type": ap.get("asset_type", ""),
            "security_attribute": ap.get("security_attribute", ""),
            "threat_type": ap.get("threat_type", ""),
            "threat_scenario": ap.get("threat_scenario", ""),
        }
        for path_idx, path_desc in enumerate(ap.get("attack_paths", [])):
            flat_eval_data.append({
                "flat_idx": len(flat_eval_data),
                "ts_idx": ts_idx,
                "path_idx": path_idx,
                "path_desc": path_desc,
                "meta": meta,
            })

    if not flat_eval_data:
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    eval_scenarios = [
        {"威胁场景": item["meta"]["threat_scenario"], "攻击路径": item["path_desc"]}
        for item in flat_eval_data
    ]

    eval_results_raw = batch_confidence_eval(
        scenarios=eval_scenarios,
        eval_template=AP_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["threat_scenario", "attack_path"],
        topo_info=topo_info,
        ext_info=ext_info,
    )

    raw_lookup = {item["id"]: item.get("raw_result", {}) for item in eval_results_raw}

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ap_feedback = {}
    # ===== 关键修复：ap_score_map 使用 (ts_idx, path_idx) 精确定位 =====
    ap_score_map = {}

    for flat_idx in range(len(flat_eval_data)):
        fd = flat_eval_data[flat_idx]
        parsed = raw_lookup.get(flat_idx, {})

        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        has_eval = flat_idx in raw_lookup
        passed = confidence_score >= CONFIDENCE_THRESHOLD if has_eval else False

        eval_item = {
            "id": flat_idx,
            "ts_idx": fd["ts_idx"],
            "path_idx": fd["path_idx"],
            "scenario": fd["path_desc"],
            "meta": fd["meta"],
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", "") if has_eval else "评估结果缺失，需要人工审核",
        }
        eval_results.append(eval_item)

        # ===== 用 (ts_idx, path_idx) 作为 key =====
        ap_score_map[(fd["ts_idx"], fd["path_idx"])] = confidence_score

        if not passed:
            failed_indices.append(flat_idx)
            meta = fd["meta"]
            key = (meta.get("function", ""), meta.get("asset_name", ""), meta.get("security_attribute", ""))
            if key not in ap_feedback:
                ap_feedback[key] = {
                    "previous_result": fd["path_desc"],
                    "feedback": eval_item["feedback"],
                }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ap_retry_count", 0)
    if not all_passed:
        for idx in failed_indices:
            fd = flat_eval_data[idx]
            ev = eval_results[idx]
            manual_review_items.append({
                "type": "attack_path",
                "id": idx,
                "ts_idx": fd["ts_idx"],
                "path_idx": fd["path_idx"],
                "confidence_score": ev.get("confidence_score", 0),
                "feedback": ev.get("feedback", ""),
                "scenario": fd["meta"],
                "meta": fd["meta"],
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 条攻击路径需要人工审核")

    save_checkpoint(eval_results, "step4_ap_confidence_eval")
    return {
        "ap_confidence": {"all_passed": all_passed, "items": eval_results},
        "ap_retry_count": retry_count,
        "ap_feedback": ap_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
        "ap_score_map": ap_score_map,
    }


def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        ds_info = json.dumps([
            {"function": d.get("function", ""), "asset_name": d.get("asset_name", ""),
             "security_attribute": d.get("security_attribute", ""),
             "damage_scenario": d.get("damage_scenario", "")}
            for d in batch
        ], ensure_ascii=False, indent=2)
        prompt = IR_BATCH_USER_TEMPLATE.format(ds_info=ds_info)
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            batch_idx = r["id"]
            batch_start = batch_idx * BATCH_SIZE_RATING
            batch_scenarios = ds_list[batch_start:batch_start + BATCH_SIZE_RATING]

            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    if idx < len(batch_scenarios) and isinstance(item, dict):
                        meta = {
                            "function": batch_scenarios[idx].get("function", ""),
                            "asset_name": batch_scenarios[idx].get("asset_name", ""),
                            "security_attribute": batch_scenarios[idx].get("security_attribute", ""),
                        }
                        ratings.append({**meta, **item})
            elif isinstance(parsed, dict):
                meta = {
                    "function": batch_scenarios[0].get("function", "") if batch_scenarios else "",
                    "asset_name": batch_scenarios[0].get("asset_name", "") if batch_scenarios else "",
                    "security_attribute": batch_scenarios[0].get("security_attribute", "") if batch_scenarios else "",
                }
                ratings.append({**meta, **parsed})
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败：{e}")

    for r in ratings:
        params = r.get("influence_parameters", {})
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        r["impact_value"] = str(impact_value)
        r["impact_level"] = impact_level

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (批量模式)")

    nested_attack_paths = state.get("attack_paths", [])
    if not nested_attack_paths:
        return {"feasibility_ratings": []}

    # ===== 关键修复：在扁平化时携带完整元信息 =====
    flat_ap_info = []
    for ts_idx, ap in enumerate(nested_attack_paths):
        for path_idx, path_desc in enumerate(ap.get("attack_paths", [])):
            flat_ap_info.append({
                "威胁场景": ap.get("threat_scenario", ""),
                "攻击路径": path_desc,
                "_ts_idx": ts_idx,
                "_path_idx": path_idx,
                "_function": ap.get("function", ""),
                "_asset_name": ap.get("asset_name", ""),
                "_asset_type": ap.get("asset_type", ""),
                "_security_attribute": ap.get("security_attribute", ""),
                "_threat_type": ap.get("threat_type", ""),
            })

    if not flat_ap_info:
        return {"feasibility_ratings": []}

    ap_info_json = json.dumps(
        [{"威胁场景": x["威胁场景"], "攻击路径": x["攻击路径"]} for x in flat_ap_info],
        ensure_ascii=False, indent=2,
    )

    tasks = [{"id": 0, "system": AF_SYSTEM, "user": AF_BATCH_USER_TEMPLATE.format(ap_info=ap_info_json)}]
    raw_results = concurrent_llm_calls(tasks, max_workers=1)

    feasibility_ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    if idx >= len(flat_ap_info):
                        break
                    attack_params = item.get("attack_parameters", {})
                    score = calculate_feasibility_score(attack_params)
                    # ===== 关键修复：写入完整元信息 =====
                    feasibility_ratings.append({
                        "ts_idx": flat_ap_info[idx]["_ts_idx"],
                        "path_idx": flat_ap_info[idx]["_path_idx"],
                        "attack_path": flat_ap_info[idx]["攻击路径"],
                        "function": flat_ap_info[idx]["_function"],
                        "asset_name": flat_ap_info[idx]["_asset_name"],
                        "asset_type": flat_ap_info[idx]["_asset_type"],
                        "security_attribute": flat_ap_info[idx]["_security_attribute"],
                        "threat_type": flat_ap_info[idx]["_threat_type"],
                        "attack_parameters": attack_params,
                        "feasibility_score": score,
                        "feasibility_level": calculate_feasibility_level(score),
                    })
        except Exception as e:
            logger.warning(f"  ⚠ 可行性评级解析失败: {e}")

    logger.info(f"  → 成功评级 {len(feasibility_ratings)}/{len(flat_ap_info)} 条攻击路径")
    save_checkpoint(feasibility_ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": feasibility_ratings}


def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险值计算与处置决策")

    nested_attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    # 构建 feasibility 查找表 (ts_idx, path_idx) → fr
    feas_lookup = {(fr["ts_idx"], fr["path_idx"]): fr for fr in feasibility_ratings}

    # ===== 关键修复：改用 key-based 查找 impact，避免 index 错位 =====
    impact_lookup = {}
    for ir in impact_ratings:
        key = (ir.get("function", ""), ir.get("asset_name", ""), ir.get("security_attribute", ""))
        influence = ir.get("influence_parameters", {})
        impact_value = calculate_impact_value(influence)
        impact_lookup[key] = {
            "influence_parameters": influence,
            "impact_value": impact_value,
            "impact_level": calculate_impact_level(impact_value),
        }

    RISK_MATRIX = {
        ("Negligible", "High"): (1, "Low risk", "Accept"), ("Negligible", "Medium"): (1, "Low risk", "Accept"),
        ("Negligible", "Low"): (1, "Low risk", "Accept"), ("Negligible", "Very Low"): (1, "Low risk", "Accept"),
        ("Moderate", "High"): (2, "Medium risk", "Reduce"), ("Moderate", "Medium"): (1, "Low risk", "Accept"),
        ("Moderate", "Low"): (1, "Low risk", "Accept"), ("Moderate", "Very Low"): (1, "Low risk", "Accept"),
        ("Major", "High"): (3, "High risk", "Reduce"), ("Major", "Medium"): (2, "Medium risk", "Reduce"),
        ("Major", "Low"): (1, "Low risk", "Accept"), ("Major", "Very Low"): (1, "Low risk", "Accept"),
        ("Severe", "High"): (4, "Very High risk", "Reduce"), ("Severe", "Medium"): (3, "High risk", "Reduce"),
        ("Severe", "Low"): (2, "Medium risk", "Reduce"), ("Severe", "Very Low"): (2, "Medium risk", "Reduce"),
    }

    risk_results = []
    for ts_idx, ap in enumerate(nested_attack_paths):
        # ===== 关键修复：用 key 而非 ts_idx 查找 impact =====
        key = (ap.get("function", ""), ap.get("asset_name", ""), ap.get("security_attribute", ""))
        impact = impact_lookup.get(key, {
            "impact_value": 1, "impact_level": "Negligible", "influence_parameters": {},
        })

        for path_idx, path_desc in enumerate(ap.get("attack_paths", [])):
            feas = feas_lookup.get((ts_idx, path_idx), {
                "feasibility_score": 0, "feasibility_level": "Very Low", "attack_parameters": {},
            })

            matrix_key = (impact["impact_level"], feas["feasibility_level"])
            risk_value, risk_level, risk_treatment = RISK_MATRIX.get(matrix_key, (1, "Low risk", "Accept"))

            risk_results.append({
                "ts_idx": ts_idx,
                "path_idx": path_idx,
                "function": ap.get("function", ""),
                "asset_name": ap.get("asset_name", ""),
                "asset_type": ap.get("asset_type", ""),
                "security_attribute": ap.get("security_attribute", ""),
                "threat_type": ap.get("threat_type", ""),
                "threat_scenario": ap.get("threat_scenario", ""),
                "attack_path": path_desc,
                "influence_parameters": impact["influence_parameters"],
                "impact_value": impact["impact_value"],
                "impact_level": impact["impact_level"],
                "attack_parameters": feas.get("attack_parameters", {}),
                "feasibility_score": feas.get("feasibility_score", 0),
                "feasibility_level": feas.get("feasibility_level", "Very Low"),
                "risk_value": risk_value,
                "risk_level": risk_level,
                "risk_treatment": risk_treatment,
            })

    treatment_counts = {}
    for r in risk_results:
        t = r["risk_treatment"]
        treatment_counts[t] = treatment_counts.get(t, 0) + 1
    logger.info(f"  → 风险计算完成: {len(risk_results)} 条路径, 处置分布: {treatment_counts}")

    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    # ── 置信度分数 ──
    ds_score_map = state.get("ds_score_map", {})
    ts_score_map = state.get("ts_score_map", {})
    ap_score_map = state.get("ap_score_map", {})   # {(ts_idx, path_idx): score}

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    # ── key → index 映射（damage_scenarios, threat_scenarios, attack_paths 是平行列表） ──
    key_to_idx = {}
    for idx, ds in enumerate(damage_scenarios):
        key_to_idx[_key(ds)] = idx

    # ── 各类查找表 ──
    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}
    feas_lookup = {(fr["ts_idx"], fr["path_idx"]): fr for fr in feasibility_ratings}
    risk_lookup = {(rr["ts_idx"], rr["path_idx"]): rr for rr in risk_results}

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for sa in asset.get("security_attributes", []):
                attr = sa["attribute_name"] if isinstance(sa, dict) else sa
                threat_type = (
                    sa.get("threat_type", ATTRIBUTE_TO_THREAT.get(attr, "未知"))
                    if isinstance(sa, dict)
                    else ATTRIBUTE_TO_THREAT.get(attr, "未知")
                )
                key = (func_name, asset["asset_name"], attr)
                ts_idx = key_to_idx.get(key)

                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})

                ap_entry = (
                    attack_paths[ts_idx]
                    if ts_idx is not None and ts_idx < len(attack_paths)
                    else {}
                )
                paths = ap_entry.get("attack_paths", [])

                attack_list = []
                for path_idx, path_desc in enumerate(paths):
                    fr = feas_lookup.get((ts_idx, path_idx), {})
                    rr = risk_lookup.get((ts_idx, path_idx), {})
                    # ===== 关键修复：用 (ts_idx, path_idx) 精确取分 =====
                    ap_score = ap_score_map.get((ts_idx, path_idx), 0)

                    attack_list.append({
                        "attack_path": path_desc,
                        "ap_score": ap_score,
                        "attack_parameters": fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": fr.get("feasibility_score", 0),
                        "feasibility_level": fr.get("feasibility_level", "Very Low"),
                        "risk_value": rr.get("risk_value", ""),
                        "risk_treatment": rr.get("risk_treatment", ""),
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "threat_type": threat_type,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "ds_score": ds_score_map.get(key, 0),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "ts_score": ts_score_map.get(key, 0),
                    "attack": attack_list,
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")

    # ── 人工审核报告 ──
    manual_items = state.get("manual_review_items", [])
    if manual_items:
        manual_report = _build_manual_review_report(
            manual_items, asset_results, damage_scenarios,
            ds_map, ts_map, ir_map,
            key_to_idx, attack_paths, feas_lookup, risk_lookup,
            ds_score_map, ts_score_map, ap_score_map,
        )
        manual_path = os.path.join(OUTPUT_DIR, "tara_manual_review.json")
        with open(manual_path, "w", encoding="utf-8") as f:
            json.dump(manual_report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 人工审核报告已保存: {manual_path}")

    return {"tara_report": report}


def _build_manual_review_report(
    manual_items, asset_results, damage_scenarios,
    ds_map, ts_map, ir_map,
    key_to_idx, attack_paths, feas_lookup, risk_lookup,
    ds_score_map, ts_score_map, ap_score_map,
):
    flagged_keys = set()
    for item in manual_items:
        for src in [item.get("scenario", {}), item.get("meta", {})]:
            if src:
                key = (src.get("function", ""), src.get("asset_name", ""), src.get("security_attribute", ""))
                if any(key):
                    flagged_keys.add(key)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}
        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for sa in asset.get("security_attributes", []):
                attr = sa["attribute_name"] if isinstance(sa, dict) else sa
                threat_type = (
                    sa.get("threat_type", ATTRIBUTE_TO_THREAT.get(attr, "未知"))
                    if isinstance(sa, dict)
                    else ATTRIBUTE_TO_THREAT.get(attr, "未知")
                )
                key = (func_name, asset["asset_name"], attr)
                if key not in flagged_keys:
                    continue

                ts_idx = key_to_idx.get(key)
                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})

                ap_entry = (
                    attack_paths[ts_idx]
                    if ts_idx is not None and ts_idx < len(attack_paths)
                    else {}
                )
                paths = ap_entry.get("attack_paths", [])

                attack_list = []
                for path_idx, path_desc in enumerate(paths):
                    fr = feas_lookup.get((ts_idx, path_idx), {})
                    rr = risk_lookup.get((ts_idx, path_idx), {})
                    attack_list.append({
                        "attack_path": path_desc,
                        "ap_score": ap_score_map.get((ts_idx, path_idx), 0),
                        "attack_parameters": fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": fr.get("feasibility_score", 0),
                        "feasibility_level": fr.get("feasibility_level", "Very Low"),
                        "risk_value": rr.get("risk_value", ""),
                        "risk_treatment": rr.get("risk_treatment", ""),
                    })

                feedbacks = []
                for item in manual_items:
                    for src in [item.get("scenario", {}), item.get("meta", {})]:
                        ik = (src.get("function", ""), src.get("asset_name", ""), src.get("security_attribute", ""))
                        if ik == key:
                            fb = item.get("feedback", "")
                            if fb:
                                feedbacks.append(fb)
                            break

                attr_entry = {
                    "attribute_name": attr,
                    "threat_type": threat_type,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "ds_score": ds_score_map.get(key, 0),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "ts_score": ts_score_map.get(key, 0),
                    "attack": attack_list,
                    "_review_flag": True,
                    "_review_feedback": "; ".join(dict.fromkeys(feedbacks)),
                }
                asset_entry["security_attributes"].append(attr_entry)
            if asset_entry["security_attributes"]:
                func_entry["assets"].append(asset_entry)
        if func_entry["assets"]:
            report.append(func_entry)
    return report
    
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：工作流构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _should_retry_ds(state: TaraState) -> str:
    ds_conf = state.get("ds_confidence", {})
    retry_count = state.get("ds_retry_count", 0)
    if not ds_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ds"
    return "next_ts"

def _should_retry_ts(state: TaraState) -> str:
    ts_conf = state.get("ts_confidence", {})
    retry_count = state.get("ts_retry_count", 0)
    if not ts_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ts"
    return "next_ap"

def _should_retry_ap(state: TaraState) -> str:
    ap_conf = state.get("ap_confidence", {})
    retry_count = state.get("ap_retry_count", 0)
    if not ap_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ap"
    return "next_ir"


def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_confidence_eval",  node_ds_confidence_eval)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_confidence_eval",  node_ts_confidence_eval)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_confidence_eval",  node_ap_confidence_eval)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")
    builder.add_edge("node_ds_generate", "node_ds_confidence_eval")
    builder.add_conditional_edges("node_ds_confidence_eval", _should_retry_ds, {
        "retry_ds": "node_ds_generate",
        "next_ts": "node_ts_generate",
    })
    builder.add_edge("node_ts_generate", "node_ts_confidence_eval")
    builder.add_conditional_edges("node_ts_confidence_eval", _should_retry_ts, {
        "retry_ts": "node_ts_generate",
        "next_ap": "node_ap_generate",
    })
    builder.add_edge("node_ap_generate", "node_ap_confidence_eval")
    builder.add_conditional_edges("node_ap_confidence_eval", _should_retry_ap, {
        "retry_ap": "node_ap_generate",
        "next_ir": "node_impact_rating",
    })
    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 10 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_dir: str | None = None,
    topology_dir: str | None = None,
    external_interface_excel: str | None = None,
    asset_results: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")
    logger.info(f"  置信度阈值: CONFIDENCE_THRESHOLD={CONFIDENCE_THRESHOLD}")
    logger.info(f"  最大重试次数: MAX_CONFIDENCE_RETRIES={MAX_CONFIDENCE_RETRIES}")

    clear_prompt_cache()

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    if asset_results is None and data_flow_dir:
        logger.info("[输入解析] DCD数据流图")
        asset_results = parse_dcd_json(data_flow_dir)

    if topology_json is None and topology_dir:
        logger.info("[输入解析] 拓扑图")
        topology_json = parse_topology_json(topology_dir)

        logger.info("[输入解析] 外部接口信息")
        external_interface_json = parse_external_interface_excel(external_interface_excel)

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or [],
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_confidence": {}, "ds_retry_count": 0, "ds_feedback": {},
        "threat_scenarios": [], "ts_confidence": {}, "ts_retry_count": 0, "ts_feedback": {},
        "attack_paths": [],    "ap_confidence": {}, "ap_retry_count": 0, "ap_feedback": {},
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
        "manual_review_items": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
            "manual_review_items": final_state.get("manual_review_items", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    manual_items = final_state.get("manual_review_items", [])
    if manual_items:
        logger.warning("=" * 60)
        logger.warning(f"⚠️  共 {len(manual_items)} 个场景需要人工审核:")
        for item in manual_items:
            logger.warning(f"  - [{item['type']}] ID: {item['id']}, 置信度: {item['confidence_score']}")
        logger.warning("=" * 60)

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":
# 🔧 测试配置开关
    USE_SAMPLE_DATA = True  # 改为 False 则使用完整数据
    
    if USE_SAMPLE_DATA:
        # ===== 样本数据 =====
        sample_assets = [
            {
                "function": "交通标识及信号灯识别",
                "assets": [
                    {"asset_type": "信号", "asset_name": "弹窗/音效"},
                ],
            },
            {
                "function": "智能泊车辅助",
                "assets": [
                    {"asset_type": "数据", "asset_name": "用户账户信息"},
                ],
            },
        ]
    
        sample_topology = [{"id": 1, "CANFD": ["AMP", "CDC", "IC"]}, {"id": 2, "A2B": ["AMP", "CDC"]}, {"id": 3, "FPDLINK": ["CDC", "IC"]}, {"id": 4, "FPDLINK": ["CDC", "IVI"]}, {"id": 5, "GMSL": ["CDC", "DMS"]}, {"id": 6, "GMSL": ["CDC", "OMS"]}, {"id": 7, "GMSL": ["CDC", "MDC"]}, {"id": 8, "ETH": ["CDC", "CEM"]}, {"id": 9, "CANFD": ["BDCR", "CDC", "CEM", "REA"]}, {"id": 10, "CAN": ["BDCR", "BLE", "CEM", "CMSL", "CMSR", "DSM", "ECALL", "PDSM", "TRM", "WLCM"]}, {"id": 11, "CANFD": ["MDC", "USS ECU"]}, {"id": 12, "CANFD": ["FR", "MDC"]}, {"id": 13, "GMSL": ["LRC-FC", "MDC"]}, {"id": 14, "GMSL": ["MDC", "SRC-FC"]}, {"id": 15, "GMSL": ["MDC", "MRC-SC"]}, {"id": 16, "GMSL": ["MDC", "MRC-RC"]}, {"id": 17, "GMSL": ["FEC", "MDC"]}, {"id": 18, "ETH": ["CEM", "MDC"]}, {"id": 19, "CANFD": ["CEM", "EPS", "IBCU", "MDC", "SRS", "VCU", "VMC"]}, {"id": 20, "ETH": ["CEM", "VCU"]}, {"id": 21, "CANFD": ["BCU", "EVCC", "FMIPU", "HPMU", "ITMS", "PDU", "RDPEU", "RMIPU", "VCU"]}, {"id": 22, "CAN": ["BCU", "EVCC"]}, {"id": 23, "CAN": ["ACP", "ITMS", "PTC"]}, {"id": 24, "CANFD": ["BDCR", "CDC", "CEM", "LBMS"]}, {"id": 25, "ETH": ["Lidar", "MDC"]}, {"id": 26, "蜂窝": ["BCALL中心", "CDC", "ECALL中心", "华为车云", "阿维塔车云"]}, {"id": 27, "CAN": ["CEM", "诊断仪"]}]
    
        sample_ext_interfaces = [
            {"component": "CDC", "interfaces": ["Cellular Network  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
            {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
            {"component": "UWB", "interfaces": ["RF  射频"]},
            {"component": "CHLIL", "interfaces": ["JTAG"]},
            {"component": "CHLIR", "interfaces": ["JTAG"]},
            {"component": "ITMS", "interfaces": ["JTAG"]},
        ]
    
        report = run_tara(
            asset_results=sample_assets,
            topology_json=sample_topology,
            external_interface_json=sample_ext_interfaces,
            output_path=os.path.join(OUTPUT_DIR, "tara_report_sample.json"),  # 区分输出文件
            build_rag_index=False,  # 样本测试关闭 RAG 索引加速
        )
    else:
        # ===== 完整数据 =====
        report = run_tara(
            data_flow_dir=DFD_DIR,
            topology_dir=TOPOLOGY_DIR,
            external_interface_excel=EXTERNAL_INTERFACE_EXCEL,
            output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
            build_rag_index=True,
        )
    
    logger.info(f"📋 最终报告包含 {len(report)} 个功能模块，已保存为 JSON 文件")


2026-04-20 09:24:19,331 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-20 09:24:19,332 [INFO] ============================================================
2026-04-20 09:24:19,333 [INFO]   并发配置: MAX_WORKERS=4, BATCH_SIZE_RATING=5
2026-04-20 09:24:19,333 [INFO]   评估开关: ENABLE_EVALUATION=True
2026-04-20 09:24:19,334 [INFO]   置信度阈值: CONFIDENCE_THRESHOLD=85
2026-04-20 09:24:19,334 [INFO]   最大重试次数: MAX_CONFIDENCE_RETRIES=1
2026-04-20 09:24:19,335 [INFO] [初始化] RAG 已禁用，跳过
2026-04-20 09:24:19,377 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-20 09:24:19,386 [INFO] ============================================================
2026-04-20 09:24:19,386 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-20 09:24:19,387 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-20 09:24:19,389 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V9\checkpoint_step1_assets.json
2026-04-20 09:24:19,390 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-20 09:24:19,390 [INFO]   → 共 7 个子任务, 并发数 4
2026-04-20 09:24:23,809 [INFO] HTTP Request: POST https://api.siliconf

In [ ]:
import json
import os
import re
import glob
import logging
import time
import hashlib
from typing import TypedDict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import lru_cache

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import pandas as pd
from requests.exceptions import ConnectionError, Timeout, HTTPError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 1 部分：全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

llm = ChatOpenAI(
    model="qwen3.5-27b",
    openai_api_key="sk-3458388d4845414387fe2e10bc8a9ee2",
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.1,
    max_tokens=4096,
    request_timeout=60,
    max_retries=3,
    stop=None,
    extra_body={"enable_thinking": False},
)

ENABLE_RAG = True
EMBEDDING_MODEL = "BAAI/bge-large-zh-v1.5"
EMBEDDING_API_KEY = "sk-wcdlrybthwbmajdynzuemqfdglxamdipniczgyllqwlqdymk"
EMBEDDING_API_BASE = "https://api.siliconflow.cn/v1"

RAG_BASE_DIR = r'D:\Jupyter profile\rag'
RAG_DIRS = {
    "tara_reports":     os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations":      os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
VECTOR_STORE_PATH = os.path.join(RAG_BASE_DIR, "vector_store")

ENABLE_EVALUATION = True

MAX_WORKERS = 5
CALL_INTERVAL = 0.8
BATCH_SIZE_RATING = 5
EVAL_BATCH_SIZE = 3

OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V8"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性":     "篡改",
    "机密性":     "信息泄露",
    "可用性":     "拒绝服务",
    "真实性":     "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性":   "越权",
}

CONFIDENCE_THRESHOLD = 85

CONFIDENCE_WEIGHTS = {
    "要素完整性": 0.30,
    "一致性": 0.30,
    "真实性": 0.30,
    "清晰度": 0.10,
}

MAX_CONFIDENCE_RETRIES = 1


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 2 部分：RAG 知识库
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class RAGKnowledgeBase:
    def __init__(self):
        self.vectorstore = None
        try:
            self.embeddings = OpenAIEmbeddings(
                model=EMBEDDING_MODEL,
                openai_api_key=EMBEDDING_API_KEY,
                openai_api_base=EMBEDDING_API_BASE,
                check_embedding_ctx_length=False,
                tiktoken_enabled=False,
                chunk_size=8,
                request_timeout=45,
                max_retries=2,
            )
        except Exception as e:
            logger.warning(f"  Embedding 初始化失败: {e}")
            self.embeddings = None

    def build_index(self, force_rebuild: bool = False):
        if not force_rebuild and os.path.exists(VECTOR_STORE_PATH):
            return self.load_index()
        if self.embeddings is None:
            return False
        documents = []
        for category, dir_path in RAG_DIRS.items():
            os.makedirs(dir_path, exist_ok=True)
            for filepath in glob.glob(os.path.join(dir_path, "**/*"), recursive=True):
                if not os.path.isfile(filepath):
                    continue
                try:
                    docs = self._load_file(filepath, category)
                    documents.extend(docs)
                except Exception as e:
                    logger.warning(f"    ✗ 加载失败 {os.path.basename(filepath)}: {e}")
        if not documents:
            return False
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=127,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
            length_function=len,
        )
        chunks = splitter.split_documents(documents)
        valid_chunks = []
        for c in chunks:
            content = c.page_content.strip()
            if 20 <= len(content) <= 128:
                valid_chunks.append(c)
            elif len(content) > 128:
                c.page_content = content[:127] + "..."
                valid_chunks.append(c)
        logger.info(f"✅ 过滤后有效 chunks: {len(valid_chunks)}/{len(chunks)}")
        self.vectorstore = FAISS.from_documents(valid_chunks, self.embeddings)
        os.makedirs(VECTOR_STORE_PATH, exist_ok=True)
        self.vectorstore.save_local(VECTOR_STORE_PATH)
        return True

    def load_index(self) -> bool:
        if self.embeddings is None:
            return False
        if os.path.exists(os.path.join(VECTOR_STORE_PATH, "index.faiss")):
            self.vectorstore = FAISS.load_local(
                VECTOR_STORE_PATH, self.embeddings, allow_dangerous_deserialization=True,
            )
            return True
        return False

    def retrieve(self, query: str, k: int = 5) -> str:
        if self.vectorstore is None:
            if not self.load_index():
                return "[知识库未就绪]"
        try:
            docs = self.vectorstore.similarity_search(query, k=k)
            if not docs:
                return "[未检索到相关参考信息]"
            return "\n---\n".join(
                f"[来源: {d.metadata.get('category', '未知')}] {d.page_content}" for d in docs
            )
        except Exception as e:
            return "[检索异常]"

    def _load_file(self, filepath: str, category: str) -> list[Document]:
        ext = os.path.splitext(filepath)[1].lower()
        metadata = {"source": filepath, "category": category, "filename": os.path.basename(filepath)}
        if ext == ".json":
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            documents = []
            if isinstance(data, list):
                for i, item in enumerate(data):
                    item_text = f"[记录 {i+1}]\n{json.dumps(item, ensure_ascii=False)}"
                    if len(item_text.strip()) >= 15:
                        documents.append(Document(page_content=item_text, metadata={**metadata, "record_idx": i}))
            elif isinstance(data, dict):
                for k, v in data.items():
                    txt = f"[{k}]\n{json.dumps(v, ensure_ascii=False)}"
                    if len(txt.strip()) >= 15:
                        documents.append(Document(page_content=txt, metadata={**metadata, "json_key": k}))
            return documents
        elif ext in (".txt", ".md"):
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()
            return [Document(page_content=text, metadata=metadata)]
        return []


rag_kb = RAGKnowledgeBase()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 3 部分：State 定义
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TaraState(TypedDict):
    data_flow_json: list
    topology_json: list
    external_interface_json: list
    asset_results: list
    damage_scenarios: list
    ds_confidence: dict
    ds_retry_count: int
    ds_feedback: dict
    threat_scenarios: list
    ts_confidence: dict
    ts_retry_count: int
    ts_feedback: dict
    attack_paths: list
    ap_confidence: dict
    ap_retry_count: int
    ap_feedback: dict
    impact_ratings: list
    feasibility_ratings: list
    risk_results: list
    tara_report: list
    manual_review_items: list
    ds_score_map: dict      # {idx: confidence_score}
    ts_score_map: dict      # {idx: confidence_score}
    ap_score_map: dict      # {(ts_idx, path_idx): confidence_score}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 4 部分：工具函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: str) -> Any:
    if text is None:
        raise ValueError("输入文本为 None")
    if not isinstance(text, str):
        text = str(text)
    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()
    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"无法解析 JSON，原始输出前200字: {text[:200]}")


def llm_call(system_prompt: str, user_prompt: str) -> str:
    try:
        resp = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        return resp.content
    except Exception as e:
        logger.error(f"  LLM 调用失败: {e}")
        return "[]"


hash_to_prompt: dict[str, str] = {}
_prompt_cache_lock = Lock()


def hash_prompt(text: str) -> str:
    if not text:
        return hashlib.md5(b"").hexdigest()
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def register_prompt_for_cache(prompt: str) -> str:
    h = hash_prompt(prompt)
    with _prompt_cache_lock:
        hash_to_prompt[h] = prompt
    return h


def clear_prompt_cache():
    with _prompt_cache_lock:
        hash_to_prompt.clear()
    llm_call_cached.cache_clear()


@lru_cache(maxsize=200)
def llm_call_cached(system_hash: str, user_hash: str) -> str:
    with _prompt_cache_lock:
        system_prompt = hash_to_prompt.get(system_hash, "")
        user_prompt = hash_to_prompt.get(user_hash, "")
    if not system_prompt or not user_prompt:
        logger.warning(f"  ⚠ 缓存调用失败: hash 未注册")
        return "[]"
    return llm_call_with_retry(system_prompt=system_prompt, user_prompt=user_prompt)


def smart_cache_key(template: str, variables: dict, key_fields: list[str] = None) -> str:
    if key_fields is None:
        key_fields = ["function", "asset_name", "security_attribute", "threat_type"]
    template_hash = hashlib.md5(template.encode('utf-8')).hexdigest()
    key_vars = {k: variables.get(k, "") for k in key_fields if k in variables}
    vars_str = json.dumps(key_vars, sort_keys=True, ensure_ascii=False)
    vars_hash = hashlib.md5(vars_str.encode('utf-8')).hexdigest()
    return f"{template_hash}:{vars_hash}"


def llm_call_with_retry(system_prompt: str, user_prompt: str, max_retries: int = 2) -> str:
    for attempt in range(max_retries + 1):
        try:
            resp = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            if hasattr(resp, 'content'):
                content = resp.content
                if hasattr(content, 'model_dump'):
                    return json.dumps(content.model_dump(), ensure_ascii=False)
                return str(content) if content is not None else ""
            elif isinstance(resp, str):
                return resp
            else:
                return str(resp)
        except (ConnectionError, Timeout, HTTPError) as e:
            logger.error(f"  ❌ 网络错误: {type(e).__name__}: {e}")
        except AttributeError as e:
            if 'model_dump' in str(e):
                logger.error(f"  ❌ 版本兼容错误: {e}")
            else:
                logger.error(f"  ❌ 属性错误: {e}")
        except Exception as e:
            logger.error(f"  ❌ 未知错误: {type(e).__name__}: {e}")
        if attempt < max_retries:
            wait = 2 ** attempt * 2
            logger.warning(f"  🔄 重试 {attempt+1}/{max_retries}，{wait}s 后...")
            time.sleep(wait)
    return "[]"


def concurrent_llm_calls(tasks: list[dict], max_workers: int = MAX_WORKERS) -> list[dict]:
    results = []
    total = len(tasks)
    if total == 0:
        return results
    start = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {}
        for i, task in enumerate(tasks):
            future = executor.submit(llm_call_with_retry, task["system"], task["user"])
            future_map[future] = task
            if i < total - 1:
                time.sleep(CALL_INTERVAL)
        done_count = 0
        for future in as_completed(future_map):
            task = future_map[future]
            done_count += 1
            try:
                raw = future.result()
                results.append({"id": task["id"], "raw": raw})
                logger.info(f"    ✅ [{done_count}/{total}] 任务 {task['id']} 完成")
            except Exception as e:
                results.append({"id": task["id"], "raw": None})
                logger.warning(f"    ❌ [{done_count}/{total}] 任务 {task['id']} 失败: {e}")
    results.sort(key=lambda x: x["id"])
    elapsed = time.time() - start
    logger.info(f"  ⏱ 并发调用完成: {len(tasks)} 任务, 耗时 {elapsed:.1f}s")
    return results


def batch_confidence_eval(
    scenarios: list,
    eval_template: str,
    system_prompt: str,
    batch_size: int = EVAL_BATCH_SIZE,
    key_fields: list[str] = None,
    **template_vars,
) -> list[dict]:
    if not scenarios:
        return []
    batches = [scenarios[i:i+batch_size] for i in range(0, len(scenarios), batch_size)]
    logger.info(f"  → 批量评估: {len(scenarios)} 条场景, 分 {len(batches)} 批")
    tasks = []
    for batch_idx, batch in enumerate(batches):
        batch_json = json.dumps(batch, ensure_ascii=False, indent=2)
        format_vars = {"batch": batch_json, "batch_count": len(batch), **template_vars}
        user_prompt = eval_template.format(**format_vars)
        cache_vars = {"batch_size": len(batch), **{k: v for k, v in template_vars.items() if k in (key_fields or [])}}
        cache_key = smart_cache_key(eval_template, cache_vars, key_fields)
        system_hash = register_prompt_for_cache(system_prompt)
        user_hash = register_prompt_for_cache(user_prompt)
        tasks.append({
            "id": batch_idx,
            "system": system_prompt,
            "user": user_prompt,
            "_cache_key": cache_key,
            "_batch_indices": list(range(batch_idx * batch_size, min((batch_idx + 1) * batch_size, len(scenarios)))),
        })
    raw_results = concurrent_llm_calls(tasks)
    eval_results = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    orig_idx = r.get("_batch_indices", [])[idx] if "_batch_indices" in r and idx < len(r["_batch_indices"]) else len(eval_results) + idx
                    eval_results.append({
                        "id": orig_idx,
                        "raw_result": item,
                        "batch_id": r["id"],
                        "cache_key": r.get("_cache_key", ""),
                    })
            else:
                eval_results.append({
                    "id": r["id"],
                    "raw_result": parsed,
                    "batch_id": r["id"],
                    "cache_key": r.get("_cache_key", ""),
                })
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败: {e}")
    eval_results.sort(key=lambda x: x["id"])
    return eval_results


def save_checkpoint(data: Any, step_name: str):
    path = os.path.join(OUTPUT_DIR, f"checkpoint_{step_name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"  💾 Checkpoint 已保存: {path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 5 部分：Python计算函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def build_confidence_lookup(confidence_data: dict) -> dict:
    """
    将置信度评估结果构建为 {id: confidence_score} 的查找表。
    
    confidence_data 结构示例:
    {
        "all_passed": True,
        "items": [
            {"id": 0, "confidence_score": 92.5, "passed": True, ...},
            {"id": 1, "confidence_score": 78.3, "passed": False, ...},
        ]
    }
    """
    if not confidence_data or "items" not in confidence_data:
        return {}
    return {item["id"]: item.get("confidence_score", 0) for item in confidence_data["items"]}
    
def calculate_impact_value(influence_parameters: dict) -> int:
    if not influence_parameters:
        return 1
    max_val = max(int(influence_parameters.get(k, 0)) for k in ["Safety", "Finance", "Operation", "Privacy"])
    return min(max(max_val, 1), 4)


def calculate_impact_level(impact_value: int) -> str:
    IMPACT_LABELS = {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}
    return IMPACT_LABELS.get(impact_value, "Negligible")


def calculate_feasibility_score(attack_parameters: dict) -> int:
    if not attack_parameters:
        return 0
    return sum(int(attack_parameters.get(k, 0)) for k in [
        "Exposure_time", "Professional_experience",
        "Required_information", "Opportunity_window", "Required_equipment",
    ])


def calculate_feasibility_level(feasibility_score: int) -> str:
    if feasibility_score <= 13:
        return "High"
    elif feasibility_score <= 19:
        return "Medium"
    elif feasibility_score <= 24:
        return "Low"
    else:
        return "Very Low"


def calculate_confidence_score(scores: dict) -> float:
    total = 0.0
    for dimension, weight in CONFIDENCE_WEIGHTS.items():
        score = scores.get(dimension, 0)
        total += score * weight
    return round(total, 2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 6 部分：输入解析函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_dcd_json(input_dir: str) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号",
        "tm.Process": "部件",
        "tm.Store": "数据",
        "tm.Actor": "接口",
    }
    all_results = []
    json_files = glob.glob(os.path.join(input_dir, '*.json'))
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                logger.warning(f"⚠️ DCD解析失败: {os.path.basename(file_path)}")
                continue
        current_result = {'function': '', 'assets': []}
        if 'detail' in data and 'diagrams' in data['detail']:
            for diagram in data['detail']['diagrams']:
                if not current_result['function'] and 'title' in diagram:
                    current_result['function'] = diagram['title']
                if 'diagramJson' in diagram and 'cells' in diagram['diagramJson']:
                    for cell in diagram['diagramJson']['cells']:
                        if cell.get('outOfScope') is True:
                            continue
                        raw_cell_type = cell.get('type', '')
                        if raw_cell_type == 'tm.Boundary':
                            continue
                        finetermval_value = ""
                        for key, value in cell.items():
                            if key.startswith('propertyList') and isinstance(value, dict):
                                if 'finetermval' in value:
                                    finetermval_value = value['finetermval']
                                    break
                        if not finetermval_value:
                            continue
                        mapped_cell_type = TYPE_MAPPING.get(raw_cell_type, raw_cell_type)
                        current_result['assets'].append({
                            'asset_type': mapped_cell_type,
                            'asset_name': finetermval_value,
                        })
        if not current_result['function']:
            current_result['function'] = os.path.splitext(os.path.basename(file_path))[0]
        if current_result['assets']:
            all_results.append(current_result)
    logger.info(f"  DCD解析完成: {len(all_results)} 个功能, 来自 {len(json_files)} 个文件")
    return all_results


def parse_topology_json(input_dir: str) -> dict:
    all_topo = {}
    json_files = glob.glob(os.path.join(input_dir, '*.json'))
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                logger.warning(f"⚠️ 拓扑图解析失败: {os.path.basename(file_path)}")
                continue
        name = os.path.splitext(os.path.basename(file_path))[0]
        if isinstance(data, dict) and '拓扑图' in data:
            all_topo[name] = data['拓扑图']
        elif isinstance(data, list):
            all_topo[name] = data
        elif isinstance(data, dict):
            all_topo[name] = data
    logger.info(f"  拓扑图解析完成: {len(all_topo)} 个文件")
    return all_topo


def parse_external_interface_excel(excel_path: str) -> list:
    if not os.path.exists(excel_path):
        logger.warning(f"⚠️ 外部接口Excel不存在: {excel_path}")
        return []
    df = pd.read_excel(excel_path, header=1)
    df.columns = [str(col).replace('\n', ' ').strip() for col in df.columns]
    interface_columns = df.columns[2:]
    results = []
    for index, row in df.iterrows():
        component_name = str(row['零部件名称']).strip()
        if pd.isna(row['零部件名称']) or not component_name:
            continue
        active_interfaces = []
        for col in interface_columns:
            cell_value = str(row[col]).strip()
            if '√' in cell_value:
                active_interfaces.append(col)
        if active_interfaces:
            results.append({
                "component": component_name,
                "interfaces": active_interfaces,
            })
    logger.info(f"  外部接口解析完成: {len(results)} 个零部件")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 7 部分：Prompt 模板
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BASE_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 中的TARA方法和 UN R155 法规。
你正在执行 TARA (Threat Analysis and Risk Assessment) 分析。

## 核心约束
1. 损害场景必须包含：(1)相关项功能与不良后果的因果关系 (2)对道路使用者的具体危害 (3)涉及的目标资产
2. 威胁场景必须包含：(1)目标资产 (2)被破坏的信息安全属性 (3)信息安全属性被破坏的原因
3. 攻击路径必须是逻辑连贯的步骤链，基于攻击树的方法推断
4. 资产类别与安全属性的对应关系：
   - 数据 → 完整性、机密性、可用性
   - 信号 → 完整性、机密性、真实性、可用性
   - 部件 → 完整性、机密性、真实性、不可抵赖性、权限属性、可用性
5. 安全属性与 STRIDE 威胁类型的映射：
   - 完整性 → 篡改
   - 机密性 → 信息泄露
   - 可用性 → 拒绝服务
   - 真实性 → 欺骗
   - 不可抵赖性 → 抵赖
   - 权限属性 → 越权

## 输出规范
- 仅输出纯 JSON，禁止输出任何解释性文字、Markdown 标记或代码块标记
- JSON 必须合法，字符串中不得包含未转义的换行符"""


DS_SINGLE_USER_TEMPLATE = """## 任务
请为以下资产的指定安全属性，生成一条具体的损害场景，损害场景是"涉及车辆或车辆功能且影响道路使用者的不良后果"。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}

## 参考知识
{rag_context}

## 生成规则
损害场景必须同时包含以下三个要素（缺一不可）且生成损害场景必须结合车辆实际域（如ADAS、T-Box、IVI、BCM、车身控制、电池管理等）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   - 必须明确提到被破坏的{asset_name}

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"
- “车辆夜间行驶时，前照灯控制功能因‘前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”

## 输出（严格 JSON 对象，不要数组）
{{
  "damage_scenario": "完整的损害场景描述"
}}"""


DS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请为以下资产的指定安全属性，重新生成一条具体的损害场景。上一次生成未通过质量评估，请根据反馈改进。

## 分析对象
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类别：{asset_type}
- 安全属性：{security_attribute}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 参考知识
{rag_context}

## 生成规则
1. 损害场景必须同时包含以下三个要素（缺一不可）且生成损害场景必须结合车辆实际域（如ADAS、T-Box、IVI、BCM、车身控制、电池管理等）：
   - {function_name}如何因为资产的{security_attribute}被破坏而产生不良后果
   - 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   - 必须明确提到被破坏的{asset_name}
2. 以{previous_result}为基础，根据{feedback}改进

## 参考示例：
- "存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。"
- “车辆夜间行驶时，前照灯控制功能因‘前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”


## 输出（严格 JSON 对象，不要包含输入字段）
{{
  "damage_scenario": "完整的损害场景描述"
}}"""


TS_SINGLE_USER_TEMPLATE = """## 任务
请根据以下损害场景，生成一条详细的威胁场景描述，威胁场景是"为实现损害场景，资产的信息安全属性遭到破坏的潜在原因"。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体「{asset_name}」
   - 「{asset_name}」被破坏的「{security_attribute}」
   - 「{security_attribute}」被破坏的原因/攻击意图
2. 威胁场景必须能直接导致给定的{damage_scenario}，描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条
3. {damage_scenario}与{asset_name}、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系息能被威胁场景包含或与威胁场景关联

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"
- "伪装信号导致发送至电源开关控制器的“灯光请求”信号的数据通信完整性丢失,可能造成前照灯意外关闭"

## 输出（严格 JSON 对象）
{{
  "threat_scenario": "详细的威胁场景描述"
}}"""


TS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请根据以下损害场景，重新生成一条详细的威胁场景描述。上一次生成未通过质量评估，请根据反馈改进。

## 输入
- 功能：{function_name}
- 资产名称：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁类型：{threat_type}
- 损害场景：{damage_scenario}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 参考知识
{rag_context}

## 生成规则
1. 威胁场景必须描述：
   - 明确指出被攻击的具体「{asset_name}」
   - 「{asset_name}」被破坏的「{security_attribute}」
   - 「{security_attribute}」被破坏的原因/攻击意图
2. 描述中必须体现"破坏该「{security_attribute}」 → 引发「{damage_scenario}」中的不良后果"逻辑链条
3. {damage_scenario}与{asset_name}、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系息能被威胁场景包含或与威胁场景关联
4. 以{previous_result}为基础，根据{feedback}改进

## 参考示例：
- "攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。"
- "伪装信号导致发送至电源开关控制器的“灯光请求”信号的数据通信完整性丢失,可能造成前照灯意外关闭"

## 输出（严格 JSON 对象，不要包含输入字段）
{{
  "threat_scenario": "详细的威胁场景描述"
}}"""


AP_SINGLE_USER_TEMPLATE = """## 任务
请为以下威胁场景结合车辆拓扑图和外部接口信息生成最短的（经过车辆拓扑图中的控制器最少）但符合现实逻辑的攻击路径，如有多条最短路径请生成多条，最多不超过4条。攻击路径是“为实现威胁场景的一组蓄意活动”。必须采用攻击树分析，从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的{threat_scenario}，每条路径的技术手段必须针对{asset_type}的特点设计
2. 路径中控制器的连接关系必须与{topo_info}一致，外部接口与控制器的连接关系必须与{ext_info}一致，不得虚构不存在的控制器或跳过拓扑中的网关/ECU。{topo_info}中不同id的通信协议，代表不同的总线，输出时只输出通信协议总线名称不要输出id。{topo_info}中通信协议总线是键，值代表这些控制器都在一条总线上可以相互传递
3. 同一个{threat_scenario}的多条攻击路径不要重复，避免冗余
4. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。"
- "1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- "1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

## 输出（严格 JSON 对象）
{{
  "attack_paths": [
    "1.完整步骤链",
    "1.完整步骤链"
  ]
}}"""


AP_SINGLE_USER_TEMPLATE_WITH_FEEDBACK = """## 任务
请为以下威胁场景重新生成攻击路径。上一次生成未通过质量评估，请根据反馈改进。

## 威胁场景
- 功能：{function_name}
- 资产：{asset_name}
- 资产类型：{asset_type}
- 安全属性：{security_attribute}
- 威胁场景：{threat_scenario}

## 上次生成结果
{previous_result}

## 评估反馈
{feedback}

## 车辆拓扑图
{topo_info}

## 外部接口信息
{ext_info}

## 参考知识
{rag_context}

## 生成规则
1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的{threat_scenario}，每条路径的技术手段必须针对{asset_type}的特点设计
2. 路径中控制器的连接关系必须与{topo_info}一致，外部接口与控制器的连接关系必须与{ext_info}一致，不得虚构不存在的控制器或跳过拓扑中的网关/ECU。{topo_info}中不同id的通信协议，代表不同的总线，输出时只输出通信协议总线名称不要输出id。{topo_info}中通信协议总线是键，值代表这些控制器都在一条总线上可以相互传递
3. 同一个{threat_scenario}的多条攻击路径不要重复，避免冗余
4. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）
5. 以{previous_result}为基础，根据{feedback}改进

## 参考示例：
- "1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。"
- "1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- "1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

## 输出（严格 JSON 对象）
{{
  "attack_paths": [
    "1.完整步骤链",
    "1.完整步骤链"
  ]
}}"""



IR_SYSTEM = """你是汽车网络安全风险评估专家，精通 ISO/SAE 21434 影响评级方法。
仅输出纯 JSON，不要任何解释性文字。"""

IR_BATCH_USER_TEMPLATE = """## 任务
影响是因损害场景造成的损害程度或物理伤害程度的估计，请对以下每个损害场景在 Safety、Finance、Operation、Privacy 四个维度进行影响评级。

## 输入：损害场景列表
{ds_info}

## 评级标准（取值 1～4 整数，必须逐字对照、严格遵守，不得主观臆断）
### Safety:
- 1 =没有受伤  
- 2 =轻度、中度伤害  
- 3 =严重的和有生命危险的伤害（可能生存）  
- 4 =威胁生命的伤害（不确定是否幸存），致命的伤害  
### Finance:
- 1 =经济损失导致的影响不大，后果可忽略不计，或与道路使用者无关  
- 2 =经济损失导致不便的后果，受影响的道路使用者将能用有限的资源来克服  
- 3 =导致经济上的大量损失，受影响的道路使用者将能够克服这些后果  
- 4 =经济损失导致的灾难性后果，受影响的道路使用者可能无法克服
利益相关者：车主、驾驶员、乘员、行人、供应商、主机厂
### Operation:
- 1 =操作上的损坏导致车辆功能没有损害或无法感知的损害  
- 2 =操作上的损坏导致了车辆功能的部分退化（例：用户满意度受到负面影响）  
- 3 =操作上的损坏导致了车辆重要功能的丧失或受损（例：司机的重大烦扰）  
- 4 =操作上的损坏导致了车辆核心功能的丧失或受损（例：车辆不工作或出现核心功能的意外行为，如启用跛行回家模式或自主驾驶到一个非预期的位置）
### Privacy:
- 1 =隐私侵犯不会给道路使用者带来不便 a)泄露的信息不敏感并且很难识别到PII主体  
- 2 =隐私侵犯给道路使用者带来很多不便 a)泄露的信息敏感但很难识别到PII主体；b)泄露的信息不敏感但很容易识别到PII主体  
- 3 =隐私侵犯给道路使用者带来很严重的影响 a)泄露的信息及其敏感但很难识别到PII主体；b)泄露的信息敏感而且很容易识别到PII主体  
- 4 =隐私侵犯会对道路使用者造成重大甚至不可逆转的影响。泄露的信息高度敏感，并且很容易识别到PII主体
考虑车主、驾驶员、乘员、行人、供应商、主机厂

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是整数 1~4

## 输出（严格 JSON 数组，只输出各维度参数）
[
  {{
    "influence_parameters": {{"Safety": 1, "Finance": 2, "Operation": 3, "Privacy": 4}}
  }}
]"""


AF_SYSTEM = """你是一名资深的汽车信息安全专家，精通 ISO/SAE 21434 和 UN R155 法规。
仅输出纯 JSON，不要任何解释性文字。"""

AF_BATCH_USER_TEMPLATE = """## 任务
请对每条攻击路径按 ISO 21434 攻击可行性法五个维度评级。

## 输入
{ap_info}

## 评级标准（必须逐字对照、严格遵守，不得主观臆断）
### Exposure_time: 
- 0=实现攻击行为的时间小于等于1天 
- 1=实现攻击行为的时间小于等于1周 
- 4=实现攻击行为的时间小于等于1个月  
- 17=实现攻击行为的时间小于等于6个月  
- 19=实现攻击行为的时间大于6个月
### Professional_experience: 
- 0=外行：与专家或专业人士相比缺乏知识，没有特别的专长。例1：普通人使用公开的攻击逐步描述  
- 3=熟悉产品或系统类型的安全行为。例2：有经验的业主，普通技术人员知道简单和流行的攻击  
- 6=熟悉底层算法、协议、硬件、结构、安全行为、密码学、经典攻击等。例3：有经验的技术人员或工程师  
- 9=一个攻击的不同步骤需要专家级别的不同专业知识。例4：多名经验丰富的工程师
### Required_information:
- 0=公共信息（例如互联网上获得的信息）  
- 3=受限制的信息（制造商和供应商共享的内部文档）  
- 7=机密信息（例如软件源代码、防盗控制系统相关信息）  
- 11=—严格保密的信息（只有少数人知道的特定客户校准或内存映射）
### Opportunity_window:
- 0=十分高——无限：通过公共/不受信任网络的高可用性，无任何时间限制（远程攻击、互联网/蜂窝接口）  
- 1=高——容易：高可用性和有限访问时间（蓝牙配对、远程软件更新）  
- 4=中——有限的物理和/或逻辑访问（进入未上锁车辆、车载诊断端口）  
- 10=低—困难：对相关项或组件的不切实际的访问（破解IC、暴力破解密钥）
### Required_equipment:
- 0=标准设备（笔记本电脑、CAN适配器、普通工具）  
- 4=专业设备（高档示波器、信号发生器、硬件调试设备）  
- 7=定制设备（厂家限制的工具、电子显微镜）  
- 9=多重定制设备，攻击不同步骤需要不同类型的定制设备

## 强约束条件
- 每个维度必须严格对照评级标准打分
- 分值必须是标准允许的整数

## 输出（严格 JSON 数组，只输出各维度参数）
[
  {{
    "attack_parameters": {{
      "Exposure_time": 1, "Professional_experience": 3,
      "Required_information": 3, "Opportunity_window": 4, "Required_equipment": 9
    }}
  }}
]"""


CONFIDENCE_EVAL_SYSTEM = """你是 TARA 分析质量评审专家。请严格评估场景质量并给出各维度的置信度评分。
仅输出 JSON 数组，不要任何其他文字。"""

DS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下损害场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
损害场景必须包含：功能与不良后果的因果关系、对道路使用者的具体危害、涉及的目标资产
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
损害场景描述是否与功能、资产、安全属性信息相符
评分标准：完全匹配100分，部分不匹配扣20-50分，严重不一致扣60-80分

### 3. 真实性（满分100分）
危害对道路使用者影响是否合理，是否有明显幻觉
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
句子是否简洁、无冗余、无语法错误
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过（"passed": false），列出具体问题"
  }}
]"""

TS_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下威胁场景列表的质量，对每条场景给出各维度的置信度评分（0-100分）。

## 待评审场景列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分100分）
威胁场景必须包含：目标资产、被破坏的信息安全属性、信息安全属性被破坏的原因
评分标准：每缺失一个要素扣30分，要素不完整扣10-20分

### 2. 一致性（满分100分）
是否直接导致损害场景，因果关系是否清晰
评分标准：完全一致100分，部分不一致扣20-50分，因果关系断裂扣60-80分

### 3. 真实性（满分100分）
攻击意图是否现实，是否有明显幻觉
评分标准：完全合理100分，部分不合理扣20-50分，存在幻觉扣60-80分

### 4. 清晰度（满分100分）
句子是否简洁、无冗余、无语法错误
评分标准：清晰流畅100分，有冗余或小错误扣10-30分，表达混乱扣40-60分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true或false（总分>=85为true）,
    "feedback": "如不通过（"passed": false），列出具体问题"
  }}
]"""

AP_CONFIDENCE_EVAL_BATCH_TEMPLATE = """## 评审任务
请评估以下攻击路径列表的质量，对**每条攻击路径**给出各维度的置信度评分（0-100 分）。

## 车辆拓扑图
{topo_info}
## 外部接口
{ext_info}

## 待评审攻击路径列表（共{batch_count}条）
{batch}

## 评估维度与评分标准

### 1. 要素完整性（满分 100 分）
攻击路径必须包含：起点（具体攻击面）、逻辑步骤、终点（目标资产）
评分标准：每缺失一个要素扣 30 分，要素不完整扣 10-20 分

### 2. 一致性（满分 100 分）
攻击路径是否能真正实现威胁场景，路径中的控制器及通信协议连接关系是否符合拓扑图，攻击入口是否存在于外部接口中
评分标准：完全一致 100 分，部分不一致扣 20-50 分，严重偏离扣 60-80 分

### 3. 真实性（满分 100 分）
攻击面是否存在，步骤是否可行，是否有明显幻觉
评分标准：完全可行 100 分，部分不可行扣 20-50 分，存在幻觉扣 60-80 分

### 4. 清晰度（满分 100 分）
句子是否简洁、无冗余、无语法错误，步骤描述是否清晰连贯
评分标准：清晰流畅 100 分，有冗余或小错误扣 10-30 分，表达混乱扣 40-60 分

## 输出格式（严格 JSON 数组，顺序与输入一致）
[
  {{
    "要素完整性": 0-100,
    "一致性": 0-100,
    "真实性": 0-100,
    "清晰度": 0-100,
    "confidence_score": 加权总分,
    "passed": true 或 false（总分>=85 为 true）,
    "feedback": "如不通过（"passed": true 或 false），列出具体问题"
  }}
]"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 8 部分：LangGraph 节点
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def node_asset_identification(state: TaraState) -> dict:
    logger.info("=" * 60)
    logger.info("[步骤1] 资产识别 (规则引擎)")

    if state.get("asset_results"):
        raw_assets = state["asset_results"]
    else:
        raw_assets = _rule_extract_assets(state.get("data_flow_json", []))

    enriched = []
    for func_item in raw_assets:
        func_entry = {"function": func_item["function"], "assets": []}
        for asset in func_item.get("assets", []):
            atype = asset["asset_type"]
            attrs = SECURITY_ATTRIBUTES_MAP.get(atype, ["完整性", "可用性"])
            security_attributes = []
            for attr in attrs:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                security_attributes.append({
                    "attribute_name": attr,
                    "threat_type": threat_type,
                })
            func_entry["assets"].append({
                "asset_name": asset["asset_name"],
                "asset_type": atype,
                "security_attributes": security_attributes,
            })
        enriched.append(func_entry)

    total_assets = sum(len(f["assets"]) for f in enriched)
    total_attrs = sum(len(a["security_attributes"]) for f in enriched for a in f["assets"])
    logger.info(f"  → {len(enriched)} 个功能, {total_assets} 个资产, {total_attrs} 个安全属性")

    save_checkpoint(enriched, "step1_assets")
    return {
        "asset_results": enriched,
        "manual_review_items": [],
    }


def _rule_extract_assets(data_flow_json: list) -> list:
    TYPE_MAPPING = {
        "tm.Flow": "信号", "tm.Process": "部件",
        "tm.Store": "数据", "tm.Actor": "接口",
    }
    results = []
    for data in data_flow_json:
        entry = {"function": "", "assets": []}
        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not entry["function"] and "title" in diagram:
                    entry["function"] = diagram["title"]
                if "diagramJson" in diagram and "cells" in diagram["diagramJson"]:
                    for cell in diagram["diagramJson"]["cells"]:
                        if cell.get("outOfScope") or cell.get("type") == "tm.Boundary":
                            continue
                        raw_type = cell.get("type", "")
                        name = ""
                        for k, v in cell.items():
                            if k.startswith("propertyList") and isinstance(v, dict):
                                name = v.get("finetermval", "")
                                if name:
                                    break
                        if name:
                            entry["assets"].append({
                                "asset_type": TYPE_MAPPING.get(raw_type, raw_type),
                                "asset_name": name,
                            })
        if entry["assets"]:
            results.append(entry)
    return results


def node_ds_generate(state: TaraState) -> dict:
    logger.info("[步骤2] 损害场景生成 (逐条并发)")

    assets = state["asset_results"]
    ds_feedback = state.get("ds_feedback", {})
    retry_count = state.get("ds_retry_count", 0)
    rag_context = rag_kb.retrieve("汽车网络安全 损害场景 STRIDE ISO21434")

    tasks = []
    task_id = 0
    for func in assets:
        for asset in func["assets"]:
            for sa in asset["security_attributes"]:
                attr_name = sa["attribute_name"]
                key = (func["function"], asset["asset_name"], attr_name)
                
                if ds_feedback and key in ds_feedback:
                    feedback_info = ds_feedback[key]
                    prompt = DS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                        function_name=func["function"],
                        asset_name=asset["asset_name"],
                        asset_type=asset["asset_type"],
                        security_attribute=attr_name,
                        previous_result=feedback_info.get("previous_result", ""),
                        feedback=feedback_info.get("feedback", ""),
                        rag_context=rag_context,
                    )
                else:
                    prompt = DS_SINGLE_USER_TEMPLATE.format(
                        function_name=func["function"],
                        asset_name=asset["asset_name"],
                        asset_type=asset["asset_type"],
                        security_attribute=attr_name,
                        rag_context=rag_context,
                    )
                tasks.append({
                    "id": task_id,
                    "system": BASE_SYSTEM,
                    "user": prompt,
                    "_meta": {
                        "function": func["function"],
                        "asset_name": asset["asset_name"],
                        "asset_type": asset["asset_type"],
                        "security_attribute": attr_name,
                        "threat_type": sa["threat_type"],
                    },
                })
                task_id += 1

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    scenarios = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                scenarios.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    scenarios.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成损害场景 {len(scenarios)}/{len(tasks)} 条")
    save_checkpoint(scenarios, "step2_damage_scenarios")
    return {
        "damage_scenarios": scenarios, 
        "ds_feedback": {},
        "ds_retry_count": state.get("ds_retry_count", 0) + 1,
    }


def node_ds_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤2-评估] 损害场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ds_confidence": {"all_passed": True, "items": []}, "ds_retry_count": 0, "ds_feedback": {}}

    scenarios = state.get("damage_scenarios", [])
    if not scenarios:
        return {"ds_confidence": {"all_passed": True, "items": []}, "ds_retry_count": 0, "ds_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=scenarios,
        eval_template=DS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"],
    )

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ds_feedback = {}

    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD

        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": scenarios[orig_id] if orig_id < len(scenarios) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
        }
        eval_results.append(eval_item)

        if not passed:
            failed_indices.append(orig_id)
            sc = scenarios[orig_id] if orig_id < len(scenarios) else {}
            key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
            ds_feedback[key] = {
                "previous_result": sc.get("damage_scenario", ""),
                "feedback": parsed.get("feedback", ""),
            }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ds_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            sc = scenarios[idx] if idx < len(scenarios) else {}
            eval_item = next((e for e in eval_results if e["id"] == idx), {})
            manual_review_items.append({
                "type": "damage_scenario",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": sc,
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 个损害场景需要人工审核")

    ds_score_map = {}
    for eval_item in eval_results:
        orig_id = eval_item["id"]
        sc = scenarios[orig_id] if orig_id < len(scenarios) else {}
        key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
        ds_score_map[key] = eval_item.get("confidence_score", 0)

    save_checkpoint(eval_results, "step2_ds_confidence_eval")
    return {
        "ds_confidence": {"all_passed": all_passed, "items": eval_results},
        "ds_retry_count": retry_count,  # ← 去掉 + 1
        "ds_feedback": ds_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
        "ds_score_map": ds_score_map,
    }


def node_ts_generate(state: TaraState) -> dict:
    logger.info("[步骤3] 威胁场景生成 (逐条并发)")

    damage_scenarios = state.get("damage_scenarios", [])
    ts_feedback = state.get("ts_feedback", {})
    rag_context = rag_kb.retrieve("汽车威胁场景 CAPEC ATT&CK CAN注入 固件篡改 中间人攻击")

    tasks = []
    for i, ds in enumerate(damage_scenarios):
        key = (ds.get("function", ""), ds.get("asset_name", ""), ds.get("security_attribute", ""))
        
        if ts_feedback and key in ts_feedback:
            feedback_info = ts_feedback[key]
            prompt = TS_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                function_name=ds.get("function", ""),
                asset_name=ds.get("asset_name", ""),
                asset_type=ds.get("asset_type", ""),        # ← 补上这行
                security_attribute=ds.get("security_attribute", ""),
                threat_type=ds.get("threat_type", ""),
                damage_scenario=ds.get("damage_scenario", ""),
                previous_result=feedback_info.get("previous_result", ""),
                feedback=feedback_info.get("feedback", ""),
                rag_context=rag_context,
            )
        else:
            prompt = TS_SINGLE_USER_TEMPLATE.format(
                function_name=ds.get("function", ""),
                asset_name=ds.get("asset_name", ""),
                asset_type=ds.get("asset_type", ""),        # ← 补上这行
                security_attribute=ds.get("security_attribute", ""),
                threat_type=ds.get("threat_type", ""),
                damage_scenario=ds.get("damage_scenario", ""),
                rag_context=rag_context,
            )
        tasks.append({
            "id": i,
            "system": BASE_SYSTEM,
            "user": prompt,
            "_meta": {
                "function": ds.get("function", ""),
                "asset_name": ds.get("asset_name", ""),
                "asset_type": ds.get("asset_type", ""),
                "security_attribute": ds.get("security_attribute", ""),
                "threat_type": ds.get("threat_type", ""),
                "damage_scenario": ds.get("damage_scenario", ""),
            },
        })

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    ts_list = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict):
                ts_list.append({**meta, **parsed})
            elif isinstance(parsed, list):
                for item in parsed:
                    ts_list.append({**meta, **item} if isinstance(item, dict) else item)
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")

    logger.info(f"  → 成功生成威胁场景 {len(ts_list)}/{len(tasks)} 条")
    save_checkpoint(ts_list, "step3_threat_scenarios")
    return {
        "threat_scenarios": ts_list, 
        "ts_feedback": {},
        "ts_retry_count": state.get("ts_retry_count", 0) + 1,
    }


def node_ts_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤3-评估] 威胁场景置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ts_confidence": {"all_passed": True, "items": []}, "ts_retry_count": 0, "ts_feedback": {}}

    ts_list = state.get("threat_scenarios", [])
    if not ts_list:
        return {"ts_confidence": {"all_passed": True, "items": []}, "ts_retry_count": 0, "ts_feedback": {}}

    eval_results_raw = batch_confidence_eval(
        scenarios=ts_list,
        eval_template=TS_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["function", "asset_name", "security_attribute", "threat_type"],
    )

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ts_feedback = {}

    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD

        orig_id = item["id"]
        eval_item = {
            "id": orig_id,
            "scenario": ts_list[orig_id] if orig_id < len(ts_list) else {},
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
        }
        eval_results.append(eval_item)

        if not passed:
            failed_indices.append(orig_id)
            ts = ts_list[orig_id] if orig_id < len(ts_list) else {}
            key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
            ts_feedback[key] = {
                "previous_result": ts.get("threat_scenario", ""),
                "feedback": parsed.get("feedback", ""),
            }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    retry_count = state.get("ts_retry_count", 0)

    if not all_passed:
        for idx in failed_indices:
            ts = ts_list[idx] if idx < len(ts_list) else {}
            eval_item = next((e for e in eval_results if e["id"] == idx), {})
            manual_review_items.append({
                "type": "threat_scenario",
                "id": idx,
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": ts,
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 个威胁场景需要人工审核")

    ts_score_map = {}
    for eval_item in eval_results:
        orig_id = eval_item["id"]
        ts = ts_list[orig_id] if orig_id < len(ts_list) else {}
        key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
        ts_score_map[key] = eval_item.get("confidence_score", 0)

    save_checkpoint(eval_results, "step3_ts_confidence_eval")
    return {
        "ts_confidence": {"all_passed": all_passed, "items": eval_results},
        "ts_retry_count": retry_count,  # ← 去掉 + 1
        "ts_feedback": ts_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
        "ts_score_map": ts_score_map,
    }


def node_ap_generate(state: TaraState) -> dict:
    logger.info("[步骤4] 攻击路径生成 (逐条并发)")

    threat_scenarios = state.get("threat_scenarios", [])
    ap_feedback = state.get("ap_feedback", {})
    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)
    rag_context = rag_kb.retrieve("汽车攻击路径 CAN总线 OBD T-Box 蓝牙 V2X 攻击树")

    tasks = []
    for i, ts in enumerate(threat_scenarios):
        key = (ts.get("function", ""), ts.get("asset_name", ""), ts.get("security_attribute", ""))
        if ap_feedback and key in ap_feedback:
            feedback_info = ap_feedback[key]
            prompt = AP_SINGLE_USER_TEMPLATE_WITH_FEEDBACK.format(
                function_name=ts.get("function", ""), asset_name=ts.get("asset_name", ""),
                asset_type=ts.get("asset_type", ""), security_attribute=ts.get("security_attribute", ""),
                threat_scenario=ts.get("threat_scenario", ""),
                previous_result=feedback_info.get("previous_result", ""),
                feedback=feedback_info.get("feedback", ""),
                topo_info=topo_info, ext_info=ext_info, rag_context=rag_context,
            )
        else:
            prompt = AP_SINGLE_USER_TEMPLATE.format(
                function_name=ts.get("function", ""), asset_name=ts.get("asset_name", ""),
                asset_type=ts.get("asset_type", ""), security_attribute=ts.get("security_attribute", ""),
                threat_scenario=ts.get("threat_scenario", ""),
                topo_info=topo_info, ext_info=ext_info, rag_context=rag_context,
            )
        tasks.append({
            "id": i, "system": BASE_SYSTEM, "user": prompt,
            "_meta": {
                "function": ts.get("function", ""), "asset_name": ts.get("asset_name", ""),
                "asset_type": ts.get("asset_type", ""), "security_attribute": ts.get("security_attribute", ""),
                "threat_type": ts.get("threat_type", ""), "threat_scenario": ts.get("threat_scenario", ""),
            },
        })

    logger.info(f"  → 共 {len(tasks)} 个子任务, 并发数 {MAX_WORKERS}")
    raw_results = concurrent_llm_calls(tasks)

    # 保持嵌套结构返回给 State
    attack_paths = []
    for r in raw_results:
        if r["raw"] is None:
            attack_paths.append({"attack_paths": []})
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            task = next((t for t in tasks if t["id"] == r["id"]), {})
            meta = task.get("_meta", {})
            if isinstance(parsed, dict) and "attack_paths" in parsed:
                paths = parsed["attack_paths"]
            elif isinstance(parsed, list):
                paths = parsed
            else:
                paths = []
            attack_paths.append({**meta, "attack_paths": paths if isinstance(paths, list) else [str(paths)]})
        except Exception as e:
            logger.warning(f"  ⚠ 任务 {r['id']} 解析失败: {e}")
            attack_paths.append({"attack_paths": []})

    total_paths = sum(len(a.get("attack_paths", [])) for a in attack_paths)
    logger.info(f"  → 成功生成: {len(attack_paths)} 个威胁场景, 共 {total_paths} 条攻击路径")
    save_checkpoint(attack_paths, "step4_attack_paths")
    return {
        "attack_paths": attack_paths, 
        "ap_feedback": {},
        "ap_retry_count": state.get("ap_retry_count", 0) + 1,  # ← 加上这行
    }


def node_ap_confidence_eval(state: TaraState) -> dict:
    logger.info("[步骤4-评估] 攻击路径置信度评估 (批量模式)")

    if not ENABLE_EVALUATION:
        logger.info("  → 评估已跳过")
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    nested_attack_paths = state.get("attack_paths", [])
    if not nested_attack_paths:
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    topo_info = json.dumps(state.get("topology_json", {}), ensure_ascii=False, indent=2)
    ext_info = json.dumps(state.get("external_interface_json", {}), ensure_ascii=False, indent=2)

    # ===== 关键修复：在内部扁平化嵌套结构，为每条路径构建评估输入 =====
    flat_eval_data = []
    for ts_idx, ap in enumerate(nested_attack_paths):
        meta = {
            "function": ap.get("function", ""),
            "asset_name": ap.get("asset_name", ""),
            "asset_type": ap.get("asset_type", ""),
            "security_attribute": ap.get("security_attribute", ""),
            "threat_type": ap.get("threat_type", ""),
            "threat_scenario": ap.get("threat_scenario", ""),
        }
        for path_idx, path_desc in enumerate(ap.get("attack_paths", [])):
            flat_eval_data.append({
                "ts_idx": ts_idx,
                "path_idx": path_idx,
                "path_desc": path_desc,  # 真实的单条路径字符串
                "meta": meta,
            })
            
    if not flat_eval_data:
        return {"ap_confidence": {"all_passed": True, "items": []}, "ap_retry_count": 0, "ap_feedback": {}}

    eval_scenarios = [
        {"威胁场景": item["meta"]["threat_scenario"], "攻击路径": item["path_desc"]}
        for item in flat_eval_data
    ]

    eval_results_raw = batch_confidence_eval(
        scenarios=eval_scenarios,
        eval_template=AP_CONFIDENCE_EVAL_BATCH_TEMPLATE,
        system_prompt=CONFIDENCE_EVAL_SYSTEM,
        batch_size=EVAL_BATCH_SIZE,
        key_fields=["threat_scenario", "attack_path"],
        topo_info=topo_info,
        ext_info=ext_info,
    )

    eval_results = []
    manual_review_items = []
    failed_indices = []
    ap_feedback = {}

    for item in eval_results_raw:
        parsed = item.get("raw_result", {})
        scores = {
            "要素完整性": parsed.get("要素完整性", 0),
            "一致性": parsed.get("一致性", 0),
            "真实性": parsed.get("真实性", 0),
            "清晰度": parsed.get("清晰度", 0),
        }
        confidence_score = calculate_confidence_score(scores)
        passed = confidence_score >= CONFIDENCE_THRESHOLD

        orig_id = item["id"]
        flat_data = flat_eval_data[orig_id] if orig_id < len(flat_eval_data) else {}

        eval_item = {
            "id": orig_id,
            "ts_idx": flat_data.get("ts_idx", -1),
            "path_idx": flat_data.get("path_idx", -1),
            "scenario": flat_data.get("path_desc", ""),
            "meta": flat_data.get("meta", {}),
            "scores": scores,
            "confidence_score": confidence_score,
            "passed": passed,
            "feedback": parsed.get("feedback", ""),
        }
        eval_results.append(eval_item)

        if not passed:
            failed_indices.append(orig_id)
            meta = flat_data.get("meta", {})
            key = (meta.get("function", ""), meta.get("asset_name", ""), meta.get("security_attribute", ""))
            # 如果同一威胁场景下有多条路径不合格，只保留第一份反馈（因为重生成是按威胁场景整体重新生成的）
            if key not in ap_feedback:
                ap_feedback[key] = {
                    "previous_result": flat_data.get("path_desc", ""),  # ← 必须是真实路径，绝不能是空字符串
                    "feedback": parsed.get("feedback", ""),
                }

    all_passed = len(failed_indices) == 0
    passed_count = len([e for e in eval_results if e["passed"]])
    logger.info(f"  → 评估完成: {passed_count}/{len(eval_results)} 通过, 阈值={CONFIDENCE_THRESHOLD}")

    # 构建 score map：key = (func, asset, attr), value = [score1, score2, ...]
    # 这个格式完美匹配 node_assemble_report 中的 ap_score_map 预期
    ap_score_map = {}
    for eval_item in eval_results:
        meta = eval_item.get("meta", {})
        key = (meta.get("function", ""), meta.get("asset_name", ""), meta.get("security_attribute", ""))
        ap_score_map.setdefault(key, []).append(eval_item.get("confidence_score", 0))

    retry_count = state.get("ap_retry_count", 0)
    if not all_passed:
        for idx in failed_indices:
            flat_data = flat_eval_data[idx] if idx < len(flat_eval_data) else {}
            eval_item = next((e for e in eval_results if e["id"] == idx), {})
            # 注意：这里 scenario 设为 meta 字典，是为了兼容 _build_manual_review_report 里的 .get() 写法
            manual_review_items.append({
                "type": "attack_path",
                "id": idx,
                "ts_idx": flat_data.get("ts_idx", -1),
                "path_idx": flat_data.get("path_idx", -1),
                "confidence_score": eval_item.get("confidence_score", 0),
                "feedback": eval_item.get("feedback", ""),
                "scenario": flat_data.get("meta", {}),  
                "meta": flat_data.get("meta", {}),
            })
        logger.warning(f"  ⚠ {len(failed_indices)} 条攻击路径需要人工审核")

    save_checkpoint(eval_results, "step4_ap_confidence_eval")
    return {
        "ap_confidence": {"all_passed": all_passed, "items": eval_results},
        "ap_retry_count": retry_count,
        "ap_feedback": ap_feedback,
        "manual_review_items": state.get("manual_review_items", []) + manual_review_items,
        "ap_score_map": ap_score_map,
    }


def node_impact_rating(state: TaraState) -> dict:
    logger.info("[步骤5] 影响评级 (分批并发)")

    ds_list = state.get("damage_scenarios", [])
    if not ds_list:
        return {"impact_ratings": []}

    batches = [ds_list[i:i+BATCH_SIZE_RATING] for i in range(0, len(ds_list), BATCH_SIZE_RATING)]
    logger.info(f"  → {len(ds_list)} 条损害场景, 分 {len(batches)} 批")

    tasks = []
    for i, batch in enumerate(batches):
        ds_info = json.dumps([
            {"function": d.get("function", ""), "asset_name": d.get("asset_name", ""),
             "security_attribute": d.get("security_attribute", ""),
             "damage_scenario": d.get("damage_scenario", "")}
            for d in batch
        ], ensure_ascii=False, indent=2)
        prompt = IR_BATCH_USER_TEMPLATE.format(ds_info=ds_info)
        tasks.append({"id": i, "system": IR_SYSTEM, "user": prompt})

    raw_results = concurrent_llm_calls(tasks)

    ratings = []
    for r in raw_results:
        if r["raw"] is None:
            continue
        try:
            parsed = safe_parse_json(r["raw"])
            batch_idx = r["id"]
            batch_start = batch_idx * BATCH_SIZE_RATING
            batch_scenarios = ds_list[batch_start:batch_start + BATCH_SIZE_RATING]

            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    if idx < len(batch_scenarios) and isinstance(item, dict):
                        meta = {
                            "function": batch_scenarios[idx].get("function", ""),
                            "asset_name": batch_scenarios[idx].get("asset_name", ""),
                            "security_attribute": batch_scenarios[idx].get("security_attribute", ""),
                        }
                        ratings.append({**meta, **item})
            elif isinstance(parsed, dict):
                meta = {
                    "function": batch_scenarios[0].get("function", "") if batch_scenarios else "",
                    "asset_name": batch_scenarios[0].get("asset_name", "") if batch_scenarios else "",
                    "security_attribute": batch_scenarios[0].get("security_attribute", "") if batch_scenarios else "",
                }
                ratings.append({**meta, **parsed})
        except Exception as e:
            logger.warning(f"  ⚠ 批次 {r['id']} 解析失败：{e}")

    for r in ratings:
        params = r.get("influence_parameters", {})
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        r["impact_value"] = str(impact_value)
        r["impact_level"] = impact_level

    logger.info(f"  → 完成影响评级 {len(ratings)} 条")
    save_checkpoint(ratings, "step5_impact_ratings")
    return {"impact_ratings": ratings}


def node_feasibility_rating(state: TaraState) -> dict:
    logger.info("[步骤6] 攻击可行性评级 (批量模式)")

    nested_attack_paths = state.get("attack_paths", [])
    if not nested_attack_paths:
        return {"feasibility_ratings": []}

    # ===== 关键：在函数内部扁平化，不改变传入的 State =====
    flat_ap_info = []
    for ts_idx, ap in enumerate(nested_attack_paths):
        for path_idx, path_desc in enumerate(ap.get("attack_paths", [])):
            flat_ap_info.append({
                "威胁场景": ap.get("threat_scenario", ""),
                "攻击路径": path_desc,
                "_ts_idx": ts_idx,
                "_path_idx": path_idx,
            })
    
    ap_info_json = json.dumps(
        [{"威胁场景": x["威胁场景"], "攻击路径": x["攻击路径"]} for x in flat_ap_info], 
        ensure_ascii=False, indent=2
    )

    tasks = [{"id": 0, "system": AF_SYSTEM, "user": AF_BATCH_USER_TEMPLATE.format(ap_info=ap_info_json)}]
    raw_results = concurrent_llm_calls(tasks, max_workers=1)

    feasibility_ratings = []
    for r in raw_results:
        if r["raw"] is None: continue
        try:
            parsed = safe_parse_json(r["raw"])
            if isinstance(parsed, list):
                for idx, item in enumerate(parsed):
                    attack_params = item.get("attack_parameters", {})
                    feasibility_ratings.append({
                        "ts_idx": flat_ap_info[idx]["_ts_idx"],
                        "path_idx": flat_ap_info[idx]["_path_idx"],
                        "attack_path": flat_ap_info[idx]["攻击路径"],
                        "attack_parameters": attack_params,
                        "feasibility_score": calculate_feasibility_score(attack_params),
                        "feasibility_level": calculate_feasibility_level(calculate_feasibility_score(attack_params)),
                    })
        except Exception as e:
            logger.warning(f"  ⚠ 可行性评级解析失败: {e}")

    logger.info(f"  → 成功评级 {len(feasibility_ratings)}/{len(flat_ap_info)} 条攻击路径")
    save_checkpoint(feasibility_ratings, "step6_feasibility_ratings")
    return {"feasibility_ratings": feasibility_ratings}


def node_risk_calculation(state: TaraState) -> dict:
    logger.info("[步骤7] 风险值计算与处置决策")

    nested_attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])

    # 构建 feasibility 查找表 (ts_idx, path_idx)
    feas_lookup = {(fr["ts_idx"], fr["path_idx"]): fr for fr in feasibility_ratings}
    # 构建 impact 查找表 (ts_idx)
    impact_lookup = {}
    for idx, ir in enumerate(impact_ratings):
        influence = ir.get("influence_parameters", {})
        impact_value = calculate_impact_value(influence)
        impact_lookup[idx] = {"influence_parameters": influence, "impact_value": impact_value, "impact_level": calculate_impact_level(impact_value)}

    RISK_MATRIX = {
        ("Negligible", "High"): (1, "Low risk", "Accept"), ("Negligible", "Medium"): (1, "Low risk", "Accept"),
        ("Negligible", "Low"): (1, "Low risk", "Accept"), ("Negligible", "Very Low"): (1, "Low risk", "Accept"),
        ("Moderate", "High"): (2, "Medium risk", "Reduce"), ("Moderate", "Medium"): (1, "Low risk", "Accept"),
        ("Moderate", "Low"): (1, "Low risk", "Accept"), ("Moderate", "Very Low"): (1, "Low risk", "Accept"),
        ("Major", "High"): (3, "High risk", "Reduce"), ("Major", "Medium"): (2, "Medium risk", "Reduce"),
        ("Major", "Low"): (1, "Low risk", "Accept"), ("Major", "Very Low"): (1, "Low risk", "Accept"),
        ("Severe", "High"): (4, "Very High risk", "Reduce"), ("Severe", "Medium"): (3, "High risk", "Reduce"),
        ("Severe", "Low"): (2, "Medium risk", "Reduce"), ("Severe", "Very Low"): (2, "Medium risk", "Reduce"),
    }

    risk_results = []
    for ts_idx, ap in enumerate(nested_attack_paths):
        impact = impact_lookup.get(ts_idx, {"impact_value": 1, "impact_level": "Negligible", "influence_parameters": {}})
        
        # 遍历该威胁场景下的每一条路径
        for path_idx, path_desc in enumerate(ap.get("attack_paths", [])):
            feas = feas_lookup.get((ts_idx, path_idx), {"feasibility_score": 0, "feasibility_level": "Very Low", "attack_parameters": {}})
            
            matrix_key = (impact["impact_level"], feas["feasibility_level"])
            risk_value, risk_level, risk_treatment = RISK_MATRIX.get(matrix_key, (1, "Low risk", "Accept"))

            risk_results.append({
                "ts_idx": ts_idx,
                "path_idx": path_idx,
                "function": ap.get("function", ""),
                "asset_name": ap.get("asset_name", ""),
                "asset_type": ap.get("asset_type", ""),
                "security_attribute": ap.get("security_attribute", ""),
                "threat_type": ap.get("threat_type", ""),
                "threat_scenario": ap.get("threat_scenario", ""),
                "attack_path": path_desc,
                "influence_parameters": impact["influence_parameters"],
                "impact_value": impact["impact_value"],
                "impact_level": impact["impact_level"],
                "attack_parameters": feas["attack_parameters"],
                "feasibility_score": feas["feasibility_score"],
                "feasibility_level": feas["feasibility_level"],
                "risk_value": risk_value,
                "risk_level": risk_level,
                "risk_treatment": risk_treatment,
            })

    treatment_counts = {}
    for r in risk_results:
        t = r["risk_treatment"]
        treatment_counts[t] = treatment_counts.get(t, 0) + 1
    logger.info(f"  → 风险计算完成: {len(risk_results)} 条路径, 处置分布: {treatment_counts}")

    save_checkpoint(risk_results, "step7_risk_results")
    return {"risk_results": risk_results}


def node_assemble_report(state: TaraState) -> dict:
    logger.info("[步骤8] TARA 报告汇总")

    asset_results = state.get("asset_results", [])
    damage_scenarios = state.get("damage_scenarios", [])
    threat_scenarios = state.get("threat_scenarios", [])
    attack_paths = state.get("attack_paths", [])
    impact_ratings = state.get("impact_ratings", [])
    feasibility_ratings = state.get("feasibility_ratings", [])
    risk_results = state.get("risk_results", [])

    # ══════════════════════════════════════════════════
    #  置信度分数查找表
    # ══════════════════════════════════════════════════
    ds_score_map: dict[tuple, float] = {}
    ds_conf = state.get("ds_confidence", {})
    if isinstance(ds_conf, dict) and "items" in ds_conf:
        for item in ds_conf["items"]:
            sc = item.get("scenario", {})
            key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
            ds_score_map[key] = item.get("confidence_score", 0)

    ts_score_map: dict[tuple, float] = {}
    ts_conf = state.get("ts_confidence", {})
    if isinstance(ts_conf, dict) and "items" in ts_conf:
        for item in ts_conf["items"]:
            sc = item.get("scenario", {})
            key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
            ts_score_map[key] = item.get("confidence_score", 0)

    ap_score_map: dict[tuple, list[float]] = state.get("ap_score_map", {})
    # ══════════════════════════════════════════════════

    def _key(item: dict) -> tuple:
        return (item.get("function", ""), item.get("asset_name", ""), item.get("security_attribute", ""))

    ds_map = {_key(d): d for d in damage_scenarios}
    ts_map = {_key(t): t for t in threat_scenarios}
    ir_map = {_key(i): i for i in impact_ratings}

    # ↓↓↓ 修改：rr_map 改为列表映射，支持多条路径各自的风险 ↓↓↓
    rr_map: dict[tuple, list[dict]] = {}
    for r in risk_results:
        k = _key(r)
        rr_map.setdefault(k, []).append(r)

    ap_map: dict[tuple, list[str]] = {}
    for a in attack_paths:
        k = _key(a)
        ap_map.setdefault(k, []).extend(a.get("attack_paths", []))

    fr_map: dict[tuple, list[dict]] = {}
    for f in feasibility_ratings:
        k = _key(f)
        fr_map.setdefault(k, []).append(f)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for sa in asset.get("security_attributes", []):
                attr = sa["attribute_name"] if isinstance(sa, dict) else sa
                threat_type = sa.get("threat_type", ATTRIBUTE_TO_THREAT.get(attr, "未知")) if isinstance(sa, dict) else ATTRIBUTE_TO_THREAT.get(attr, "未知")
                key = (func_name, asset["asset_name"], attr)

                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])
                rrs = rr_map.get(key, [])  # ← 修改：获取风险列表

                ap_scores = ap_score_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    matched_rr = rrs[idx] if idx < len(rrs) else {}  # ← 修改：按索引取对应路径的风险
                    
                    attack_list.append({
                        "attack_path": path_desc,
                        "ap_score": ap_scores[idx] if idx < len(ap_scores) else 0,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                        "risk_value": matched_rr.get("risk_value", ""),          # ← 修改：每条路径都有
                        "risk_treatment": matched_rr.get("risk_treatment", ""),  # ← 修改：每条路径都有
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "threat_type": threat_type,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "ds_score": ds_score_map.get(key, 0),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "ts_score": ts_score_map.get(key, 0),
                    "attack": attack_list,
                }
                asset_entry["security_attributes"].append(attr_entry)
            func_entry["assets"].append(asset_entry)
        report.append(func_entry)

    logger.info(f"  → 报告组装完成：{len(report)} 个功能模块")

    manual_items = state.get("manual_review_items", [])
    if manual_items:
        manual_report = _build_manual_review_report(
            manual_items, asset_results, ds_map, ts_map, ap_map, ir_map, fr_map, rr_map,
            ds_score_map, ts_score_map, ap_score_map,
        )
        manual_path = os.path.join(OUTPUT_DIR, "tara_manual_review.json")
        with open(manual_path, "w", encoding="utf-8") as f:
            json.dump(manual_report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 人工审核报告已保存: {manual_path}")

    return {"tara_report": report}


def _build_manual_review_report(
    manual_items: list,
    asset_results: list,
    ds_map: dict,
    ts_map: dict,
    ap_map: dict,
    ir_map: dict,
    fr_map: dict,
    rr_map: dict,        # ← 现在是 dict[tuple, list[dict]]
    ds_score_map: dict,
    ts_score_map: dict,
    ap_score_map: dict,
) -> list:
    flagged_keys = set()
    for item in manual_items:
        sc = item.get("scenario", {})
        meta = item.get("meta", {})
        if sc:
            key = (sc.get("function", ""), sc.get("asset_name", ""), sc.get("security_attribute", ""))
            flagged_keys.add(key)
        if meta:
            key = (meta.get("function", ""), meta.get("asset_name", ""), meta.get("security_attribute", ""))
            flagged_keys.add(key)

    report = []
    for func_item in asset_results:
        func_name = func_item["function"]
        func_entry = {"function": func_name, "assets": []}

        for asset in func_item.get("assets", []):
            asset_entry = {
                "asset_name": asset["asset_name"],
                "asset_type": asset["asset_type"],
                "security_attributes": [],
            }
            for sa in asset.get("security_attributes", []):
                attr = sa["attribute_name"] if isinstance(sa, dict) else sa
                threat_type = sa.get("threat_type", ATTRIBUTE_TO_THREAT.get(attr, "未知")) if isinstance(sa, dict) else ATTRIBUTE_TO_THREAT.get(attr, "未知")
                key = (func_name, asset["asset_name"], attr)

                if key not in flagged_keys:
                    continue

                ds = ds_map.get(key, {})
                ts = ts_map.get(key, {})
                ir = ir_map.get(key, {})
                paths = ap_map.get(key, [])
                frs = fr_map.get(key, [])
                rrs = rr_map.get(key, [])  # ← 修改：获取风险列表

                ap_scores = ap_score_map.get(key, [])

                attack_list = []
                for idx, path_desc in enumerate(paths):
                    matched_fr = frs[idx] if idx < len(frs) else {}
                    matched_rr = rrs[idx] if idx < len(rrs) else {}  # ← 修改
                    
                    attack_list.append({
                        "attack_path": path_desc,
                        "ap_score": ap_scores[idx] if idx < len(ap_scores) else 0,
                        "attack_parameters": matched_fr.get("attack_parameters", {
                            "Exposure_time": 0, "Professional_experience": 0,
                            "Required_information": 0, "Opportunity_window": 0,
                            "Required_equipment": 0,
                        }),
                        "feasibility_score": matched_fr.get("feasibility_score", 0),
                        "feasibility_level": matched_fr.get("feasibility_level", "Very Low"),
                        "risk_value": matched_rr.get("risk_value", ""),          # ← 修改
                        "risk_treatment": matched_rr.get("risk_treatment", ""),  # ← 修改
                    })

                attr_entry = {
                    "attribute_name": attr,
                    "threat_type": threat_type,
                    "damage_scenario": ds.get("damage_scenario", ""),
                    "ds_score": ds_score_map.get(key, 0),
                    "influence_parameters": ir.get("influence_parameters", {
                        "Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0,
                    }),
                    "impact_value": ir.get("impact_value", "1"),
                    "impact_level": ir.get("impact_level", "Negligible"),
                    "threat_scenarios": ts.get("threat_scenario", ""),
                    "ts_score": ts_score_map.get(key, 0),
                    "attack": attack_list,
                    "_review_flag": True,
                    "_review_feedback": "; ".join([
                        item.get("feedback", "") for item in manual_items
                        if (item.get("scenario", {}).get("function", ""), item.get("scenario", {}).get("asset_name", ""), item.get("scenario", {}).get("security_attribute", "")) == key
                        or (item.get("meta", {}).get("function", ""), item.get("meta", {}).get("asset_name", ""), item.get("meta", {}).get("security_attribute", "")) == key
                    ]),
                }
                asset_entry["security_attributes"].append(attr_entry)
            if asset_entry["security_attributes"]:
                func_entry["assets"].append(asset_entry)
        if func_entry["assets"]:
            report.append(func_entry)
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 9 部分：工作流构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _should_retry_ds(state: TaraState) -> str:
    ds_conf = state.get("ds_confidence", {})
    retry_count = state.get("ds_retry_count", 0)
    if not ds_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ds"
    return "next_ts"

def _should_retry_ts(state: TaraState) -> str:
    ts_conf = state.get("ts_confidence", {})
    retry_count = state.get("ts_retry_count", 0)
    if not ts_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ts"
    return "next_ap"

def _should_retry_ap(state: TaraState) -> str:
    ap_conf = state.get("ap_confidence", {})
    retry_count = state.get("ap_retry_count", 0)
    if not ap_conf.get("all_passed", True) and retry_count < MAX_CONFIDENCE_RETRIES:
        return "retry_ap"
    return "next_ir"


def build_tara_graph():
    builder = StateGraph(TaraState)

    builder.add_node("node_asset_id",            node_asset_identification)
    builder.add_node("node_ds_generate",         node_ds_generate)
    builder.add_node("node_ds_confidence_eval",  node_ds_confidence_eval)
    builder.add_node("node_ts_generate",         node_ts_generate)
    builder.add_node("node_ts_confidence_eval",  node_ts_confidence_eval)
    builder.add_node("node_ap_generate",         node_ap_generate)
    builder.add_node("node_ap_confidence_eval",  node_ap_confidence_eval)
    builder.add_node("node_impact_rating",       node_impact_rating)
    builder.add_node("node_feasibility_rating",  node_feasibility_rating)
    builder.add_node("node_risk_calc",           node_risk_calculation)
    builder.add_node("node_report",              node_assemble_report)

    builder.set_entry_point("node_asset_id")
    builder.add_edge("node_asset_id", "node_ds_generate")
    builder.add_edge("node_ds_generate", "node_ds_confidence_eval")
    builder.add_conditional_edges("node_ds_confidence_eval", _should_retry_ds, {
        "retry_ds": "node_ds_generate",
        "next_ts": "node_ts_generate",
    })
    builder.add_edge("node_ts_generate", "node_ts_confidence_eval")
    builder.add_conditional_edges("node_ts_confidence_eval", _should_retry_ts, {
        "retry_ts": "node_ts_generate",
        "next_ap": "node_ap_generate",
    })
    builder.add_edge("node_ap_generate", "node_ap_confidence_eval")
    builder.add_conditional_edges("node_ap_confidence_eval", _should_retry_ap, {
        "retry_ap": "node_ap_generate",
        "next_ir": "node_impact_rating",
    })
    builder.add_edge("node_impact_rating", "node_feasibility_rating")
    builder.add_edge("node_feasibility_rating", "node_risk_calc")
    builder.add_edge("node_risk_calc", "node_report")
    builder.add_edge("node_report", END)

    return builder.compile()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  第 10 部分：主入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_dir: str | None = None,
    topology_dir: str | None = None,
    external_interface_excel: str | None = None,
    asset_results: list | None = None,
    topology_json: dict | None = None,
    external_interface_json: list | None = None,
    output_path: str | None = None,
    build_rag_index: bool = True,
) -> list:
    logger.info("🚗 TARA 多智能体工作流启动")
    logger.info("=" * 60)

    logger.info(f"  并发配置: MAX_WORKERS={MAX_WORKERS}, BATCH_SIZE_RATING={BATCH_SIZE_RATING}")
    logger.info(f"  评估开关: ENABLE_EVALUATION={ENABLE_EVALUATION}")
    logger.info(f"  置信度阈值: CONFIDENCE_THRESHOLD={CONFIDENCE_THRESHOLD}")
    logger.info(f"  最大重试次数: MAX_CONFIDENCE_RETRIES={MAX_CONFIDENCE_RETRIES}")

    clear_prompt_cache()

    if ENABLE_RAG and build_rag_index:
        logger.info("[初始化] RAG 知识库")
        for cat, path in RAG_DIRS.items():
            os.makedirs(path, exist_ok=True)
        rag_kb.build_index()
    elif ENABLE_RAG:
        rag_kb.load_index()
    else:
        logger.info("[初始化] RAG 已禁用，跳过")

    if asset_results is None and data_flow_dir:
        logger.info("[输入解析] DCD数据流图")
        asset_results = parse_dcd_json(data_flow_dir)

    if topology_json is None and topology_dir:
        logger.info("[输入解析] 拓扑图")
        topology_json = parse_topology_json(topology_dir)

        logger.info("[输入解析] 外部接口信息")
        external_interface_json = parse_external_interface_excel(external_interface_excel)

    graph = build_tara_graph()

    initial_state: TaraState = {
        "data_flow_json": [],
        "topology_json": topology_json or {},
        "external_interface_json": external_interface_json or [],
        "asset_results": asset_results or [],
        "damage_scenarios": [], "ds_confidence": {}, "ds_retry_count": 0, "ds_feedback": {},
        "threat_scenarios": [], "ts_confidence": {}, "ts_retry_count": 0, "ts_feedback": {},
        "attack_paths": [],    "ap_confidence": {}, "ap_retry_count": 0, "ap_feedback": {},
        "impact_ratings": [], "feasibility_ratings": [],
        "risk_results": [], "tara_report": [],
        "manual_review_items": [],
    }

    logger.info("[执行] 开始运行 LangGraph 工作流")
    start_time = time.time()

    final_state = graph.invoke(initial_state)
    report = final_state["tara_report"]

    elapsed = time.time() - start_time
    logger.info(f"  ⏱ 总耗时: {elapsed:.1f}s ({elapsed/60:.1f}min)")

    if output_path:
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        logger.info(f"📄 TARA 报告已保存: {os.path.abspath(output_path)}")

    full_state_path = os.path.join(OUTPUT_DIR, "tara_full_state.json")
    with open(full_state_path, "w", encoding="utf-8") as f:
        json.dump({
            "asset_results": final_state.get("asset_results", []),
            "damage_scenarios": final_state.get("damage_scenarios", []),
            "threat_scenarios": final_state.get("threat_scenarios", []),
            "attack_paths": final_state.get("attack_paths", []),
            "impact_ratings": final_state.get("impact_ratings", []),
            "feasibility_ratings": final_state.get("feasibility_ratings", []),
            "risk_results": final_state.get("risk_results", []),
            "manual_review_items": final_state.get("manual_review_items", []),
        }, f, ensure_ascii=False, indent=2)
    logger.info(f"📄 完整中间状态已保存: {full_state_path}")

    manual_items = final_state.get("manual_review_items", [])
    if manual_items:
        logger.warning("=" * 60)
        logger.warning(f"⚠️  共 {len(manual_items)} 个场景需要人工审核:")
        for item in manual_items:
            logger.warning(f"  - [{item['type']}] ID: {item['id']}, 置信度: {item['confidence_score']}")
        logger.warning("=" * 60)

    logger.info("=" * 60)
    logger.info("✅ TARA 工作流执行完毕")
    return report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  主程序入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":
# 🔧 测试配置开关
    USE_SAMPLE_DATA = True  # 改为 False 则使用完整数据
    
    if USE_SAMPLE_DATA:
        # ===== 样本数据 =====
        sample_assets = [
            {
                "function": "交通标识及信号灯识别",
                "assets": [
                    {"asset_type": "信号", "asset_name": "弹窗/音效"},
                ],
            },
            {
                "function": "智能泊车辅助",
                "assets": [
                    {"asset_type": "数据", "asset_name": "用户账户信息"},
                ],
            },
        ]
    
        sample_topology = [{"id": 1, "CANFD": ["AMP", "CDC", "IC"]}, {"id": 2, "A2B": ["AMP", "CDC"]}, {"id": 3, "FPDLINK": ["CDC", "IC"]}, {"id": 4, "FPDLINK": ["CDC", "IVI"]}, {"id": 5, "GMSL": ["CDC", "DMS"]}, {"id": 6, "GMSL": ["CDC", "OMS"]}, {"id": 7, "GMSL": ["CDC", "MDC"]}, {"id": 8, "ETH": ["CDC", "CEM"]}, {"id": 9, "CANFD": ["BDCR", "CDC", "CEM", "REA"]}, {"id": 10, "CAN": ["BDCR", "BLE", "CEM", "CMSL", "CMSR", "DSM", "ECALL", "PDSM", "TRM", "WLCM"]}, {"id": 11, "CANFD": ["MDC", "USS ECU"]}, {"id": 12, "CANFD": ["FR", "MDC"]}, {"id": 13, "GMSL": ["LRC-FC", "MDC"]}, {"id": 14, "GMSL": ["MDC", "SRC-FC"]}, {"id": 15, "GMSL": ["MDC", "MRC-SC"]}, {"id": 16, "GMSL": ["MDC", "MRC-RC"]}, {"id": 17, "GMSL": ["FEC", "MDC"]}, {"id": 18, "ETH": ["CEM", "MDC"]}, {"id": 19, "CANFD": ["CEM", "EPS", "IBCU", "MDC", "SRS", "VCU", "VMC"]}, {"id": 20, "ETH": ["CEM", "VCU"]}, {"id": 21, "CANFD": ["BCU", "EVCC", "FMIPU", "HPMU", "ITMS", "PDU", "RDPEU", "RMIPU", "VCU"]}, {"id": 22, "CAN": ["BCU", "EVCC"]}, {"id": 23, "CAN": ["ACP", "ITMS", "PTC"]}, {"id": 24, "CANFD": ["BDCR", "CDC", "CEM", "LBMS"]}, {"id": 25, "ETH": ["Lidar", "MDC"]}, {"id": 26, "蜂窝": ["BCALL中心", "CDC", "ECALL中心", "华为车云", "阿维塔车云"]}, {"id": 27, "CAN": ["CEM", "诊断仪"]}]
    
        sample_ext_interfaces = [
            {"component": "CDC", "interfaces": ["Cellular Network  蜂窝网络", "WiFi", "Bluetooth", "USB", "SD卡"]},
            {"component": "BDC2.0", "interfaces": ["OBD  车载自动诊断系统"]},
            {"component": "UWB", "interfaces": ["RF  射频"]},
            {"component": "CHLIL", "interfaces": ["JTAG"]},
            {"component": "CHLIR", "interfaces": ["JTAG"]},
            {"component": "ITMS", "interfaces": ["JTAG"]},
        ]
    
        report = run_tara(
            asset_results=sample_assets,
            topology_json=sample_topology,
            external_interface_json=sample_ext_interfaces,
            output_path=os.path.join(OUTPUT_DIR, "tara_report_sample.json"),  # 区分输出文件
            build_rag_index=True,  # 样本测试关闭 RAG 索引加速
        )
    else:
        # ===== 完整数据 =====
        report = run_tara(
            data_flow_dir=DFD_DIR,
            topology_dir=TOPOLOGY_DIR,
            external_interface_excel=EXTERNAL_INTERFACE_EXCEL,
            output_path=os.path.join(OUTPUT_DIR, "tara_report.json"),
            build_rag_index=True,
        )
    
    logger.info(f"📋 最终报告包含 {len(report)} 个功能模块，已保存为 JSON 文件")


2026-04-17 13:58:38,047 [INFO] 🚗 TARA 多智能体工作流启动
2026-04-17 13:58:38,048 [INFO] ============================================================
2026-04-17 13:58:38,049 [INFO]   并发配置: MAX_WORKERS=5, BATCH_SIZE_RATING=5
2026-04-17 13:58:38,049 [INFO]   评估开关: ENABLE_EVALUATION=True
2026-04-17 13:58:38,050 [INFO]   置信度阈值: CONFIDENCE_THRESHOLD=85
2026-04-17 13:58:38,050 [INFO]   最大重试次数: MAX_CONFIDENCE_RETRIES=1
2026-04-17 13:58:38,051 [INFO] [初始化] RAG 知识库
2026-04-17 13:58:38,097 [INFO] [执行] 开始运行 LangGraph 工作流
2026-04-17 13:58:38,105 [INFO] ============================================================
2026-04-17 13:58:38,106 [INFO] [步骤1] 资产识别 (规则引擎)
2026-04-17 13:58:38,106 [INFO]   → 2 个功能, 2 个资产, 7 个安全属性
2026-04-17 13:58:38,109 [INFO]   💾 Checkpoint 已保存: D:\Jupyter profile\汽车信息安全风险评估\outputs\V8\checkpoint_step1_assets.json
2026-04-17 13:58:38,110 [INFO] [步骤2] 损害场景生成 (逐条并发)
2026-04-17 13:58:38,110 [INFO]   → 共 7 个子任务, 并发数 5
2026-04-17 13:59:39,442 [INFO] Retrying request to /chat/completions in 0

# v10
新框架

In [15]:

import copy
import glob
import hashlib
import json
import logging
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from difflib import SequenceMatcher
from threading import Lock, Semaphore
from typing import Any, Optional

try:
    import pandas as pd
except Exception:
    pd = None
from urllib import error as urlerror
from urllib import request as urlrequest


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1) 全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA-V10")


MODEL_NAME = os.getenv("TARA_MODEL", "qwen3.5-35b-a3b")
API_KEY = os.getenv("TARA_API_KEY", "sk-3458388d4845414387fe2e10bc8a9ee2")
API_BASE = os.getenv("TARA_API_BASE", "https://dashscope.aliyuncs.com/compatible-mode/v1")

if not API_KEY:
    logger.warning("未检测到环境变量 TARA_API_KEY。")

REQUEST_TIMEOUT = int(os.getenv("TARA_REQUEST_TIMEOUT", "90"))
MAX_TOKENS = int(os.getenv("TARA_MAX_TOKENS", "4096"))
TEMPERATURE = float(os.getenv("TARA_TEMPERATURE", "0.1"))

MAX_WORKERS = int(os.getenv("TARA_MAX_WORKERS", "6"))
MAX_CONCURRENT_LLM = int(os.getenv("TARA_MAX_CONCURRENT_LLM", "3"))
CALL_INTERVAL = float(os.getenv("TARA_CALL_INTERVAL", "0.4"))

BATCH_SIZE_INFLUENCE = int(os.getenv("TARA_BATCH_SIZE_INFLUENCE", "8"))
BATCH_SIZE_ATTACK = int(os.getenv("TARA_BATCH_SIZE_ATTACK", "8"))

# 可选：attack_potential / attack_vector / cvss
FEASIBILITY_METHOD = "attack_potential"

OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V11"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ENABLE_RAG = os.getenv("TARA_ENABLE_RAG", "1").strip() not in {"0", "false", "False"}
RAG_BASE_DIR = os.getenv("TARA_RAG_BASE_DIR", r"D:\Jupyter profile\rag")
RAG_DIRS = {
    "tara_reports": os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations": os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
RAG_TOP_K = int(os.getenv("TARA_RAG_TOP_K", "5"))
RAG_MAX_CONTEXT_CHARS = int(os.getenv("TARA_RAG_MAX_CHARS", "1800"))

llm_semaphore = Semaphore(MAX_CONCURRENT_LLM)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1.5) 计时工具
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class Timer:
    """分阶段计时器：with 块自动记录耗时并日志输出。"""

    def __init__(self, stage_name: str):
        self.stage_name = stage_name
        self.start = 0.0

    def __enter__(self):
        self.start = time.perf_counter()
        logger.info("═══ [计时] %s 开始 ═══", self.stage_name)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.perf_counter() - self.start
        if exc_type is None:
            logger.info("═══ [计时] %s 完成，耗时 %.2f 秒 (%.2f 分钟) ═══", self.stage_name, elapsed, elapsed / 60)
        else:
            logger.error("═══ [计时] %s 异常退出，耗时 %.2f 秒，错误: %s ═══", self.stage_name, elapsed, exc_val)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2) 业务常量
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性": "篡改",
    "机密性": "信息泄露",
    "可用性": "拒绝服务",
    "真实性": "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性": "越权",
}

# Attack Potential 五维取值规范（ISO/SAE 21434 常见量表）
AP_ALLOWED_VALUES = {
    "Exposure_time": [0, 1, 4, 17, 19],
    "Professional_experience": [0, 3, 6, 9],
    "Required_information": [0, 3, 7, 11],
    "Opportunity_window": [0, 1, 4, 10],
    "Required_equipment": [0, 4, 7, 9],
}

# CVSS 取值
CVSS_V = {
    "network": 0.85,
    "adjacent": 0.62,
    "local": 0.55,
    "physical": 0.20,
}
CVSS_C = {"low": 0.77, "high": 0.44}
CVSS_P = {"none": 0.85, "low": 0.62, "high": 0.27}
CVSS_U = {"none": 0.85, "required": 0.62}

# 攻击向量 -> 可行性等级映射（ISO 21434 G.4）
ATTACK_VECTOR_TO_LEVEL = {
    "network": "High",
    "adjacent": "Medium",
    "local": "Low",
    "physical": "Very Low",
}

RISK_MATRIX = {
    (4, "High"): 5, (4, "Medium"): 5, (4, "Low"): 4, (4, "Very Low"): 4,
    (3, "High"): 5, (3, "Medium"): 4, (3, "Low"): 4, (3, "Very Low"): 3,
    (2, "High"): 4, (2, "Medium"): 4, (2, "Low"): 3, (2, "Very Low"): 2,
    (1, "High"): 3, (1, "Medium"): 2, (1, "Low"): 2, (1, "Very Low"): 1,
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3) 语义缓存
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SemanticCache:
    """轻量语义缓存：规范化文本 + 相似度匹配。"""

    def __init__(self, similarity_threshold: float = 0.985, max_bucket_size: int = 1000):
        self.similarity_threshold = similarity_threshold
        self.max_bucket_size = max_bucket_size
        self._store: dict[str, list[dict]] = defaultdict(list)
        self._lock = Lock()

    @staticmethod
    def _normalize(obj: Any) -> str:
        if isinstance(obj, (dict, list, tuple)):
            text = json.dumps(obj, ensure_ascii=False, sort_keys=True)
        else:
            text = str(obj)
        text = text.lower().strip()
        text = re.sub(r"\s+", "", text)
        text = text.replace("：", ":")
        return text

    def get(self, bucket: str, key_obj: Any) -> Optional[Any]:
        signature = self._normalize(key_obj)
        key_hash = hashlib.sha256(signature.encode("utf-8")).hexdigest()

        with self._lock:
            items = self._store.get(bucket, [])
            for item in items:
                if item["hash"] == key_hash:
                    return copy.deepcopy(item["value"])
                ratio = SequenceMatcher(None, signature, item["signature"]).ratio()
                if ratio >= self.similarity_threshold:
                    return copy.deepcopy(item["value"])
        return None

    def set(self, bucket: str, key_obj: Any, value: Any) -> None:
        signature = self._normalize(key_obj)
        key_hash = hashlib.sha256(signature.encode("utf-8")).hexdigest()

        with self._lock:
            items = self._store[bucket]
            for item in items:
                if item["hash"] == key_hash:
                    item["value"] = copy.deepcopy(value)
                    return
            items.append({"hash": key_hash, "signature": signature, "value": copy.deepcopy(value)})
            if len(items) > self.max_bucket_size:
                del items[0 : len(items) - self.max_bucket_size]


semantic_cache = SemanticCache()

# 攻击路径缓存：相同 asset_name 的攻击路径相同，跨功能/属性复用
_attack_path_cache: dict[str, list[str]] = {}
_attack_path_cache_lock = Lock()


def _get_attack_path_cache(asset_name: str) -> list[str] | None:
    with _attack_path_cache_lock:
        return copy.deepcopy(_attack_path_cache.get(asset_name))


def _set_attack_path_cache(asset_name: str, paths: list[str]) -> None:
    with _attack_path_cache_lock:
        _attack_path_cache[asset_name] = copy.deepcopy(paths)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.5) 轻量 RAG（兼容无第三方依赖环境）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SimpleRAGKnowledgeBase:
    def __init__(self, rag_dirs: dict[str, str]):
        self.rag_dirs = rag_dirs
        self.docs: list[dict] = []
        self._loaded = False
        self._lock = Lock()

    def _safe_read_text(self, path: str) -> str:
        for enc in ("utf-8", "utf-8-sig", "gbk", "gb18030"):
            try:
                with open(path, "r", encoding=enc) as f:
                    return f.read()
            except Exception:
                continue
        return ""

    def _load_json_text(self, path: str) -> str:
        try:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, (dict, list)):
                return json.dumps(obj, ensure_ascii=False)
            return str(obj)
        except Exception:
            return self._safe_read_text(path)

    def load(self, force: bool = False) -> None:
        with self._lock:
            if self._loaded and not force:
                return
            self.docs = []
            for category, d in self.rag_dirs.items():
                if not os.path.isdir(d):
                    continue
                for p in glob.glob(os.path.join(d, "**", "*"), recursive=True):
                    if not os.path.isfile(p):
                        continue
                    ext = os.path.splitext(p)[1].lower()
                    if ext in {".txt", ".md"}:
                        txt = self._safe_read_text(p)
                    elif ext == ".json":
                        txt = self._load_json_text(p)
                    else:
                        continue
                    txt = (txt or "").strip()
                    if len(txt) < 20:
                        continue
                    self.docs.append(
                        {
                            "category": category,
                            "path": p,
                            "text": txt[:12000],  # 限制单文档最大长度
                        }
                    )
            self._loaded = True
            logger.info(f"RAG文档加载完成: {len(self.docs)} 条")

    def retrieve(self, query: str, top_k: int = 5, max_chars: int = 1800) -> str:
        if not ENABLE_RAG:
            return "[RAG已关闭]"
        if not self._loaded:
            self.load()
        if not self.docs:
            return "[RAG为空]"

        q = (query or "").strip().lower()
        q_tokens = [t for t in re.split(r"[\s,，。;；:：|/\\-]+", q) if t]
        if not q_tokens:
            q_tokens = [q]

        scored = []
        for d in self.docs:
            text = d["text"].lower()
            score = 0
            for t in q_tokens:
                if t and t in text:
                    score += 3
            # 轻微偏置：法规库在场景生成里常更有用
            if d["category"] == "regulations":
                score += 1
            if score > 0:
                scored.append((score, d))

        if not scored:
            # 没命中时给前几个法规类兜底
            fallback = [d for d in self.docs if d["category"] == "regulations"][:top_k]
            scored_docs = fallback if fallback else self.docs[:top_k]
        else:
            scored.sort(key=lambda x: x[0], reverse=True)
            scored_docs = [d for _, d in scored[:top_k]]

        chunks = []
        total = 0
        for d in scored_docs:
            snippet = d["text"][:500].strip()
            line = f"[来源:{d['category']}] {snippet}"
            if total + len(line) > max_chars:
                break
            chunks.append(line)
            total += len(line)

        return "\n---\n".join(chunks) if chunks else "[RAG未命中]"


rag_kb = SimpleRAGKnowledgeBase(RAG_DIRS)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4) 数据结构
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class TaskUnit:
    task_id: str
    function: str
    asset_name: str
    asset_type: str
    attribute_name: str
    threat_type: str
    target: str = ""
    source: str = ""

    damage_scenario: str = ""
    threat_scenarios: list[str] = field(default_factory=list)
    attack_paths: list[str] = field(default_factory=list)
    selected_path_frameworks: list[dict] = field(default_factory=list)

    influence_parameters: dict = field(default_factory=dict)
    impact_value: int = 1
    impact_level: str = "Negligible"

    attack_details: list[dict] = field(default_factory=list)

    risk_value: int = 1
    risk_treatment: str = "接受"

    errors: list[str] = field(default_factory=list)


def _task_to_dict(t: TaskUnit) -> dict:
    return {
        "task_id": t.task_id,
        "function": t.function,
        "asset_name": t.asset_name,
        "asset_type": t.asset_type,
        "attribute_name": t.attribute_name,
        "threat_type": t.threat_type,
        "target": t.target,
        "source": t.source,
        "damage_scenario": t.damage_scenario,
        "threat_scenarios": t.threat_scenarios,
        "attack_paths": t.attack_paths,
        "selected_path_frameworks": t.selected_path_frameworks,
        "influence_parameters": t.influence_parameters,
        "impact_value": t.impact_value,
        "impact_level": t.impact_level,
        "attack_details": t.attack_details,
        "risk_value": t.risk_value,
        "risk_treatment": t.risk_treatment,
        "errors": t.errors,
    }


def _task_from_dict(d: dict) -> TaskUnit:
    return TaskUnit(
        task_id=d.get("task_id", ""),
        function=d.get("function", ""),
        asset_name=d.get("asset_name", ""),
        asset_type=d.get("asset_type", ""),
        attribute_name=d.get("attribute_name", ""),
        threat_type=d.get("threat_type", ""),
        target=d.get("target", ""),
        source=d.get("source", ""),
        damage_scenario=d.get("damage_scenario", ""),
        threat_scenarios=list(d.get("threat_scenarios", [])),
        attack_paths=list(d.get("attack_paths", [])),
        selected_path_frameworks=list(d.get("selected_path_frameworks", [])),
        influence_parameters=dict(d.get("influence_parameters", {})),
        impact_value=int(d.get("impact_value", 1)),
        impact_level=str(d.get("impact_level", "Negligible")),
        attack_details=list(d.get("attack_details", [])),
        risk_value=int(d.get("risk_value", 1)),
        risk_treatment=str(d.get("risk_treatment", "接受")),
        errors=list(d.get("errors", [])),
    )


def _tasks_to_dict(tasks: list[TaskUnit]) -> list[dict]:
    return [_task_to_dict(t) for t in tasks]


def _tasks_from_dict(data: list[dict]) -> list[TaskUnit]:
    return [_task_from_dict(d) for d in data]


@dataclass
class TopologyElementVO:
    id: str
    name: str = ""
    type: int | None = None
    color: str = ""
    source_id: str = ""
    target_id: str = ""
    is_gateway: bool = False
    ids: list[str] = field(default_factory=list)


@dataclass
class PathNodeVO:
    component_name_id: str
    component_name: str
    pre_component_name_id: str
    pre_component_name: str
    line_id: str
    color: str
    is_gateway: bool


class TopologyElementType:
    COMPONENT = 1
    GATEWAY = 2
    EXTERNAL_COMPONENT = 3
    LINE = 4


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5) 通用工具
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: Any) -> Any:
    if text is None:
        raise ValueError("输入为空")
    if not isinstance(text, str):
        text = str(text)

    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()

    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                continue

    raise ValueError(f"JSON解析失败，原文前200字: {cleaned[:200]}")


def llm_call_with_semaphore(system_prompt: str, user_prompt: str, retries: int = 2) -> str:
    with llm_semaphore:
        for attempt in range(retries + 1):
            try:
                endpoint = API_BASE.rstrip("/") + "/chat/completions"
                payload = {
                    "model": MODEL_NAME,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "extra_body": {"enable_thinking": False},
                }
                body = json.dumps(payload).encode("utf-8")
                req = urlrequest.Request(
                    endpoint,
                    data=body,
                    headers={
                        "Authorization": f"Bearer {API_KEY}",
                        "Content-Type": "application/json",
                    },
                    method="POST",
                )
                with urlrequest.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
                    data = json.loads(resp.read().decode("utf-8"))
                content = (
                    data.get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                )
                return str(content) if content is not None else ""
            except (urlerror.URLError, TimeoutError) as e:
                logger.warning(f"LLM网络错误: {type(e).__name__}: {e}")
            except Exception as e:
                logger.warning(f"LLM调用异常: {type(e).__name__}: {e}")

            if attempt < retries:
                wait = 2 ** attempt
                time.sleep(wait)

    return "{}"


def cached_llm_json(
    *,
    bucket: str,
    cache_key: Any,
    system_prompt: str,
    user_prompt: str,
    default: Any,
) -> tuple[Any, bool]:
    cached = semantic_cache.get(bucket, cache_key)
    if cached is not None:
        return cached, True

    raw = llm_call_with_semaphore(system_prompt, user_prompt)
    try:
        parsed = safe_parse_json(raw)
    except Exception:
        parsed = copy.deepcopy(default)

    semantic_cache.set(bucket, cache_key, parsed)
    return parsed, False


def normalize_attack_vector(value: str) -> str:
    s = (value or "").strip().lower()
    # Network（远程/蜂窝/卫星）
    if any(k in s for k in ["network", "网络", "remote", "ota", "蜂窝", "4g", "5g", "lte", "ethernet", "以太网", "cloud", "卫星定位"]):
        return "network"
    # Adjacent（短距无线）
    if any(k in s for k in ["adjacent", "相邻", "bluetooth", "蓝牙", "ble", "wifi", "wlan", "星闪", "v2x", "射频信道"]):
        return "adjacent"
    # Local（物理端口/本地接入）
    if any(k in s for k in ["local", "本地", "usb", "tf卡", "sd卡", "ic卡", "充电口", "obd", "内部通讯", "nfc", "登录"]):
        return "local"
    # Physical（侵入式调试）
    if any(k in s for k in ["physical", "物理", "debug", "jtag", "内部调试"]):
        return "physical"
    return "local"


def calculate_impact_value(influence_parameters: dict) -> int:
    if not influence_parameters:
        return 1
    v = max(
        int(influence_parameters.get("Safety", 0)),
        int(influence_parameters.get("Finance", 0)),
        int(influence_parameters.get("Operation", 0)),
        int(influence_parameters.get("Privacy", 0)),
    )
    return max(1, min(4, v))


def calculate_impact_level(impact_value: int) -> str:
    return {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}.get(impact_value, "Negligible")


def calculate_attack_potential_score(params: dict) -> int:
    return sum(int(params.get(k, 0)) for k in AP_ALLOWED_VALUES.keys())


def calculate_attack_potential_level(total_score: int) -> str:
    # 用户要求的映射
    if total_score <= 9:
        return "High"
    if total_score <= 13:
        return "Medium"
    if total_score <= 19:
        return "Low"
    return "Very Low"


def calculate_cvss_score(params: dict) -> float:
    v = CVSS_V.get(normalize_attack_vector(params.get("攻击向量", "local")), 0.55)
    c = CVSS_C.get((params.get("攻击复杂度", "high") or "high").strip().lower(), 0.44)
    p = CVSS_P.get((params.get("权限要求", "none") or "none").strip().lower(), 0.85)
    u = CVSS_U.get((params.get("用户交互", "none") or "none").strip().lower(), 0.85)
    score = 8.22 * v * c * p * u
    return round(score, 2)


def calculate_cvss_level(score: float) -> str:
    if 2.96 <= score <= 3.89:
        return "High"
    if 2.00 <= score <= 2.95:
        return "Medium"
    if 1.06 <= score <= 1.99:
        return "Low"
    if 0.12 <= score <= 1.05:
        return "Very Low"
    return "Very Low" if score < 2.96 else "High"


def calculate_attack_vector_level(vector: str) -> str:
    return ATTACK_VECTOR_TO_LEVEL.get(normalize_attack_vector(vector), "Low")


def calculate_risk_value(impact_value: int, feasibility_level: str) -> int:
    return RISK_MATRIX.get((impact_value, feasibility_level), 1)


def get_risk_treatment(risk_value: int) -> str:
    if risk_value >= 5:
        return "风险缓解"
    if risk_value == 4:
        return "风险缓解"
    if risk_value == 3:
        return "风险缓解"
    if risk_value == 2:
        return "风险接受"
    return "风险接受"


def save_json(data: Any, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5.5) Checkpoint（中间结果检查点）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CHECKPOINT_DIR = os.getenv("TARA_CHECKPOINT_DIR", os.path.join(OUTPUT_DIR, ".checkpoints"))
_CHECKPOINT_META_FILE = os.path.join(CHECKPOINT_DIR, "_meta.json")
_CHECKPOINT_FILES = {
    "stage_A_gen": "tasks_after_stageA.json",
    "stage_B_impact": "tasks_after_stageB.json",
    "stage_C_attack": "tasks_after_stageC.json",
}


def _checkpoint_path(stage: str) -> str:
    return os.path.join(CHECKPOINT_DIR, _CHECKPOINT_FILES[stage])


def _checkpoint_meta(method: str, total_tasks: int) -> dict:
    """生成当前运行的元信息快照。"""
    return {
        "method": method,
        "total_tasks": total_tasks,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "checkpoint_version": "v10.1",
    }


def save_checkpoint(tasks: list[TaskUnit], stage: str, method: str) -> str:
    """将 tasks 序列化保存到对应阶段的 checkpoint 文件。"""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    path = _checkpoint_path(stage)
    data = {
        "meta": _checkpoint_meta(method, len(tasks)),
        "tasks": _tasks_to_dict(tasks),
    }
    save_json(data, path)
    logger.info("[Checkpoint] %s 已保存 -> %s (%d 个任务)", stage, path, len(tasks))
    return path


def load_checkpoint(stage: str) -> list[TaskUnit] | None:
    """加载指定阶段的 checkpoint，不存在或损坏时返回 None。"""
    path = _checkpoint_path(stage)
    if not os.path.isfile(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tasks_data = data.get("tasks", [])
        if not tasks_data:
            logger.warning("[Checkpoint] %s 文件为空", stage)
            return None
        tasks = _tasks_from_dict(tasks_data)
        logger.info("[Checkpoint] %s 已加载 <- %s (%d 个任务)", stage, path, len(tasks))
        return tasks
    except Exception as e:
        logger.warning("[Checkpoint] %s 加载失败: %s，将重新运行该阶段", stage, e)
        return None


def resume_from_checkpoint(
    stages: list[str],
    method: str,
) -> tuple[int, list[TaskUnit] | None]:
    """按优先级检查已有的 checkpoint，找到第一个有效的 checkpoint。

    Args:
        stages: 按优先级排列的阶段名列表（从最新到最旧）。
        method: 期望的可行性方法。

    Returns:
        (stage_index, tasks) — stage_index 是需要继续的阶段的索引（0=S的开始），
        tasks 为已有 checkpoint 的 task 列表（stage_index > 0 时非 None）。
    """
    for i, stage in enumerate(stages):
        tasks = load_checkpoint(stage)
        if tasks is not None:
            # 校验元信息（方法是否一致）
            meta_path = _checkpoint_path(stage)
            try:
                with open(meta_path, "r", encoding="utf-8") as f:
                    meta = json.load(f).get("meta", {})
                saved_method = meta.get("method", "")
                if saved_method and saved_method != method:
                    logger.warning(
                        "[Checkpoint] %s 的方法(%s)与当前(%s)不一致，忽略",
                        stage, saved_method, method,
                    )
                    continue
            except Exception:
                pass

            # stage 在 stages 列表中的索引就是还需要跑的阶段范围
            remain_idx = i + 1
            if remain_idx >= len(stages):
                logger.info("[Checkpoint] 所有阶段已完成，直接使用最终结果")
                return remain_idx, tasks
            logger.info(
                "[Checkpoint] 从 %s checkpoint 恢复，将运行剩余 %d 个阶段",
                stage, len(stages) - remain_idx,
            )
            return remain_idx, tasks

    return 0, None


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6) 输入解析（DFD.py 逻辑）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_dfd_json(input_dir: str) -> list[dict]:
    type_mapping = {
        "tm.Flow": "信号",
        "tm.Process": "部件",
        "tm.Store": "数据",
        "tm.Actor": "接口",
    }

    all_results: list[dict] = []
    json_files = glob.glob(os.path.join(input_dir, "*.json"))

    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except Exception:
                logger.warning(f"DFD解析失败，跳过: {os.path.basename(file_path)}")
                continue

        current_result = {"function": "", "assets": []}

        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not current_result["function"] and "title" in diagram:
                    current_result["function"] = diagram["title"]

                if "diagramJson" not in diagram or "cells" not in diagram["diagramJson"]:
                    continue

                cells = diagram["diagramJson"]["cells"]
                id_to_cell = {c.get("id"): c for c in cells if c.get("id")}

                for cell in cells:
                    if cell.get("outOfScope") is True:
                        continue

                    raw_type = cell.get("type", "")
                    if raw_type == "tm.Boundary":
                        continue

                    asset_details = ""
                    finetermval_value = ""
                    for key, value in cell.items():
                        if key.startswith("propertyList") and isinstance(value, dict):
                            asset_details = value.get("assetDetails", "")
                            finetermval_value = value.get("finetermval", "")
                            break

                    if not finetermval_value:
                        continue

                    mapped_type = type_mapping.get(raw_type, raw_type)
                    target_value = ""
                    source_value = ""

                    if mapped_type == "部件":
                        target_value = finetermval_value

                    elif mapped_type == "数据":
                        cell_id = cell.get("id", "")
                        for flow_cell in cells:
                            if flow_cell.get("type") == "tm.Flow" and flow_cell.get("source", {}).get("id") == cell_id:
                                target_id = flow_cell.get("target", {}).get("id", "")
                                if target_id in id_to_cell:
                                    target_cell = id_to_cell[target_id]
                                    for k, v in target_cell.items():
                                        if k.startswith("propertyList") and isinstance(v, dict):
                                            target_value = v.get("finetermval", "")
                                            if target_value:
                                                break
                                if target_value:
                                    break

                    elif mapped_type == "信号":
                        target_id = cell.get("target", {}).get("id", "")
                        source_id = cell.get("source", {}).get("id", "")

                        if target_id in id_to_cell:
                            for k, v in id_to_cell[target_id].items():
                                if k.startswith("propertyList") and isinstance(v, dict):
                                    target_value = v.get("finetermval", "")
                                    if target_value:
                                        break

                        if source_id in id_to_cell:
                            for k, v in id_to_cell[source_id].items():
                                if k.startswith("propertyList") and isinstance(v, dict):
                                    source_value = v.get("finetermval", "")
                                    if source_value:
                                        break

                    item = {
                        "asset_type": mapped_type,
                        "asset_name": finetermval_value,
                        "assetDetails": asset_details,
                    }
                    if target_value:
                        item["target"] = target_value
                    if source_value:
                        item["source"] = source_value

                    current_result["assets"].append(item)

        if not current_result["function"]:
            current_result["function"] = os.path.splitext(os.path.basename(file_path))[0]

        if current_result["assets"]:
            all_results.append(current_result)

    logger.info(f"DFD解析完成: {len(all_results)} 个功能")
    return all_results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7) 拓扑解析
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TopologyMapUtil:
    @staticmethod
    def list_element_vo(cells: list[dict]) -> list[TopologyElementVO]:
        result: list[TopologyElementVO] = []
        for cell in cells:
            ele = TopologyElementVO(id=cell.get("id"))
            shape = cell.get("shape", "")
            attrs = cell.get("attrs", {})

            if shape == "custom-rounded-rect":
                stroke = attrs.get("body", {}).get("stroke", "")
                if stroke == "#016AFF":
                    ele.type = TopologyElementType.GATEWAY
                    ele.is_gateway = True
                else:
                    ele.type = TopologyElementType.COMPONENT
                ele.color = stroke
                ele.name = attrs.get("text", {}).get("text", "")

            elif shape == "custom-rounded-rect-dash":
                ele.type = TopologyElementType.EXTERNAL_COMPONENT
                ele.color = attrs.get("body", {}).get("stroke", "")
                ele.name = attrs.get("text", {}).get("text", "")

            elif shape == "edge":
                ele.type = TopologyElementType.LINE
                ele.color = attrs.get("line", {}).get("stroke", "")
                ele.source_id = cell.get("source", {}).get("cell")
                ele.target_id = cell.get("target", {}).get("cell")

            result.append(ele)
        return result

    @staticmethod
    def get_distinct_components(element_vos: list[TopologyElementVO]) -> list[TopologyElementVO]:
        components = [
            e for e in element_vos
            if e.type in [
                TopologyElementType.COMPONENT,
                TopologyElementType.GATEWAY,
                TopologyElementType.EXTERNAL_COMPONENT,
            ] and e.name
        ]

        name_map: dict[str, list[TopologyElementVO]] = defaultdict(list)
        for c in components:
            name_map[c.name].append(c)

        distinct: list[TopologyElementVO] = []
        for _, elements in name_map.items():
            first = elements[0]
            first.ids = [e.id for e in elements]
            distinct.append(first)
        return distinct

    @staticmethod
    def generate_full_path(
        external_interfaces: list[dict],
        element_vos: list[TopologyElementVO],
        protocol_legends: list[dict],
        distinct_components: list[TopologyElementVO],
    ) -> list[list[PathNodeVO]]:
        many_to_one: dict[str, str] = {}
        for comp in distinct_components:
            for old_id in comp.ids:
                many_to_one[old_id] = comp.id

        valid_colors = [p.get("color") for p in protocol_legends if p.get("color")]
        lines = [e for e in element_vos if e.type == TopologyElementType.LINE and e.color in valid_colors]

        for line in lines:
            line.source_id = many_to_one.get(line.source_id, line.source_id)
            line.target_id = many_to_one.get(line.target_id, line.target_id)

        ext_names = [ext["related_component"] for ext in external_interfaces]
        ext_ids = [c.id for c in distinct_components if c.name in ext_names]

        com_to_lines: dict[str, set] = defaultdict(set)
        line_to_coms: dict[str, set] = defaultdict(set)

        for line in lines:
            if line.source_id:
                com_to_lines[line.source_id].add(line.color)
                line_to_coms[line.color].add(line.source_id)
            if line.target_id:
                com_to_lines[line.target_id].add(line.color)
                line_to_coms[line.color].add(line.target_id)

        starts = {k: v for k, v in com_to_lines.items() if k in ext_ids}
        com_dic = {c.id: c for c in distinct_components}

        return TopologyMapUtil.get_path_by_bfs(starts, com_to_lines, line_to_coms, com_dic)

    @staticmethod
    def get_path_by_bfs(
        starts: dict[str, set],
        com_to_lines: dict[str, set],
        line_to_coms: dict[str, set],
        com_dic: dict[str, TopologyElementVO],
    ) -> list[list[PathNodeVO]]:
        all_paths: list[list[PathNodeVO]] = []

        for start_id in starts.keys():
            queue = deque([(start_id, set(), [])])
            while queue:
                current_com_id, visited, path = queue.popleft()

                direct_points = set()
                line_ids = com_to_lines.get(current_com_id, set())
                for l_id in line_ids:
                    direct_points.update(line_to_coms.get(l_id, set()))

                for l_id in line_ids:
                    if l_id in visited:
                        continue

                    for next_com in line_to_coms.get(l_id, set()):
                        if next_com not in com_dic or next_com == current_com_id or next_com in visited:
                            continue

                        node = PathNodeVO(
                            component_name_id=next_com,
                            component_name=com_dic[next_com].name,
                            pre_component_name_id=current_com_id,
                            pre_component_name=com_dic[current_com_id].name,
                            line_id=l_id,
                            color=l_id,
                            is_gateway=com_dic[next_com].is_gateway,
                        )

                        new_path = copy.deepcopy(path)
                        new_path.append(node)
                        all_paths.append(new_path)

                        new_visited = set(visited)
                        new_visited.update(line_ids)
                        new_visited.update(direct_points)
                        queue.append((next_com, new_visited, new_path))

        return all_paths


def _determine_attack_vector(ext_interface: str, color_name_map: dict, path_nodes: list[PathNodeVO]) -> str:
    interface_lower = (ext_interface or "").lower()

    # Network
    if any(k in interface_lower for k in ["4g", "5g", "lte", "蜂窝", "ota", "cloud", "远程", "ethernet", "以太网", "卫星定位"]):
        return "network"
    # Adjacent
    if any(k in interface_lower for k in ["wifi", "wlan", "蓝牙", "bluetooth", "ble", "星闪", "v2x", "射频信道"]):
        return "adjacent"
    # Local
    if any(k in interface_lower for k in ["usb", "tf卡", "sd卡", "ic卡", "充电口", "obd", "内部通讯", "nfc", "local", "本地登录"]):
        return "local"
    # Physical
    if any(k in interface_lower for k in ["jtag", "debug", "内部调试", "uart", "spi", "i2c", "物理"]):
        return "physical"

    if path_nodes:
        protocol_name = color_name_map.get(path_nodes[0].color, "").lower()
        if any(k in protocol_name for k in ["wifi", "wlan", "4g", "5g", "ethernet", "以太网"]):
            return "network"
        if any(k in protocol_name for k in ["蓝牙", "ble", "nfc"]):
            return "adjacent"
        if any(k in protocol_name for k in ["usb", "obd"]):
            return "local"

    return "local"


def _norm_text(v: Any) -> str:
    if v is None:
        return ""
    s = str(v).replace("\n", " ").replace("\r", " ").replace("\u3000", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def _is_checked(v: Any) -> bool:
    s = _norm_text(v).lower()
    if not s:
        return False
    if any(mark in s for mark in ["√", "✓", "☑", "✅"]):
        return True
    return s in {"1", "y", "yes", "true", "是", "有", "开", "通过"}


def _excel_col_to_idx(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha()).upper()
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord("A") + 1)
    return max(0, idx - 1)


def _parse_xlsx_rows_stdlib(excel_path: str) -> list[dict]:
    """
    无 pandas/openpyxl 时，使用标准库解析 xlsx。
    返回：[{row_num:int, cells:{col_idx: value}}...]
    """
    ns = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rel_ns = {"r": "http://schemas.openxmlformats.org/package/2006/relationships"}

    with zipfile.ZipFile(excel_path, "r") as zf:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall(".//x:si", ns):
                text = "".join((t.text or "") for t in si.findall(".//x:t", ns))
                shared_strings.append(text)

        # 找第一个工作表文件路径
        sheet_target = "worksheets/sheet1.xml"
        try:
            wb_root = ET.fromstring(zf.read("xl/workbook.xml"))
            sheets = wb_root.findall(".//x:sheets/x:sheet", ns)
            first_sheet = sheets[0] if sheets else None
            rid = first_sheet.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id") if first_sheet is not None else None
            if rid and "xl/_rels/workbook.xml.rels" in zf.namelist():
                rel_root = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
                for rel in rel_root.findall(".//r:Relationship", rel_ns):
                    if rel.attrib.get("Id") == rid:
                        sheet_target = rel.attrib.get("Target", sheet_target)
                        break
        except Exception:
            pass

        sheet_path = "xl/" + sheet_target.lstrip("/")
        if sheet_path not in zf.namelist():
            # 兜底
            candidates = [n for n in zf.namelist() if n.startswith("xl/worksheets/sheet") and n.endswith(".xml")]
            if not candidates:
                return []
            sheet_path = sorted(candidates)[0]

        sheet_root = ET.fromstring(zf.read(sheet_path))
        out_rows: list[dict] = []

        for row in sheet_root.findall(".//x:sheetData/x:row", ns):
            row_num = int(row.attrib.get("r", "0") or 0)
            cells: dict[int, str] = {}
            for c in row.findall("x:c", ns):
                ref = c.attrib.get("r", "")
                col_idx = _excel_col_to_idx(ref) if ref else 0
                ctype = c.attrib.get("t", "")
                value = ""
                if ctype == "s":
                    v = c.find("x:v", ns)
                    if v is not None and v.text and v.text.isdigit():
                        ss_idx = int(v.text)
                        if 0 <= ss_idx < len(shared_strings):
                            value = shared_strings[ss_idx]
                elif ctype == "inlineStr":
                    is_node = c.find("x:is", ns)
                    if is_node is not None:
                        value = "".join((t.text or "") for t in is_node.findall(".//x:t", ns))
                else:
                    v = c.find("x:v", ns)
                    value = v.text if (v is not None and v.text is not None) else ""
                cells[col_idx] = _norm_text(value)
            out_rows.append({"row_num": row_num, "cells": cells})

        return out_rows


def _load_external_interfaces(excel_path: str) -> list[dict]:
    """读取外部接口清单：pandas -> openpyxl -> 标准库 xlsx 解析。"""
    # 1) pandas 路径
    if pd is not None:
        try:
            df = pd.read_excel(excel_path, header=1)
            df.columns = [_norm_text(c) for c in df.columns]

            direct_info_col = next((c for c in df.columns if "外部接口" in c and "信息" in c), None)
            direct_comp_col = next((c for c in df.columns if "关联部件" in c), None)
            if direct_info_col and direct_comp_col:
                direct_items = []
                for _, row in df.iterrows():
                    ext = _norm_text(row.get(direct_info_col, ""))
                    comp = _norm_text(row.get(direct_comp_col, ""))
                    if ext and comp and comp.lower() != "nan":
                        direct_items.append(
                            {
                                "external_interface": ext,
                                "related_component": comp,
                                "all_interfaces": [ext],
                            }
                        )
                if direct_items:
                    return direct_items

            comp_col = next((c for c in df.columns if "零部件" in c and "名称" in c), None)
            if comp_col is None and len(df.columns) >= 2:
                comp_col = df.columns[1]
            if comp_col is None:
                comp_col = df.columns[0] if len(df.columns) else None

            if comp_col is None:
                return []

            interface_columns = [c for c in df.columns if c != comp_col and _norm_text(c)]
            # 对齐原逻辑：倾向跳过前两个说明列
            if len(df.columns) > 2:
                interface_columns = [c for c in df.columns[2:] if c != comp_col and _norm_text(c)]

            external_interfaces: list[dict] = []
            for _, row in df.iterrows():
                comp = _norm_text(row.get(comp_col, ""))
                if not comp or comp.lower() == "nan":
                    continue
                active = [col for col in interface_columns if _is_checked(row.get(col, ""))]
                if active:
                    external_interfaces.append(
                        {
                            "external_interface": active[0],
                            "related_component": comp,
                            "all_interfaces": active,
                        }
                    )
            if external_interfaces:
                return external_interfaces
        except Exception as e:
            logger.warning(f"pandas读取外部接口失败，尝试其他方式: {e}")

    # 2) openpyxl 路径
    try:
        from openpyxl import load_workbook

        wb = load_workbook(excel_path, data_only=True)
        ws = wb.active

        # 先尝试“外部接口信息 + 外部接口关联部件”直接映射格式
        direct_header_row = None
        info_idx = None
        comp_idx_direct = None
        for r in range(1, min(ws.max_row, 10) + 1):
            vals = [_norm_text(c.value) for c in ws[r]]
            for i, v in enumerate(vals):
                if "外部接口" in v and "信息" in v:
                    info_idx = i
                if "关联部件" in v:
                    comp_idx_direct = i
            if info_idx is not None and comp_idx_direct is not None:
                direct_header_row = r
                break

        if direct_header_row is not None:
            direct_items = []
            for row in ws.iter_rows(min_row=direct_header_row + 1, values_only=True):
                vals = [_norm_text(v) for v in row]
                ext = vals[info_idx] if info_idx < len(vals) else ""
                comp = vals[comp_idx_direct] if comp_idx_direct < len(vals) else ""
                if ext and comp:
                    direct_items.append(
                        {
                            "external_interface": ext,
                            "related_component": comp,
                            "all_interfaces": [ext],
                        }
                    )
            if direct_items:
                return direct_items

        # 优先寻找包含“零部件名称”的表头行
        header_row_idx = None
        comp_idx = None
        max_scan = min(ws.max_row, 20)
        for r in range(1, max_scan + 1):
            vals = [_norm_text(c.value) for c in ws[r]]
            for i, v in enumerate(vals):
                if "零部件" in v and "名称" in v:
                    header_row_idx = r
                    comp_idx = i
                    break
            if header_row_idx is not None:
                break

        if header_row_idx is None:
            header_row_idx = 2
            vals = [_norm_text(c.value) for c in ws[header_row_idx]]
            comp_idx = vals.index("零部件名称") if "零部件名称" in vals else 1

        header_vals = [_norm_text(c.value) for c in ws[header_row_idx]]
        interface_cols = [(i, h) for i, h in enumerate(header_vals) if i > comp_idx and h]

        external_interfaces: list[dict] = []
        for row in ws.iter_rows(min_row=header_row_idx + 1, values_only=True):
            row_vals = [_norm_text(v) for v in row]
            if comp_idx >= len(row_vals):
                continue
            comp = row_vals[comp_idx]
            if not comp:
                continue
            active = [h for i, h in interface_cols if i < len(row_vals) and _is_checked(row_vals[i])]
            if active:
                external_interfaces.append(
                    {
                        "external_interface": active[0],
                        "related_component": comp,
                        "all_interfaces": active,
                    }
                )
        if external_interfaces:
            return external_interfaces
    except Exception:
        pass

    # 3) 标准库兜底（无 pandas / openpyxl）
    try:
        rows = _parse_xlsx_rows_stdlib(excel_path)
    except Exception as e:
        logger.warning(f"标准库解析Excel失败: {e}")
        return []

    if not rows:
        return []

    # A) “外部接口信息 + 外部接口关联部件”直接映射格式
    direct_header = None
    info_idx = None
    comp_idx_direct = None
    for r in rows[:20]:
        for idx, v in r["cells"].items():
            if "外部接口" in v and "信息" in v:
                info_idx = idx
            if "关联部件" in v:
                comp_idx_direct = idx
        if info_idx is not None and comp_idx_direct is not None:
            direct_header = r["row_num"]
            break

    if direct_header is not None:
        direct_items = []
        for r in rows:
            if r["row_num"] <= direct_header:
                continue
            cells = r["cells"]
            ext = _norm_text(cells.get(info_idx, ""))
            comp = _norm_text(cells.get(comp_idx_direct, ""))
            if ext and comp:
                direct_items.append(
                    {
                        "external_interface": ext,
                        "related_component": comp,
                        "all_interfaces": [ext],
                    }
                )
        if direct_items:
            return direct_items

    # B) 勾选矩阵格式
    header_row_num = None
    comp_idx = None
    for r in rows[:30]:
        for idx, v in r["cells"].items():
            if "零部件" in v and "名称" in v:
                header_row_num = r["row_num"]
                comp_idx = idx
                break
        if header_row_num is not None:
            break

    if header_row_num is None:
        # 尝试第2行兜底
        row2 = next((r for r in rows if r["row_num"] == 2), None)
        if row2:
            header_row_num = 2
            for idx, v in row2["cells"].items():
                if "零部件" in v and "名称" in v:
                    comp_idx = idx
                    break
            if comp_idx is None:
                comp_idx = 1
        else:
            return []

    header_row = next((r for r in rows if r["row_num"] == header_row_num), None)
    if not header_row:
        return []

    interface_cols = [(idx, name) for idx, name in sorted(header_row["cells"].items()) if idx > comp_idx and name]

    external_interfaces: list[dict] = []
    for r in rows:
        if r["row_num"] <= header_row_num:
            continue
        cells = r["cells"]
        comp = _norm_text(cells.get(comp_idx, ""))
        if not comp:
            continue
        active = [name for idx, name in interface_cols if _is_checked(cells.get(idx, ""))]
        if active:
            external_interfaces.append(
                {
                    "external_interface": active[0],
                    "related_component": comp,
                    "all_interfaces": active,
                }
            )

    if not external_interfaces:
        logger.warning("外部接口Excel已读取，但未识别到勾选项；请检查表头与勾选符号。")
    return external_interfaces


def parse_topology_and_generate_frameworks(topology_file: str, external_interface_excel: str) -> list[dict]:
    with open(topology_file, "r", encoding="utf-8") as f:
        topo_data = json.load(f).get("data", {})

    cells = topo_data.get("cells", [])
    protocol_legends = topo_data.get("lineData", [])
    color_name_map = {p.get("color"): p.get("name") for p in protocol_legends}

    if not os.path.exists(external_interface_excel):
        logger.warning(f"外部接口文件不存在: {external_interface_excel}")
        return []
    external_interfaces = _load_external_interfaces(external_interface_excel)
    if not external_interfaces:
        logger.warning("外部接口清单为空，攻击路径框架将为空。")
        return []

    element_vos = TopologyMapUtil.list_element_vo(cells)
    distinct_components = TopologyMapUtil.get_distinct_components(element_vos)
    path_node_vos = TopologyMapUtil.generate_full_path(
        external_interfaces, element_vos, protocol_legends, distinct_components
    )

    source_target_map: dict[str, list[list[PathNodeVO]]] = defaultdict(list)
    for p in path_node_vos:
        source_id = p[0].pre_component_name_id
        target_id = p[-1].component_name_id
        source_target_map[f"{source_id}_{target_id}"].append(p)

    comp_to_interfaces: dict[str, list[str]] = defaultdict(list)
    for ext in external_interfaces:
        comp_to_interfaces[ext["related_component"]].append(ext["external_interface"])

    frameworks: list[dict] = []

    for _, paths in source_target_map.items():
        min_len = min(len(p) for p in paths) if paths else 0
        min_paths = [p for p in paths if len(p) == min_len]

        has_gateway = any(any(n.is_gateway for n in p) for p in min_paths)
        filtered = [p for p in min_paths if any(n.is_gateway for n in p)] if has_gateway else min_paths

        for p in filtered:
            start_comp_name = p[0].pre_component_name
            ext_list = comp_to_interfaces.get(start_comp_name, ["未知接口"])

            for ext in ext_list:
                desc = [f"[{ext}] -> {start_comp_name}"]
                for node in p:
                    proto = color_name_map.get(node.color, "未知协议")
                    desc.append(f" -> [{proto}] -> {node.component_name}")
                path_desc = "".join(desc)

                path_nodes = [start_comp_name] + [n.component_name for n in p]
                attack_vector = _determine_attack_vector(ext, color_name_map, p)

                frameworks.append({
                    "path_id": hashlib.sha1(path_desc.encode("utf-8")).hexdigest()[:16],
                    "path_description": path_desc,
                    "path_nodes": path_nodes,
                    "last_node": path_nodes[-1] if path_nodes else "",
                    "start_component": start_comp_name,
                    "external_interface": ext,
                    "components": [n.component_name for n in p],
                    "attack_vector": attack_vector,
                })

    unique = []
    seen = set()
    for fw in frameworks:
        if fw["path_description"] in seen:
            continue
        seen.add(fw["path_description"])
        unique.append(fw)

    logger.info(f"拓扑攻击路径框架生成完成: {len(unique)} 条")
    return unique


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8) Prompt（生成+内嵌自评）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DS_SYSTEM_PROMPT = """你是汽车网络安全专家，负责TARA损害场景建模。损害场景是”涉及车辆或车辆功能且影响道路使用者的不良后果”。
在输出前必须进行自检，仅输出通过自检的结果。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类别、安全属性、威胁类型，以及相关参考知识（RAG）。

## 生成规则

损害场景必须同时包含以下三个要素（缺一不可），且必须结合车辆实际域（如ADAS、车身控制、动力系统等）：

   1. 功能如何因为资产的安全属性被破坏而产生不良后果
   2. 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   3. 必须明确提到被破坏的资产名称

## 参考示例：

- “存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。”
- “车辆夜间行驶时，前照灯控制功能因’前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”

【自检3条件】
1) 必须同时体现 function、asset_name、attribute_name 与 threat_type 的逻辑关联。
2) 必须明确具体危害后果（道路使用者/车辆运营/隐私或财务等）。
3) 描述必须具体且可验证，避免空泛结论。

仅输出JSON：
{"damage_scenario": "..."}
"""

TS_SYSTEM_PROMPT = """你是汽车网络安全专家，负责TARA威胁场景建模。威胁场景是”为实现损害场景，资产的信息安全属性遭到破坏的潜在原因”。
在输出前必须进行自检，仅输出通过自检的结果。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类型、安全属性、威胁类型、损害场景，以及相关参考知识（RAG）。

## 生成规则

1. 威胁场景必须描述：
   - 明确指出被攻击的具体资产名称
   - 该资产被破坏的安全属性
   - 该安全属性被破坏的原因/攻击意图
2. 威胁场景必须能直接导致给定的损害场景，描述中必须体现”破坏该安全属性 -> 引发损害场景中的不良后果”逻辑链条
3. 损害场景与资产名称、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系应能被威胁场景包含或与威胁场景关联

## 参考示例：

- “攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。”
- “伪装信号导致发送至电源开关控制器的’灯光请求’信号的数据通信完整性丢失,可能造成前照灯意外关闭”

【自检3条件】
1) 必须明确攻击对象与受损安全属性（attribute_name）。
2) 必须说明威胁实现方式，并与threat_type保持一致。
3) 必须与damage_scenario保持因果一致。

仅输出JSON：
{"threat_scenarios": ["...", "..."]}
"""

AP_SYSTEM_PROMPT = """你是汽车攻击路径专家。
在输出前必须进行自检，仅输出通过自检的结果。攻击路径是”为实现威胁场景的一组蓄意活动”。必须采用攻击树分析，从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类型、安全属性、威胁场景列表、攻击路径框架（path_frameworks），以及相关参考知识（RAG）。

## 生成规则

1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的威胁场景，每条路径的技术手段必须针对资产类型的特点设计
2. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）
3. 同一条威胁场景的多条攻击路径不要重复，避免冗余
4. 必须基于提供的路径框架（path_frameworks），从中选取一条或多条框架，将具体的攻击手段填入框架中的每个节点，构成完整的攻击链。

## 参考示例：

- “1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。”
- “1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- “1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

【自检3条件】
1) 攻击步骤必须与threat_scenarios一致。
2) 必须基于提供的路径框架，步骤链逻辑闭合。
3) 每条路径都要可执行、具体。

仅输出JSON：
{"attack_path": ["步骤链1", "步骤链2"]}
"""

IMPACT_SYSTEM_PROMPT = """你是ISO/SAE 21434影响评级专家。
根据损害场景输出四维评分（0~4）：Safety/Finance/Operation/Privacy。
仅输出JSON：
{"influence_parameters": {"Safety":0,"Finance":0,"Operation":0,"Privacy":0}}
"""

AP_FEASIBILITY_SYSTEM_PROMPT = """你是ISO/SAE 21434攻击潜力评估专家。
请输出五维分值，且必须从给定离散值中选择：
- Exposure_time: 0/1/4/17/19
- Professional_experience: 0/3/6/9
- Required_information: 0/3/7/11
- Opportunity_window: 0/1/4/10
- Required_equipment: 0/4/7/9
仅输出JSON：
{"attack_parameters": {"Exposure_time":0,"Professional_experience":0,"Required_information":0,"Opportunity_window":0,"Required_equipment":0}}
"""

CVSS_FEASIBILITY_SYSTEM_PROMPT = """你是CVSS可利用性评估专家。
请根据攻击路径分析并输出全部4个CVSS可利用性参数：
- 攻击向量: network(远程/蜂窝/卫星)/adjacent(短距无线)/local(本地端口)/physical(侵入式调试)
- 攻击复杂度: low(低)/high(高)
- 权限要求: none(无)/low(低)/high(高)
- 用户交互: none(无)/required(需要)

仅输出JSON：
{"attack_parameters": {"攻击向量":"network","攻击复杂度":"high","权限要求":"none","用户交互":"none"}}
"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 9) 生成与评估核心函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_damage_scenario(task: TaskUnit, rag_context: str = "") -> str:
    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}

参考知识（RAG）:
{rag_context}

请输出 damage_scenario。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_type": task.threat_type,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"damage_scenario task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="damage_scenario",
            cache_key=cache_key,
            system_prompt=DS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"damage_scenario": ""},
        )

    if cache_hit:
        logger.debug(f"[cache] damage_scenario task={task.task_id}")

    return str(parsed.get("damage_scenario", "")).strip()


def generate_threat_scenarios(task: TaskUnit, rag_context: str = "") -> list[str]:
    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}
- damage_scenario: {task.damage_scenario}

参考知识（RAG）:
{rag_context}

请输出 threat_scenarios。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_type": task.threat_type,
        "damage_scenario": task.damage_scenario,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"threat_scenarios task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="threat_scenarios",
            cache_key=cache_key,
            system_prompt=TS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"threat_scenarios": []},
        )

    if cache_hit:
        logger.debug(f"[cache] threat_scenarios task={task.task_id}")

    scenarios = parsed.get("threat_scenarios", parsed.get("threat_scenario", []))
    if isinstance(scenarios, str):
        return [scenarios]
    if isinstance(scenarios, list):
        return [str(s).strip() for s in scenarios if str(s).strip()]
    return []


def generate_attack_paths(task: TaskUnit, frameworks: list[dict], rag_context: str = "") -> list[str]:
    fw_brief = [
        {
            "path_id": fw["path_id"],
            "path_description": fw["path_description"],
            "last_node": fw["last_node"],
            "attack_vector": fw["attack_vector"],
            "path_nodes": fw["path_nodes"],
        }
        for fw in frameworks[:8]
    ]

    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- path_frameworks: {json.dumps(fw_brief, ensure_ascii=False)}

参考知识（RAG）:
{rag_context}

请输出 attack_path。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_scenarios": task.threat_scenarios,
        "path_frameworks": fw_brief,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"attack_path task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="attack_path",
            cache_key=cache_key,
            system_prompt=AP_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_path": []},
        )

    if cache_hit:
        logger.debug(f"[cache] attack_path task={task.task_id}")

    paths = parsed.get("attack_path", parsed.get("attack_paths", []))
    if isinstance(paths, str):
        return [paths]
    if isinstance(paths, list):
        return [str(p).strip() for p in paths if str(p).strip()]
    return []


def generate_influence_parameters(task: TaskUnit) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- asset_type: {task.asset_type}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}
- damage_scenario: {task.damage_scenario}

请输出 influence_parameters。"""

    cache_key = {
        "asset_name": task.asset_name,
        "asset_type": task.asset_type,
        "attribute_name": task.attribute_name,
        "damage_scenario": task.damage_scenario,
    }

    with Timer(f"influence task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="influence_parameters",
            cache_key=cache_key,
            system_prompt=IMPACT_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"influence_parameters": {"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}},
        )

    params = parsed.get("influence_parameters", {})
    return {
        "Safety": int(params.get("Safety", 0)),
        "Finance": int(params.get("Finance", 0)),
        "Operation": int(params.get("Operation", 0)),
        "Privacy": int(params.get("Privacy", 0)),
    }


def generate_attack_parameters_attack_potential(task: TaskUnit, attack_path: str) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- attack_path: {attack_path}

请输出 attack_parameters。"""

    cache_key = {
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "attack_path": attack_path,
    }

    with Timer(f"feasibility(ap) task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="attack_parameters_ap",
            cache_key=cache_key,
            system_prompt=AP_FEASIBILITY_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_parameters": {k: 0 for k in AP_ALLOWED_VALUES}},
        )

    params = parsed.get("attack_parameters", {})
    fixed = {}
    for k, allowed in AP_ALLOWED_VALUES.items():
        v = int(params.get(k, 0))
        # 强制吸附到最近合法值
        fixed[k] = min(allowed, key=lambda x: abs(x - v))
    return fixed


def generate_attack_parameters_cvss(task: TaskUnit, attack_path: str) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- attack_path: {attack_path}

请分析上述攻击路径，输出全部4个CVSS可利用性参数。"""

    cache_key = {
        "asset_name": task.asset_name,
        "attack_path": attack_path,
    }

    with Timer(f"feasibility(cvss) task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="attack_parameters_cvss",
            cache_key=cache_key,
            system_prompt=CVSS_FEASIBILITY_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_parameters": {"攻击向量": "local", "攻击复杂度": "high", "权限要求": "none", "用户交互": "none"}},
        )

    p = parsed.get("attack_parameters", {})
    return {
        "攻击向量": normalize_attack_vector(p.get("攻击向量", "local")),
        "攻击复杂度": str(p.get("攻击复杂度", "high")).strip().lower(),
        "权限要求": str(p.get("权限要求", "none")).strip().lower(),
        "用户交互": str(p.get("用户交互", "none")).strip().lower(),
    }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 10) 任务构建 + 路径筛选
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_task_units(asset_results: list[dict]) -> list[TaskUnit]:
    tasks: list[TaskUnit] = []
    idx = 0
    for func_data in asset_results:
        function_name = func_data.get("function", "")
        for asset in func_data.get("assets", []):
            asset_type = asset.get("asset_type", "")
            asset_name = asset.get("asset_name", "")
            target = asset.get("target", "")
            source = asset.get("source", "")

            attrs = SECURITY_ATTRIBUTES_MAP.get(asset_type, ["完整性"])
            for attr in attrs:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                tasks.append(
                    TaskUnit(
                        task_id=f"{idx:05d}",
                        function=function_name,
                        asset_name=asset_name,
                        asset_type=asset_type,
                        attribute_name=attr,
                        threat_type=threat_type,
                        target=target,
                        source=source,
                    )
                )
                idx += 1

    logger.info(f"任务单元构建完成: {len(tasks)}")
    return tasks


def filter_path_frameworks(task: TaskUnit, all_frameworks: list[dict]) -> list[dict]:
    target = (task.target or "").strip()
    source = (task.source or "").strip()

    selected: list[dict] = []

    for fw in all_frameworks:
        last_node = (fw.get("last_node") or "").strip()
        path_nodes = [str(n).strip() for n in fw.get("path_nodes", [])]

        if task.asset_type in ("部件", "数据"):
            if target and last_node == target:
                selected.append(fw)

        elif task.asset_type == "信号":
            cond_target = target and last_node == target
            cond_source = source and source in path_nodes
            if cond_target and cond_source:
                selected.append(fw)

        else:
            if target and last_node == target:
                selected.append(fw)

    # 回退策略：如果严格筛选为空，放宽到 target 命中路径字符串
    if not selected and target:
        selected = [fw for fw in all_frameworks if target in fw.get("path_description", "")]

    # 仍为空时，再按资产名尝试
    if not selected and task.asset_name:
        selected = [fw for fw in all_frameworks if task.asset_name in fw.get("path_description", "")]

    if len(selected) > 8:
        selected = selected[:8]

    return selected


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 11) 并发流水线
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def process_task_generation(
    task: TaskUnit,
    all_frameworks: list[dict],
    rag_contexts: Optional[dict[str, str]] = None,
) -> TaskUnit:
    """阶段A：每个资产独立并发（损害->威胁->攻击路径生成）"""
    rag_contexts = rag_contexts or {}
    rag_ds = rag_contexts.get("damage", "")
    rag_ts = rag_contexts.get("threat", "")
    rag_ap = rag_contexts.get("attack_path", "")
    try:
        task.damage_scenario = generate_damage_scenario(task, rag_ds)
        if not task.damage_scenario:
            task.errors.append("damage_scenario为空")

        task.threat_scenarios = generate_threat_scenarios(task, rag_ts)
        if not task.threat_scenarios:
            task.errors.append("threat_scenarios为空")

        selected = filter_path_frameworks(task, all_frameworks)
        task.selected_path_frameworks = selected

        task.attack_paths = generate_attack_paths(task, selected, rag_ap)
        if not task.attack_paths:
            task.errors.append("attack_path为空")

    except Exception as e:
        task.errors.append(f"生成阶段异常: {e}")

    return task


def process_task_full_pipeline(
    task: TaskUnit,
    all_frameworks: list[dict],
    rag_contexts: dict[str, str],
    method: str,
) -> TaskUnit:
    """单资产全流水线：损害→威胁→攻击路径(缓存)→影响评级→可行性评估→风险矩阵"""
    try:
        # ── Stage A: 场景生成 ──────────────────────────
        task.damage_scenario = generate_damage_scenario(task, rag_contexts.get("damage", ""))
        if not task.damage_scenario:
            task.errors.append("damage_scenario为空")

        task.threat_scenarios = generate_threat_scenarios(task, rag_contexts.get("threat", ""))
        if not task.threat_scenarios:
            task.errors.append("threat_scenarios为空")

        selected = filter_path_frameworks(task, all_frameworks)
        task.selected_path_frameworks = selected

        # 攻击路径缓存：同一 asset_name 的路径相同，跨功能复用
        cached_paths = _get_attack_path_cache(task.asset_name)
        if cached_paths is not None:
            task.attack_paths = cached_paths
            logger.debug(f"[path_cache] task={task.task_id} 复用 asset={task.asset_name} 的攻击路径")
        else:
            task.attack_paths = generate_attack_paths(task, selected, rag_contexts.get("attack_path", ""))
            if task.attack_paths:
                _set_attack_path_cache(task.asset_name, task.attack_paths)

        if not task.attack_paths:
            task.errors.append("attack_path为空")

        # ── Stage B: 影响评级 ──────────────────────────
        params = generate_influence_parameters(task)
        task.influence_parameters = params
        task.impact_value = calculate_impact_value(params)
        task.impact_level = calculate_impact_level(task.impact_value)

        # ── Stage C: 攻击可行性评估 + 风险矩阵 ─────────
        task.attack_details = []
        if task.attack_paths:
            for ai, ap_text in enumerate(task.attack_paths):
                vector = "local"
                if task.selected_path_frameworks:
                    fw = task.selected_path_frameworks[ai % len(task.selected_path_frameworks)]
                    vector = fw.get("attack_vector", "local")

                if method == "attack_vector":
                    ap_params = {"攻击向量": normalize_attack_vector(vector)}
                    feasibility_score = None
                    feasibility_level = calculate_attack_vector_level(vector)
                elif method == "cvss":
                    ap_params = generate_attack_parameters_cvss(task, ap_text)
                    feasibility_score = calculate_cvss_score(ap_params)
                    feasibility_level = calculate_cvss_level(feasibility_score)
                else:  # attack_potential
                    ap_params = generate_attack_parameters_attack_potential(task, ap_text)
                    feasibility_score = calculate_attack_potential_score(ap_params)
                    feasibility_level = calculate_attack_potential_level(feasibility_score)

                risk_value = calculate_risk_value(task.impact_value, feasibility_level)
                risk_treatment = get_risk_treatment(risk_value)

                task.attack_details.append({
                    "attack_path": ap_text,
                    "attack_vector": normalize_attack_vector(vector),
                    "attack_parameters": ap_params,
                    "feasibility_score": feasibility_score,
                    "feasibility_level": feasibility_level,
                    "risk_value": risk_value,
                    "risk_treatment": risk_treatment,
                })

        if task.attack_details:
            task.risk_value = max(d["risk_value"] for d in task.attack_details)
        else:
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")
        task.risk_treatment = get_risk_treatment(task.risk_value)

    except Exception as e:
        task.errors.append(f"全流水线异常: {e}")

    return task


def process_in_batches(items: list[Any], batch_size: int, worker_fn, stage_name: str) -> list[Any]:
    """连续批处理：上一批结束后再提交下一批。"""
    outputs: list[Any] = [None] * len(items)
    total = len(items)
    if total == 0:
        return outputs

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = items[start:end]
        logger.info(f"[{stage_name}] 连续批处理 {start + 1}-{end}/{total}")

        with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(batch))) as executor:
            future_map = {executor.submit(worker_fn, i, item): i for i, item in enumerate(batch, start=start)}
            for future in as_completed(future_map):
                i = future_map[future]
                try:
                    outputs[i] = future.result()
                except Exception as e:
                    outputs[i] = e

        if end < total:
            time.sleep(CALL_INTERVAL)

    return outputs


def stage_b_influence_batch(tasks: list[TaskUnit]) -> None:
    """阶段B：influence_parameters 连续批处理。"""

    def worker(index: int, task: TaskUnit):
        params = generate_influence_parameters(task)
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        return {
            "index": index,
            "params": params,
            "impact_value": impact_value,
            "impact_level": impact_level,
        }

    results = process_in_batches(tasks, BATCH_SIZE_INFLUENCE, worker, "Impact")
    for item in results:
        if isinstance(item, dict):
            task = tasks[item["index"]]
            task.influence_parameters = item["params"]
            task.impact_value = item["impact_value"]
            task.impact_level = item["impact_level"]


def stage_c_attack_batch(tasks: list[TaskUnit], method: str) -> None:
    """阶段C：attack_parameters 连续批处理 + 可行性计算 + 风险矩阵。"""

    jobs: list[dict] = []
    for ti, task in enumerate(tasks):
        if not task.attack_paths:
            # 没有攻击路径时，给一个兜底记录
            task.attack_details = []
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")
            task.risk_treatment = get_risk_treatment(task.risk_value)
            continue

        for ai, ap_text in enumerate(task.attack_paths):
            vector = "local"
            if task.selected_path_frameworks:
                fw = task.selected_path_frameworks[ai % len(task.selected_path_frameworks)]
                vector = fw.get("attack_vector", "local")

            jobs.append({
                "task_index": ti,
                "attack_index": ai,
                "attack_path": ap_text,
                "attack_vector": vector,
            })

    def worker(index: int, job: dict):
        task = tasks[job["task_index"]]
        attack_path = job["attack_path"]
        attack_vector = job["attack_vector"]

        if method == "attack_vector":
            params = {"攻击向量": normalize_attack_vector(attack_vector)}
            feasibility_score = None
            feasibility_level = calculate_attack_vector_level(attack_vector)

        elif method == "cvss":
            params = generate_attack_parameters_cvss(task, attack_path)
            feasibility_score = calculate_cvss_score(params)
            feasibility_level = calculate_cvss_level(feasibility_score)

        else:  # attack_potential
            params = generate_attack_parameters_attack_potential(task, attack_path)
            feasibility_score = calculate_attack_potential_score(params)
            feasibility_level = calculate_attack_potential_level(feasibility_score)

        risk_value = calculate_risk_value(task.impact_value, feasibility_level)
        risk_treatment = get_risk_treatment(risk_value)

        return {
            "job": job,
            "attack_parameters": params,
            "feasibility_score": feasibility_score,
            "feasibility_level": feasibility_level,
            "risk_value": risk_value,
            "risk_treatment": risk_treatment,
        }

    results = process_in_batches(jobs, BATCH_SIZE_ATTACK, worker, "Feasibility")

    grouped: dict[int, list[dict]] = defaultdict(list)
    for r in results:
        if not isinstance(r, dict):
            continue
        ti = r["job"]["task_index"]
        grouped[ti].append(r)

    for ti, task in enumerate(tasks):
        details = []
        for r in sorted(grouped.get(ti, []), key=lambda x: x["job"]["attack_index"]):
            job = r["job"]
            details.append({
                "attack_path": job["attack_path"],
                "attack_vector": normalize_attack_vector(job["attack_vector"]),
                "attack_parameters": r["attack_parameters"],
                "feasibility_score": r["feasibility_score"],
                "feasibility_level": r["feasibility_level"],
                "risk_value": r["risk_value"],
                "risk_treatment": r["risk_treatment"],
            })

        task.attack_details = details

        if details:
            task.risk_value = max(d["risk_value"] for d in details)
        else:
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")

        task.risk_treatment = get_risk_treatment(task.risk_value)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 12) 报告构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_report(tasks: list[TaskUnit], feasibility_method: str) -> list[dict]:
    function_map: dict[str, dict] = {}

    for task in tasks:
        if task.function not in function_map:
            function_map[task.function] = {"function": task.function, "assets": {}}

        asset_key = (task.asset_name, task.asset_type, task.target, task.source)
        assets = function_map[task.function]["assets"]

        if asset_key not in assets:
            assets[asset_key] = {
                "asset_name": task.asset_name,
                "asset_type": task.asset_type,
                "target": task.target,
                "source": task.source,
                "security_attributes": [],
            }

        detail = {
            "attribute_name": task.attribute_name,
            "threat_type": task.threat_type,
            "damage_scenario": task.damage_scenario,
            "threat_scenarios": task.threat_scenarios,
            "influence_parameters": task.influence_parameters,
            "impact_value": task.impact_value,
            "impact_level": task.impact_level,
            "feasibility_method": feasibility_method,
            "attack": task.attack_details,
            "risk_value": task.risk_value,
            "risk_treatment": task.risk_treatment,
            "errors": task.errors,
        }

        assets[asset_key]["security_attributes"].append(detail)

    report = []
    for func_data in function_map.values():
        report.append({
            "function": func_data["function"],
            "assets": list(func_data["assets"].values()),
        })

    return report


def build_error_report(tasks: list[TaskUnit]) -> list[dict]:
    rows = []
    for task in tasks:
        if not task.errors:
            continue
        rows.append({
            "task_id": task.task_id,
            "function": task.function,
            "asset_name": task.asset_name,
            "asset_type": task.asset_type,
            "attribute_name": task.attribute_name,
            "threat_type": task.threat_type,
            "errors": task.errors,
        })
    return rows


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 13) 主流程
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_dir: str,
    topology_file: str,
    external_interface_excel: str,
    output_path: str,
    feasibility_method: str = FEASIBILITY_METHOD,
    max_assets_for_test: Optional[int] = None,
    skip_checkpoint_resume: bool = False,
) -> tuple[list[dict], list[dict]]:
    method = (feasibility_method or "attack_potential").strip().lower()
    if method not in {"attack_potential", "attack_vector", "cvss"}:
        logger.warning(f"未知可行性方法: {method}，回退为 attack_potential")
        method = "attack_potential"

    _pipeline_start = time.perf_counter()

    logger.info("=" * 72)
    logger.info("TARA v10 开始")
    logger.info(f"可行性评估方法: {method}")
    logger.info("=" * 72)

    # ─── Step 1: DFD资产识别 ─────────────────────────
    with Timer("Step 1 — DFD资产识别") as _:
        asset_results = parse_dfd_json(data_flow_dir)

    # 测试模式：仅取前N个DFD资产
    if max_assets_for_test is not None and max_assets_for_test > 0:
        limited_results = []
        count = 0
        for func_data in asset_results:
            assets = func_data.get("assets", [])
            if count >= max_assets_for_test:
                break
            remain = max_assets_for_test - count
            if len(assets) > remain:
                copy_func = dict(func_data)
                copy_func["assets"] = assets[:remain]
                limited_results.append(copy_func)
                count += remain
                break
            else:
                limited_results.append(func_data)
                count += len(assets)
        asset_results = limited_results
        logger.info(f"测试模式启用：仅使用前 {max_assets_for_test} 个DFD资产，实际纳入 {count} 个")

    # ─── Step 2: 拓扑攻击路径框架 ────────────────────
    with Timer("Step 2 — 拓扑攻击路径框架") as _:
        path_frameworks = []
        if os.path.exists(topology_file) and os.path.exists(external_interface_excel):
            path_frameworks = parse_topology_and_generate_frameworks(topology_file, external_interface_excel)
        else:
            logger.warning("拓扑文件或接口清单不存在，攻击路径框架为空")

    # ─── Step 3: 扩展属性 + RAG ──────────────────────
    with Timer("Step 3 — 任务构建 + RAG") as _:
        tasks = build_task_units(asset_results)
        rag_contexts = {
            "damage": rag_kb.retrieve(
                "汽车网络安全 损害场景 STRIDE ISO21434",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
            "threat": rag_kb.retrieve(
                "汽车威胁场景 CAPEC ATT&CK",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
            "attack_path": rag_kb.retrieve(
                "汽车攻击路径 CAN总线 OBD T-Box",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
        }
        logger.info(
            "RAG上下文长度: damage=%d, threat=%d, attack_path=%d",
            len(rag_contexts["damage"]),
            len(rag_contexts["threat"]),
            len(rag_contexts["attack_path"]),
        )

    # ─── Checkpoint 恢复 ──────────────────────────────
    # 全流水线模式下只有一个 checkpoint 点：stage_C_attack（最终完成态）
    checkpoint_stages = ["stage_C_attack"]
    current_task_data: list[TaskUnit] | None = None
    resume_from = 0

    if not skip_checkpoint_resume:
        resume_from, current_task_data = resume_from_checkpoint(checkpoint_stages, method)
    else:
        logger.info("Checkpoint恢复已跳过（skip_checkpoint_resume=True）")

    # ─── Step 4-6: 全流水线并发生成 (Stages A+B+C) ──
    # 每个资产独立、完整地走完损害→威胁→攻击路径→影响→可行性→风险矩阵
    if resume_from <= 0:
        with Timer("全流水线并发生成 (A+B+C)") as _:
            logger.info(f"全流水线并行：每个资产独立完成全部 TARA 阶段，资产数={len(tasks)}")
            completed_tasks: list[TaskUnit] = []

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_map = {
                    executor.submit(process_task_full_pipeline, t, path_frameworks, rag_contexts, method): t
                    for t in tasks
                }
                done = 0
                total = len(future_map)
                for future in as_completed(future_map):
                    t = future_map[future]
                    done += 1
                    try:
                        completed_tasks.append(future.result())
                    except Exception as e:
                        t.errors.append(f"全流水线异常: {e}")
                        completed_tasks.append(t)
                    if done < total:
                        time.sleep(CALL_INTERVAL)

            completed_tasks.sort(key=lambda x: x.task_id)

        save_checkpoint(completed_tasks, "stage_C_attack", method)
        current_task_data = completed_tasks
    else:
        logger.info("全流水线跳过（从 checkpoint 恢复，全部已完成）")

    assert current_task_data is not None, "当前任务数据为空，无法继续"

    # ─── Step 7: 报告 ────────────────────────────────
    with Timer("Step 7 — 报告构建") as _:
        report = build_tara_report(current_task_data, method)
        err_report = build_error_report(current_task_data)

        save_json(report, output_path)
        save_json(err_report, output_path.replace(".json", "_errors.json"))

    total_elapsed = time.perf_counter() - _pipeline_start
    logger.info("=" * 72)
    logger.info(f"TARA v10 完成: 总任务={len(current_task_data)}, 异常任务={len(err_report)}")
    logger.info(f"总耗时: {total_elapsed:.2f} 秒 ({total_elapsed / 60:.2f} 分钟)")
    logger.info(f"主报告: {output_path}")
    logger.info(f"异常报告: {output_path.replace('.json', '_errors.json')}")
    logger.info("=" * 72)

    return report, err_report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 14) 入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":
    topology_file = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\topology\拓扑图数据导出2026_5_9 10_43_29.json"
    external_interface_excel = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\information\对外接口清单.xlsx"
    data_flow_dir = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\DFD"

    # 你可以在这里切换：attack_potential / attack_vector / cvss
    method = os.getenv("TARA_FEASIBILITY_METHOD", FEASIBILITY_METHOD)
    # 测试时仅使用前N个DFD资产（默认2，设为0或负数则关闭）
    test_max_assets = int(os.getenv("TARA_TEST_MAX_ASSETS", "1"))
    if test_max_assets <= 0:
        test_max_assets = None
    # Checkpoint恢复：设 TARA_SKIP_CHECKPOINT=1 可强制重跑所有阶段
    skip_cp = os.getenv("TARA_SKIP_CHECKPOINT", "0").strip() in {"1", "true", "True"}

    run_tara(
        data_flow_dir=data_flow_dir,
        topology_file=topology_file,
        external_interface_excel=external_interface_excel,
        output_path=os.path.join(OUTPUT_DIR, "tara_report_v10.json"),
        feasibility_method=method,
        max_assets_for_test=test_max_assets,
        skip_checkpoint_resume=skip_cp,
    )

2026-05-14 18:06:13,935 [INFO] ========================================================================
2026-05-14 18:06:13,936 [INFO] TARA v10 开始
2026-05-14 18:06:13,937 [INFO] 可行性评估方法: attack_potential
2026-05-14 18:06:13,937 [INFO] ========================================================================
2026-05-14 18:06:13,938 [INFO] ═══ [计时] Step 1 — DFD资产识别 开始 ═══
2026-05-14 18:06:13,942 [INFO] DFD解析完成: 2 个功能
2026-05-14 18:06:13,943 [INFO] ═══ [计时] Step 1 — DFD资产识别 完成，耗时 0.00 秒 (0.00 分钟) ═══
2026-05-14 18:06:13,943 [INFO] 测试模式启用：仅使用前 1 个DFD资产，实际纳入 1 个
2026-05-14 18:06:13,944 [INFO] ═══ [计时] Step 2 — 拓扑攻击路径框架 开始 ═══
2026-05-14 18:06:14,184 [INFO] 拓扑攻击路径框架生成完成: 1169 条
2026-05-14 18:06:14,187 [INFO] ═══ [计时] Step 2 — 拓扑攻击路径框架 完成，耗时 0.24 秒 (0.00 分钟) ═══
2026-05-14 18:06:14,188 [INFO] ═══ [计时] Step 3 — 任务构建 + RAG 开始 ═══
2026-05-14 18:06:14,189 [INFO] 任务单元构建完成: 6
2026-05-14 18:06:14,196 [INFO] RAG文档加载完成: 1 条
2026-05-14 18:06:14,196 [INFO] RAG上下文长度: damage=518, threat=518, attack_path=51

In [4]:

import copy
import glob
import hashlib
import json
import logging
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from difflib import SequenceMatcher
from threading import Lock, Semaphore
from typing import Any, Optional

try:
    import pandas as pd
except Exception:
    pd = None
from urllib import error as urlerror
from urllib import request as urlrequest


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1) 全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA-V10")


MODEL_NAME = os.getenv("TARA_MODEL", "qwen3.6-35b-a3b")
API_KEY = os.getenv("TARA_API_KEY", "sk-3458388d4845414387fe2e10bc8a9ee2")
API_BASE = os.getenv("TARA_API_BASE", "https://dashscope.aliyuncs.com/compatible-mode/v1")

if not API_KEY:
    logger.warning("未检测到环境变量 TARA_API_KEY。")

REQUEST_TIMEOUT = int(os.getenv("TARA_REQUEST_TIMEOUT", "90"))
MAX_TOKENS = int(os.getenv("TARA_MAX_TOKENS", "4096"))
TEMPERATURE = float(os.getenv("TARA_TEMPERATURE", "0.1"))

MAX_WORKERS = int(os.getenv("TARA_MAX_WORKERS", "6"))
MAX_CONCURRENT_LLM = int(os.getenv("TARA_MAX_CONCURRENT_LLM", "3"))
CALL_INTERVAL = float(os.getenv("TARA_CALL_INTERVAL", "0.4"))

BATCH_SIZE_INFLUENCE = int(os.getenv("TARA_BATCH_SIZE_INFLUENCE", "8"))
BATCH_SIZE_ATTACK = int(os.getenv("TARA_BATCH_SIZE_ATTACK", "8"))

# 可选：attack_potential / attack_vector / cvss
FEASIBILITY_METHOD = "cvss"

OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V13"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ENABLE_RAG = os.getenv("TARA_ENABLE_RAG", "1").strip() not in {"0", "false", "False"}
RAG_BASE_DIR = os.getenv("TARA_RAG_BASE_DIR", r"D:\Jupyter profile\rag")
RAG_DIRS = {
    "tara_reports": os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations": os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
RAG_TOP_K = int(os.getenv("TARA_RAG_TOP_K", "5"))
RAG_MAX_CONTEXT_CHARS = int(os.getenv("TARA_RAG_MAX_CHARS", "1800"))

llm_semaphore = Semaphore(MAX_CONCURRENT_LLM)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1.5) 计时工具
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class Timer:
    """分阶段计时器：with 块自动记录耗时并日志输出。"""

    def __init__(self, stage_name: str):
        self.stage_name = stage_name
        self.start = 0.0

    def __enter__(self):
        self.start = time.perf_counter()
        logger.info("═══ [计时] %s 开始 ═══", self.stage_name)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.perf_counter() - self.start
        if exc_type is None:
            logger.info("═══ [计时] %s 完成，耗时 %.2f 秒 (%.2f 分钟) ═══", self.stage_name, elapsed, elapsed / 60)
        else:
            logger.error("═══ [计时] %s 异常退出，耗时 %.2f 秒，错误: %s ═══", self.stage_name, elapsed, exc_val)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2) 业务常量
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性": "篡改",
    "机密性": "信息泄露",
    "可用性": "拒绝服务",
    "真实性": "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性": "越权",
}

# Attack Potential 五维取值规范（ISO/SAE 21434 常见量表）
AP_ALLOWED_VALUES = {
    "Exposure_time": [0, 1, 4, 17, 19],
    "Professional_experience": [0, 3, 6, 9],
    "Required_information": [0, 3, 7, 11],
    "Opportunity_window": [0, 1, 4, 10],
    "Required_equipment": [0, 4, 7, 9],
}

# CVSS 取值
CVSS_V = {
    "network": 0.85,
    "adjacent": 0.62,
    "local": 0.55,
    "physical": 0.20,
}
CVSS_C = {"low": 0.77, "high": 0.44}
CVSS_P = {"none": 0.85, "low": 0.62, "high": 0.27}
CVSS_U = {"none": 0.85, "required": 0.62}

# 攻击向量 -> 可行性等级映射（ISO 21434 G.4）
ATTACK_VECTOR_TO_LEVEL = {
    "network": "High",
    "adjacent": "Medium",
    "local": "Low",
    "physical": "Very Low",
}

RISK_MATRIX = {
    (4, "High"): 5, (4, "Medium"): 5, (4, "Low"): 4, (4, "Very Low"): 4,
    (3, "High"): 5, (3, "Medium"): 4, (3, "Low"): 4, (3, "Very Low"): 3,
    (2, "High"): 4, (2, "Medium"): 4, (2, "Low"): 3, (2, "Very Low"): 2,
    (1, "High"): 3, (1, "Medium"): 2, (1, "Low"): 2, (1, "Very Low"): 1,
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3) 语义缓存
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SemanticCache:
    """轻量语义缓存：规范化文本 + 相似度匹配。"""

    def __init__(self, similarity_threshold: float = 0.985, max_bucket_size: int = 1000):
        self.similarity_threshold = similarity_threshold
        self.max_bucket_size = max_bucket_size
        self._store: dict[str, list[dict]] = defaultdict(list)
        self._lock = Lock()

    @staticmethod
    def _normalize(obj: Any) -> str:
        if isinstance(obj, (dict, list, tuple)):
            text = json.dumps(obj, ensure_ascii=False, sort_keys=True)
        else:
            text = str(obj)
        text = text.lower().strip()
        text = re.sub(r"\s+", "", text)
        text = text.replace("：", ":")
        return text

    def get(self, bucket: str, key_obj: Any) -> Optional[Any]:
        signature = self._normalize(key_obj)
        key_hash = hashlib.sha256(signature.encode("utf-8")).hexdigest()

        with self._lock:
            items = self._store.get(bucket, [])
            for item in items:
                if item["hash"] == key_hash:
                    return copy.deepcopy(item["value"])
                ratio = SequenceMatcher(None, signature, item["signature"]).ratio()
                if ratio >= self.similarity_threshold:
                    return copy.deepcopy(item["value"])
        return None

    def set(self, bucket: str, key_obj: Any, value: Any) -> None:
        signature = self._normalize(key_obj)
        key_hash = hashlib.sha256(signature.encode("utf-8")).hexdigest()

        with self._lock:
            items = self._store[bucket]
            for item in items:
                if item["hash"] == key_hash:
                    item["value"] = copy.deepcopy(value)
                    return
            items.append({"hash": key_hash, "signature": signature, "value": copy.deepcopy(value)})
            if len(items) > self.max_bucket_size:
                del items[0 : len(items) - self.max_bucket_size]


semantic_cache = SemanticCache()

# 攻击路径缓存：相同 asset_name 的攻击路径相同，跨功能/属性复用
_attack_path_cache: dict[str, list[str]] = {}
_attack_path_cache_lock = Lock()


def _get_attack_path_cache(asset_name: str) -> list[str] | None:
    with _attack_path_cache_lock:
        return copy.deepcopy(_attack_path_cache.get(asset_name))


def _set_attack_path_cache(asset_name: str, paths: list[str]) -> None:
    with _attack_path_cache_lock:
        _attack_path_cache[asset_name] = copy.deepcopy(paths)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.5) 轻量 RAG（兼容无第三方依赖环境）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SimpleRAGKnowledgeBase:
    def __init__(self, rag_dirs: dict[str, str]):
        self.rag_dirs = rag_dirs
        self.docs: list[dict] = []
        self._loaded = False
        self._lock = Lock()

    def _safe_read_text(self, path: str) -> str:
        for enc in ("utf-8", "utf-8-sig", "gbk", "gb18030"):
            try:
                with open(path, "r", encoding=enc) as f:
                    return f.read()
            except Exception:
                continue
        return ""

    def _load_json_text(self, path: str) -> str:
        try:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, (dict, list)):
                return json.dumps(obj, ensure_ascii=False)
            return str(obj)
        except Exception:
            return self._safe_read_text(path)

    def load(self, force: bool = False) -> None:
        with self._lock:
            if self._loaded and not force:
                return
            self.docs = []
            for category, d in self.rag_dirs.items():
                if not os.path.isdir(d):
                    continue
                for p in glob.glob(os.path.join(d, "**", "*"), recursive=True):
                    if not os.path.isfile(p):
                        continue
                    ext = os.path.splitext(p)[1].lower()
                    if ext in {".txt", ".md"}:
                        txt = self._safe_read_text(p)
                    elif ext == ".json":
                        txt = self._load_json_text(p)
                    else:
                        continue
                    txt = (txt or "").strip()
                    if len(txt) < 20:
                        continue
                    self.docs.append(
                        {
                            "category": category,
                            "path": p,
                            "text": txt[:12000],  # 限制单文档最大长度
                        }
                    )
            self._loaded = True
            logger.info(f"RAG文档加载完成: {len(self.docs)} 条")

    def retrieve(self, query: str, top_k: int = 5, max_chars: int = 1800) -> str:
        if not ENABLE_RAG:
            return "[RAG已关闭]"
        if not self._loaded:
            self.load()
        if not self.docs:
            return "[RAG为空]"

        q = (query or "").strip().lower()
        q_tokens = [t for t in re.split(r"[\s,，。;；:：|/\\-]+", q) if t]
        if not q_tokens:
            q_tokens = [q]

        scored = []
        for d in self.docs:
            text = d["text"].lower()
            score = 0
            for t in q_tokens:
                if t and t in text:
                    score += 3
            # 轻微偏置：法规库在场景生成里常更有用
            if d["category"] == "regulations":
                score += 1
            if score > 0:
                scored.append((score, d))

        if not scored:
            # 没命中时给前几个法规类兜底
            fallback = [d for d in self.docs if d["category"] == "regulations"][:top_k]
            scored_docs = fallback if fallback else self.docs[:top_k]
        else:
            scored.sort(key=lambda x: x[0], reverse=True)
            scored_docs = [d for _, d in scored[:top_k]]

        chunks = []
        total = 0
        for d in scored_docs:
            snippet = d["text"][:500].strip()
            line = f"[来源:{d['category']}] {snippet}"
            if total + len(line) > max_chars:
                break
            chunks.append(line)
            total += len(line)

        return "\n---\n".join(chunks) if chunks else "[RAG未命中]"


rag_kb = SimpleRAGKnowledgeBase(RAG_DIRS)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4) 数据结构
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class TaskUnit:
    task_id: str
    function: str
    asset_name: str
    asset_type: str
    attribute_name: str
    threat_type: str
    target: str = ""
    source: str = ""

    damage_scenario: str = ""
    threat_scenarios: list[str] = field(default_factory=list)
    attack_paths: list[str] = field(default_factory=list)
    selected_path_frameworks: list[dict] = field(default_factory=list)

    influence_parameters: dict = field(default_factory=dict)
    impact_value: int = 1
    impact_level: str = "Negligible"

    attack_details: list[dict] = field(default_factory=list)

    risk_value: int = 1
    risk_treatment: str = "接受"

    errors: list[str] = field(default_factory=list)


def _task_to_dict(t: TaskUnit) -> dict:
    return {
        "task_id": t.task_id,
        "function": t.function,
        "asset_name": t.asset_name,
        "asset_type": t.asset_type,
        "attribute_name": t.attribute_name,
        "threat_type": t.threat_type,
        "target": t.target,
        "source": t.source,
        "damage_scenario": t.damage_scenario,
        "threat_scenarios": t.threat_scenarios,
        "attack_paths": t.attack_paths,
        "selected_path_frameworks": t.selected_path_frameworks,
        "influence_parameters": t.influence_parameters,
        "impact_value": t.impact_value,
        "impact_level": t.impact_level,
        "attack_details": t.attack_details,
        "risk_value": t.risk_value,
        "risk_treatment": t.risk_treatment,
        "errors": t.errors,
    }


def _task_from_dict(d: dict) -> TaskUnit:
    return TaskUnit(
        task_id=d.get("task_id", ""),
        function=d.get("function", ""),
        asset_name=d.get("asset_name", ""),
        asset_type=d.get("asset_type", ""),
        attribute_name=d.get("attribute_name", ""),
        threat_type=d.get("threat_type", ""),
        target=d.get("target", ""),
        source=d.get("source", ""),
        damage_scenario=d.get("damage_scenario", ""),
        threat_scenarios=list(d.get("threat_scenarios", [])),
        attack_paths=list(d.get("attack_paths", [])),
        selected_path_frameworks=list(d.get("selected_path_frameworks", [])),
        influence_parameters=dict(d.get("influence_parameters", {})),
        impact_value=int(d.get("impact_value", 1)),
        impact_level=str(d.get("impact_level", "Negligible")),
        attack_details=list(d.get("attack_details", [])),
        risk_value=int(d.get("risk_value", 1)),
        risk_treatment=str(d.get("risk_treatment", "接受")),
        errors=list(d.get("errors", [])),
    )


def _tasks_to_dict(tasks: list[TaskUnit]) -> list[dict]:
    return [_task_to_dict(t) for t in tasks]


def _tasks_from_dict(data: list[dict]) -> list[TaskUnit]:
    return [_task_from_dict(d) for d in data]


@dataclass
class TopologyElementVO:
    id: str
    name: str = ""
    type: int | None = None
    color: str = ""
    source_id: str = ""
    target_id: str = ""
    is_gateway: bool = False
    ids: list[str] = field(default_factory=list)


@dataclass
class PathNodeVO:
    component_name_id: str
    component_name: str
    pre_component_name_id: str
    pre_component_name: str
    line_id: str
    color: str
    is_gateway: bool


class TopologyElementType:
    COMPONENT = 1
    GATEWAY = 2
    EXTERNAL_COMPONENT = 3
    LINE = 4


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5) 通用工具
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: Any) -> Any:
    if text is None:
        raise ValueError("输入为空")
    if not isinstance(text, str):
        text = str(text)

    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()

    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                continue

    raise ValueError(f"JSON解析失败，原文前200字: {cleaned[:200]}")


def llm_call_with_semaphore(system_prompt: str, user_prompt: str, retries: int = 2) -> str:
    with llm_semaphore:
        for attempt in range(retries + 1):
            try:
                endpoint = API_BASE.rstrip("/") + "/chat/completions"
                payload = {
                    "model": MODEL_NAME,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "extra_body": {"enable_thinking": False},
                }
                body = json.dumps(payload).encode("utf-8")
                req = urlrequest.Request(
                    endpoint,
                    data=body,
                    headers={
                        "Authorization": f"Bearer {API_KEY}",
                        "Content-Type": "application/json",
                    },
                    method="POST",
                )
                with urlrequest.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
                    data = json.loads(resp.read().decode("utf-8"))
                content = (
                    data.get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                )
                return str(content) if content is not None else ""
            except (urlerror.URLError, TimeoutError) as e:
                logger.warning(f"LLM网络错误: {type(e).__name__}: {e}")
            except Exception as e:
                logger.warning(f"LLM调用异常: {type(e).__name__}: {e}")

            if attempt < retries:
                wait = 2 ** attempt
                time.sleep(wait)

    return "{}"


def cached_llm_json(
    *,
    bucket: str,
    cache_key: Any,
    system_prompt: str,
    user_prompt: str,
    default: Any,
) -> tuple[Any, bool]:
    cached = semantic_cache.get(bucket, cache_key)
    if cached is not None:
        return cached, True

    raw = llm_call_with_semaphore(system_prompt, user_prompt)
    try:
        parsed = safe_parse_json(raw)
    except Exception:
        parsed = copy.deepcopy(default)

    semantic_cache.set(bucket, cache_key, parsed)
    return parsed, False


def normalize_attack_vector(value: str) -> str:
    s = (value or "").strip().lower()
    # Network（远程/蜂窝/卫星）
    if any(k in s for k in ["network", "网络", "remote", "ota", "蜂窝", "4g", "5g", "lte", "ethernet", "以太网", "cloud", "卫星定位"]):
        return "network"
    # Adjacent（短距无线）
    if any(k in s for k in ["adjacent", "相邻", "bluetooth", "蓝牙", "ble", "wifi", "wlan", "星闪", "v2x", "射频信道"]):
        return "adjacent"
    # Local（物理端口/本地接入）
    if any(k in s for k in ["local", "本地", "usb", "tf卡", "sd卡", "ic卡", "充电口", "obd", "内部通讯", "nfc", "登录"]):
        return "local"
    # Physical（侵入式调试）
    if any(k in s for k in ["physical", "物理", "debug", "jtag", "内部调试"]):
        return "physical"
    return "local"


def calculate_impact_value(influence_parameters: dict) -> int:
    if not influence_parameters:
        return 1
    v = max(
        int(influence_parameters.get("Safety", 0)),
        int(influence_parameters.get("Finance", 0)),
        int(influence_parameters.get("Operation", 0)),
        int(influence_parameters.get("Privacy", 0)),
    )
    return max(1, min(4, v))


def calculate_impact_level(impact_value: int) -> str:
    return {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}.get(impact_value, "Negligible")


def calculate_attack_potential_score(params: dict) -> int:
    return sum(int(params.get(k, 0)) for k in AP_ALLOWED_VALUES.keys())


def calculate_attack_potential_level(total_score: int) -> str:
    # 用户要求的映射
    if total_score <= 9:
        return "High"
    if total_score <= 13:
        return "Medium"
    if total_score <= 19:
        return "Low"
    return "Very Low"


def calculate_cvss_score(params: dict) -> float:
    v = CVSS_V.get(normalize_attack_vector(params.get("攻击向量", "local")), 0.55)
    c = CVSS_C.get((params.get("攻击复杂度", "high") or "high").strip().lower(), 0.44)
    p = CVSS_P.get((params.get("权限要求", "none") or "none").strip().lower(), 0.85)
    u = CVSS_U.get((params.get("用户交互", "none") or "none").strip().lower(), 0.85)
    score = 8.22 * v * c * p * u
    return round(score, 2)


def calculate_cvss_level(score: float) -> str:
    if 2.96 <= score <= 3.89:
        return "High"
    if 2.00 <= score <= 2.95:
        return "Medium"
    if 1.06 <= score <= 1.99:
        return "Low"
    if 0.12 <= score <= 1.05:
        return "Very Low"
    return "Very Low" if score < 2.96 else "High"


def calculate_attack_vector_level(vector: str) -> str:
    return ATTACK_VECTOR_TO_LEVEL.get(normalize_attack_vector(vector), "Low")


def calculate_risk_value(impact_value: int, feasibility_level: str) -> int:
    return RISK_MATRIX.get((impact_value, feasibility_level), 1)


def get_risk_treatment(risk_value: int) -> str:
    if risk_value >= 5:
        return "风险缓解"
    if risk_value == 4:
        return "风险缓解"
    if risk_value == 3:
        return "风险缓解"
    if risk_value == 2:
        return "风险接受"
    return "风险接受"


def save_json(data: Any, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5.5) Checkpoint（中间结果检查点）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CHECKPOINT_DIR = os.getenv("TARA_CHECKPOINT_DIR", os.path.join(OUTPUT_DIR, ".checkpoints"))
_CHECKPOINT_META_FILE = os.path.join(CHECKPOINT_DIR, "_meta.json")
_CHECKPOINT_FILES = {
    "stage_A_gen": "tasks_after_stageA.json",
    "stage_B_impact": "tasks_after_stageB.json",
    "stage_C_attack": "tasks_after_stageC.json",
}


def _checkpoint_path(stage: str) -> str:
    return os.path.join(CHECKPOINT_DIR, _CHECKPOINT_FILES[stage])


def _checkpoint_meta(method: str, total_tasks: int) -> dict:
    """生成当前运行的元信息快照。"""
    return {
        "method": method,
        "total_tasks": total_tasks,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "checkpoint_version": "v10.1",
    }


def save_checkpoint(tasks: list[TaskUnit], stage: str, method: str) -> str:
    """将 tasks 序列化保存到对应阶段的 checkpoint 文件。"""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    path = _checkpoint_path(stage)
    data = {
        "meta": _checkpoint_meta(method, len(tasks)),
        "tasks": _tasks_to_dict(tasks),
    }
    save_json(data, path)
    logger.info("[Checkpoint] %s 已保存 -> %s (%d 个任务)", stage, path, len(tasks))
    return path


def load_checkpoint(stage: str) -> list[TaskUnit] | None:
    """加载指定阶段的 checkpoint，不存在或损坏时返回 None。"""
    path = _checkpoint_path(stage)
    if not os.path.isfile(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tasks_data = data.get("tasks", [])
        if not tasks_data:
            logger.warning("[Checkpoint] %s 文件为空", stage)
            return None
        tasks = _tasks_from_dict(tasks_data)
        logger.info("[Checkpoint] %s 已加载 <- %s (%d 个任务)", stage, path, len(tasks))
        return tasks
    except Exception as e:
        logger.warning("[Checkpoint] %s 加载失败: %s，将重新运行该阶段", stage, e)
        return None


def resume_from_checkpoint(
    stages: list[str],
    method: str,
) -> tuple[int, list[TaskUnit] | None]:
    """按优先级检查已有的 checkpoint，找到第一个有效的 checkpoint。

    Args:
        stages: 按优先级排列的阶段名列表（从最新到最旧）。
        method: 期望的可行性方法。

    Returns:
        (stage_index, tasks) — stage_index 是需要继续的阶段的索引（0=S的开始），
        tasks 为已有 checkpoint 的 task 列表（stage_index > 0 时非 None）。
    """
    for i, stage in enumerate(stages):
        tasks = load_checkpoint(stage)
        if tasks is not None:
            # 校验元信息（方法是否一致）
            meta_path = _checkpoint_path(stage)
            try:
                with open(meta_path, "r", encoding="utf-8") as f:
                    meta = json.load(f).get("meta", {})
                saved_method = meta.get("method", "")
                if saved_method and saved_method != method:
                    logger.warning(
                        "[Checkpoint] %s 的方法(%s)与当前(%s)不一致，忽略",
                        stage, saved_method, method,
                    )
                    continue
            except Exception:
                pass

            # stage 在 stages 列表中的索引就是还需要跑的阶段范围
            remain_idx = i + 1
            if remain_idx >= len(stages):
                logger.info("[Checkpoint] 所有阶段已完成，直接使用最终结果")
                return remain_idx, tasks
            logger.info(
                "[Checkpoint] 从 %s checkpoint 恢复，将运行剩余 %d 个阶段",
                stage, len(stages) - remain_idx,
            )
            return remain_idx, tasks

    return 0, None


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6) 输入解析（DFD.py 逻辑）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_dfd_json(input_dir: str) -> list[dict]:
    type_mapping = {
        "tm.Flow": "信号",
        "tm.Process": "部件",
        "tm.Store": "数据",
        "tm.Actor": "接口",
    }

    all_results: list[dict] = []
    json_files = glob.glob(os.path.join(input_dir, "*.json"))

    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except Exception:
                logger.warning(f"DFD解析失败，跳过: {os.path.basename(file_path)}")
                continue

        current_result = {"function": "", "assets": []}

        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not current_result["function"] and "title" in diagram:
                    current_result["function"] = diagram["title"]

                if "diagramJson" not in diagram or "cells" not in diagram["diagramJson"]:
                    continue

                cells = diagram["diagramJson"]["cells"]
                id_to_cell = {c.get("id"): c for c in cells if c.get("id")}

                for cell in cells:
                    if cell.get("outOfScope") is True:
                        continue

                    raw_type = cell.get("type", "")
                    if raw_type == "tm.Boundary":
                        continue

                    asset_details = ""
                    finetermval_value = ""
                    for key, value in cell.items():
                        if key.startswith("propertyList") and isinstance(value, dict):
                            asset_details = value.get("assetDetails", "")
                            finetermval_value = value.get("finetermval", "")
                            break

                    if not finetermval_value:
                        continue

                    mapped_type = type_mapping.get(raw_type, raw_type)
                    target_value = ""
                    source_value = ""

                    if mapped_type == "部件":
                        target_value = finetermval_value

                    elif mapped_type == "数据":
                        cell_id = cell.get("id", "")
                        for flow_cell in cells:
                            if flow_cell.get("type") == "tm.Flow" and flow_cell.get("source", {}).get("id") == cell_id:
                                target_id = flow_cell.get("target", {}).get("id", "")
                                if target_id in id_to_cell:
                                    target_cell = id_to_cell[target_id]
                                    for k, v in target_cell.items():
                                        if k.startswith("propertyList") and isinstance(v, dict):
                                            target_value = v.get("finetermval", "")
                                            if target_value:
                                                break
                                if target_value:
                                    break

                    elif mapped_type == "信号":
                        target_id = cell.get("target", {}).get("id", "")
                        source_id = cell.get("source", {}).get("id", "")

                        if target_id in id_to_cell:
                            for k, v in id_to_cell[target_id].items():
                                if k.startswith("propertyList") and isinstance(v, dict):
                                    target_value = v.get("finetermval", "")
                                    if target_value:
                                        break

                        if source_id in id_to_cell:
                            for k, v in id_to_cell[source_id].items():
                                if k.startswith("propertyList") and isinstance(v, dict):
                                    source_value = v.get("finetermval", "")
                                    if source_value:
                                        break

                    item = {
                        "asset_type": mapped_type,
                        "asset_name": finetermval_value,
                        "assetDetails": asset_details,
                    }
                    if target_value:
                        item["target"] = target_value
                    if source_value:
                        item["source"] = source_value

                    current_result["assets"].append(item)

        if not current_result["function"]:
            current_result["function"] = os.path.splitext(os.path.basename(file_path))[0]

        if current_result["assets"]:
            all_results.append(current_result)

    logger.info(f"DFD解析完成: {len(all_results)} 个功能")
    return all_results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7) 拓扑解析
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TopologyMapUtil:
    @staticmethod
    def list_element_vo(cells: list[dict]) -> list[TopologyElementVO]:
        result: list[TopologyElementVO] = []
        for cell in cells:
            ele = TopologyElementVO(id=cell.get("id"))
            shape = cell.get("shape", "")
            attrs = cell.get("attrs", {})

            if shape == "custom-rounded-rect":
                stroke = attrs.get("body", {}).get("stroke", "")
                if stroke == "#016AFF":
                    ele.type = TopologyElementType.GATEWAY
                    ele.is_gateway = True
                else:
                    ele.type = TopologyElementType.COMPONENT
                ele.color = stroke
                ele.name = attrs.get("text", {}).get("text", "")

            elif shape == "custom-rounded-rect-dash":
                ele.type = TopologyElementType.EXTERNAL_COMPONENT
                ele.color = attrs.get("body", {}).get("stroke", "")
                ele.name = attrs.get("text", {}).get("text", "")

            elif shape == "edge":
                ele.type = TopologyElementType.LINE
                ele.color = attrs.get("line", {}).get("stroke", "")
                ele.source_id = cell.get("source", {}).get("cell")
                ele.target_id = cell.get("target", {}).get("cell")

            result.append(ele)
        return result

    @staticmethod
    def get_distinct_components(element_vos: list[TopologyElementVO]) -> list[TopologyElementVO]:
        components = [
            e for e in element_vos
            if e.type in [
                TopologyElementType.COMPONENT,
                TopologyElementType.GATEWAY,
                TopologyElementType.EXTERNAL_COMPONENT,
            ] and e.name
        ]

        name_map: dict[str, list[TopologyElementVO]] = defaultdict(list)
        for c in components:
            name_map[c.name].append(c)

        distinct: list[TopologyElementVO] = []
        for _, elements in name_map.items():
            first = elements[0]
            first.ids = [e.id for e in elements]
            distinct.append(first)
        return distinct

    @staticmethod
    def generate_full_path(
        external_interfaces: list[dict],
        element_vos: list[TopologyElementVO],
        protocol_legends: list[dict],
        distinct_components: list[TopologyElementVO],
    ) -> list[list[PathNodeVO]]:
        many_to_one: dict[str, str] = {}
        for comp in distinct_components:
            for old_id in comp.ids:
                many_to_one[old_id] = comp.id

        valid_colors = [p.get("color") for p in protocol_legends if p.get("color")]
        lines = [e for e in element_vos if e.type == TopologyElementType.LINE and e.color in valid_colors]

        for line in lines:
            line.source_id = many_to_one.get(line.source_id, line.source_id)
            line.target_id = many_to_one.get(line.target_id, line.target_id)

        ext_names = [ext["related_component"] for ext in external_interfaces]
        ext_ids = [c.id for c in distinct_components if c.name in ext_names]

        com_to_lines: dict[str, set] = defaultdict(set)
        line_to_coms: dict[str, set] = defaultdict(set)

        for line in lines:
            if line.source_id:
                com_to_lines[line.source_id].add(line.color)
                line_to_coms[line.color].add(line.source_id)
            if line.target_id:
                com_to_lines[line.target_id].add(line.color)
                line_to_coms[line.color].add(line.target_id)

        starts = {k: v for k, v in com_to_lines.items() if k in ext_ids}
        com_dic = {c.id: c for c in distinct_components}

        return TopologyMapUtil.get_path_by_bfs(starts, com_to_lines, line_to_coms, com_dic)

    @staticmethod
    def get_path_by_bfs(
        starts: dict[str, set],
        com_to_lines: dict[str, set],
        line_to_coms: dict[str, set],
        com_dic: dict[str, TopologyElementVO],
    ) -> list[list[PathNodeVO]]:
        all_paths: list[list[PathNodeVO]] = []

        for start_id in starts.keys():
            queue = deque([(start_id, set(), [])])
            while queue:
                current_com_id, visited, path = queue.popleft()

                direct_points = set()
                line_ids = com_to_lines.get(current_com_id, set())
                for l_id in line_ids:
                    direct_points.update(line_to_coms.get(l_id, set()))

                for l_id in line_ids:
                    if l_id in visited:
                        continue

                    for next_com in line_to_coms.get(l_id, set()):
                        if next_com not in com_dic or next_com == current_com_id or next_com in visited:
                            continue

                        node = PathNodeVO(
                            component_name_id=next_com,
                            component_name=com_dic[next_com].name,
                            pre_component_name_id=current_com_id,
                            pre_component_name=com_dic[current_com_id].name,
                            line_id=l_id,
                            color=l_id,
                            is_gateway=com_dic[next_com].is_gateway,
                        )

                        new_path = copy.deepcopy(path)
                        new_path.append(node)
                        all_paths.append(new_path)

                        new_visited = set(visited)
                        new_visited.update(line_ids)
                        new_visited.update(direct_points)
                        queue.append((next_com, new_visited, new_path))

        return all_paths


def _determine_attack_vector(ext_interface: str, color_name_map: dict, path_nodes: list[PathNodeVO]) -> str:
    interface_lower = (ext_interface or "").lower()

    # Network
    if any(k in interface_lower for k in ["4g", "5g", "lte", "蜂窝", "ota", "cloud", "远程", "ethernet", "以太网", "卫星定位"]):
        return "network"
    # Adjacent
    if any(k in interface_lower for k in ["wifi", "wlan", "蓝牙", "bluetooth", "ble", "星闪", "v2x", "射频信道"]):
        return "adjacent"
    # Local
    if any(k in interface_lower for k in ["usb", "tf卡", "sd卡", "ic卡", "充电口", "obd", "内部通讯", "nfc", "local", "本地登录"]):
        return "local"
    # Physical
    if any(k in interface_lower for k in ["jtag", "debug", "内部调试", "uart", "spi", "i2c", "物理"]):
        return "physical"

    if path_nodes:
        protocol_name = color_name_map.get(path_nodes[0].color, "").lower()
        if any(k in protocol_name for k in ["wifi", "wlan", "4g", "5g", "ethernet", "以太网"]):
            return "network"
        if any(k in protocol_name for k in ["蓝牙", "ble", "nfc"]):
            return "adjacent"
        if any(k in protocol_name for k in ["usb", "obd"]):
            return "local"

    return "local"


def _norm_text(v: Any) -> str:
    if v is None:
        return ""
    s = str(v).replace("\n", " ").replace("\r", " ").replace("\u3000", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def _is_checked(v: Any) -> bool:
    s = _norm_text(v).lower()
    if not s:
        return False
    if any(mark in s for mark in ["√", "✓", "☑", "✅"]):
        return True
    return s in {"1", "y", "yes", "true", "是", "有", "开", "通过"}


def _excel_col_to_idx(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha()).upper()
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord("A") + 1)
    return max(0, idx - 1)


def _parse_xlsx_rows_stdlib(excel_path: str) -> list[dict]:
    """
    无 pandas/openpyxl 时，使用标准库解析 xlsx。
    返回：[{row_num:int, cells:{col_idx: value}}...]
    """
    ns = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rel_ns = {"r": "http://schemas.openxmlformats.org/package/2006/relationships"}

    with zipfile.ZipFile(excel_path, "r") as zf:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall(".//x:si", ns):
                text = "".join((t.text or "") for t in si.findall(".//x:t", ns))
                shared_strings.append(text)

        # 找第一个工作表文件路径
        sheet_target = "worksheets/sheet1.xml"
        try:
            wb_root = ET.fromstring(zf.read("xl/workbook.xml"))
            sheets = wb_root.findall(".//x:sheets/x:sheet", ns)
            first_sheet = sheets[0] if sheets else None
            rid = first_sheet.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id") if first_sheet is not None else None
            if rid and "xl/_rels/workbook.xml.rels" in zf.namelist():
                rel_root = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
                for rel in rel_root.findall(".//r:Relationship", rel_ns):
                    if rel.attrib.get("Id") == rid:
                        sheet_target = rel.attrib.get("Target", sheet_target)
                        break
        except Exception:
            pass

        sheet_path = "xl/" + sheet_target.lstrip("/")
        if sheet_path not in zf.namelist():
            # 兜底
            candidates = [n for n in zf.namelist() if n.startswith("xl/worksheets/sheet") and n.endswith(".xml")]
            if not candidates:
                return []
            sheet_path = sorted(candidates)[0]

        sheet_root = ET.fromstring(zf.read(sheet_path))
        out_rows: list[dict] = []

        for row in sheet_root.findall(".//x:sheetData/x:row", ns):
            row_num = int(row.attrib.get("r", "0") or 0)
            cells: dict[int, str] = {}
            for c in row.findall("x:c", ns):
                ref = c.attrib.get("r", "")
                col_idx = _excel_col_to_idx(ref) if ref else 0
                ctype = c.attrib.get("t", "")
                value = ""
                if ctype == "s":
                    v = c.find("x:v", ns)
                    if v is not None and v.text and v.text.isdigit():
                        ss_idx = int(v.text)
                        if 0 <= ss_idx < len(shared_strings):
                            value = shared_strings[ss_idx]
                elif ctype == "inlineStr":
                    is_node = c.find("x:is", ns)
                    if is_node is not None:
                        value = "".join((t.text or "") for t in is_node.findall(".//x:t", ns))
                else:
                    v = c.find("x:v", ns)
                    value = v.text if (v is not None and v.text is not None) else ""
                cells[col_idx] = _norm_text(value)
            out_rows.append({"row_num": row_num, "cells": cells})

        return out_rows


def _load_external_interfaces(excel_path: str) -> list[dict]:
    """读取外部接口清单：pandas -> openpyxl -> 标准库 xlsx 解析。"""
    # 1) pandas 路径
    if pd is not None:
        try:
            df = pd.read_excel(excel_path, header=1)
            df.columns = [_norm_text(c) for c in df.columns]

            direct_info_col = next((c for c in df.columns if "外部接口" in c and "信息" in c), None)
            direct_comp_col = next((c for c in df.columns if "关联部件" in c), None)
            if direct_info_col and direct_comp_col:
                direct_items = []
                for _, row in df.iterrows():
                    ext = _norm_text(row.get(direct_info_col, ""))
                    comp = _norm_text(row.get(direct_comp_col, ""))
                    if ext and comp and comp.lower() != "nan":
                        direct_items.append(
                            {
                                "external_interface": ext,
                                "related_component": comp,
                                "all_interfaces": [ext],
                            }
                        )
                if direct_items:
                    return direct_items

            comp_col = next((c for c in df.columns if "零部件" in c and "名称" in c), None)
            if comp_col is None and len(df.columns) >= 2:
                comp_col = df.columns[1]
            if comp_col is None:
                comp_col = df.columns[0] if len(df.columns) else None

            if comp_col is None:
                return []

            interface_columns = [c for c in df.columns if c != comp_col and _norm_text(c)]
            # 对齐原逻辑：倾向跳过前两个说明列
            if len(df.columns) > 2:
                interface_columns = [c for c in df.columns[2:] if c != comp_col and _norm_text(c)]

            external_interfaces: list[dict] = []
            for _, row in df.iterrows():
                comp = _norm_text(row.get(comp_col, ""))
                if not comp or comp.lower() == "nan":
                    continue
                active = [col for col in interface_columns if _is_checked(row.get(col, ""))]
                if active:
                    external_interfaces.append(
                        {
                            "external_interface": active[0],
                            "related_component": comp,
                            "all_interfaces": active,
                        }
                    )
            if external_interfaces:
                return external_interfaces
        except Exception as e:
            logger.warning(f"pandas读取外部接口失败，尝试其他方式: {e}")

    # 2) openpyxl 路径
    try:
        from openpyxl import load_workbook

        wb = load_workbook(excel_path, data_only=True)
        ws = wb.active

        # 先尝试“外部接口信息 + 外部接口关联部件”直接映射格式
        direct_header_row = None
        info_idx = None
        comp_idx_direct = None
        for r in range(1, min(ws.max_row, 10) + 1):
            vals = [_norm_text(c.value) for c in ws[r]]
            for i, v in enumerate(vals):
                if "外部接口" in v and "信息" in v:
                    info_idx = i
                if "关联部件" in v:
                    comp_idx_direct = i
            if info_idx is not None and comp_idx_direct is not None:
                direct_header_row = r
                break

        if direct_header_row is not None:
            direct_items = []
            for row in ws.iter_rows(min_row=direct_header_row + 1, values_only=True):
                vals = [_norm_text(v) for v in row]
                ext = vals[info_idx] if info_idx < len(vals) else ""
                comp = vals[comp_idx_direct] if comp_idx_direct < len(vals) else ""
                if ext and comp:
                    direct_items.append(
                        {
                            "external_interface": ext,
                            "related_component": comp,
                            "all_interfaces": [ext],
                        }
                    )
            if direct_items:
                return direct_items

        # 优先寻找包含“零部件名称”的表头行
        header_row_idx = None
        comp_idx = None
        max_scan = min(ws.max_row, 20)
        for r in range(1, max_scan + 1):
            vals = [_norm_text(c.value) for c in ws[r]]
            for i, v in enumerate(vals):
                if "零部件" in v and "名称" in v:
                    header_row_idx = r
                    comp_idx = i
                    break
            if header_row_idx is not None:
                break

        if header_row_idx is None:
            header_row_idx = 2
            vals = [_norm_text(c.value) for c in ws[header_row_idx]]
            comp_idx = vals.index("零部件名称") if "零部件名称" in vals else 1

        header_vals = [_norm_text(c.value) for c in ws[header_row_idx]]
        interface_cols = [(i, h) for i, h in enumerate(header_vals) if i > comp_idx and h]

        external_interfaces: list[dict] = []
        for row in ws.iter_rows(min_row=header_row_idx + 1, values_only=True):
            row_vals = [_norm_text(v) for v in row]
            if comp_idx >= len(row_vals):
                continue
            comp = row_vals[comp_idx]
            if not comp:
                continue
            active = [h for i, h in interface_cols if i < len(row_vals) and _is_checked(row_vals[i])]
            if active:
                external_interfaces.append(
                    {
                        "external_interface": active[0],
                        "related_component": comp,
                        "all_interfaces": active,
                    }
                )
        if external_interfaces:
            return external_interfaces
    except Exception:
        pass

    # 3) 标准库兜底（无 pandas / openpyxl）
    try:
        rows = _parse_xlsx_rows_stdlib(excel_path)
    except Exception as e:
        logger.warning(f"标准库解析Excel失败: {e}")
        return []

    if not rows:
        return []

    # A) “外部接口信息 + 外部接口关联部件”直接映射格式
    direct_header = None
    info_idx = None
    comp_idx_direct = None
    for r in rows[:20]:
        for idx, v in r["cells"].items():
            if "外部接口" in v and "信息" in v:
                info_idx = idx
            if "关联部件" in v:
                comp_idx_direct = idx
        if info_idx is not None and comp_idx_direct is not None:
            direct_header = r["row_num"]
            break

    if direct_header is not None:
        direct_items = []
        for r in rows:
            if r["row_num"] <= direct_header:
                continue
            cells = r["cells"]
            ext = _norm_text(cells.get(info_idx, ""))
            comp = _norm_text(cells.get(comp_idx_direct, ""))
            if ext and comp:
                direct_items.append(
                    {
                        "external_interface": ext,
                        "related_component": comp,
                        "all_interfaces": [ext],
                    }
                )
        if direct_items:
            return direct_items

    # B) 勾选矩阵格式
    header_row_num = None
    comp_idx = None
    for r in rows[:30]:
        for idx, v in r["cells"].items():
            if "零部件" in v and "名称" in v:
                header_row_num = r["row_num"]
                comp_idx = idx
                break
        if header_row_num is not None:
            break

    if header_row_num is None:
        # 尝试第2行兜底
        row2 = next((r for r in rows if r["row_num"] == 2), None)
        if row2:
            header_row_num = 2
            for idx, v in row2["cells"].items():
                if "零部件" in v and "名称" in v:
                    comp_idx = idx
                    break
            if comp_idx is None:
                comp_idx = 1
        else:
            return []

    header_row = next((r for r in rows if r["row_num"] == header_row_num), None)
    if not header_row:
        return []

    interface_cols = [(idx, name) for idx, name in sorted(header_row["cells"].items()) if idx > comp_idx and name]

    external_interfaces: list[dict] = []
    for r in rows:
        if r["row_num"] <= header_row_num:
            continue
        cells = r["cells"]
        comp = _norm_text(cells.get(comp_idx, ""))
        if not comp:
            continue
        active = [name for idx, name in interface_cols if _is_checked(cells.get(idx, ""))]
        if active:
            external_interfaces.append(
                {
                    "external_interface": active[0],
                    "related_component": comp,
                    "all_interfaces": active,
                }
            )

    if not external_interfaces:
        logger.warning("外部接口Excel已读取，但未识别到勾选项；请检查表头与勾选符号。")
    return external_interfaces


def parse_topology_and_generate_frameworks(
    topology_file: str,
    external_interface_excel: str,
    asset_results: list[dict] | None = None,
) -> list[dict]:
    with open(topology_file, "r", encoding="utf-8") as f:
        topo_data = json.load(f).get("data", {})

    cells = topo_data.get("cells", [])
    protocol_legends = topo_data.get("lineData", [])
    color_name_map = {p.get("color"): p.get("name") for p in protocol_legends}

    if not os.path.exists(external_interface_excel):
        logger.warning(f"外部接口文件不存在: {external_interface_excel}")
        return []
    external_interfaces = _load_external_interfaces(external_interface_excel)
    if not external_interfaces:
        logger.warning("外部接口清单为空，攻击路径框架将为空。")
        return []

    # 从DFD解析结果中提取所有 target 名称，用于后续框架预匹配
    dfd_targets: set[str] = set()
    if asset_results:
        for func_data in asset_results:
            for asset in func_data.get("assets", []):
                t = (asset.get("target") or "").strip()
                if t:
                    dfd_targets.add(t)

    element_vos = TopologyMapUtil.list_element_vo(cells)
    distinct_components = TopologyMapUtil.get_distinct_components(element_vos)
    path_node_vos = TopologyMapUtil.generate_full_path(
        external_interfaces, element_vos, protocol_legends, distinct_components
    )

    source_target_map: dict[str, list[list[PathNodeVO]]] = defaultdict(list)
    for p in path_node_vos:
        source_id = p[0].pre_component_name_id
        target_id = p[-1].component_name_id
        source_target_map[f"{source_id}_{target_id}"].append(p)

    comp_to_interfaces: dict[str, list[str]] = defaultdict(list)
    for ext in external_interfaces:
        comp_to_interfaces[ext["related_component"]].append(ext["external_interface"])

    frameworks: list[dict] = []

    for _, paths in source_target_map.items():
        min_len = min(len(p) for p in paths) if paths else 0
        min_paths = [p for p in paths if len(p) == min_len]

        has_gateway = any(any(n.is_gateway for n in p) for p in min_paths)
        filtered = [p for p in min_paths if any(n.is_gateway for n in p)] if has_gateway else min_paths

        for p in filtered:
            start_comp_name = p[0].pre_component_name
            ext_list = comp_to_interfaces.get(start_comp_name, ["未知接口"])

            for ext in ext_list:
                desc = [f"[{ext}] -> {start_comp_name}"]
                for node in p:
                    proto = color_name_map.get(node.color, "未知协议")
                    desc.append(f" -> [{proto}] -> {node.component_name}")
                path_desc = "".join(desc)

                path_nodes = [start_comp_name] + [n.component_name for n in p]
                last_node = path_nodes[-1] if path_nodes else ""
                second_to_last_node = path_nodes[-2] if len(path_nodes) >= 2 else ""
                attack_vector = _determine_attack_vector(ext, color_name_map, p)

                # 预计算此框架匹配哪些 DFD target（last_node == target）
                matched_targets = [t for t in dfd_targets if t == last_node]

                frameworks.append({
                    "path_id": hashlib.sha1(path_desc.encode("utf-8")).hexdigest()[:16],
                    "path_description": path_desc,
                    "path_nodes": path_nodes,
                    "last_node": last_node,
                    "second_to_last_node": second_to_last_node,
                    "start_component": start_comp_name,
                    "external_interface": ext,
                    "components": [n.component_name for n in p],
                    "attack_vector": attack_vector,
                    "matched_targets": matched_targets,
                })

    unique = []
    seen = set()
    for fw in frameworks:
        if fw["path_description"] in seen:
            continue
        seen.add(fw["path_description"])
        unique.append(fw)

    logger.info(f"拓扑攻击路径框架生成完成: {len(unique)} 条")
    return unique


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8) Prompt（生成+内嵌自评）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DS_SYSTEM_PROMPT = """你是汽车网络安全专家，负责TARA损害场景建模。损害场景是”涉及车辆或车辆功能且影响道路使用者的不良后果”。
在输出前必须进行自检，仅输出通过自检的结果。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类别、安全属性、威胁类型，以及相关参考知识（RAG）。

## 生成规则

损害场景必须同时包含以下三个要素（缺一不可），且必须结合车辆实际域（如ADAS、车身控制、动力系统等）：

   1. 功能如何因为资产的安全属性被破坏而产生不良后果
   2. 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   3. 必须明确提到被破坏的资产名称

## 参考示例：

- “存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。”
- “车辆夜间行驶时，前照灯控制功能因’前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”

【自检3条件】
1) 必须同时体现 function、asset_name、attribute_name 与 threat_type 的逻辑关联。
2) 必须明确具体危害后果（道路使用者/车辆运营/隐私或财务等）。
3) 描述必须具体且可验证，避免空泛结论。

仅输出JSON：
{"damage_scenario": "..."}
"""

TS_SYSTEM_PROMPT = """你是汽车网络安全专家，负责TARA威胁场景建模。威胁场景是”为实现损害场景，资产的信息安全属性遭到破坏的潜在原因”。
在输出前必须进行自检，仅输出通过自检的结果。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类型、安全属性、威胁类型、损害场景，以及相关参考知识（RAG）。

## 生成规则

1. 威胁场景必须描述：
   - 明确指出被攻击的具体资产名称
   - 该资产被破坏的安全属性
   - 该安全属性被破坏的原因/攻击意图
2. 威胁场景必须能直接导致给定的损害场景，描述中必须体现”破坏该安全属性 -> 引发损害场景中的不良后果”逻辑链条
3. 损害场景与资产名称、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系应能被威胁场景包含或与威胁场景关联

## 参考示例：

- “攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。”
- “伪装信号导致发送至电源开关控制器的’灯光请求’信号的数据通信完整性丢失,可能造成前照灯意外关闭”

【自检3条件】
1) 必须明确攻击对象与受损安全属性（attribute_name）。
2) 必须说明威胁实现方式，并与threat_type保持一致。
3) 必须与damage_scenario保持因果一致。

仅输出JSON：
{"threat_scenarios": ["..."]}
"""

AP_SYSTEM_PROMPT = """你是汽车攻击路径专家。
在输出前必须进行自检，仅输出通过自检的结果。攻击路径是”为实现威胁场景的一组蓄意活动”。必须采用攻击树分析，从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类型、安全属性、威胁场景列表、攻击路径框架（path_frameworks），以及相关参考知识（RAG）。

## 生成规则

1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的威胁场景，每条路径的技术手段必须针对资产类型的特点设计
2. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）
3. 同一条威胁场景的多条攻击路径不要重复，避免冗余
4. 必须基于提供的路径框架（path_frameworks），从中选取一条或多条框架，将具体的攻击手段填入框架中的每个节点，构成完整的攻击链。

## 参考示例：

- “1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。”
- “1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- “1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

【自检3条件】
1) 攻击步骤必须与threat_scenarios一致。
2) 必须基于提供的路径框架，步骤链逻辑闭合。
3) 每条路径都要可执行、具体。

仅输出JSON：
{"attack_path": ["步骤链1", "步骤链2"]}
"""

IMPACT_SYSTEM_PROMPT = """你是ISO/SAE 21434影响评级专家。
根据损害场景输出四维评分（0~4）：Safety/Finance/Operation/Privacy。
仅输出JSON：
{"influence_parameters": {"Safety":0,"Finance":0,"Operation":0,"Privacy":0}}
"""

AP_FEASIBILITY_SYSTEM_PROMPT = """你是ISO/SAE 21434攻击潜力评估专家。
请输出五维分值，且必须从给定离散值中选择：
- Exposure_time: 0/1/4/17/19
- Professional_experience: 0/3/6/9
- Required_information: 0/3/7/11
- Opportunity_window: 0/1/4/10
- Required_equipment: 0/4/7/9
仅输出JSON：
{"attack_parameters": {"Exposure_time":0,"Professional_experience":0,"Required_information":0,"Opportunity_window":0,"Required_equipment":0}}
"""

CVSS_FEASIBILITY_SYSTEM_PROMPT = """你是CVSS可利用性评估专家。
请根据攻击路径分析并输出以下3个CVSS可利用性参数（攻击向量已由外部接口映射确定，无需输出）：
- 攻击复杂度: low(低)/high(高)
- 权限要求: none(无)/low(低)/high(高)
- 用户交互: none(无)/required(需要)

仅输出JSON：
{"attack_parameters": {"攻击复杂度":"high","权限要求":"none","用户交互":"none"}}
"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 9) 生成与评估核心函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_damage_scenario(task: TaskUnit, rag_context: str = "") -> str:
    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}

参考知识（RAG）:
{rag_context}

请输出 damage_scenario。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_type": task.threat_type,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"damage_scenario task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="damage_scenario",
            cache_key=cache_key,
            system_prompt=DS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"damage_scenario": ""},
        )

    if cache_hit:
        logger.debug(f"[cache] damage_scenario task={task.task_id}")

    return str(parsed.get("damage_scenario", "")).strip()


def generate_threat_scenarios(task: TaskUnit, rag_context: str = "") -> list[str]:
    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}
- damage_scenario: {task.damage_scenario}

参考知识（RAG）:
{rag_context}

请输出 threat_scenarios。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_type": task.threat_type,
        "damage_scenario": task.damage_scenario,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"threat_scenarios task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="threat_scenarios",
            cache_key=cache_key,
            system_prompt=TS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"threat_scenarios": []},
        )

    if cache_hit:
        logger.debug(f"[cache] threat_scenarios task={task.task_id}")

    scenarios = parsed.get("threat_scenarios", parsed.get("threat_scenario", []))
    if isinstance(scenarios, str):
        return [scenarios]
    if isinstance(scenarios, list):
        seen = set()
        deduped = []
        for s in scenarios:
            s = str(s).strip()
            if s and s not in seen:
                seen.add(s)
                deduped.append(s)
        return deduped
    return []


def generate_attack_paths(task: TaskUnit, frameworks: list[dict], rag_context: str = "") -> list[str]:
    fw_brief = [
        {
            "path_id": fw["path_id"],
            "path_description": fw["path_description"],
            "last_node": fw["last_node"],
            "attack_vector": fw["attack_vector"],
            "path_nodes": fw["path_nodes"],
        }
        for fw in frameworks[:8]
    ]

    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- path_frameworks: {json.dumps(fw_brief, ensure_ascii=False)}

参考知识（RAG）:
{rag_context}

请输出 attack_path。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_scenarios": task.threat_scenarios,
        "path_frameworks": fw_brief,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"attack_path task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="attack_path",
            cache_key=cache_key,
            system_prompt=AP_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_path": []},
        )

    if cache_hit:
        logger.debug(f"[cache] attack_path task={task.task_id}")

    paths = parsed.get("attack_path", parsed.get("attack_paths", []))
    if isinstance(paths, str):
        return [paths]
    if isinstance(paths, list):
        return [str(p).strip() for p in paths if str(p).strip()]
    return []


def generate_influence_parameters(task: TaskUnit) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- asset_type: {task.asset_type}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}
- damage_scenario: {task.damage_scenario}

请输出 influence_parameters。"""

    cache_key = {
        "asset_name": task.asset_name,
        "asset_type": task.asset_type,
        "attribute_name": task.attribute_name,
        "damage_scenario": task.damage_scenario,
    }

    with Timer(f"influence task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="influence_parameters",
            cache_key=cache_key,
            system_prompt=IMPACT_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"influence_parameters": {"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}},
        )

    params = parsed.get("influence_parameters", {})
    return {
        "Safety": int(params.get("Safety", 0)),
        "Finance": int(params.get("Finance", 0)),
        "Operation": int(params.get("Operation", 0)),
        "Privacy": int(params.get("Privacy", 0)),
    }


def generate_attack_parameters_attack_potential(task: TaskUnit, attack_path: str) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- attack_path: {attack_path}

请输出 attack_parameters。"""

    cache_key = {
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "attack_path": attack_path,
    }

    with Timer(f"feasibility(ap) task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="attack_parameters_ap",
            cache_key=cache_key,
            system_prompt=AP_FEASIBILITY_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_parameters": {k: 0 for k in AP_ALLOWED_VALUES}},
        )

    params = parsed.get("attack_parameters", {})
    fixed = {}
    for k, allowed in AP_ALLOWED_VALUES.items():
        v = int(params.get(k, 0))
        # 强制吸附到最近合法值
        fixed[k] = min(allowed, key=lambda x: abs(x - v))
    return fixed


def generate_attack_parameters_cvss(task: TaskUnit, attack_path: str, attack_vector: str = "local") -> dict:
    """生成CVSS参数。攻击向量从外部接口映射预先确定（不调用大模型）。"""
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- attack_path: {attack_path}
- 攻击向量（已从外部接口映射确定）: {attack_vector}

请分析上述攻击路径，输出剩余3个CVSS可利用性参数。"""

    cache_key = {
        "asset_name": task.asset_name,
        "attack_path": attack_path,
    }

    with Timer(f"feasibility(cvss) task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="attack_parameters_cvss",
            cache_key=cache_key,
            system_prompt=CVSS_FEASIBILITY_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_parameters": {"攻击复杂度": "high", "权限要求": "none", "用户交互": "none"}},
        )

    p = parsed.get("attack_parameters", {})
    return {
        "攻击向量": attack_vector,  # 使用预映射值，非LLM生成
        "攻击复杂度": str(p.get("攻击复杂度", "high")).strip().lower(),
        "权限要求": str(p.get("权限要求", "none")).strip().lower(),
        "用户交互": str(p.get("用户交互", "none")).strip().lower(),
    }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 10) 任务构建 + 路径筛选
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_task_units(asset_results: list[dict]) -> list[TaskUnit]:
    tasks: list[TaskUnit] = []
    idx = 0
    for func_data in asset_results:
        function_name = func_data.get("function", "")
        for asset in func_data.get("assets", []):
            asset_type = asset.get("asset_type", "")
            asset_name = asset.get("asset_name", "")
            target = asset.get("target", "")
            source = asset.get("source", "")

            attrs = SECURITY_ATTRIBUTES_MAP.get(asset_type, ["完整性"])
            for attr in attrs:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                tasks.append(
                    TaskUnit(
                        task_id=f"{idx:05d}",
                        function=function_name,
                        asset_name=asset_name,
                        asset_type=asset_type,
                        attribute_name=attr,
                        threat_type=threat_type,
                        target=target,
                        source=source,
                    )
                )
                idx += 1

    logger.info(f"任务单元构建完成: {len(tasks)}")
    return tasks


def filter_path_frameworks(task: TaskUnit, all_frameworks: list[dict]) -> list[dict]:
    target = (task.target or "").strip()
    source = (task.source or "").strip()
    is_signal = task.asset_type == "信号"

    selected: list[dict] = []

    for fw in all_frameworks:
        # 优先使用框架预计算的 matched_targets（与 DFD 解析结果联动）
        matched = fw.get("matched_targets", [])
        target_ok = bool(matched) and target in matched
        if not target_ok:
            # 回退到动态匹配（无 DFD 数据时的兼容）
            last_node = (fw.get("last_node") or "").strip()
            target_ok = bool(target) and last_node == target

        if not target_ok:
            continue

        # 信号类额外检查：source 必须等于路径倒数第二个节点
        if is_signal and source:
            second_to_last = fw.get("second_to_last_node", "")
            if source != second_to_last:
                continue

        selected.append(fw)

    # 回退策略：如果严格筛选为空，放宽到 target 命中路径字符串
    if not selected and target:
        selected = [fw for fw in all_frameworks if target in fw.get("path_description", "")]

    # 仍为空时，再按资产名尝试
    if not selected and task.asset_name:
        selected = [fw for fw in all_frameworks if task.asset_name in fw.get("path_description", "")]

    if len(selected) > 8:
        selected = selected[:8]

    return selected


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 11) 并发流水线
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def process_task_generation(
    task: TaskUnit,
    all_frameworks: list[dict],
    rag_contexts: Optional[dict[str, str]] = None,
) -> TaskUnit:
    """阶段A：每个资产独立并发（损害->威胁->攻击路径生成）"""
    rag_contexts = rag_contexts or {}
    rag_ds = rag_contexts.get("damage", "")
    rag_ts = rag_contexts.get("threat", "")
    rag_ap = rag_contexts.get("attack_path", "")
    try:
        task.damage_scenario = generate_damage_scenario(task, rag_ds)
        if not task.damage_scenario:
            task.errors.append("damage_scenario为空")

        task.threat_scenarios = generate_threat_scenarios(task, rag_ts)
        if not task.threat_scenarios:
            task.errors.append("threat_scenarios为空")

        selected = filter_path_frameworks(task, all_frameworks)
        task.selected_path_frameworks = selected

        task.attack_paths = generate_attack_paths(task, selected, rag_ap)
        if not task.attack_paths:
            task.errors.append("attack_path为空")

    except Exception as e:
        task.errors.append(f"生成阶段异常: {e}")

    return task


def process_task_gen_impact(
    task: TaskUnit,
    all_frameworks: list[dict],
    rag_contexts: dict[str, str],
) -> TaskUnit:
    """单资产 A+B 阶段：损害场景→威胁场景→攻击路径(缓存)→影响评级（不含可行性评估）。"""
    try:
        # ── Stage A: 场景生成 ──────────────────────────
        task.damage_scenario = generate_damage_scenario(task, rag_contexts.get("damage", ""))
        if not task.damage_scenario:
            task.errors.append("damage_scenario为空")

        task.threat_scenarios = generate_threat_scenarios(task, rag_contexts.get("threat", ""))
        if not task.threat_scenarios:
            task.errors.append("threat_scenarios为空")

        selected = filter_path_frameworks(task, all_frameworks)
        task.selected_path_frameworks = selected

        # 攻击路径缓存：同一 asset_name 的路径相同，跨功能复用
        cached_paths = _get_attack_path_cache(task.asset_name)
        if cached_paths is not None:
            task.attack_paths = cached_paths
            logger.debug(f"[path_cache] task={task.task_id} 复用 asset={task.asset_name} 的攻击路径")
        else:
            task.attack_paths = generate_attack_paths(task, selected, rag_contexts.get("attack_path", ""))
            if task.attack_paths:
                _set_attack_path_cache(task.asset_name, task.attack_paths)

        if not task.attack_paths:
            task.errors.append("attack_path为空")

        # ── Stage B: 影响评级 ──────────────────────────
        params = generate_influence_parameters(task)
        task.influence_parameters = params
        task.impact_value = calculate_impact_value(params)
        task.impact_level = calculate_impact_level(task.impact_value)

    except Exception as e:
        task.errors.append(f"A+B阶段异常: {e}")

    return task


def process_in_batches(items: list[Any], batch_size: int, worker_fn, stage_name: str) -> list[Any]:
    """连续批处理：上一批结束后再提交下一批。"""
    outputs: list[Any] = [None] * len(items)
    total = len(items)
    if total == 0:
        return outputs

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = items[start:end]
        logger.info(f"[{stage_name}] 连续批处理 {start + 1}-{end}/{total}")

        with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(batch))) as executor:
            future_map = {executor.submit(worker_fn, i, item): i for i, item in enumerate(batch, start=start)}
            for future in as_completed(future_map):
                i = future_map[future]
                try:
                    outputs[i] = future.result()
                except Exception as e:
                    outputs[i] = e

        if end < total:
            time.sleep(CALL_INTERVAL)

    return outputs


def stage_b_influence_batch(tasks: list[TaskUnit]) -> None:
    """阶段B：influence_parameters 连续批处理。"""

    def worker(index: int, task: TaskUnit):
        params = generate_influence_parameters(task)
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        return {
            "index": index,
            "params": params,
            "impact_value": impact_value,
            "impact_level": impact_level,
        }

    results = process_in_batches(tasks, BATCH_SIZE_INFLUENCE, worker, "Impact")
    for item in results:
        if isinstance(item, dict):
            task = tasks[item["index"]]
            task.influence_parameters = item["params"]
            task.impact_value = item["impact_value"]
            task.impact_level = item["impact_level"]


def stage_c_attack_batch(tasks: list[TaskUnit], method: str) -> None:
    """阶段C：attack_parameters 连续批处理 + 可行性计算 + 风险矩阵。"""

    jobs: list[dict] = []
    for ti, task in enumerate(tasks):
        if not task.attack_paths:
            # 没有攻击路径时，给一个兜底记录
            task.attack_details = []
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")
            task.risk_treatment = get_risk_treatment(task.risk_value)
            continue

        for ai, ap_text in enumerate(task.attack_paths):
            vector = "local"
            if task.selected_path_frameworks:
                fw = task.selected_path_frameworks[ai % len(task.selected_path_frameworks)]
                vector = fw.get("attack_vector", "local")

            jobs.append({
                "task_index": ti,
                "attack_index": ai,
                "attack_path": ap_text,
                "attack_vector": vector,
            })

    def worker(index: int, job: dict):
        task = tasks[job["task_index"]]
        attack_path = job["attack_path"]
        attack_vector = job["attack_vector"]

        feasibility_display = ""

        if method == "attack_vector":
            params = {"攻击向量": normalize_attack_vector(attack_vector)}
            feasibility_score = None
            feasibility_level = calculate_attack_vector_level(attack_vector)

        elif method == "cvss":
            params = generate_attack_parameters_cvss(task, attack_path, attack_vector)
            feasibility_score = calculate_cvss_score(params)
            feasibility_level = calculate_cvss_level(feasibility_score)
            # CVSS公式展示：E = 8.22 × V × C × P × U
            v = CVSS_V.get(params.get("攻击向量", "local"), 0.55)
            c = CVSS_C.get(params.get("攻击复杂度", "high"), 0.44)
            p = CVSS_P.get(params.get("权限要求", "none"), 0.85)
            u = CVSS_U.get(params.get("用户交互", "none"), 0.85)

        else:  # attack_potential
            params = generate_attack_parameters_attack_potential(task, attack_path)
            feasibility_score = calculate_attack_potential_score(params)
            feasibility_level = calculate_attack_potential_level(feasibility_score)

        risk_value = calculate_risk_value(task.impact_value, feasibility_level)
        risk_treatment = get_risk_treatment(risk_value)

        return {
            "job": job,
            "attack_parameters": params,
            "feasibility_score": feasibility_score,
            "feasibility_level": feasibility_level,
            "risk_value": risk_value,
            "risk_treatment": risk_treatment,
            "feasibility_display": feasibility_display if method == "cvss" else "",
        }

    results = process_in_batches(jobs, BATCH_SIZE_ATTACK, worker, "Feasibility")

    grouped: dict[int, list[dict]] = defaultdict(list)
    for r in results:
        if not isinstance(r, dict):
            continue
        ti = r["job"]["task_index"]
        grouped[ti].append(r)

    for ti, task in enumerate(tasks):
        details = []
        for r in sorted(grouped.get(ti, []), key=lambda x: x["job"]["attack_index"]):
            job = r["job"]
            details.append({
                "attack_path": job["attack_path"],
                "attack_vector": normalize_attack_vector(job["attack_vector"]),
                "attack_parameters": r["attack_parameters"],
                "feasibility_score": r["feasibility_score"],
                "feasibility_level": r["feasibility_level"],
                "risk_value": r["risk_value"],
                "risk_treatment": r["risk_treatment"],
                "feasibility_display": r.get("feasibility_display", ""),
            })

        task.attack_details = details

        if details:
            task.risk_value = max(d["risk_value"] for d in details)
        else:
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")

        task.risk_treatment = get_risk_treatment(task.risk_value)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 12) 报告构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_report(tasks: list[TaskUnit], feasibility_method: str) -> list[dict]:
    function_map: dict[str, dict] = {}

    for task in tasks:
        if task.function not in function_map:
            function_map[task.function] = {"function": task.function, "assets": {}}

        asset_key = (task.asset_name, task.asset_type, task.target, task.source)
        assets = function_map[task.function]["assets"]

        if asset_key not in assets:
            assets[asset_key] = {
                "asset_name": task.asset_name,
                "asset_type": task.asset_type,
                "target": task.target,
                "source": task.source,
                "security_attributes": [],
            }

        detail = {
            "attribute_name": task.attribute_name,
            "threat_type": task.threat_type,
            "damage_scenario": task.damage_scenario,
            "threat_scenarios": task.threat_scenarios,
            "influence_parameters": task.influence_parameters,
            "impact_value": task.impact_value,
            "impact_level": task.impact_level,
            "feasibility_method": feasibility_method,
            "attack": task.attack_details,
        }
        if task.errors:
            detail["errors"] = task.errors

        assets[asset_key]["security_attributes"].append(detail)

    report = []
    for func_data in function_map.values():
        report.append({
            "function": func_data["function"],
            "assets": list(func_data["assets"].values()),
        })

    return report


def build_error_report(tasks: list[TaskUnit]) -> list[dict]:
    rows = []
    for task in tasks:
        if not task.errors:
            continue
        rows.append({
            "task_id": task.task_id,
            "function": task.function,
            "asset_name": task.asset_name,
            "asset_type": task.asset_type,
            "attribute_name": task.attribute_name,
            "threat_type": task.threat_type,
            "errors": task.errors,
        })
    return rows


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 13) 主流程
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_dir: str,
    topology_file: str,
    external_interface_excel: str,
    output_path: str,
    feasibility_method: str = FEASIBILITY_METHOD,
    max_assets_for_test: Optional[int] = None,
    skip_checkpoint_resume: bool = False,
) -> tuple[list[dict], list[dict]]:
    method = (feasibility_method or "attack_potential").strip().lower()
    if method not in {"attack_potential", "attack_vector", "cvss"}:
        logger.warning(f"未知可行性方法: {method}，回退为 attack_potential")
        method = "attack_potential"

    _pipeline_start = time.perf_counter()

    logger.info("=" * 72)
    logger.info("TARA v10 开始")
    logger.info(f"可行性评估方法: {method}")
    logger.info("=" * 72)

    # ─── Step 1: DFD资产识别 ─────────────────────────
    with Timer("Step 1 — DFD资产识别") as _:
        asset_results = parse_dfd_json(data_flow_dir)

    # 测试模式：仅取前N个DFD资产
    if max_assets_for_test is not None and max_assets_for_test > 0:
        limited_results = []
        count = 0
        for func_data in asset_results:
            assets = func_data.get("assets", [])
            if count >= max_assets_for_test:
                break
            remain = max_assets_for_test - count
            if len(assets) > remain:
                copy_func = dict(func_data)
                copy_func["assets"] = assets[:remain]
                limited_results.append(copy_func)
                count += remain
                break
            else:
                limited_results.append(func_data)
                count += len(assets)
        asset_results = limited_results
        logger.info(f"测试模式启用：仅使用前 {max_assets_for_test} 个DFD资产，实际纳入 {count} 个")

    # ─── Step 2: 拓扑攻击路径框架 ────────────────────
    with Timer("Step 2 — 拓扑攻击路径框架") as _:
        path_frameworks = []
        if os.path.exists(topology_file) and os.path.exists(external_interface_excel):
            path_frameworks = parse_topology_and_generate_frameworks(topology_file, external_interface_excel, asset_results)
        else:
            logger.warning("拓扑文件或接口清单不存在，攻击路径框架为空")

    # ─── Step 3: 扩展属性 + RAG ──────────────────────
    with Timer("Step 3 — 任务构建 + RAG") as _:
        tasks = build_task_units(asset_results)
        rag_contexts = {
            "damage": rag_kb.retrieve(
                "汽车网络安全 损害场景 STRIDE ISO21434",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
            "threat": rag_kb.retrieve(
                "汽车威胁场景 CAPEC ATT&CK",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
            "attack_path": rag_kb.retrieve(
                "汽车攻击路径 CAN总线 OBD T-Box",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
        }
        logger.info(
            "RAG上下文长度: damage=%d, threat=%d, attack_path=%d",
            len(rag_contexts["damage"]),
            len(rag_contexts["threat"]),
            len(rag_contexts["attack_path"]),
        )

    # ─── Checkpoint 恢复 ──────────────────────────────
    # 全流水线模式下只有一个 checkpoint 点：stage_C_attack（最终完成态）
    checkpoint_stages = ["stage_C_attack"]
    current_task_data: list[TaskUnit] | None = None
    resume_from = 0

    if not skip_checkpoint_resume:
        resume_from, current_task_data = resume_from_checkpoint(checkpoint_stages, method)
    else:
        logger.info("Checkpoint恢复已跳过（skip_checkpoint_resume=True）")

    # ─── Step 4-6: A+B per-asset 并行 + C 批量评估 ──
    if resume_from <= 0:
        # ── Stage A+B: per-asset 并行 ──────────────────
        with Timer("Stage A+B — 场景生成+影响评级 (per-asset 并行)") as _:
            logger.info(f"Stage A+B：每个资产独立完成生成+影响评级，资产数={len(tasks)}")
            completed_tasks: list[TaskUnit] = []

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_map = {
                    executor.submit(process_task_gen_impact, t, path_frameworks, rag_contexts): t
                    for t in tasks
                }
                done = 0
                total = len(future_map)
                for future in as_completed(future_map):
                    t = future_map[future]
                    done += 1
                    try:
                        completed_tasks.append(future.result())
                    except Exception as e:
                        t.errors.append(f"A+B阶段异常: {e}")
                        completed_tasks.append(t)
                    if done < total:
                        time.sleep(CALL_INTERVAL)

            completed_tasks.sort(key=lambda x: x.task_id)

        # ── Stage C: 攻击可行性批量评估 ─────────────────
        with Timer("Stage C — 攻击可行性批量评估") as _:
            total_aps = sum(len(t.attack_paths or []) for t in completed_tasks)
            logger.info(f"Stage C：批量评估攻击可行性，总攻击路径数={total_aps}")
            stage_c_attack_batch(completed_tasks, method)

        save_checkpoint(completed_tasks, "stage_C_attack", method)
        current_task_data = completed_tasks
    else:
        logger.info("全流水线跳过（从 checkpoint 恢复，全部已完成）")

    assert current_task_data is not None, "当前任务数据为空，无法继续"

    # ─── Step 7: 报告 ────────────────────────────────
    with Timer("Step 7 — 报告构建") as _:
        report = build_tara_report(current_task_data, method)
        err_report = build_error_report(current_task_data)

        save_json(report, output_path)
        save_json(err_report, output_path.replace(".json", "_errors.json"))

    total_elapsed = time.perf_counter() - _pipeline_start
    logger.info("=" * 72)
    logger.info(f"TARA v10 完成: 总任务={len(current_task_data)}, 异常任务={len(err_report)}")
    logger.info(f"总耗时: {total_elapsed:.2f} 秒 ({total_elapsed / 60:.2f} 分钟)")
    logger.info(f"主报告: {output_path}")
    logger.info(f"异常报告: {output_path.replace('.json', '_errors.json')}")
    logger.info("=" * 72)

    return report, err_report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 14) 入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":
    topology_file = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\topology\拓扑图数据导出2026_5_9 10_43_29.json"
    external_interface_excel = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\information\对外接口清单.xlsx"
    data_flow_dir = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\DFD"

    # 你可以在这里切换：attack_potential / attack_vector / cvss
    method = os.getenv("TARA_FEASIBILITY_METHOD", FEASIBILITY_METHOD)
    # 测试时仅使用前N个DFD资产（默认1，设为0或负数则关闭）
    test_max_assets = int(os.getenv("TARA_TEST_MAX_ASSETS", "1"))
    if test_max_assets <= 0:
        test_max_assets = None
    # Checkpoint恢复：设 TARA_SKIP_CHECKPOINT=1 可强制重跑所有阶段
    skip_cp = os.getenv("TARA_SKIP_CHECKPOINT", "0").strip() in {"1", "true", "True"}

    run_tara(
        data_flow_dir=data_flow_dir,
        topology_file=topology_file,
        external_interface_excel=external_interface_excel,
        output_path=os.path.join(OUTPUT_DIR, "tara_report_v10.json"),
        feasibility_method=method,
        max_assets_for_test=test_max_assets,
        skip_checkpoint_resume=skip_cp,
    )

2026-05-15 11:00:27,004 [INFO] ========================================================================
2026-05-15 11:00:27,005 [INFO] TARA v10 开始
2026-05-15 11:00:27,005 [INFO] 可行性评估方法: cvss
2026-05-15 11:00:27,006 [INFO] ========================================================================
2026-05-15 11:00:27,006 [INFO] ═══ [计时] Step 1 — DFD资产识别 开始 ═══
2026-05-15 11:00:27,008 [INFO] DFD解析完成: 2 个功能
2026-05-15 11:00:27,009 [INFO] ═══ [计时] Step 1 — DFD资产识别 完成，耗时 0.00 秒 (0.00 分钟) ═══
2026-05-15 11:00:27,009 [INFO] 测试模式启用：仅使用前 1 个DFD资产，实际纳入 1 个
2026-05-15 11:00:27,009 [INFO] ═══ [计时] Step 2 — 拓扑攻击路径框架 开始 ═══
2026-05-15 11:00:27,130 [INFO] 拓扑攻击路径框架生成完成: 1169 条
2026-05-15 11:00:27,132 [INFO] ═══ [计时] Step 2 — 拓扑攻击路径框架 完成，耗时 0.12 秒 (0.00 分钟) ═══
2026-05-15 11:00:27,132 [INFO] ═══ [计时] Step 3 — 任务构建 + RAG 开始 ═══
2026-05-15 11:00:27,133 [INFO] 任务单元构建完成: 6
2026-05-15 11:00:27,139 [INFO] RAG文档加载完成: 1 条
2026-05-15 11:00:27,139 [INFO] RAG上下文长度: damage=518, threat=518, attack_path=518
2026-05-15

In [7]:

import copy
import glob
import hashlib
import json
import logging
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from difflib import SequenceMatcher
from threading import Lock, Semaphore
from typing import Any, Optional

try:
    import pandas as pd
except Exception:
    pd = None
from urllib import error as urlerror
from urllib import request as urlrequest


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1) 全局配置
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("TARA-V10")


MODEL_NAME = os.getenv("TARA_MODEL", "qwen3.6-35b-a3b")
API_KEY = os.getenv("TARA_API_KEY", "sk-3458388d4845414387fe2e10bc8a9ee2")
API_BASE = os.getenv("TARA_API_BASE", "https://dashscope.aliyuncs.com/compatible-mode/v1")

if not API_KEY:
    logger.warning("未检测到环境变量 TARA_API_KEY。")

REQUEST_TIMEOUT = int(os.getenv("TARA_REQUEST_TIMEOUT", "90"))
MAX_TOKENS = int(os.getenv("TARA_MAX_TOKENS", "4096"))
TEMPERATURE = float(os.getenv("TARA_TEMPERATURE", "0.1"))

MAX_WORKERS = int(os.getenv("TARA_MAX_WORKERS", "6"))
MAX_CONCURRENT_LLM = int(os.getenv("TARA_MAX_CONCURRENT_LLM", "3"))
CALL_INTERVAL = float(os.getenv("TARA_CALL_INTERVAL", "0.4"))

BATCH_SIZE_INFLUENCE = int(os.getenv("TARA_BATCH_SIZE_INFLUENCE", "8"))
BATCH_SIZE_ATTACK = int(os.getenv("TARA_BATCH_SIZE_ATTACK", "8"))

# 可选：attack_potential / attack_vector / cvss
FEASIBILITY_METHOD = "cvss"

OUTPUT_DIR = r"D:\Jupyter profile\汽车信息安全风险评估\outputs\V14"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ENABLE_RAG = os.getenv("TARA_ENABLE_RAG", "1").strip() not in {"0", "false", "False"}
RAG_BASE_DIR = os.getenv("TARA_RAG_BASE_DIR", r"D:\Jupyter profile\rag")
RAG_DIRS = {
    "tara_reports": os.path.join(RAG_BASE_DIR, "tara_reports"),
    "regulations": os.path.join(RAG_BASE_DIR, "regulations"),
    "attack_databases": os.path.join(RAG_BASE_DIR, "attack_databases"),
}
RAG_TOP_K = int(os.getenv("TARA_RAG_TOP_K", "5"))
RAG_MAX_CONTEXT_CHARS = int(os.getenv("TARA_RAG_MAX_CHARS", "1800"))

llm_semaphore = Semaphore(MAX_CONCURRENT_LLM)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1.5) 计时工具
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class Timer:
    """分阶段计时器：with 块自动记录耗时并日志输出。"""

    def __init__(self, stage_name: str):
        self.stage_name = stage_name
        self.start = 0.0

    def __enter__(self):
        self.start = time.perf_counter()
        logger.info("═══ [计时] %s 开始 ═══", self.stage_name)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.perf_counter() - self.start
        if exc_type is None:
            logger.info("═══ [计时] %s 完成，耗时 %.2f 秒 (%.2f 分钟) ═══", self.stage_name, elapsed, elapsed / 60)
        else:
            logger.error("═══ [计时] %s 异常退出，耗时 %.2f 秒，错误: %s ═══", self.stage_name, elapsed, exc_val)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2) 业务常量
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SECURITY_ATTRIBUTES_MAP: dict[str, list[str]] = {
    "数据": ["完整性", "机密性", "可用性"],
    "信号": ["完整性", "机密性", "真实性", "可用性"],
    "部件": ["完整性", "机密性", "真实性", "不可抵赖性", "权限属性", "可用性"],
    "接口": ["完整性", "机密性", "真实性", "可用性"],
}

ATTRIBUTE_TO_THREAT: dict[str, str] = {
    "完整性": "篡改",
    "机密性": "信息泄露",
    "可用性": "拒绝服务",
    "真实性": "欺骗",
    "不可抵赖性": "抵赖",
    "权限属性": "越权",
}

# Attack Potential 五维取值规范（ISO/SAE 21434 常见量表）
AP_ALLOWED_VALUES = {
    "Exposure_time": [0, 1, 4, 17, 19],
    "Professional_experience": [0, 3, 6, 9],
    "Required_information": [0, 3, 7, 11],
    "Opportunity_window": [0, 1, 4, 10],
    "Required_equipment": [0, 4, 7, 9],
}

# CVSS 取值
CVSS_V = {
    "network": 0.85,
    "adjacent": 0.62,
    "local": 0.55,
    "physical": 0.20,
}
CVSS_C = {"low": 0.77, "high": 0.44}
CVSS_P = {"none": 0.85, "low": 0.62, "high": 0.27}
CVSS_U = {"none": 0.85, "required": 0.62}

# 攻击向量 -> 可行性等级映射（ISO 21434 G.4）
ATTACK_VECTOR_TO_LEVEL = {
    "network": "High",
    "adjacent": "Medium",
    "local": "Low",
    "physical": "Very Low",
}

RISK_MATRIX = {
    (4, "High"): 5, (4, "Medium"): 5, (4, "Low"): 4, (4, "Very Low"): 4,
    (3, "High"): 5, (3, "Medium"): 4, (3, "Low"): 4, (3, "Very Low"): 3,
    (2, "High"): 4, (2, "Medium"): 4, (2, "Low"): 3, (2, "Very Low"): 2,
    (1, "High"): 3, (1, "Medium"): 2, (1, "Low"): 2, (1, "Very Low"): 1,
}


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3) 语义缓存
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SemanticCache:
    """轻量语义缓存：规范化文本 + 相似度匹配。"""

    def __init__(self, similarity_threshold: float = 0.985, max_bucket_size: int = 1000):
        self.similarity_threshold = similarity_threshold
        self.max_bucket_size = max_bucket_size
        self._store: dict[str, list[dict]] = defaultdict(list)
        self._lock = Lock()

    @staticmethod
    def _normalize(obj: Any) -> str:
        if isinstance(obj, (dict, list, tuple)):
            text = json.dumps(obj, ensure_ascii=False, sort_keys=True)
        else:
            text = str(obj)
        text = text.lower().strip()
        text = re.sub(r"\s+", "", text)
        text = text.replace("：", ":")
        return text

    def get(self, bucket: str, key_obj: Any) -> Optional[Any]:
        signature = self._normalize(key_obj)
        key_hash = hashlib.sha256(signature.encode("utf-8")).hexdigest()

        with self._lock:
            items = self._store.get(bucket, [])
            for item in items:
                if item["hash"] == key_hash:
                    return copy.deepcopy(item["value"])
                ratio = SequenceMatcher(None, signature, item["signature"]).ratio()
                if ratio >= self.similarity_threshold:
                    return copy.deepcopy(item["value"])
        return None

    def set(self, bucket: str, key_obj: Any, value: Any) -> None:
        signature = self._normalize(key_obj)
        key_hash = hashlib.sha256(signature.encode("utf-8")).hexdigest()

        with self._lock:
            items = self._store[bucket]
            for item in items:
                if item["hash"] == key_hash:
                    item["value"] = copy.deepcopy(value)
                    return
            items.append({"hash": key_hash, "signature": signature, "value": copy.deepcopy(value)})
            if len(items) > self.max_bucket_size:
                del items[0 : len(items) - self.max_bucket_size]


semantic_cache = SemanticCache()

# 攻击路径缓存：相同 asset_name 的攻击路径相同，跨功能/属性复用
_attack_path_cache: dict[str, list[str]] = {}
_attack_path_cache_lock = Lock()


def _get_attack_path_cache(asset_name: str) -> list[str] | None:
    with _attack_path_cache_lock:
        return copy.deepcopy(_attack_path_cache.get(asset_name))


def _set_attack_path_cache(asset_name: str, paths: list[str]) -> None:
    with _attack_path_cache_lock:
        _attack_path_cache[asset_name] = copy.deepcopy(paths)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.5) 轻量 RAG（兼容无第三方依赖环境）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SimpleRAGKnowledgeBase:
    def __init__(self, rag_dirs: dict[str, str]):
        self.rag_dirs = rag_dirs
        self.docs: list[dict] = []
        self._loaded = False
        self._lock = Lock()

    def _safe_read_text(self, path: str) -> str:
        for enc in ("utf-8", "utf-8-sig", "gbk", "gb18030"):
            try:
                with open(path, "r", encoding=enc) as f:
                    return f.read()
            except Exception:
                continue
        return ""

    def _load_json_text(self, path: str) -> str:
        try:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, (dict, list)):
                return json.dumps(obj, ensure_ascii=False)
            return str(obj)
        except Exception:
            return self._safe_read_text(path)

    def load(self, force: bool = False) -> None:
        with self._lock:
            if self._loaded and not force:
                return
            self.docs = []
            for category, d in self.rag_dirs.items():
                if not os.path.isdir(d):
                    continue
                for p in glob.glob(os.path.join(d, "**", "*"), recursive=True):
                    if not os.path.isfile(p):
                        continue
                    ext = os.path.splitext(p)[1].lower()
                    if ext in {".txt", ".md"}:
                        txt = self._safe_read_text(p)
                    elif ext == ".json":
                        txt = self._load_json_text(p)
                    else:
                        continue
                    txt = (txt or "").strip()
                    if len(txt) < 20:
                        continue
                    self.docs.append(
                        {
                            "category": category,
                            "path": p,
                            "text": txt[:12000],  # 限制单文档最大长度
                        }
                    )
            self._loaded = True
            logger.info(f"RAG文档加载完成: {len(self.docs)} 条")

    def retrieve(self, query: str, top_k: int = 5, max_chars: int = 1800) -> str:
        if not ENABLE_RAG:
            return "[RAG已关闭]"
        if not self._loaded:
            self.load()
        if not self.docs:
            return "[RAG为空]"

        q = (query or "").strip().lower()
        q_tokens = [t for t in re.split(r"[\s,，。;；:：|/\\-]+", q) if t]
        if not q_tokens:
            q_tokens = [q]

        scored = []
        for d in self.docs:
            text = d["text"].lower()
            score = 0
            for t in q_tokens:
                if t and t in text:
                    score += 3
            # 轻微偏置：法规库在场景生成里常更有用
            if d["category"] == "regulations":
                score += 1
            if score > 0:
                scored.append((score, d))

        if not scored:
            # 没命中时给前几个法规类兜底
            fallback = [d for d in self.docs if d["category"] == "regulations"][:top_k]
            scored_docs = fallback if fallback else self.docs[:top_k]
        else:
            scored.sort(key=lambda x: x[0], reverse=True)
            scored_docs = [d for _, d in scored[:top_k]]

        chunks = []
        total = 0
        for d in scored_docs:
            snippet = d["text"][:500].strip()
            line = f"[来源:{d['category']}] {snippet}"
            if total + len(line) > max_chars:
                break
            chunks.append(line)
            total += len(line)

        return "\n---\n".join(chunks) if chunks else "[RAG未命中]"


rag_kb = SimpleRAGKnowledgeBase(RAG_DIRS)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4) 数据结构
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class TaskUnit:
    task_id: str
    function: str
    asset_name: str
    asset_type: str
    attribute_name: str
    threat_type: str
    target: str = ""
    source: str = ""

    damage_scenario: str = ""
    threat_scenarios: list[str] = field(default_factory=list)
    attack_paths: list[str] = field(default_factory=list)
    selected_path_frameworks: list[dict] = field(default_factory=list)

    influence_parameters: dict = field(default_factory=dict)
    impact_value: int = 1
    impact_level: str = "Negligible"

    attack_details: list[dict] = field(default_factory=list)

    risk_value: int = 1
    risk_treatment: str = "接受"

    errors: list[str] = field(default_factory=list)


def _task_to_dict(t: TaskUnit) -> dict:
    return {
        "task_id": t.task_id,
        "function": t.function,
        "asset_name": t.asset_name,
        "asset_type": t.asset_type,
        "attribute_name": t.attribute_name,
        "threat_type": t.threat_type,
        "target": t.target,
        "source": t.source,
        "damage_scenario": t.damage_scenario,
        "threat_scenarios": t.threat_scenarios,
        "attack_paths": t.attack_paths,
        "selected_path_frameworks": t.selected_path_frameworks,
        "influence_parameters": t.influence_parameters,
        "impact_value": t.impact_value,
        "impact_level": t.impact_level,
        "attack_details": t.attack_details,
        "risk_value": t.risk_value,
        "risk_treatment": t.risk_treatment,
        "errors": t.errors,
    }


def _task_from_dict(d: dict) -> TaskUnit:
    return TaskUnit(
        task_id=d.get("task_id", ""),
        function=d.get("function", ""),
        asset_name=d.get("asset_name", ""),
        asset_type=d.get("asset_type", ""),
        attribute_name=d.get("attribute_name", ""),
        threat_type=d.get("threat_type", ""),
        target=d.get("target", ""),
        source=d.get("source", ""),
        damage_scenario=d.get("damage_scenario", ""),
        threat_scenarios=list(d.get("threat_scenarios", [])),
        attack_paths=list(d.get("attack_paths", [])),
        selected_path_frameworks=list(d.get("selected_path_frameworks", [])),
        influence_parameters=dict(d.get("influence_parameters", {})),
        impact_value=int(d.get("impact_value", 1)),
        impact_level=str(d.get("impact_level", "Negligible")),
        attack_details=list(d.get("attack_details", [])),
        risk_value=int(d.get("risk_value", 1)),
        risk_treatment=str(d.get("risk_treatment", "接受")),
        errors=list(d.get("errors", [])),
    )


def _tasks_to_dict(tasks: list[TaskUnit]) -> list[dict]:
    return [_task_to_dict(t) for t in tasks]


def _tasks_from_dict(data: list[dict]) -> list[TaskUnit]:
    return [_task_from_dict(d) for d in data]


@dataclass
class TopologyElementVO:
    id: str
    name: str = ""
    type: int | None = None
    color: str = ""
    source_id: str = ""
    target_id: str = ""
    is_gateway: bool = False
    ids: list[str] = field(default_factory=list)


@dataclass
class PathNodeVO:
    component_name_id: str
    component_name: str
    pre_component_name_id: str
    pre_component_name: str
    line_id: str
    color: str
    is_gateway: bool


class TopologyElementType:
    COMPONENT = 1
    GATEWAY = 2
    EXTERNAL_COMPONENT = 3
    LINE = 4


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5) 通用工具
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def safe_parse_json(text: Any) -> Any:
    if text is None:
        raise ValueError("输入为空")
    if not isinstance(text, str):
        text = str(text)

    cleaned = text.strip()
    cleaned = re.sub(r"<think.*?</think.*?>", "", cleaned, flags=re.DOTALL).strip()

    md_match = re.search(r"```(?:json|JSON)?\s*\n?(.*?)```", cleaned, re.DOTALL)
    if md_match:
        cleaned = md_match.group(1).strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    for pattern in [r"(\[[\s\S]*\])", r"(\{[\s\S]*\})"]:
        m = re.search(pattern, cleaned)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                continue

    raise ValueError(f"JSON解析失败，原文前200字: {cleaned[:200]}")


def llm_call_with_semaphore(system_prompt: str, user_prompt: str, retries: int = 2) -> str:
    with llm_semaphore:
        for attempt in range(retries + 1):
            try:
                endpoint = API_BASE.rstrip("/") + "/chat/completions"
                payload = {
                    "model": MODEL_NAME,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "extra_body": {"enable_thinking": False},
                }
                body = json.dumps(payload).encode("utf-8")
                req = urlrequest.Request(
                    endpoint,
                    data=body,
                    headers={
                        "Authorization": f"Bearer {API_KEY}",
                        "Content-Type": "application/json",
                    },
                    method="POST",
                )
                with urlrequest.urlopen(req, timeout=REQUEST_TIMEOUT) as resp:
                    data = json.loads(resp.read().decode("utf-8"))
                content = (
                    data.get("choices", [{}])[0]
                    .get("message", {})
                    .get("content", "")
                )
                return str(content) if content is not None else ""
            except (urlerror.URLError, TimeoutError) as e:
                logger.warning(f"LLM网络错误: {type(e).__name__}: {e}")
            except Exception as e:
                logger.warning(f"LLM调用异常: {type(e).__name__}: {e}")

            if attempt < retries:
                wait = 2 ** attempt
                time.sleep(wait)

    return "{}"


def cached_llm_json(
    *,
    bucket: str,
    cache_key: Any,
    system_prompt: str,
    user_prompt: str,
    default: Any,
) -> tuple[Any, bool]:
    cached = semantic_cache.get(bucket, cache_key)
    if cached is not None:
        return cached, True

    raw = llm_call_with_semaphore(system_prompt, user_prompt)
    try:
        parsed = safe_parse_json(raw)
    except Exception:
        parsed = copy.deepcopy(default)

    semantic_cache.set(bucket, cache_key, parsed)
    return parsed, False


def normalize_attack_vector(value: str) -> str:
    s = (value or "").strip().lower()
    # Network（远程/蜂窝/卫星）
    if any(k in s for k in ["network", "网络", "remote", "ota", "蜂窝", "4g", "5g", "lte", "ethernet", "以太网", "cloud", "卫星定位"]):
        return "network"
    # Adjacent（短距无线）
    if any(k in s for k in ["adjacent", "相邻", "bluetooth", "蓝牙", "ble", "wifi", "wlan", "星闪", "v2x", "射频信道"]):
        return "adjacent"
    # Local（物理端口/本地接入）
    if any(k in s for k in ["local", "本地", "usb", "tf卡", "sd卡", "ic卡", "充电口", "obd", "内部通讯", "nfc", "登录"]):
        return "local"
    # Physical（侵入式调试）
    if any(k in s for k in ["physical", "物理", "debug", "jtag", "内部调试"]):
        return "physical"
    return "local"


def calculate_impact_value(influence_parameters: dict) -> int:
    if not influence_parameters:
        return 1
    v = max(
        int(influence_parameters.get("Safety", 0)),
        int(influence_parameters.get("Finance", 0)),
        int(influence_parameters.get("Operation", 0)),
        int(influence_parameters.get("Privacy", 0)),
    )
    return max(1, min(4, v))


def calculate_impact_level(impact_value: int) -> str:
    return {1: "Negligible", 2: "Moderate", 3: "Major", 4: "Severe"}.get(impact_value, "Negligible")


def calculate_attack_potential_score(params: dict) -> int:
    return sum(int(params.get(k, 0)) for k in AP_ALLOWED_VALUES.keys())


def calculate_attack_potential_level(total_score: int) -> str:
    # 用户要求的映射
    if total_score <= 9:
        return "High"
    if total_score <= 13:
        return "Medium"
    if total_score <= 19:
        return "Low"
    return "Very Low"


def calculate_cvss_score(params: dict) -> float:
    v = CVSS_V.get(normalize_attack_vector(params.get("attack_vector", "local")), 0.55)
    c = CVSS_C.get((params.get("complexity", "high") or "high").strip().lower(), 0.44)
    p = CVSS_P.get((params.get("privileges", "none") or "none").strip().lower(), 0.85)
    u = CVSS_U.get((params.get("user_interaction", "none") or "none").strip().lower(), 0.85)
    score = 8.22 * v * c * p * u
    return round(score, 2)


def calculate_cvss_level(score: float) -> str:
    if 2.96 <= score <= 3.89:
        return "High"
    if 2.00 <= score <= 2.95:
        return "Medium"
    if 1.06 <= score <= 1.99:
        return "Low"
    if 0.12 <= score <= 1.05:
        return "Very Low"
    return "Very Low" if score < 2.96 else "High"


def calculate_attack_vector_level(vector: str) -> str:
    return ATTACK_VECTOR_TO_LEVEL.get(normalize_attack_vector(vector), "Low")


def calculate_risk_value(impact_value: int, feasibility_level: str) -> int:
    return RISK_MATRIX.get((impact_value, feasibility_level), 1)


def get_risk_treatment(risk_value: int) -> str:
    if risk_value >= 5:
        return "风险缓解"
    if risk_value == 4:
        return "风险缓解"
    if risk_value == 3:
        return "风险缓解"
    if risk_value == 2:
        return "风险接受"
    return "风险接受"


def save_json(data: Any, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5.5) Checkpoint（中间结果检查点）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CHECKPOINT_DIR = os.getenv("TARA_CHECKPOINT_DIR", os.path.join(OUTPUT_DIR, ".checkpoints"))
_CHECKPOINT_META_FILE = os.path.join(CHECKPOINT_DIR, "_meta.json")
_CHECKPOINT_FILES = {
    "stage_A_gen": "tasks_after_stageA.json",
    "stage_B_impact": "tasks_after_stageB.json",
    "stage_C_attack": "tasks_after_stageC.json",
}


def _checkpoint_path(stage: str) -> str:
    return os.path.join(CHECKPOINT_DIR, _CHECKPOINT_FILES[stage])


def _checkpoint_meta(method: str, total_tasks: int) -> dict:
    """生成当前运行的元信息快照。"""
    return {
        "method": method,
        "total_tasks": total_tasks,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "checkpoint_version": "v10.1",
    }


def save_checkpoint(tasks: list[TaskUnit], stage: str, method: str) -> str:
    """将 tasks 序列化保存到对应阶段的 checkpoint 文件。"""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    path = _checkpoint_path(stage)
    data = {
        "meta": _checkpoint_meta(method, len(tasks)),
        "tasks": _tasks_to_dict(tasks),
    }
    save_json(data, path)
    logger.info("[Checkpoint] %s 已保存 -> %s (%d 个任务)", stage, path, len(tasks))
    return path


def load_checkpoint(stage: str) -> list[TaskUnit] | None:
    """加载指定阶段的 checkpoint，不存在或损坏时返回 None。"""
    path = _checkpoint_path(stage)
    if not os.path.isfile(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tasks_data = data.get("tasks", [])
        if not tasks_data:
            logger.warning("[Checkpoint] %s 文件为空", stage)
            return None
        tasks = _tasks_from_dict(tasks_data)
        logger.info("[Checkpoint] %s 已加载 <- %s (%d 个任务)", stage, path, len(tasks))
        return tasks
    except Exception as e:
        logger.warning("[Checkpoint] %s 加载失败: %s，将重新运行该阶段", stage, e)
        return None


def resume_from_checkpoint(
    stages: list[str],
    method: str,
) -> tuple[int, list[TaskUnit] | None]:
    """按优先级检查已有的 checkpoint，找到第一个有效的 checkpoint。

    Args:
        stages: 按优先级排列的阶段名列表（从最新到最旧）。
        method: 期望的可行性方法。

    Returns:
        (stage_index, tasks) — stage_index 是需要继续的阶段的索引（0=S的开始），
        tasks 为已有 checkpoint 的 task 列表（stage_index > 0 时非 None）。
    """
    for i, stage in enumerate(stages):
        tasks = load_checkpoint(stage)
        if tasks is not None:
            # 校验元信息（方法是否一致）
            meta_path = _checkpoint_path(stage)
            try:
                with open(meta_path, "r", encoding="utf-8") as f:
                    meta = json.load(f).get("meta", {})
                saved_method = meta.get("method", "")
                if saved_method and saved_method != method:
                    logger.warning(
                        "[Checkpoint] %s 的方法(%s)与当前(%s)不一致，忽略",
                        stage, saved_method, method,
                    )
                    continue
            except Exception:
                pass

            # stage 在 stages 列表中的索引就是还需要跑的阶段范围
            remain_idx = i + 1
            if remain_idx >= len(stages):
                logger.info("[Checkpoint] 所有阶段已完成，直接使用最终结果")
                return remain_idx, tasks
            logger.info(
                "[Checkpoint] 从 %s checkpoint 恢复，将运行剩余 %d 个阶段",
                stage, len(stages) - remain_idx,
            )
            return remain_idx, tasks

    return 0, None


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6) 输入解析（DFD.py 逻辑）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_dfd_json(input_dir: str) -> list[dict]:
    type_mapping = {
        "tm.Flow": "信号",
        "tm.Process": "部件",
        "tm.Store": "数据",
        "tm.Actor": "接口",
    }

    all_results: list[dict] = []
    json_files = glob.glob(os.path.join(input_dir, "*.json"))

    for file_path in json_files:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except Exception:
                logger.warning(f"DFD解析失败，跳过: {os.path.basename(file_path)}")
                continue

        current_result = {"function": "", "assets": []}

        if "detail" in data and "diagrams" in data["detail"]:
            for diagram in data["detail"]["diagrams"]:
                if not current_result["function"] and "title" in diagram:
                    current_result["function"] = diagram["title"]

                if "diagramJson" not in diagram or "cells" not in diagram["diagramJson"]:
                    continue

                cells = diagram["diagramJson"]["cells"]
                id_to_cell = {c.get("id"): c for c in cells if c.get("id")}

                for cell in cells:
                    if cell.get("outOfScope") is True:
                        continue

                    raw_type = cell.get("type", "")
                    if raw_type == "tm.Boundary":
                        continue

                    asset_details = ""
                    finetermval_value = ""
                    for key, value in cell.items():
                        if key.startswith("propertyList") and isinstance(value, dict):
                            asset_details = value.get("assetDetails", "")
                            finetermval_value = value.get("finetermval", "")
                            break

                    if not finetermval_value:
                        continue

                    mapped_type = type_mapping.get(raw_type, raw_type)
                    target_value = ""
                    source_value = ""

                    if mapped_type == "部件":
                        target_value = finetermval_value

                    elif mapped_type == "数据":
                        cell_id = cell.get("id", "")
                        for flow_cell in cells:
                            if flow_cell.get("type") == "tm.Flow" and flow_cell.get("source", {}).get("id") == cell_id:
                                target_id = flow_cell.get("target", {}).get("id", "")
                                if target_id in id_to_cell:
                                    target_cell = id_to_cell[target_id]
                                    for k, v in target_cell.items():
                                        if k.startswith("propertyList") and isinstance(v, dict):
                                            target_value = v.get("finetermval", "")
                                            if target_value:
                                                break
                                if target_value:
                                    break

                    elif mapped_type == "信号":
                        target_id = cell.get("target", {}).get("id", "")
                        source_id = cell.get("source", {}).get("id", "")

                        if target_id in id_to_cell:
                            for k, v in id_to_cell[target_id].items():
                                if k.startswith("propertyList") and isinstance(v, dict):
                                    target_value = v.get("finetermval", "")
                                    if target_value:
                                        break

                        if source_id in id_to_cell:
                            for k, v in id_to_cell[source_id].items():
                                if k.startswith("propertyList") and isinstance(v, dict):
                                    source_value = v.get("finetermval", "")
                                    if source_value:
                                        break

                    item = {
                        "asset_type": mapped_type,
                        "asset_name": finetermval_value,
                        "assetDetails": asset_details,
                    }
                    if target_value:
                        item["target"] = target_value
                    if source_value:
                        item["source"] = source_value

                    current_result["assets"].append(item)

        if not current_result["function"]:
            current_result["function"] = os.path.splitext(os.path.basename(file_path))[0]

        if current_result["assets"]:
            all_results.append(current_result)

    logger.info(f"DFD解析完成: {len(all_results)} 个功能")
    return all_results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7) 拓扑解析
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class TopologyMapUtil:
    @staticmethod
    def list_element_vo(cells: list[dict]) -> list[TopologyElementVO]:
        result: list[TopologyElementVO] = []
        for cell in cells:
            ele = TopologyElementVO(id=cell.get("id"))
            shape = cell.get("shape", "")
            attrs = cell.get("attrs", {})

            if shape == "custom-rounded-rect":
                stroke = attrs.get("body", {}).get("stroke", "")
                if stroke == "#016AFF":
                    ele.type = TopologyElementType.GATEWAY
                    ele.is_gateway = True
                else:
                    ele.type = TopologyElementType.COMPONENT
                ele.color = stroke
                ele.name = attrs.get("text", {}).get("text", "")

            elif shape == "custom-rounded-rect-dash":
                ele.type = TopologyElementType.EXTERNAL_COMPONENT
                ele.color = attrs.get("body", {}).get("stroke", "")
                ele.name = attrs.get("text", {}).get("text", "")

            elif shape == "edge":
                ele.type = TopologyElementType.LINE
                ele.color = attrs.get("line", {}).get("stroke", "")
                ele.source_id = cell.get("source", {}).get("cell")
                ele.target_id = cell.get("target", {}).get("cell")

            result.append(ele)
        return result

    @staticmethod
    def get_distinct_components(element_vos: list[TopologyElementVO]) -> list[TopologyElementVO]:
        components = [
            e for e in element_vos
            if e.type in [
                TopologyElementType.COMPONENT,
                TopologyElementType.GATEWAY,
                TopologyElementType.EXTERNAL_COMPONENT,
            ] and e.name
        ]

        name_map: dict[str, list[TopologyElementVO]] = defaultdict(list)
        for c in components:
            name_map[c.name].append(c)

        distinct: list[TopologyElementVO] = []
        for _, elements in name_map.items():
            first = elements[0]
            first.ids = [e.id for e in elements]
            distinct.append(first)
        return distinct

    @staticmethod
    def generate_full_path(
        external_interfaces: list[dict],
        element_vos: list[TopologyElementVO],
        protocol_legends: list[dict],
        distinct_components: list[TopologyElementVO],
    ) -> list[list[PathNodeVO]]:
        many_to_one: dict[str, str] = {}
        for comp in distinct_components:
            for old_id in comp.ids:
                many_to_one[old_id] = comp.id

        valid_colors = [p.get("color") for p in protocol_legends if p.get("color")]
        lines = [e for e in element_vos if e.type == TopologyElementType.LINE and e.color in valid_colors]

        for line in lines:
            line.source_id = many_to_one.get(line.source_id, line.source_id)
            line.target_id = many_to_one.get(line.target_id, line.target_id)

        ext_names = [ext["related_component"] for ext in external_interfaces]
        ext_ids = [c.id for c in distinct_components if c.name in ext_names]

        com_to_lines: dict[str, set] = defaultdict(set)
        line_to_coms: dict[str, set] = defaultdict(set)

        for line in lines:
            if line.source_id:
                com_to_lines[line.source_id].add(line.color)
                line_to_coms[line.color].add(line.source_id)
            if line.target_id:
                com_to_lines[line.target_id].add(line.color)
                line_to_coms[line.color].add(line.target_id)

        starts = {k: v for k, v in com_to_lines.items() if k in ext_ids}
        com_dic = {c.id: c for c in distinct_components}

        return TopologyMapUtil.get_path_by_bfs(starts, com_to_lines, line_to_coms, com_dic)

    @staticmethod
    def get_path_by_bfs(
        starts: dict[str, set],
        com_to_lines: dict[str, set],
        line_to_coms: dict[str, set],
        com_dic: dict[str, TopologyElementVO],
    ) -> list[list[PathNodeVO]]:
        all_paths: list[list[PathNodeVO]] = []

        for start_id in starts.keys():
            queue = deque([(start_id, set(), [])])
            while queue:
                current_com_id, visited, path = queue.popleft()

                direct_points = set()
                line_ids = com_to_lines.get(current_com_id, set())
                for l_id in line_ids:
                    direct_points.update(line_to_coms.get(l_id, set()))

                for l_id in line_ids:
                    if l_id in visited:
                        continue

                    for next_com in line_to_coms.get(l_id, set()):
                        if next_com not in com_dic or next_com == current_com_id or next_com in visited:
                            continue

                        node = PathNodeVO(
                            component_name_id=next_com,
                            component_name=com_dic[next_com].name,
                            pre_component_name_id=current_com_id,
                            pre_component_name=com_dic[current_com_id].name,
                            line_id=l_id,
                            color=l_id,
                            is_gateway=com_dic[next_com].is_gateway,
                        )

                        new_path = copy.deepcopy(path)
                        new_path.append(node)
                        all_paths.append(new_path)

                        new_visited = set(visited)
                        new_visited.update(line_ids)
                        new_visited.update(direct_points)
                        queue.append((next_com, new_visited, new_path))

        return all_paths


def _determine_attack_vector(ext_interface: str, color_name_map: dict, path_nodes: list[PathNodeVO]) -> str:
    interface_lower = (ext_interface or "").lower()

    # Network
    if any(k in interface_lower for k in ["4g", "5g", "lte", "蜂窝", "ota", "cloud", "远程", "ethernet", "以太网", "卫星定位"]):
        return "network"
    # Adjacent
    if any(k in interface_lower for k in ["wifi", "wlan", "蓝牙", "bluetooth", "ble", "星闪", "v2x", "射频信道"]):
        return "adjacent"
    # Local
    if any(k in interface_lower for k in ["usb", "tf卡", "sd卡", "ic卡", "充电口", "obd", "内部通讯", "nfc", "local", "本地登录"]):
        return "local"
    # Physical
    if any(k in interface_lower for k in ["jtag", "debug", "内部调试", "uart", "spi", "i2c", "物理"]):
        return "physical"

    if path_nodes:
        protocol_name = color_name_map.get(path_nodes[0].color, "").lower()
        if any(k in protocol_name for k in ["wifi", "wlan", "4g", "5g", "ethernet", "以太网"]):
            return "network"
        if any(k in protocol_name for k in ["蓝牙", "ble", "nfc"]):
            return "adjacent"
        if any(k in protocol_name for k in ["usb", "obd"]):
            return "local"

    return "local"


def _norm_text(v: Any) -> str:
    if v is None:
        return ""
    s = str(v).replace("\n", " ").replace("\r", " ").replace("\u3000", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def _is_checked(v: Any) -> bool:
    s = _norm_text(v).lower()
    if not s:
        return False
    if any(mark in s for mark in ["√", "✓", "☑", "✅"]):
        return True
    return s in {"1", "y", "yes", "true", "是", "有", "开", "通过"}


def _excel_col_to_idx(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha()).upper()
    idx = 0
    for ch in letters:
        idx = idx * 26 + (ord(ch) - ord("A") + 1)
    return max(0, idx - 1)


def _parse_xlsx_rows_stdlib(excel_path: str) -> list[dict]:
    """
    无 pandas/openpyxl 时，使用标准库解析 xlsx。
    返回：[{row_num:int, cells:{col_idx: value}}...]
    """
    ns = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rel_ns = {"r": "http://schemas.openxmlformats.org/package/2006/relationships"}

    with zipfile.ZipFile(excel_path, "r") as zf:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall(".//x:si", ns):
                text = "".join((t.text or "") for t in si.findall(".//x:t", ns))
                shared_strings.append(text)

        # 找第一个工作表文件路径
        sheet_target = "worksheets/sheet1.xml"
        try:
            wb_root = ET.fromstring(zf.read("xl/workbook.xml"))
            sheets = wb_root.findall(".//x:sheets/x:sheet", ns)
            first_sheet = sheets[0] if sheets else None
            rid = first_sheet.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id") if first_sheet is not None else None
            if rid and "xl/_rels/workbook.xml.rels" in zf.namelist():
                rel_root = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
                for rel in rel_root.findall(".//r:Relationship", rel_ns):
                    if rel.attrib.get("Id") == rid:
                        sheet_target = rel.attrib.get("Target", sheet_target)
                        break
        except Exception:
            pass

        sheet_path = "xl/" + sheet_target.lstrip("/")
        if sheet_path not in zf.namelist():
            # 兜底
            candidates = [n for n in zf.namelist() if n.startswith("xl/worksheets/sheet") and n.endswith(".xml")]
            if not candidates:
                return []
            sheet_path = sorted(candidates)[0]

        sheet_root = ET.fromstring(zf.read(sheet_path))
        out_rows: list[dict] = []

        for row in sheet_root.findall(".//x:sheetData/x:row", ns):
            row_num = int(row.attrib.get("r", "0") or 0)
            cells: dict[int, str] = {}
            for c in row.findall("x:c", ns):
                ref = c.attrib.get("r", "")
                col_idx = _excel_col_to_idx(ref) if ref else 0
                ctype = c.attrib.get("t", "")
                value = ""
                if ctype == "s":
                    v = c.find("x:v", ns)
                    if v is not None and v.text and v.text.isdigit():
                        ss_idx = int(v.text)
                        if 0 <= ss_idx < len(shared_strings):
                            value = shared_strings[ss_idx]
                elif ctype == "inlineStr":
                    is_node = c.find("x:is", ns)
                    if is_node is not None:
                        value = "".join((t.text or "") for t in is_node.findall(".//x:t", ns))
                else:
                    v = c.find("x:v", ns)
                    value = v.text if (v is not None and v.text is not None) else ""
                cells[col_idx] = _norm_text(value)
            out_rows.append({"row_num": row_num, "cells": cells})

        return out_rows


def _load_external_interfaces(excel_path: str) -> list[dict]:
    """读取外部接口清单：pandas -> openpyxl -> 标准库 xlsx 解析。"""
    # 1) pandas 路径
    if pd is not None:
        try:
            df = pd.read_excel(excel_path, header=1)
            df.columns = [_norm_text(c) for c in df.columns]

            direct_info_col = next((c for c in df.columns if "外部接口" in c and "信息" in c), None)
            direct_comp_col = next((c for c in df.columns if "关联部件" in c), None)
            if direct_info_col and direct_comp_col:
                direct_items = []
                for _, row in df.iterrows():
                    ext = _norm_text(row.get(direct_info_col, ""))
                    comp = _norm_text(row.get(direct_comp_col, ""))
                    if ext and comp and comp.lower() != "nan":
                        direct_items.append(
                            {
                                "external_interface": ext,
                                "related_component": comp,
                                "all_interfaces": [ext],
                            }
                        )
                if direct_items:
                    return direct_items

            comp_col = next((c for c in df.columns if "零部件" in c and "名称" in c), None)
            if comp_col is None and len(df.columns) >= 2:
                comp_col = df.columns[1]
            if comp_col is None:
                comp_col = df.columns[0] if len(df.columns) else None

            if comp_col is None:
                return []

            interface_columns = [c for c in df.columns if c != comp_col and _norm_text(c)]
            # 对齐原逻辑：倾向跳过前两个说明列
            if len(df.columns) > 2:
                interface_columns = [c for c in df.columns[2:] if c != comp_col and _norm_text(c)]

            external_interfaces: list[dict] = []
            for _, row in df.iterrows():
                comp = _norm_text(row.get(comp_col, ""))
                if not comp or comp.lower() == "nan":
                    continue
                active = [col for col in interface_columns if _is_checked(row.get(col, ""))]
                if active:
                    external_interfaces.append(
                        {
                            "external_interface": active[0],
                            "related_component": comp,
                            "all_interfaces": active,
                        }
                    )
            if external_interfaces:
                return external_interfaces
        except Exception as e:
            logger.warning(f"pandas读取外部接口失败，尝试其他方式: {e}")

    # 2) openpyxl 路径
    try:
        from openpyxl import load_workbook

        wb = load_workbook(excel_path, data_only=True)
        ws = wb.active

        # 先尝试“外部接口信息 + 外部接口关联部件”直接映射格式
        direct_header_row = None
        info_idx = None
        comp_idx_direct = None
        for r in range(1, min(ws.max_row, 10) + 1):
            vals = [_norm_text(c.value) for c in ws[r]]
            for i, v in enumerate(vals):
                if "外部接口" in v and "信息" in v:
                    info_idx = i
                if "关联部件" in v:
                    comp_idx_direct = i
            if info_idx is not None and comp_idx_direct is not None:
                direct_header_row = r
                break

        if direct_header_row is not None:
            direct_items = []
            for row in ws.iter_rows(min_row=direct_header_row + 1, values_only=True):
                vals = [_norm_text(v) for v in row]
                ext = vals[info_idx] if info_idx < len(vals) else ""
                comp = vals[comp_idx_direct] if comp_idx_direct < len(vals) else ""
                if ext and comp:
                    direct_items.append(
                        {
                            "external_interface": ext,
                            "related_component": comp,
                            "all_interfaces": [ext],
                        }
                    )
            if direct_items:
                return direct_items

        # 优先寻找包含“零部件名称”的表头行
        header_row_idx = None
        comp_idx = None
        max_scan = min(ws.max_row, 20)
        for r in range(1, max_scan + 1):
            vals = [_norm_text(c.value) for c in ws[r]]
            for i, v in enumerate(vals):
                if "零部件" in v and "名称" in v:
                    header_row_idx = r
                    comp_idx = i
                    break
            if header_row_idx is not None:
                break

        if header_row_idx is None:
            header_row_idx = 2
            vals = [_norm_text(c.value) for c in ws[header_row_idx]]
            comp_idx = vals.index("零部件名称") if "零部件名称" in vals else 1

        header_vals = [_norm_text(c.value) for c in ws[header_row_idx]]
        interface_cols = [(i, h) for i, h in enumerate(header_vals) if i > comp_idx and h]

        external_interfaces: list[dict] = []
        for row in ws.iter_rows(min_row=header_row_idx + 1, values_only=True):
            row_vals = [_norm_text(v) for v in row]
            if comp_idx >= len(row_vals):
                continue
            comp = row_vals[comp_idx]
            if not comp:
                continue
            active = [h for i, h in interface_cols if i < len(row_vals) and _is_checked(row_vals[i])]
            if active:
                external_interfaces.append(
                    {
                        "external_interface": active[0],
                        "related_component": comp,
                        "all_interfaces": active,
                    }
                )
        if external_interfaces:
            return external_interfaces
    except Exception:
        pass

    # 3) 标准库兜底（无 pandas / openpyxl）
    try:
        rows = _parse_xlsx_rows_stdlib(excel_path)
    except Exception as e:
        logger.warning(f"标准库解析Excel失败: {e}")
        return []

    if not rows:
        return []

    # A) “外部接口信息 + 外部接口关联部件”直接映射格式
    direct_header = None
    info_idx = None
    comp_idx_direct = None
    for r in rows[:20]:
        for idx, v in r["cells"].items():
            if "外部接口" in v and "信息" in v:
                info_idx = idx
            if "关联部件" in v:
                comp_idx_direct = idx
        if info_idx is not None and comp_idx_direct is not None:
            direct_header = r["row_num"]
            break

    if direct_header is not None:
        direct_items = []
        for r in rows:
            if r["row_num"] <= direct_header:
                continue
            cells = r["cells"]
            ext = _norm_text(cells.get(info_idx, ""))
            comp = _norm_text(cells.get(comp_idx_direct, ""))
            if ext and comp:
                direct_items.append(
                    {
                        "external_interface": ext,
                        "related_component": comp,
                        "all_interfaces": [ext],
                    }
                )
        if direct_items:
            return direct_items

    # B) 勾选矩阵格式
    header_row_num = None
    comp_idx = None
    for r in rows[:30]:
        for idx, v in r["cells"].items():
            if "零部件" in v and "名称" in v:
                header_row_num = r["row_num"]
                comp_idx = idx
                break
        if header_row_num is not None:
            break

    if header_row_num is None:
        # 尝试第2行兜底
        row2 = next((r for r in rows if r["row_num"] == 2), None)
        if row2:
            header_row_num = 2
            for idx, v in row2["cells"].items():
                if "零部件" in v and "名称" in v:
                    comp_idx = idx
                    break
            if comp_idx is None:
                comp_idx = 1
        else:
            return []

    header_row = next((r for r in rows if r["row_num"] == header_row_num), None)
    if not header_row:
        return []

    interface_cols = [(idx, name) for idx, name in sorted(header_row["cells"].items()) if idx > comp_idx and name]

    external_interfaces: list[dict] = []
    for r in rows:
        if r["row_num"] <= header_row_num:
            continue
        cells = r["cells"]
        comp = _norm_text(cells.get(comp_idx, ""))
        if not comp:
            continue
        active = [name for idx, name in interface_cols if _is_checked(cells.get(idx, ""))]
        if active:
            external_interfaces.append(
                {
                    "external_interface": active[0],
                    "related_component": comp,
                    "all_interfaces": active,
                }
            )

    if not external_interfaces:
        logger.warning("外部接口Excel已读取，但未识别到勾选项；请检查表头与勾选符号。")
    return external_interfaces


def parse_topology_and_generate_frameworks(
    topology_file: str,
    external_interface_excel: str,
    asset_results: list[dict] | None = None,
) -> list[dict]:
    with open(topology_file, "r", encoding="utf-8") as f:
        topo_data = json.load(f).get("data", {})

    cells = topo_data.get("cells", [])
    protocol_legends = topo_data.get("lineData", [])
    color_name_map = {p.get("color"): p.get("name") for p in protocol_legends}

    if not os.path.exists(external_interface_excel):
        logger.warning(f"外部接口文件不存在: {external_interface_excel}")
        return []
    external_interfaces = _load_external_interfaces(external_interface_excel)
    if not external_interfaces:
        logger.warning("外部接口清单为空，攻击路径框架将为空。")
        return []

    # 从DFD解析结果中提取所有 target 名称，用于后续框架预匹配
    dfd_targets: set[str] = set()
    if asset_results:
        for func_data in asset_results:
            for asset in func_data.get("assets", []):
                t = (asset.get("target") or "").strip()
                if t:
                    dfd_targets.add(t)

    element_vos = TopologyMapUtil.list_element_vo(cells)
    distinct_components = TopologyMapUtil.get_distinct_components(element_vos)
    path_node_vos = TopologyMapUtil.generate_full_path(
        external_interfaces, element_vos, protocol_legends, distinct_components
    )

    source_target_map: dict[str, list[list[PathNodeVO]]] = defaultdict(list)
    for p in path_node_vos:
        source_id = p[0].pre_component_name_id
        target_id = p[-1].component_name_id
        source_target_map[f"{source_id}_{target_id}"].append(p)

    comp_to_interfaces: dict[str, list[str]] = defaultdict(list)
    for ext in external_interfaces:
        comp_to_interfaces[ext["related_component"]].append(ext["external_interface"])

    frameworks: list[dict] = []

    for _, paths in source_target_map.items():
        min_len = min(len(p) for p in paths) if paths else 0
        min_paths = [p for p in paths if len(p) == min_len]

        has_gateway = any(any(n.is_gateway for n in p) for p in min_paths)
        filtered = [p for p in min_paths if any(n.is_gateway for n in p)] if has_gateway else min_paths

        for p in filtered:
            start_comp_name = p[0].pre_component_name
            ext_list = comp_to_interfaces.get(start_comp_name, ["未知接口"])

            for ext in ext_list:
                desc = [f"[{ext}] -> {start_comp_name}"]
                for node in p:
                    proto = color_name_map.get(node.color, "未知协议")
                    desc.append(f" -> [{proto}] -> {node.component_name}")
                path_desc = "".join(desc)

                path_nodes = [start_comp_name] + [n.component_name for n in p]
                last_node = path_nodes[-1] if path_nodes else ""
                second_to_last_node = path_nodes[-2] if len(path_nodes) >= 2 else ""
                attack_vector = _determine_attack_vector(ext, color_name_map, p)

                # 预计算此框架匹配哪些 DFD target（last_node == target）
                matched_targets = [t for t in dfd_targets if t == last_node]

                frameworks.append({
                    "path_id": hashlib.sha1(path_desc.encode("utf-8")).hexdigest()[:16],
                    "path_description": path_desc,
                    "path_nodes": path_nodes,
                    "last_node": last_node,
                    "second_to_last_node": second_to_last_node,
                    "start_component": start_comp_name,
                    "external_interface": ext,
                    "components": [n.component_name for n in p],
                    "attack_vector": attack_vector,
                    "matched_targets": matched_targets,
                })

    unique = []
    seen = set()
    for fw in frameworks:
        if fw["path_description"] in seen:
            continue
        seen.add(fw["path_description"])
        unique.append(fw)

    logger.info(f"拓扑攻击路径框架生成完成: {len(unique)} 条")
    return unique


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8) Prompt（生成+内嵌自评）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DS_SYSTEM_PROMPT = """你是汽车网络安全专家，负责TARA损害场景建模。损害场景是”涉及车辆或车辆功能且影响道路使用者的不良后果”。
在输出前必须进行自检，仅输出通过自检的结果。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类别、安全属性、威胁类型，以及相关参考知识（RAG）。

## 生成规则

损害场景必须同时包含以下三个要素（缺一不可），且必须结合车辆实际域（如ADAS、车身控制、动力系统等）：

   1. 功能如何因为资产的安全属性被破坏而产生不良后果
   2. 对道路使用者的损害说明（必须明确指出对车主、驾驶员、乘员、行人或其他道路使用者的具体损害）
   3. 必须明确提到被破坏的资产名称

## 参考示例：

- “存储在信息娱乐系统中的个人信息(客户个人偏好)失去机密性,在未经客户同意的情况下,披露客户个人信息。”
- “车辆夜间行驶时，前照灯控制功能因’前照灯请求信号’完整性被破坏而意外关闭，导致驾驶员在无照明条件下高速行驶，与静止障碍物发生正面碰撞的风险。”

【自检3条件】
1) 必须同时体现 function、asset_name、attribute_name 与 threat_type 的逻辑关联。
2) 必须明确具体危害后果（道路使用者/车辆运营/隐私或财务等）。
3) 描述必须具体且可验证，避免空泛结论。

仅输出JSON：
{"damage_scenario": "..."}
"""

TS_SYSTEM_PROMPT = """你是汽车网络安全专家，负责TARA威胁场景建模。威胁场景是”为实现损害场景，资产的信息安全属性遭到破坏的潜在原因”。
在输出前必须进行自检，仅输出通过自检的结果。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类型、安全属性、威胁类型、损害场景，以及相关参考知识（RAG）。

## 生成规则

1. 威胁场景必须描述：
   - 明确指出被攻击的具体资产名称
   - 该资产被破坏的安全属性
   - 该安全属性被破坏的原因/攻击意图
2. 威胁场景必须能直接导致给定的损害场景，描述中必须体现”破坏该安全属性 -> 引发损害场景中的不良后果”逻辑链条
3. 损害场景与资产名称、攻击者、攻击方法、攻击工具及攻击面之间的依赖关系应能被威胁场景包含或与威胁场景关联

## 参考示例：

- “攻击者从制动ECU方面对CAN消息进行欺骗，导致CAN消息的完整性缺失,从而导致制动功能的完整性缺失。”
- “伪装信号导致发送至电源开关控制器的’灯光请求’信号的数据通信完整性丢失,可能造成前照灯意外关闭”

【自检3条件】
1) 必须明确攻击对象与受损安全属性（attribute_name）。
2) 必须说明威胁实现方式，并与threat_type保持一致。
3) 必须与damage_scenario保持因果一致。

仅输出JSON：
{"threat_scenarios": ["..."]}
"""

AP_SYSTEM_PROMPT = """你是汽车攻击路径专家。
在输出前必须进行自检，仅输出通过自检的结果。攻击路径是”为实现威胁场景的一组蓄意活动”。必须采用攻击树分析，从外部攻击面开始，沿车辆拓扑逻辑连贯地推进到目标资产，最终实现给定的威胁场景。

## 输入说明

以下信息将在 user_prompt 中提供：功能名称、资产名称、资产类型、安全属性、威胁场景列表、攻击路径框架（path_frameworks），以及相关参考知识（RAG）。

## 生成规则

1. 每条攻击路径必须是完整步骤链，沿车辆拓扑从外部攻击入口到目标资产，每条路径必须能直接实现给定的威胁场景，每条路径的技术手段必须针对资产类型的特点设计
2. 必须引用真实攻击手段（如CAN报文注入、固件逆向、中间人攻击、重放攻击、伪造身份、DoS泛洪、OBD物理注入、蓝牙/蜂窝/USB/OTA入侵等）。必须结合车辆实际攻击面（外部接口：蜂窝网络、Bluetooth、Wi-Fi、OBD、USB、TF卡、射频信道等）
3. 同一条威胁场景的多条攻击路径不要重复，避免冗余
4. 必须基于提供的路径框架（path_frameworks），从中选取一条或多条框架，将具体的攻击手段填入框架中的每个节点，构成完整的攻击链。

## 参考示例：

- “1.利用蜂窝接口损害远程通信ECU;2.利用远程通信ECU的CAN通信损害网关ECU;3.网关ECU转发恶意制动请求信号。”
- “1.攻击者通过蜂窝网络接口入侵了导航ECU;2.被入侵的导航ECU发送恶意控制信号;3.网关ECU转发恶意控制信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”
- “1.攻击者可以本地访问OBD连接器;2.攻击者通过OBD连接器发送恶意控制信号;3.网关ECU转发恶意信号至电源开关执行器;4.恶意信号伪装成灯光请求(关灯)。”

【自检3条件】
1) 攻击步骤必须与threat_scenarios一致。
2) 必须基于提供的路径框架，步骤链逻辑闭合。
3) 每条路径都要可执行、具体。

仅输出JSON：
{"attack_path": ["步骤链1", "步骤链2"]}
"""

IMPACT_SYSTEM_PROMPT = """你是ISO/SAE 21434影响评级专家。
根据损害场景输出四维评分（0~4）：Safety/Finance/Operation/Privacy。
仅输出JSON：
{"influence_parameters": {"Safety":0,"Finance":0,"Operation":0,"Privacy":0}}
"""

AP_FEASIBILITY_SYSTEM_PROMPT = """你是ISO/SAE 21434攻击潜力评估专家。
请输出五维分值，且必须从给定离散值中选择：
- Exposure_time: 0/1/4/17/19
- Professional_experience: 0/3/6/9
- Required_information: 0/3/7/11
- Opportunity_window: 0/1/4/10
- Required_equipment: 0/4/7/9
仅输出JSON：
{"attack_parameters": {"Exposure_time":0,"Professional_experience":0,"Required_information":0,"Opportunity_window":0,"Required_equipment":0}}
"""

CVSS_FEASIBILITY_SYSTEM_PROMPT = """你是CVSS可利用性评估专家。
请根据攻击路径分析并输出以下3个CVSS可利用性参数（攻击向量已由外部接口映射确定，无需输出）：
- complexity: low/high
- privileges: none/low/high
- user_interaction: none/required

仅输出JSON：
{"attack_parameters": {"complexity":"high","privileges":"none","user_interaction":"none"}}
"""


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 9) 生成与评估核心函数
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_damage_scenario(task: TaskUnit, rag_context: str = "") -> str:
    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}

参考知识（RAG）:
{rag_context}

请输出 damage_scenario。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_type": task.threat_type,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"damage_scenario task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="damage_scenario",
            cache_key=cache_key,
            system_prompt=DS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"damage_scenario": ""},
        )

    if cache_hit:
        logger.debug(f"[cache] damage_scenario task={task.task_id}")

    return str(parsed.get("damage_scenario", "")).strip()


def generate_threat_scenarios(task: TaskUnit, rag_context: str = "") -> list[str]:
    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}
- damage_scenario: {task.damage_scenario}

参考知识（RAG）:
{rag_context}

请输出 threat_scenarios。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_type": task.threat_type,
        "damage_scenario": task.damage_scenario,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"threat_scenarios task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="threat_scenarios",
            cache_key=cache_key,
            system_prompt=TS_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"threat_scenarios": []},
        )

    if cache_hit:
        logger.debug(f"[cache] threat_scenarios task={task.task_id}")

    scenarios = parsed.get("threat_scenarios", parsed.get("threat_scenario", []))
    if isinstance(scenarios, str):
        return [scenarios]
    if isinstance(scenarios, list):
        seen = set()
        deduped = []
        for s in scenarios:
            s = str(s).strip()
            if s and s not in seen:
                seen.add(s)
                deduped.append(s)
        return deduped
    return []


def generate_attack_paths(task: TaskUnit, frameworks: list[dict], rag_context: str = "") -> list[str]:
    fw_brief = [
        {
            "path_id": fw["path_id"],
            "path_description": fw["path_description"],
            "last_node": fw["last_node"],
            "attack_vector": fw["attack_vector"],
            "path_nodes": fw["path_nodes"],
        }
        for fw in frameworks[:8]
    ]

    user_prompt = f"""输入：
- function: {task.function}
- asset_type: {task.asset_type}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- path_frameworks: {json.dumps(fw_brief, ensure_ascii=False)}

参考知识（RAG）:
{rag_context}

请输出 attack_path。"""

    cache_key = {
        "function": task.function,
        "asset_type": task.asset_type,
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "threat_scenarios": task.threat_scenarios,
        "path_frameworks": fw_brief,
        "rag_context_hash": hashlib.md5((rag_context or "").encode("utf-8")).hexdigest(),
    }

    with Timer(f"attack_path task={task.task_id}"):
        parsed, cache_hit = cached_llm_json(
            bucket="attack_path",
            cache_key=cache_key,
            system_prompt=AP_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_path": []},
        )

    if cache_hit:
        logger.debug(f"[cache] attack_path task={task.task_id}")

    paths = parsed.get("attack_path", parsed.get("attack_paths", []))
    if isinstance(paths, str):
        return [paths]
    if isinstance(paths, list):
        return [str(p).strip() for p in paths if str(p).strip()]
    return []


def generate_influence_parameters(task: TaskUnit) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- asset_type: {task.asset_type}
- attribute_name: {task.attribute_name}
- threat_type: {task.threat_type}
- damage_scenario: {task.damage_scenario}

请输出 influence_parameters。"""

    cache_key = {
        "asset_name": task.asset_name,
        "asset_type": task.asset_type,
        "attribute_name": task.attribute_name,
        "damage_scenario": task.damage_scenario,
    }

    with Timer(f"influence task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="influence_parameters",
            cache_key=cache_key,
            system_prompt=IMPACT_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"influence_parameters": {"Safety": 0, "Finance": 0, "Operation": 0, "Privacy": 0}},
        )

    params = parsed.get("influence_parameters", {})
    return {
        "Safety": int(params.get("Safety", 0)),
        "Finance": int(params.get("Finance", 0)),
        "Operation": int(params.get("Operation", 0)),
        "Privacy": int(params.get("Privacy", 0)),
    }


def generate_attack_parameters_attack_potential(task: TaskUnit, attack_path: str) -> dict:
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- attribute_name: {task.attribute_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- attack_path: {attack_path}

请输出 attack_parameters。"""

    cache_key = {
        "asset_name": task.asset_name,
        "attribute_name": task.attribute_name,
        "attack_path": attack_path,
    }

    with Timer(f"feasibility(ap) task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="attack_parameters_ap",
            cache_key=cache_key,
            system_prompt=AP_FEASIBILITY_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_parameters": {k: 0 for k in AP_ALLOWED_VALUES}},
        )

    params = parsed.get("attack_parameters", {})
    fixed = {}
    for k, allowed in AP_ALLOWED_VALUES.items():
        v = int(params.get(k, 0))
        # 强制吸附到最近合法值
        fixed[k] = min(allowed, key=lambda x: abs(x - v))
    return fixed


def generate_attack_parameters_cvss(task: TaskUnit, attack_path: str, attack_vector: str = "local") -> dict:
    """生成CVSS参数。攻击向量从外部接口映射预先确定（不调用大模型）。"""
    user_prompt = f"""输入：
- function: {task.function}
- asset_name: {task.asset_name}
- threat_scenarios: {json.dumps(task.threat_scenarios, ensure_ascii=False)}
- attack_path: {attack_path}
- 攻击向量（已从外部接口映射确定）: {attack_vector}

请分析上述攻击路径，输出剩余3个CVSS可利用性参数。"""

    cache_key = {
        "asset_name": task.asset_name,
        "attack_path": attack_path,
    }

    with Timer(f"feasibility(cvss) task={task.task_id}"):
        parsed, _ = cached_llm_json(
            bucket="attack_parameters_cvss",
            cache_key=cache_key,
            system_prompt=CVSS_FEASIBILITY_SYSTEM_PROMPT,
            user_prompt=user_prompt,
            default={"attack_parameters": {"complexity": "high", "privileges": "none", "user_interaction": "none"}},
        )

    p = parsed.get("attack_parameters", {})
    return {
        "attack_vector": attack_vector,
        "complexity": str(p.get("complexity", "high")).strip().lower(),
        "privileges": str(p.get("privileges", "none")).strip().lower(),
        "user_interaction": str(p.get("user_interaction", "none")).strip().lower(),
    }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 10) 任务构建 + 路径筛选
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_task_units(asset_results: list[dict]) -> list[TaskUnit]:
    tasks: list[TaskUnit] = []
    idx = 0
    for func_data in asset_results:
        function_name = func_data.get("function", "")
        for asset in func_data.get("assets", []):
            asset_type = asset.get("asset_type", "")
            asset_name = asset.get("asset_name", "")
            target = asset.get("target", "")
            source = asset.get("source", "")

            attrs = SECURITY_ATTRIBUTES_MAP.get(asset_type, ["完整性"])
            for attr in attrs:
                threat_type = ATTRIBUTE_TO_THREAT.get(attr, "未知")
                tasks.append(
                    TaskUnit(
                        task_id=f"{idx:05d}",
                        function=function_name,
                        asset_name=asset_name,
                        asset_type=asset_type,
                        attribute_name=attr,
                        threat_type=threat_type,
                        target=target,
                        source=source,
                    )
                )
                idx += 1

    logger.info(f"任务单元构建完成: {len(tasks)}")
    return tasks


def filter_path_frameworks(task: TaskUnit, all_frameworks: list[dict]) -> list[dict]:
    target = (task.target or "").strip()
    source = (task.source or "").strip()
    is_signal = task.asset_type == "信号"

    selected: list[dict] = []

    for fw in all_frameworks:
        # 优先使用框架预计算的 matched_targets（与 DFD 解析结果联动）
        matched = fw.get("matched_targets", [])
        target_ok = bool(matched) and target in matched
        if not target_ok:
            # 回退到动态匹配（无 DFD 数据时的兼容）
            last_node = (fw.get("last_node") or "").strip()
            target_ok = bool(target) and last_node == target

        if not target_ok:
            continue

        # 信号类额外检查：source 必须等于路径倒数第二个节点
        if is_signal and source:
            second_to_last = fw.get("second_to_last_node", "")
            if source != second_to_last:
                continue

        selected.append(fw)

    # 回退策略：如果严格筛选为空，放宽到 target 命中路径字符串
    if not selected and target:
        selected = [fw for fw in all_frameworks if target in fw.get("path_description", "")]

    # 仍为空时，再按资产名尝试
    if not selected and task.asset_name:
        selected = [fw for fw in all_frameworks if task.asset_name in fw.get("path_description", "")]

    if len(selected) > 8:
        selected = selected[:8]

    return selected


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 11) 并发流水线
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def process_task_generation(
    task: TaskUnit,
    all_frameworks: list[dict],
    rag_contexts: Optional[dict[str, str]] = None,
) -> TaskUnit:
    """阶段A：每个资产独立并发（损害->威胁->攻击路径生成）"""
    rag_contexts = rag_contexts or {}
    rag_ds = rag_contexts.get("damage", "")
    rag_ts = rag_contexts.get("threat", "")
    rag_ap = rag_contexts.get("attack_path", "")
    try:
        task.damage_scenario = generate_damage_scenario(task, rag_ds)
        if not task.damage_scenario:
            task.errors.append("damage_scenario为空")

        task.threat_scenarios = generate_threat_scenarios(task, rag_ts)
        if not task.threat_scenarios:
            task.errors.append("threat_scenarios为空")

        selected = filter_path_frameworks(task, all_frameworks)
        task.selected_path_frameworks = selected

        task.attack_paths = generate_attack_paths(task, selected, rag_ap)
        if not task.attack_paths:
            task.errors.append("attack_path为空")

    except Exception as e:
        task.errors.append(f"生成阶段异常: {e}")

    return task


def process_task_gen_impact(
    task: TaskUnit,
    all_frameworks: list[dict],
    rag_contexts: dict[str, str],
) -> TaskUnit:
    """单资产 A+B 阶段：损害场景→威胁场景→攻击路径(缓存)→影响评级（不含可行性评估）。"""
    try:
        # ── Stage A: 场景生成 ──────────────────────────
        task.damage_scenario = generate_damage_scenario(task, rag_contexts.get("damage", ""))
        if not task.damage_scenario:
            task.errors.append("damage_scenario为空")

        task.threat_scenarios = generate_threat_scenarios(task, rag_contexts.get("threat", ""))
        if not task.threat_scenarios:
            task.errors.append("threat_scenarios为空")

        selected = filter_path_frameworks(task, all_frameworks)
        task.selected_path_frameworks = selected

        # 攻击路径缓存：同一 asset_name 的路径相同，跨功能复用
        cached_paths = _get_attack_path_cache(task.asset_name)
        if cached_paths is not None:
            task.attack_paths = cached_paths
            logger.debug(f"[path_cache] task={task.task_id} 复用 asset={task.asset_name} 的攻击路径")
        else:
            task.attack_paths = generate_attack_paths(task, selected, rag_contexts.get("attack_path", ""))
            if task.attack_paths:
                _set_attack_path_cache(task.asset_name, task.attack_paths)

        if not task.attack_paths:
            task.errors.append("attack_path为空")

        # ── Stage B: 影响评级 ──────────────────────────
        params = generate_influence_parameters(task)
        task.influence_parameters = params
        task.impact_value = calculate_impact_value(params)
        task.impact_level = calculate_impact_level(task.impact_value)

    except Exception as e:
        task.errors.append(f"A+B阶段异常: {e}")

    return task


def process_in_batches(items: list[Any], batch_size: int, worker_fn, stage_name: str) -> list[Any]:
    """连续批处理：上一批结束后再提交下一批。"""
    outputs: list[Any] = [None] * len(items)
    total = len(items)
    if total == 0:
        return outputs

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = items[start:end]
        logger.info(f"[{stage_name}] 连续批处理 {start + 1}-{end}/{total}")

        with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(batch))) as executor:
            future_map = {executor.submit(worker_fn, i, item): i for i, item in enumerate(batch, start=start)}
            for future in as_completed(future_map):
                i = future_map[future]
                try:
                    outputs[i] = future.result()
                except Exception as e:
                    outputs[i] = e

        if end < total:
            time.sleep(CALL_INTERVAL)

    return outputs


def stage_b_influence_batch(tasks: list[TaskUnit]) -> None:
    """阶段B：influence_parameters 连续批处理。"""

    def worker(index: int, task: TaskUnit):
        params = generate_influence_parameters(task)
        impact_value = calculate_impact_value(params)
        impact_level = calculate_impact_level(impact_value)
        return {
            "index": index,
            "params": params,
            "impact_value": impact_value,
            "impact_level": impact_level,
        }

    results = process_in_batches(tasks, BATCH_SIZE_INFLUENCE, worker, "Impact")
    for item in results:
        if isinstance(item, dict):
            task = tasks[item["index"]]
            task.influence_parameters = item["params"]
            task.impact_value = item["impact_value"]
            task.impact_level = item["impact_level"]


def stage_c_attack_batch(tasks: list[TaskUnit], method: str) -> None:
    """阶段C：attack_parameters 连续批处理 + 可行性计算 + 风险矩阵。"""

    jobs: list[dict] = []
    for ti, task in enumerate(tasks):
        if not task.attack_paths:
            # 没有攻击路径时，给一个兜底记录
            task.attack_details = []
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")
            task.risk_treatment = get_risk_treatment(task.risk_value)
            continue

        for ai, ap_text in enumerate(task.attack_paths):
            vector = "local"
            if task.selected_path_frameworks:
                fw = task.selected_path_frameworks[ai % len(task.selected_path_frameworks)]
                vector = fw.get("attack_vector", "local")

            jobs.append({
                "task_index": ti,
                "attack_index": ai,
                "attack_path": ap_text,
                "attack_vector": vector,
            })

    def worker(index: int, job: dict):
        task = tasks[job["task_index"]]
        attack_path = job["attack_path"]
        attack_vector = job["attack_vector"]

        if method == "attack_vector":
            params = {"attack_vector": normalize_attack_vector(attack_vector)}
            feasibility_score = None
            feasibility_level = calculate_attack_vector_level(attack_vector)

        elif method == "cvss":
            params = generate_attack_parameters_cvss(task, attack_path, attack_vector)
            feasibility_score = calculate_cvss_score(params)
            feasibility_level = calculate_cvss_level(feasibility_score)

        else:  # attack_potential
            params = generate_attack_parameters_attack_potential(task, attack_path)
            feasibility_score = calculate_attack_potential_score(params)
            feasibility_level = calculate_attack_potential_level(feasibility_score)

        risk_value = calculate_risk_value(task.impact_value, feasibility_level)
        risk_treatment = get_risk_treatment(risk_value)

        return {
            "job": job,
            "attack_parameters": params,
            "feasibility_score": feasibility_score,
            "feasibility_level": feasibility_level,
            "risk_value": risk_value,
            "risk_treatment": risk_treatment,
        }

    results = process_in_batches(jobs, BATCH_SIZE_ATTACK, worker, "Feasibility")

    grouped: dict[int, list[dict]] = defaultdict(list)
    for r in results:
        if not isinstance(r, dict):
            continue
        ti = r["job"]["task_index"]
        grouped[ti].append(r)

    for ti, task in enumerate(tasks):
        details = []
        for r in sorted(grouped.get(ti, []), key=lambda x: x["job"]["attack_index"]):
            job = r["job"]
            details.append({
                "attack_path": job["attack_path"],
                "attack_parameters": r["attack_parameters"],
                "feasibility_score": r["feasibility_score"],
                "feasibility_level": r["feasibility_level"],
                "risk_value": r["risk_value"],
                "risk_treatment": r["risk_treatment"],
            })

        task.attack_details = details

        if details:
            task.risk_value = max(d["risk_value"] for d in details)
        else:
            task.risk_value = calculate_risk_value(task.impact_value, "Very Low")

        task.risk_treatment = get_risk_treatment(task.risk_value)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 12) 报告构建
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_tara_report(tasks: list[TaskUnit], feasibility_method: str) -> list[dict]:
    function_map: dict[str, dict] = {}

    for task in tasks:
        if task.function not in function_map:
            function_map[task.function] = {"function": task.function, "assets": {}}

        asset_key = (task.asset_name, task.asset_type, task.target, task.source)
        assets = function_map[task.function]["assets"]

        if asset_key not in assets:
            assets[asset_key] = {
                "asset_name": task.asset_name,
                "asset_type": task.asset_type,
                "security_attributes": [],
            }

        detail = {
            "attribute_name": task.attribute_name,
            "threat_type": task.threat_type,
            "damage_scenario": task.damage_scenario,
            "threat_scenarios": task.threat_scenarios,
            "influence_parameters": task.influence_parameters,
            "impact_value": task.impact_value,
            "impact_level": task.impact_level,
            "feasibility_method": feasibility_method,
            "attack": task.attack_details,
        }
        if task.errors:
            detail["errors"] = task.errors

        assets[asset_key]["security_attributes"].append(detail)

    report = []
    for func_data in function_map.values():
        report.append({
            "function": func_data["function"],
            "assets": list(func_data["assets"].values()),
        })

    return report


def build_error_report(tasks: list[TaskUnit]) -> list[dict]:
    rows = []
    for task in tasks:
        if not task.errors:
            continue
        rows.append({
            "task_id": task.task_id,
            "function": task.function,
            "asset_name": task.asset_name,
            "asset_type": task.asset_type,
            "attribute_name": task.attribute_name,
            "threat_type": task.threat_type,
            "errors": task.errors,
        })
    return rows


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 13) 主流程
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_tara(
    data_flow_dir: str,
    topology_file: str,
    external_interface_excel: str,
    output_path: str,
    feasibility_method: str = FEASIBILITY_METHOD,
    max_assets_for_test: Optional[int] = None,
    skip_checkpoint_resume: bool = False,
) -> tuple[list[dict], list[dict]]:
    method = (feasibility_method or "attack_potential").strip().lower()
    if method not in {"attack_potential", "attack_vector", "cvss"}:
        logger.warning(f"未知可行性方法: {method}，回退为 attack_potential")
        method = "attack_potential"

    _pipeline_start = time.perf_counter()

    logger.info("=" * 72)
    logger.info("TARA v10 开始")
    logger.info(f"可行性评估方法: {method}")
    logger.info("=" * 72)

    # ─── Step 1: DFD资产识别 ─────────────────────────
    with Timer("Step 1 — DFD资产识别") as _:
        asset_results = parse_dfd_json(data_flow_dir)

    # 测试模式：仅取前N个DFD资产
    if max_assets_for_test is not None and max_assets_for_test > 0:
        limited_results = []
        count = 0
        for func_data in asset_results:
            assets = func_data.get("assets", [])
            if count >= max_assets_for_test:
                break
            remain = max_assets_for_test - count
            if len(assets) > remain:
                copy_func = dict(func_data)
                copy_func["assets"] = assets[:remain]
                limited_results.append(copy_func)
                count += remain
                break
            else:
                limited_results.append(func_data)
                count += len(assets)
        asset_results = limited_results
        logger.info(f"测试模式启用：仅使用前 {max_assets_for_test} 个DFD资产，实际纳入 {count} 个")

    # ─── Step 2: 拓扑攻击路径框架 ────────────────────
    with Timer("Step 2 — 拓扑攻击路径框架") as _:
        path_frameworks = []
        if os.path.exists(topology_file) and os.path.exists(external_interface_excel):
            path_frameworks = parse_topology_and_generate_frameworks(topology_file, external_interface_excel, asset_results)
        else:
            logger.warning("拓扑文件或接口清单不存在，攻击路径框架为空")

    # ─── Step 3: 扩展属性 + RAG ──────────────────────
    with Timer("Step 3 — 任务构建 + RAG") as _:
        tasks = build_task_units(asset_results)
        rag_contexts = {
            "damage": rag_kb.retrieve(
                "汽车网络安全 损害场景 STRIDE ISO21434",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
            "threat": rag_kb.retrieve(
                "汽车威胁场景 CAPEC ATT&CK",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
            "attack_path": rag_kb.retrieve(
                "汽车攻击路径 CAN总线 OBD T-Box",
                top_k=RAG_TOP_K,
                max_chars=RAG_MAX_CONTEXT_CHARS,
            ),
        }
        logger.info(
            "RAG上下文长度: damage=%d, threat=%d, attack_path=%d",
            len(rag_contexts["damage"]),
            len(rag_contexts["threat"]),
            len(rag_contexts["attack_path"]),
        )

    # ─── Checkpoint 恢复 ──────────────────────────────
    # 全流水线模式下只有一个 checkpoint 点：stage_C_attack（最终完成态）
    checkpoint_stages = ["stage_C_attack"]
    current_task_data: list[TaskUnit] | None = None
    resume_from = 0

    if not skip_checkpoint_resume:
        resume_from, current_task_data = resume_from_checkpoint(checkpoint_stages, method)
    else:
        logger.info("Checkpoint恢复已跳过（skip_checkpoint_resume=True）")

    # ─── Step 4-6: A+B per-asset 并行 + C 批量评估 ──
    if resume_from <= 0:
        # ── Stage A+B: per-asset 并行 ──────────────────
        with Timer("Stage A+B — 场景生成+影响评级 (per-asset 并行)") as _:
            logger.info(f"Stage A+B：每个资产独立完成生成+影响评级，资产数={len(tasks)}")
            completed_tasks: list[TaskUnit] = []

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_map = {
                    executor.submit(process_task_gen_impact, t, path_frameworks, rag_contexts): t
                    for t in tasks
                }
                done = 0
                total = len(future_map)
                for future in as_completed(future_map):
                    t = future_map[future]
                    done += 1
                    try:
                        completed_tasks.append(future.result())
                    except Exception as e:
                        t.errors.append(f"A+B阶段异常: {e}")
                        completed_tasks.append(t)
                    if done < total:
                        time.sleep(CALL_INTERVAL)

            completed_tasks.sort(key=lambda x: x.task_id)

        # ── Stage C: 攻击可行性批量评估 ─────────────────
        with Timer("Stage C — 攻击可行性批量评估") as _:
            total_aps = sum(len(t.attack_paths or []) for t in completed_tasks)
            logger.info(f"Stage C：批量评估攻击可行性，总攻击路径数={total_aps}")
            stage_c_attack_batch(completed_tasks, method)

        save_checkpoint(completed_tasks, "stage_C_attack", method)
        current_task_data = completed_tasks
    else:
        logger.info("全流水线跳过（从 checkpoint 恢复，全部已完成）")

    assert current_task_data is not None, "当前任务数据为空，无法继续"

    # ─── Step 7: 报告 ────────────────────────────────
    with Timer("Step 7 — 报告构建") as _:
        report = build_tara_report(current_task_data, method)
        err_report = build_error_report(current_task_data)

        save_json(report, output_path)
        save_json(err_report, output_path.replace(".json", "_errors.json"))

    total_elapsed = time.perf_counter() - _pipeline_start
    logger.info("=" * 72)
    logger.info(f"TARA v10 完成: 总任务={len(current_task_data)}, 异常任务={len(err_report)}")
    logger.info(f"总耗时: {total_elapsed:.2f} 秒 ({total_elapsed / 60:.2f} 分钟)")
    logger.info(f"主报告: {output_path}")
    logger.info(f"异常报告: {output_path.replace('.json', '_errors.json')}")
    logger.info("=" * 72)

    return report, err_report


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 14) 入口
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

if __name__ == "__main__":
    topology_file = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\topology\拓扑图数据导出2026_5_9 10_43_29.json"
    external_interface_excel = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\information\对外接口清单.xlsx"
    data_flow_dir = r"D:\Jupyter profile\汽车信息安全风险评估\data\input\DFD"

    # 你可以在这里切换：attack_potential / attack_vector / cvss
    method = os.getenv("TARA_FEASIBILITY_METHOD", FEASIBILITY_METHOD)
    # 测试时仅使用前N个DFD资产（默认1，设为0或负数则关闭）
    test_max_assets = int(os.getenv("TARA_TEST_MAX_ASSETS", "1"))
    if test_max_assets <= 0:
        test_max_assets = None
    # Checkpoint恢复：设 TARA_SKIP_CHECKPOINT=1 可强制重跑所有阶段
    skip_cp = os.getenv("TARA_SKIP_CHECKPOINT", "1").strip() in {"1", "true", "True"}

    run_tara(
        data_flow_dir=data_flow_dir,
        topology_file=topology_file,
        external_interface_excel=external_interface_excel,
        output_path=os.path.join(OUTPUT_DIR, "tara_report_v10.json"),
        feasibility_method=method,
        max_assets_for_test=test_max_assets,
        skip_checkpoint_resume=skip_cp,
    )

2026-05-15 11:31:15,571 [INFO] ========================================================================
2026-05-15 11:31:15,572 [INFO] TARA v10 开始
2026-05-15 11:31:15,572 [INFO] 可行性评估方法: cvss
2026-05-15 11:31:15,572 [INFO] ========================================================================
2026-05-15 11:31:15,573 [INFO] ═══ [计时] Step 1 — DFD资产识别 开始 ═══
2026-05-15 11:31:15,576 [INFO] DFD解析完成: 2 个功能
2026-05-15 11:31:15,576 [INFO] ═══ [计时] Step 1 — DFD资产识别 完成，耗时 0.00 秒 (0.00 分钟) ═══
2026-05-15 11:31:15,576 [INFO] 测试模式启用：仅使用前 1 个DFD资产，实际纳入 1 个
2026-05-15 11:31:15,576 [INFO] ═══ [计时] Step 2 — 拓扑攻击路径框架 开始 ═══
2026-05-15 11:31:15,640 [INFO] 拓扑攻击路径框架生成完成: 1169 条
2026-05-15 11:31:15,642 [INFO] ═══ [计时] Step 2 — 拓扑攻击路径框架 完成，耗时 0.07 秒 (0.00 分钟) ═══
2026-05-15 11:31:15,642 [INFO] ═══ [计时] Step 3 — 任务构建 + RAG 开始 ═══
2026-05-15 11:31:15,643 [INFO] 任务单元构建完成: 6
2026-05-15 11:31:15,647 [INFO] RAG文档加载完成: 1 条
2026-05-15 11:31:15,648 [INFO] RAG上下文长度: damage=518, threat=518, attack_path=518
2026-05-15